In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mdshahrirhabib/phase2b-t2-frozen-modelling-input-20260910/phase2B_T2_FROZEN_CORPUS.zip.sha256.txt
/kaggle/input/datasets/mdshahrirhabib/phase2b-t2-frozen-modelling-input-20260910/phase2B_T2_FROZEN_CORPUS/phase2B_T2_FROZEN_CORPUS/SHA256SUMS.txt
/kaggle/input/datasets/mdshahrirhabib/phase2b-t2-frozen-modelling-input-20260910/phase2B_T2_FROZEN_CORPUS/phase2B_T2_FROZEN_CORPUS/corpus_manifest.json
/kaggle/input/datasets/mdshahrirhabib/phase2b-t2-frozen-modelling-input-20260910/phase2B_T2_FROZEN_CORPUS/phase2B_T2_FROZEN_CORPUS/llama_sensitivity_assignments.csv
/kaggle/input/datasets/mdshahrirhabib/phase2b-t2-frozen-modelling-input-20260910/phase2B_T2_FROZEN_CORPUS/phase2B_T2_FROZEN_CORPUS/outer_fold_assignments.csv
/kaggle/input/datasets/mdshahrirhabib/phase2b-t2-frozen-modelling-input-20260910/phase2B_T2_FROZEN_CORPUS/phase2B_T2_FROZEN_CORPUS/exclusion_ledger.csv
/kaggle/input/datasets/mdshahrirhabib/phase2b-t2-frozen-modelling-input-20260910/phase2B_T2_FROZEN_CORPUS/

## PHASE 2B — STEP M0B: STRICT FROZEN T2 MODELLING-INPUT VERIFICATION

In [4]:
# PHASE 2B — STEP M0B
# Strict verification of the frozen T2 modelling input.
# No corpus reconstruction, splitting, transformation or modelling.

from collections import Counter, defaultdict
from pathlib import Path
import csv
import hashlib
import json
import re
import sys
import unicodedata


W = 116
ROOT = Path("/kaggle/input")

ZIP_SHA = (
    "5d8011d289c78327cf5d461e160aced8f930c51680ec7751f8c2a755beaf19ea"
)
MANIFEST_PAYLOAD_SHA = (
    "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
)
PRIMARY_FP = (
    "ebd02167d656029372e0c5c56aca3fb4d0ae900bea32bd33f2f7c0dd22cbc129"
)
LLAMA_FP = (
    "6406f02bfe431d95f447a0cce09bdf2d93e079775b8cb949d9c464ac7eda1c51"
)
SUMS_SHA = (
    "3d064e1fd63fad55e96399b6c30383e30fba63fc62606db88e088a4860f4a799"
)

HASHES = {
    "corpus_manifest.json":
        "bb2a38473fce80d8d832461680d45702d73ecd95e4d6c628973fd1a77215d246",
    "exclusion_ledger.csv":
        "c9f2289c761412d3ff7f364a89d68f6f7acd70ee4dad40a413ce7b6ee4d0f426",
    "label_contract.json":
        "1eb6f36ee8f63bd4472e7611e9aed28cd6e35a138f20af04dc73a2e86b1e0c3a",
    "llama_sensitivity_assignments.csv":
        "cbaa8600be78726bdc4b8abc5d03f7d7735cfbe7c7b6bfc96f072a7f1c487a77",
    "llama_unseen_generator_sensitivity.jsonl":
        "37500d0ae8b48b632c18f9b57c931e2f88e6793e51975c6ef0243ab179bca01a",
    "outer_fold_assignments.csv":
        "92b8ffe242731d544b473b1604e5f7bc3279b386c25d2934ca2547ea1a1fc84b",
    "primary_tripartite_corpus.jsonl":
        "b836eb86f1d34ed2db2c8fce4f1b04e6bec515632403962ba594e5c0adaba9ef",
}

LABELS = {
    "presumed_truthful": 0,
    "human_deceptive": 1,
    "ai_generated": 2,
}

PRIMARY_COUNTS = {
    "presumed_truthful": 796,
    "human_deceptive": 800,
    "ai_generated": 800,
}

PRIMARY_FOLDS = {
    1: 479,
    2: 479,
    3: 478,
    4: 480,
    5: 480,
}

LLAMA_FOLDS = {
    1: 160,
    2: 158,
    3: 158,
    4: 157,
    5: 157,
}

TRAIN_TEST = {
    1: (1917, 479),
    2: (1917, 479),
    3: (1918, 478),
    4: (1916, 480),
    5: (1916, 480),
}

HOTELS = {
    1: {"hilton", "james", "monaco", "sofitel"},
    2: {"affinia", "ambassador", "hardrock", "talbott"},
    3: {"conrad", "fairmont", "hyatt", "omni"},
    4: {"homewood", "knickerbocker", "sheraton", "swissotel"},
    5: {"allegro", "amalfi", "intercontinental", "palmer"},
}

UPSTREAM = {
    "t0r_tree_sha256":
        "d9072b7a35bffa852e0d70f97aad3fff9f9fd67ee94bb7592a56f9b7a99a1661",
    "t0r_payload_sha256":
        "c20c6f55e459c059fc8882d5fcfaed5fdb53132d99598d2307064da3d4265296",
    "t1a_payload_sha256":
        "11c62be4fcdf3565def1c9321e920604f61a039830081f7688e7f840842cb8a7",
    "t1b_payload_sha256":
        "d15026836234131d7d0c947e56e3897d5bfe416abda9083bafb8eeae53164922",
    "t1b_r_payload_sha256":
        "918bce5f0cc30b153b6d0a2691a8323debd78a8404ca35a9cbbf4b23c21bb97b",
    "t1c_payload_sha256":
        "e2e7c1f32ad310e2f18b438e70d9c059b106d20a2f3548bfeb1f4c7137136e2a",
}


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * W)


def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def text_sha256(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def canonical_sha256(value):
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return text_sha256(payload)


def audit_identity(text):
    text = unicodedata.normalize(
        "NFKC",
        str(text),
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def load_jsonl(path):
    rows = []

    with path.open(
        "r",
        encoding="utf-8",
        errors="strict",
        newline="",
    ) as handle:

        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            require(
                line.endswith("\n") and line.strip(),
                f"{path.name}: invalid line {line_number}",
            )

            rows.append(json.loads(line))

    return rows


def load_csv(path):
    with path.open(
        "r",
        encoding="utf-8",
        errors="strict",
        newline="",
    ) as handle:

        return list(csv.DictReader(handle))


print("=" * W)
print(
    "PHASE 2B — STEP M0B: "
    "STRICT FROZEN T2 MODELLING-INPUT VERIFICATION"
)
print("=" * W)

print(
    "Python:",
    sys.version.split()[0],
    "| Input root:",
    ROOT,
)

require(
    ROOT.is_dir(),
    "/kaggle/input is unavailable",
)


# ------------------------------------------------------------------
# Input discovery
# ------------------------------------------------------------------

section("INPUT DISCOVERY AND ZIP PROVENANCE")

manifest_candidates = sorted(
    ROOT.rglob("corpus_manifest.json")
)

require(
    len(manifest_candidates) == 1,
    "expected one corpus manifest; "
    f"found {len(manifest_candidates)}",
)

T2_ROOT = manifest_candidates[0].parent

sidecar_candidates = sorted(
    ROOT.rglob(
        "phase2B_T2_FROZEN_CORPUS.zip.sha256.txt"
    )
)

require(
    len(sidecar_candidates) == 1,
    "expected one ZIP checksum file; "
    f"found {len(sidecar_candidates)}",
)

recorded_zip_sha = (
    sidecar_candidates[0]
    .read_text(
        encoding="utf-8",
        errors="strict",
    )
    .strip()
    .split()[0]
    .lower()
)

require(
    recorded_zip_sha == ZIP_SHA,
    "ZIP checksum sidecar differs",
)

zip_candidates = sorted(
    ROOT.rglob(
        "phase2B_T2_FROZEN_CORPUS.zip"
    )
)

require(
    len(zip_candidates) <= 1,
    "multiple frozen ZIPs found",
)

if zip_candidates:
    require(
        file_sha256(zip_candidates[0]) == ZIP_SHA,
        "mounted ZIP checksum differs",
    )

print("Artifact root       :", T2_ROOT)
print("Recorded ZIP SHA256 :", recorded_zip_sha)

print(
    "ZIP runtime status  :",
    (
        "byte-verified"
        if zip_candidates
        else
        "Kaggle-extracted; "
        "externally verified ZIP recorded"
    ),
)


# ------------------------------------------------------------------
# Internal checksums
# ------------------------------------------------------------------

section("INTERNAL SHA-256 REGISTRY")

for filename, expected_hash in HASHES.items():
    path = T2_ROOT / filename

    require(
        path.is_file(),
        f"missing {filename}",
    )

    actual_hash = file_sha256(path)

    require(
        actual_hash == expected_hash,
        f"checksum mismatch: {filename}",
    )

    print(f"{filename:48s} PASS")


checksums_path = T2_ROOT / "SHA256SUMS.txt"

require(
    file_sha256(checksums_path) == SUMS_SHA,
    "SHA256SUMS.txt checksum mismatch",
)

registered_hashes = {}

for line in checksums_path.read_text(
    encoding="utf-8"
).splitlines():

    digest, filename = line.split(
        None,
        1,
    )

    registered_hashes[
        filename.strip()
    ] = digest.lower()


require(
    registered_hashes == HASHES,
    "internal checksum registry differs",
)

print(f"{'SHA256SUMS.txt':48s} PASS")


# ------------------------------------------------------------------
# Manifest and label contract
# ------------------------------------------------------------------

section("MANIFEST, LABELS AND UPSTREAM CHAIN")

manifest = json.loads(
    (T2_ROOT / "corpus_manifest.json")
    .read_text(encoding="utf-8")
)

stored_manifest_payload = manifest.pop(
    "manifest_payload_sha256"
)

require(
    stored_manifest_payload == MANIFEST_PAYLOAD_SHA,
    "stored manifest payload differs",
)

require(
    canonical_sha256(manifest)
    == MANIFEST_PAYLOAD_SHA,
    "recomputed manifest payload differs",
)

require(
    manifest["phase"] == "Phase 2B"
    and manifest["gate"] == "T2",
    "phase or gate differs",
)

require(
    manifest["status"]
    == "FROZEN_CORPUS_CONSTRUCTED",
    "manifest status differs",
)

require(
    manifest["upstream"] == UPSTREAM,
    "T0R-to-T1C chain differs",
)


label_contract = json.loads(
    (T2_ROOT / "label_contract.json")
    .read_text(encoding="utf-8")
)

require(
    label_contract["mapping"] == LABELS,
    "label mapping differs",
)

require(
    label_contract["llama_reference_label_id"] == 2,
    "Llama reference label differs",
)

print(
    "Manifest payload:",
    stored_manifest_payload,
)
print("Upstream chain  : PASS")
print(
    "Label mapping   :",
    label_contract["mapping"],
)


# ------------------------------------------------------------------
# Frozen corpus verification
# ------------------------------------------------------------------

section("PRIMARY AND LLAMA CORPUS VERIFICATION")

primary_path = (
    T2_ROOT
    / "primary_tripartite_corpus.jsonl"
)

llama_path = (
    T2_ROOT
    / "llama_unseen_generator_sensitivity.jsonl"
)

primary_rows = load_jsonl(primary_path)
llama_rows = load_jsonl(llama_path)


require(
    len(primary_rows) == 2396,
    "primary row count differs",
)

require(
    len({
        row["row_id"]
        for row in primary_rows
    }) == 2396,
    "primary row IDs are not unique",
)

require(
    len(llama_rows) == 790,
    "Llama row count differs",
)

require(
    len({
        row["row_id"]
        for row in llama_rows
    }) == 790,
    "Llama row IDs are not unique",
)


for row in primary_rows:

    require(
        row["label_id"]
        == LABELS[row["scientific_class"]],
        f"{row['row_id']}: label mismatch",
    )

    require(
        isinstance(row["text"], str)
        and row["text"].strip(),
        f"{row['row_id']}: empty text",
    )

    require(
        text_sha256(row["text"])
        == row["model_text_sha256"],
        f"{row['row_id']}: text hash mismatch",
    )

    require(
        text_sha256(
            audit_identity(row["text"])
        )
        == row["audit_identity_sha256"],
        f"{row['row_id']}: audit hash mismatch",
    )


for row in llama_rows:

    require(
        row["reference_label_id"] == 2,
        f"{row['row_id']}: reference label mismatch",
    )

    require(
        row["evaluation_role"]
        == "unseen_generator_sensitivity",
        f"{row['row_id']}: role mismatch",
    )

    require(
        isinstance(row["text"], str)
        and row["text"].strip(),
        f"{row['row_id']}: empty text",
    )

    require(
        text_sha256(row["text"])
        == row["model_text_sha256"],
        f"{row['row_id']}: text hash mismatch",
    )

    require(
        text_sha256(
            audit_identity(row["text"])
        )
        == row["audit_identity_sha256"],
        f"{row['row_id']}: audit hash mismatch",
    )


primary_counts = dict(
    Counter(
        row["scientific_class"]
        for row in primary_rows
    )
)

primary_fold_counts = dict(
    sorted(
        Counter(
            row["official_fold"]
            for row in primary_rows
        ).items()
    )
)

llama_fold_counts = dict(
    sorted(
        Counter(
            row["official_fold"]
            for row in llama_rows
        ).items()
    )
)

llama_polarity_counts = dict(
    sorted(
        Counter(
            row["polarity"]
            for row in llama_rows
        ).items()
    )
)


require(
    primary_counts == PRIMARY_COUNTS,
    "primary class counts differ",
)

require(
    primary_fold_counts == PRIMARY_FOLDS,
    "primary fold counts differ",
)

require(
    llama_fold_counts == LLAMA_FOLDS,
    "Llama fold counts differ",
)

require(
    llama_polarity_counts
    == {
        "negative": 394,
        "positive": 396,
    },
    "Llama polarity counts differ",
)


primary_fingerprint = canonical_sha256([
    {
        "row_id": row["row_id"],
        "label_id": row["label_id"],
        "official_fold": row["official_fold"],
        "model_text_sha256":
            row["model_text_sha256"],
    }
    for row in primary_rows
])

llama_fingerprint = canonical_sha256([
    {
        "row_id": row["row_id"],
        "official_fold": row["official_fold"],
        "model_text_sha256":
            row["model_text_sha256"],
    }
    for row in llama_rows
])


require(
    primary_fingerprint == PRIMARY_FP,
    "primary corpus fingerprint differs",
)

require(
    llama_fingerprint == LLAMA_FP,
    "Llama corpus fingerprint differs",
)


actual_routing = defaultdict(set)

for row in primary_rows:
    actual_routing[
        row["official_fold"]
    ].add(row["hotel"])


require(
    dict(actual_routing) == HOTELS,
    "hotel-to-fold mapping differs",
)


primary_representation_differences = sum(
    row["model_text_sha256"]
    != row["audit_identity_sha256"]
    for row in primary_rows
)

llama_representation_differences = sum(
    row["model_text_sha256"]
    != row["audit_identity_sha256"]
    for row in llama_rows
)


require(
    primary_representation_differences == 693,
    "primary representation audit differs",
)

require(
    llama_representation_differences == 3,
    "Llama representation audit differs",
)


print(
    "Primary:",
    len(primary_rows),
    "| classes:",
    primary_counts,
    "| folds:",
    primary_fold_counts,
)

print(
    "Primary fingerprint:",
    primary_fingerprint,
)

print(
    "Llama  :",
    len(llama_rows),
    "| polarity:",
    llama_polarity_counts,
    "| folds:",
    llama_fold_counts,
)

print(
    "Llama fingerprint  :",
    llama_fingerprint,
)


# ------------------------------------------------------------------
# Fold-routing verification
# ------------------------------------------------------------------

section(
    "FOLD ROUTING, "
    "SENSITIVITY ROUTING AND EXCLUSIONS"
)

primary_by_id = {
    row["row_id"]: row
    for row in primary_rows
}

outer_assignments = load_csv(
    T2_ROOT
    / "outer_fold_assignments.csv"
)

require(
    len(outer_assignments) == 11980,
    "outer assignment count differs",
)

seen_assignments = set()
assignments_per_row = defaultdict(Counter)
assignments_per_fold = defaultdict(Counter)


for assignment_row in outer_assignments:

    outer_fold = int(
        assignment_row["outer_test_fold"]
    )

    row_id = assignment_row["row_id"]

    official_fold = int(
        assignment_row["official_fold"]
    )

    require(
        row_id in primary_by_id,
        f"unknown assignment row: {row_id}",
    )

    require(
        (outer_fold, row_id)
        not in seen_assignments,
        "repeated outer assignment",
    )

    seen_assignments.add(
        (outer_fold, row_id)
    )

    require(
        official_fold
        == primary_by_id[
            row_id
        ]["official_fold"],
        f"{row_id}: official-fold mismatch",
    )

    expected_assignment = (
        "test"
        if official_fold == outer_fold
        else "train"
    )

    require(
        assignment_row["assignment"]
        == expected_assignment,
        f"{row_id}: routing mismatch",
    )

    assignments_per_row[
        row_id
    ][expected_assignment] += 1

    assignments_per_fold[
        outer_fold
    ][expected_assignment] += 1


require(
    all(
        counts
        == Counter({
            "train": 4,
            "test": 1,
        })
        for counts
        in assignments_per_row.values()
    ),
    "a primary row lacks four-train/"
    "one-test routing",
)


for fold, (train_n, test_n) in TRAIN_TEST.items():

    require(
        (
            assignments_per_fold[
                fold
            ]["train"],
            assignments_per_fold[
                fold
            ]["test"],
        )
        == (train_n, test_n),
        f"fold {fold}: size mismatch",
    )

    train_hotels = {
        primary_by_id[
            row["row_id"]
        ]["hotel"]
        for row in outer_assignments
        if int(
            row["outer_test_fold"]
        ) == fold
        and row["assignment"] == "train"
    }

    test_hotels = {
        primary_by_id[
            row["row_id"]
        ]["hotel"]
        for row in outer_assignments
        if int(
            row["outer_test_fold"]
        ) == fold
        and row["assignment"] == "test"
    }

    require(
        train_hotels.isdisjoint(
            test_hotels
        ),
        f"fold {fold}: hotel leakage",
    )

    require(
        test_hotels == HOTELS[fold],
        f"fold {fold}: held-out hotels differ",
    )

    print(
        f"Fold {fold}: "
        f"train={train_n}, "
        f"test={test_n}, "
        "hotel overlap=0  PASS"
    )


# ------------------------------------------------------------------
# Llama routing and exclusions
# ------------------------------------------------------------------

llama_assignments = load_csv(
    T2_ROOT
    / "llama_sensitivity_assignments.csv"
)

llama_by_id = {
    row["row_id"]: row
    for row in llama_rows
}

require(
    len(llama_assignments) == 790,
    "Llama assignment count differs",
)

require(
    len({
        row["row_id"]
        for row in llama_assignments
    }) == 790,
    "Llama assignment IDs repeat",
)


for assignment_row in llama_assignments:

    row_id = assignment_row["row_id"]

    require(
        row_id in llama_by_id,
        f"unknown Llama row: {row_id}",
    )

    require(
        assignment_row["assignment"]
        == "sensitivity_test_only",
        f"{row_id}: invalid sensitivity role",
    )

    require(
        int(
            assignment_row[
                "evaluation_fold"
            ]
        )
        == llama_by_id[
            row_id
        ]["official_fold"],
        f"{row_id}: sensitivity-fold mismatch",
    )


exclusions = load_csv(
    T2_ROOT
    / "exclusion_ledger.csv"
)

exclusion_reasons = dict(
    Counter(
        row["exclusion_reason"]
        for row in exclusions
    )
)

require(
    len(exclusions) == 14,
    "exclusion count differs",
)

require(
    exclusion_reasons
    == {
        "excess_exact_duplicate": 4,
        "empty_text": 10,
    },
    "exclusion reasons differ",
)

eligible_ids = {
    row["row_id"]
    for row in primary_rows
}.union({
    row["row_id"]
    for row in llama_rows
})

excluded_ids = {
    row["row_id"]
    for row in exclusions
}

require(
    eligible_ids.isdisjoint(
        excluded_ids
    ),
    "excluded row appears in eligible data",
)


print(
    "Llama sensitivity-only assignments: "
    "790  PASS"
)

print(
    "Exclusions:",
    exclusion_reasons,
    "| eligible overlap: 0  PASS",
)


# ------------------------------------------------------------------
# Retain verified paths for subsequent modelling cells
# ------------------------------------------------------------------

PHASE2B_T2_ROOT = T2_ROOT

PHASE2B_PRIMARY_PATH = (
    T2_ROOT
    / "primary_tripartite_corpus.jsonl"
)

PHASE2B_LLAMA_PATH = (
    T2_ROOT
    / "llama_unseen_generator_sensitivity.jsonl"
)

PHASE2B_PRIMARY_ROWS = primary_rows
PHASE2B_LLAMA_ROWS = llama_rows

PHASE2B_T2_INPUTS = {
    "root": str(T2_ROOT),
    "manifest_payload_sha256":
        MANIFEST_PAYLOAD_SHA,
    "primary_fingerprint":
        primary_fingerprint,
    "llama_fingerprint":
        llama_fingerprint,
}


print("\n" + "=" * W)
print(
    "STEP M0B DECISION: PASS — "
    "FROZEN T2 INPUT VERIFIED "
    "FOR MODEL-CONTRACT USE"
)
print("=" * W)

print(
    "No file written, extracted, transformed, "
    "relabelled, resampled, split or modelled."
)

print(
    "STOP — share this complete output before "
    "the Phase 2B model contract is frozen."
)

PHASE 2B — STEP M0B: STRICT FROZEN T2 MODELLING-INPUT VERIFICATION
Python: 3.12.13 | Input root: /kaggle/input

INPUT DISCOVERY AND ZIP PROVENANCE
--------------------------------------------------------------------------------------------------------------------
Artifact root       : /kaggle/input/datasets/mdshahrirhabib/phase2b-t2-frozen-modelling-input-20260910/phase2B_T2_FROZEN_CORPUS/phase2B_T2_FROZEN_CORPUS
Recorded ZIP SHA256 : 5d8011d289c78327cf5d461e160aced8f930c51680ec7751f8c2a755beaf19ea
ZIP runtime status  : Kaggle-extracted; externally verified ZIP recorded

INTERNAL SHA-256 REGISTRY
--------------------------------------------------------------------------------------------------------------------
corpus_manifest.json                             PASS
exclusion_ledger.csv                             PASS
label_contract.json                              PASS
llama_sensitivity_assignments.csv                PASS
llama_unseen_generator_sensitivity.jsonl         PASS
outer_fol

M0B passed completely. The mounted corpus is byte-consistent at the internal-file level, the manifest chain is intact, all counts and fingerprints match, and the fold routing contains no hotel leakage.

## PHASE 2B — STEP M1A: MODELLING-ENVIRONMENT AND CHECKPOINT-SOURCE AUDIT

In [5]:
# PHASE 2B — STEP M1A
# Read-only modelling-environment and checkpoint-source audit.
# No installation, download, model loading, training or evaluation.

from importlib import metadata, util
from pathlib import Path
import json
import os
import platform
import sys


WIDTH = 116

EXPECTED_T2_MANIFEST_PAYLOAD = (
    "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
)

PROPOSED_MODEL_ID = "microsoft/deberta-v3-base"

PROPOSED_MODEL_REVISION = (
    "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
)

PACKAGES = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "joblib",
    "torch",
    "transformers",
    "tokenizers",
    "accelerate",
    "safetensors",
    "sentencepiece",
]


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def package_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return "NOT INSTALLED"


print("=" * WIDTH)
print(
    "PHASE 2B — STEP M1A: "
    "MODELLING-ENVIRONMENT AND CHECKPOINT-SOURCE AUDIT"
)
print("=" * WIDTH)


# ------------------------------------------------------------------
# Frozen T2 checkpoint continuity
# ------------------------------------------------------------------

section("FROZEN T2 CONTINUITY")

require(
    "PHASE2B_T2_INPUTS" in globals(),
    "verified PHASE2B_T2_INPUTS is missing; "
    "run M0B in this runtime",
)

require(
    PHASE2B_T2_INPUTS[
        "manifest_payload_sha256"
    ] == EXPECTED_T2_MANIFEST_PAYLOAD,
    "T2 manifest payload differs",
)

require(
    Path(
        PHASE2B_T2_INPUTS["root"]
    ).is_dir(),
    "verified T2 root is unavailable",
)

print(
    "T2 root             :",
    PHASE2B_T2_INPUTS["root"],
)

print(
    "T2 manifest payload :",
    PHASE2B_T2_INPUTS[
        "manifest_payload_sha256"
    ],
)

print("T2 continuity       : PASS")


# ------------------------------------------------------------------
# Runtime identity
# ------------------------------------------------------------------

section("RUNTIME IDENTITY")

print("Python executable :", sys.executable)
print("Python version    :", platform.python_version())
print("Platform          :", platform.platform())
print("Processor         :", platform.processor() or "not reported")
print("CPU count         :", os.cpu_count())


# ------------------------------------------------------------------
# Package inventory
# ------------------------------------------------------------------

section("CRITICAL PACKAGE INVENTORY")

PACKAGE_VERSIONS = {}

for package_name in PACKAGES:
    version = package_version(package_name)
    PACKAGE_VERSIONS[package_name] = version

    print(
        f"{package_name:20s}: {version}"
    )


require(
    PACKAGE_VERSIONS["numpy"]
    != "NOT INSTALLED",
    "NumPy is unavailable",
)

require(
    PACKAGE_VERSIONS["pandas"]
    != "NOT INSTALLED",
    "pandas is unavailable",
)

require(
    PACKAGE_VERSIONS["scikit-learn"]
    != "NOT INSTALLED",
    "scikit-learn is unavailable",
)

require(
    PACKAGE_VERSIONS["torch"]
    != "NOT INSTALLED",
    "PyTorch is unavailable",
)

require(
    PACKAGE_VERSIONS["transformers"]
    != "NOT INSTALLED",
    "transformers is unavailable",
)


# ------------------------------------------------------------------
# PyTorch and accelerator status
# ------------------------------------------------------------------

section("PYTORCH AND ACCELERATOR STATUS")

import torch

print("PyTorch version       :", torch.__version__)
print("CUDA build            :", torch.version.cuda)
print("CUDA available        :", torch.cuda.is_available())
print("CUDA device count     :", torch.cuda.device_count())
print(
    "cuDNN version         :",
    (
        torch.backends.cudnn.version()
        if torch.backends.cudnn.is_available()
        else "not available"
    ),
)

if torch.cuda.is_available():

    for device_index in range(
        torch.cuda.device_count()
    ):
        properties = torch.cuda.get_device_properties(
            device_index
        )

        print(
            f"Device {device_index} name     :",
            properties.name,
        )

        print(
            f"Device {device_index} memory   :",
            properties.total_memory,
            "bytes",
        )

else:
    print(
        "Expected at this stage : "
        "CPU-only contract session"
    )


# ------------------------------------------------------------------
# Existing checkpoint-source discovery
# ------------------------------------------------------------------

section("EXISTING DEBERTA CHECKPOINT SOURCES")

model_candidates = []

for path in sorted(
    Path("/kaggle/input").rglob("config.json")
):
    searchable_path = str(path).lower()

    if "deberta" in searchable_path:
        model_candidates.append(path)


if model_candidates:

    for path in model_candidates:

        print("Candidate:", path)

        try:
            config = json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

            print(
                "  architectures:",
                config.get("architectures"),
            )

            print(
                "  num_labels:",
                config.get("num_labels"),
            )

        except Exception as error:
            print(
                "  config inspection failed:",
                repr(error),
            )

else:
    print(
        "No DeBERTa checkpoint is currently "
        "attached under /kaggle/input."
    )


# ------------------------------------------------------------------
# Proposed fresh checkpoint contract
# ------------------------------------------------------------------

section("PROPOSED PRIMARY CHECKPOINT")

print("Model ID :", PROPOSED_MODEL_ID)
print("Revision :", PROPOSED_MODEL_REVISION)

print(
    "Initialisation policy : "
    "fresh pretrained base weights with a "
    "new three-class classification head"
)

print(
    "RS-1 reuse            : "
    "prohibited as primary Phase 2B initialisation"
)

print(
    "Outer-fold tuning     : prohibited"
)

print(
    "Model acquisition     : "
    "not performed in this cell"
)


# ------------------------------------------------------------------
# Contract-readiness summary
# ------------------------------------------------------------------

missing_optional = [
    package_name
    for package_name in [
        "accelerate",
        "safetensors",
        "sentencepiece",
    ]
    if PACKAGE_VERSIONS[
        package_name
    ] == "NOT INSTALLED"
]


print("\n" + "=" * WIDTH)
print("STEP M1A COMPLETE — READ-ONLY ENVIRONMENT AUDIT")
print("=" * WIDTH)

print(
    "Optional transformer dependencies missing:",
    (
        missing_optional
        if missing_optional
        else "none"
    ),
)

print(
    "DeBERTa input candidates:",
    len(model_candidates),
)

print(
    "No package installed, model downloaded, "
    "checkpoint loaded, file written, "
    "training performed or prediction generated."
)

print(
    "STOP — share this complete output before "
    "the frozen model and hyperparameter "
    "contract is written."
)# PHASE 2B — STEP M1A
# Read-only modelling-environment and checkpoint-source audit.
# No installation, download, model loading, training or evaluation.

from importlib import metadata, util
from pathlib import Path
import json
import os
import platform
import sys


WIDTH = 116

EXPECTED_T2_MANIFEST_PAYLOAD = (
    "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
)

PROPOSED_MODEL_ID = "microsoft/deberta-v3-base"

PROPOSED_MODEL_REVISION = (
    "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
)

PACKAGES = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "joblib",
    "torch",
    "transformers",
    "tokenizers",
    "accelerate",
    "safetensors",
    "sentencepiece",
]


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def package_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return "NOT INSTALLED"


print("=" * WIDTH)
print(
    "PHASE 2B — STEP M1A: "
    "MODELLING-ENVIRONMENT AND CHECKPOINT-SOURCE AUDIT"
)
print("=" * WIDTH)


# ------------------------------------------------------------------
# Frozen T2 checkpoint continuity
# ------------------------------------------------------------------

section("FROZEN T2 CONTINUITY")

require(
    "PHASE2B_T2_INPUTS" in globals(),
    "verified PHASE2B_T2_INPUTS is missing; "
    "run M0B in this runtime",
)

require(
    PHASE2B_T2_INPUTS[
        "manifest_payload_sha256"
    ] == EXPECTED_T2_MANIFEST_PAYLOAD,
    "T2 manifest payload differs",
)

require(
    Path(
        PHASE2B_T2_INPUTS["root"]
    ).is_dir(),
    "verified T2 root is unavailable",
)

print(
    "T2 root             :",
    PHASE2B_T2_INPUTS["root"],
)

print(
    "T2 manifest payload :",
    PHASE2B_T2_INPUTS[
        "manifest_payload_sha256"
    ],
)

print("T2 continuity       : PASS")


# ------------------------------------------------------------------
# Runtime identity
# ------------------------------------------------------------------

section("RUNTIME IDENTITY")

print("Python executable :", sys.executable)
print("Python version    :", platform.python_version())
print("Platform          :", platform.platform())
print("Processor         :", platform.processor() or "not reported")
print("CPU count         :", os.cpu_count())


# ------------------------------------------------------------------
# Package inventory
# ------------------------------------------------------------------

section("CRITICAL PACKAGE INVENTORY")

PACKAGE_VERSIONS = {}

for package_name in PACKAGES:
    version = package_version(package_name)
    PACKAGE_VERSIONS[package_name] = version

    print(
        f"{package_name:20s}: {version}"
    )


require(
    PACKAGE_VERSIONS["numpy"]
    != "NOT INSTALLED",
    "NumPy is unavailable",
)

require(
    PACKAGE_VERSIONS["pandas"]
    != "NOT INSTALLED",
    "pandas is unavailable",
)

require(
    PACKAGE_VERSIONS["scikit-learn"]
    != "NOT INSTALLED",
    "scikit-learn is unavailable",
)

require(
    PACKAGE_VERSIONS["torch"]
    != "NOT INSTALLED",
    "PyTorch is unavailable",
)

require(
    PACKAGE_VERSIONS["transformers"]
    != "NOT INSTALLED",
    "transformers is unavailable",
)


# ------------------------------------------------------------------
# PyTorch and accelerator status
# ------------------------------------------------------------------

section("PYTORCH AND ACCELERATOR STATUS")

import torch

print("PyTorch version       :", torch.__version__)
print("CUDA build            :", torch.version.cuda)
print("CUDA available        :", torch.cuda.is_available())
print("CUDA device count     :", torch.cuda.device_count())
print(
    "cuDNN version         :",
    (
        torch.backends.cudnn.version()
        if torch.backends.cudnn.is_available()
        else "not available"
    ),
)

if torch.cuda.is_available():

    for device_index in range(
        torch.cuda.device_count()
    ):
        properties = torch.cuda.get_device_properties(
            device_index
        )

        print(
            f"Device {device_index} name     :",
            properties.name,
        )

        print(
            f"Device {device_index} memory   :",
            properties.total_memory,
            "bytes",
        )

else:
    print(
        "Expected at this stage : "
        "CPU-only contract session"
    )


# ------------------------------------------------------------------
# Existing checkpoint-source discovery
# ------------------------------------------------------------------

section("EXISTING DEBERTA CHECKPOINT SOURCES")

model_candidates = []

for path in sorted(
    Path("/kaggle/input").rglob("config.json")
):
    searchable_path = str(path).lower()

    if "deberta" in searchable_path:
        model_candidates.append(path)


if model_candidates:

    for path in model_candidates:

        print("Candidate:", path)

        try:
            config = json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

            print(
                "  architectures:",
                config.get("architectures"),
            )

            print(
                "  num_labels:",
                config.get("num_labels"),
            )

        except Exception as error:
            print(
                "  config inspection failed:",
                repr(error),
            )

else:
    print(
        "No DeBERTa checkpoint is currently "
        "attached under /kaggle/input."
    )


# ------------------------------------------------------------------
# Proposed fresh checkpoint contract
# ------------------------------------------------------------------

section("PROPOSED PRIMARY CHECKPOINT")

print("Model ID :", PROPOSED_MODEL_ID)
print("Revision :", PROPOSED_MODEL_REVISION)

print(
    "Initialisation policy : "
    "fresh pretrained base weights with a "
    "new three-class classification head"
)

print(
    "RS-1 reuse            : "
    "prohibited as primary Phase 2B initialisation"
)

print(
    "Outer-fold tuning     : prohibited"
)

print(
    "Model acquisition     : "
    "not performed in this cell"
)


# ------------------------------------------------------------------
# Contract-readiness summary
# ------------------------------------------------------------------

missing_optional = [
    package_name
    for package_name in [
        "accelerate",
        "safetensors",
        "sentencepiece",
    ]
    if PACKAGE_VERSIONS[
        package_name
    ] == "NOT INSTALLED"
]


print("\n" + "=" * WIDTH)
print("STEP M1A COMPLETE — READ-ONLY ENVIRONMENT AUDIT")
print("=" * WIDTH)

print(
    "Optional transformer dependencies missing:",
    (
        missing_optional
        if missing_optional
        else "none"
    ),
)

print(
    "DeBERTa input candidates:",
    len(model_candidates),
)

print(
    "No package installed, model downloaded, "
    "checkpoint loaded, file written, "
    "training performed or prediction generated."
)

print(
    "STOP — share this complete output before "
    "the frozen model and hyperparameter "
    "contract is written."
)

PHASE 2B — STEP M1A: MODELLING-ENVIRONMENT AND CHECKPOINT-SOURCE AUDIT

FROZEN T2 CONTINUITY
--------------------------------------------------------------------------------------------------------------------
T2 root             : /kaggle/input/datasets/mdshahrirhabib/phase2b-t2-frozen-modelling-input-20260910/phase2B_T2_FROZEN_CORPUS/phase2B_T2_FROZEN_CORPUS
T2 manifest payload : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8
T2 continuity       : PASS

RUNTIME IDENTITY
--------------------------------------------------------------------------------------------------------------------
Python executable : /usr/bin/python3
Python version    : 3.12.13
Platform          : Linux-6.12.90+-x86_64-with-glibc2.35
Processor         : x86_64
CPU count         : 4

CRITICAL PACKAGE INVENTORY
--------------------------------------------------------------------------------------------------------------------
numpy               : 2.0.2
pandas              : 2.3.3
scipy           

In [6]:
# PHASE 2B — STEP M1B
# Acquire and preserve the exact pinned DeBERTa-v3-base checkpoint.
# No model is instantiated, trained or evaluated.

from importlib import metadata
from pathlib import Path, PurePosixPath
import hashlib
import json
import shutil
import sys
import tempfile
import zipfile

from huggingface_hub import HfApi, snapshot_download


WIDTH = 118

MODEL_ID = "microsoft/deberta-v3-base"

MODEL_REVISION = (
    "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
)

EXPECTED_T2_MANIFEST_PAYLOAD = (
    "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
)

WORKING_ROOT = Path("/kaggle/working")

OUTPUT_DIR = (
    WORKING_ROOT
    / "phase2B_DEBERTA_V3_BASE_PINNED"
)

ZIP_PATH = (
    WORKING_ROOT
    / "phase2B_DEBERTA_V3_BASE_PINNED.zip"
)

ZIP_SHA_PATH = (
    WORKING_ROOT
    / "phase2B_DEBERTA_V3_BASE_PINNED.zip.sha256.txt"
)


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


def safe_repository_name(name):
    path = PurePosixPath(name)

    return (
        not path.is_absolute()
        and ".." not in path.parts
        and len(path.parts) == 1
    )


def deterministic_zip(source_directory, destination):
    with zipfile.ZipFile(
        destination,
        "w",
        allowZip64=True,
    ) as archive:

        files = sorted(
            path
            for path in source_directory.rglob("*")
            if path.is_file()
        )

        for path in files:

            archive_name = (
                PurePosixPath(
                    source_directory.name
                )
                / path.relative_to(
                    source_directory
                )
            )

            information = zipfile.ZipInfo(
                str(archive_name),
                date_time=(
                    1980,
                    1,
                    1,
                    0,
                    0,
                    0,
                ),
            )

            information.compress_type = (
                zipfile.ZIP_STORED
            )

            information.create_system = 3

            information.external_attr = (
                0o100644 << 16
            )

            with (
                path.open("rb") as source,
                archive.open(
                    information,
                    "w",
                ) as target,
            ):
                shutil.copyfileobj(
                    source,
                    target,
                    length=1024 * 1024,
                )


print("=" * WIDTH)
print(
    "PHASE 2B — STEP M1B: "
    "PINNED DEBERTA BASE ACQUISITION "
    "AND PRESERVATION"
)
print("=" * WIDTH)


require(
    sys.version_info[:2] == (3, 12),
    "Python major/minor version differs",
)


# --------------------------------------------------------------
# T2 continuity
# --------------------------------------------------------------

manifest_candidates = sorted(
    Path("/kaggle/input").rglob(
        "corpus_manifest.json"
    )
)

require(
    len(manifest_candidates) == 1,
    "expected exactly one frozen T2 manifest",
)

t2_manifest = json.loads(
    manifest_candidates[0].read_text(
        encoding="utf-8"
    )
)

require(
    t2_manifest.get(
        "manifest_payload_sha256"
    )
    == EXPECTED_T2_MANIFEST_PAYLOAD,
    "T2 manifest continuity check failed",
)


# --------------------------------------------------------------
# Existing-output protection
# --------------------------------------------------------------

for target in [
    OUTPUT_DIR,
    ZIP_PATH,
    ZIP_SHA_PATH,
]:
    require(
        not target.exists(),
        "output already exists: "
        f"{target}; preserve and inspect it",
    )


section("SOURCE REQUEST")

print("Repository        :", MODEL_ID)
print("Requested revision:", MODEL_REVISION)
print("T2 continuity     : PASS")

print(
    "huggingface_hub   :",
    metadata.version(
        "huggingface-hub"
    ),
)


# --------------------------------------------------------------
# Resolve pinned repository
# --------------------------------------------------------------

api = HfApi()

try:
    repository_information = api.repo_info(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
    )

    repository_files = sorted(
        api.list_repo_files(
            repo_id=MODEL_ID,
            revision=MODEL_REVISION,
        )
    )

except Exception as error:
    raise RuntimeError(
        "STOP — pinned Hugging Face source "
        "could not be reached. Confirm that "
        "Kaggle Internet is ON. Original error: "
        + repr(error)
    ) from error


require(
    repository_information.sha
    == MODEL_REVISION,
    "resolved commit differs: "
    f"{repository_information.sha}",
)

require(
    "config.json" in repository_files,
    "config.json is absent from "
    "the pinned repository",
)


# --------------------------------------------------------------
# Select one authoritative weight format
# --------------------------------------------------------------

if "model.safetensors" in repository_files:

    weight_filename = "model.safetensors"
    weight_format = "safetensors"

elif "pytorch_model.bin" in repository_files:

    weight_filename = "pytorch_model.bin"
    weight_format = "pytorch_bin"

else:
    raise RuntimeError(
        "STOP — no supported single-file "
        "model weights were found"
    )


tokenizer_allowlist = {
    "spm.model",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "added_tokens.json",
}

documentation_allowlist = {
    "README.md",
    "LICENSE",
    "LICENSE.txt",
}


selected_filenames = [
    "config.json",
    weight_filename,
]

selected_filenames += sorted(
    filename
    for filename in tokenizer_allowlist
    if filename in repository_files
)

selected_filenames += sorted(
    filename
    for filename in documentation_allowlist
    if filename in repository_files
)


require(
    (
        "spm.model" in selected_filenames
        or "tokenizer.json"
        in selected_filenames
    ),
    "no supported tokenizer vocabulary "
    "file was found",
)

require(
    all(
        safe_repository_name(filename)
        for filename in selected_filenames
    ),
    "unsafe or nested selected path",
)


section("PINNED REPOSITORY RESOLUTION")

print(
    "Resolved commit :",
    repository_information.sha,
)

print(
    "Repository files:",
    len(repository_files),
)

print(
    "Weight file     :",
    weight_filename,
)

print(
    "Weight format   :",
    weight_format,
)

print(
    "Selected files  :",
    len(selected_filenames),
)

for filename in selected_filenames:
    print(" ", filename)


# --------------------------------------------------------------
# Download selected files
# --------------------------------------------------------------

with tempfile.TemporaryDirectory(
    prefix="phase2B_M1B_",
    dir=WORKING_ROOT,
) as temporary_directory:

    temporary_root = Path(
        temporary_directory
    )

    download_root = (
        temporary_root
        / "download"
    )

    section("PINNED DOWNLOAD")

    try:
        snapshot_download(
            repo_id=MODEL_ID,
            revision=MODEL_REVISION,
            local_dir=str(
                download_root
            ),
            allow_patterns=(
                selected_filenames
            ),
        )

    except Exception as error:
        raise RuntimeError(
            "STOP — pinned checkpoint download "
            "failed. Do not substitute another "
            "revision. Original error: "
            + repr(error)
        ) from error


    require(
        all(
            (
                download_root
                / filename
            ).is_file()
            for filename
            in selected_filenames
        ),
        "download is incomplete",
    )


    configuration = json.loads(
        (
            download_root
            / "config.json"
        ).read_text(
            encoding="utf-8"
        )
    )

    require(
        configuration.get(
            "model_type"
        ) == "deberta-v2",
        "unexpected model_type in config",
    )


    source_registry = {}

    for filename in selected_filenames:

        path = (
            download_root
            / filename
        )

        source_registry[
            filename
        ] = {
            "size_bytes":
                path.stat().st_size,
            "sha256":
                file_sha256(path),
        }


    checkpoint_manifest_payload = {
        "phase": "Phase 2B",
        "gate": "M1B",
        "status":
            "PINNED_BASE_CHECKPOINT_PRESERVED",
        "frozen_date": "2026-09-10",

        "upstream_t2_manifest_payload_sha256":
            EXPECTED_T2_MANIFEST_PAYLOAD,

        "source": {
            "provider":
                "Hugging Face Hub",
            "repo_id":
                MODEL_ID,
            "requested_revision":
                MODEL_REVISION,
            "resolved_commit_sha":
                repository_information.sha,
        },

        "selection": {
            "weight_file":
                weight_filename,
            "weight_format":
                weight_format,
            "selected_repository_files":
                selected_filenames,
            "unselected_repository_files_are_not_part_of_the_model_input":
                True,
        },

        "model_config_audit": {
            "model_type":
                configuration.get(
                    "model_type"
                ),
            "architectures":
                configuration.get(
                    "architectures"
                ),
            "hidden_size":
                configuration.get(
                    "hidden_size"
                ),
            "num_hidden_layers":
                configuration.get(
                    "num_hidden_layers"
                ),
            "num_attention_heads":
                configuration.get(
                    "num_attention_heads"
                ),
            "vocab_size":
                configuration.get(
                    "vocab_size"
                ),
        },

        "acquisition_environment": {
            "python":
                ".".join(
                    map(
                        str,
                        sys.version_info[:3],
                    )
                ),
            "huggingface_hub":
                metadata.version(
                    "huggingface-hub"
                ),
        },

        "source_files":
            source_registry,

        "model_loaded":
            False,

        "classification_head_created":
            False,

        "training_or_inference_performed":
            False,
    }


    checkpoint_manifest_payload_sha = (
        canonical_sha256(
            checkpoint_manifest_payload
        )
    )

    checkpoint_manifest = dict(
        checkpoint_manifest_payload
    )

    checkpoint_manifest[
        "manifest_payload_sha256"
    ] = checkpoint_manifest_payload_sha


    OUTPUT_DIR.mkdir(
        parents=False,
        exist_ok=False,
    )


    for filename in selected_filenames:

        shutil.copyfile(
            download_root / filename,
            OUTPUT_DIR / filename,
        )


    checkpoint_manifest_path = (
        OUTPUT_DIR
        / "checkpoint_manifest.json"
    )

    checkpoint_manifest_path.write_text(
        json.dumps(
            checkpoint_manifest,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )


    artifact_hashes = {
        path.name:
            file_sha256(path)

        for path in sorted(
            OUTPUT_DIR.iterdir()
        )

        if path.is_file()
    }


    checksum_registry_path = (
        OUTPUT_DIR
        / "SHA256SUMS.txt"
    )

    checksum_registry_path.write_text(
        "".join(
            f"{artifact_hashes[filename]}  "
            f"{filename}\n"

            for filename
            in sorted(artifact_hashes)
        ),
        encoding="utf-8",
    )


section("FROZEN CHECKPOINT FILE REGISTRY")

for path in sorted(
    item
    for item in OUTPUT_DIR.iterdir()
    if item.is_file()
):

    print(
        f"{path.name:32s} "
        f"{path.stat().st_size:12d} bytes  "
        f"{file_sha256(path)}"
    )


print(
    "Checkpoint manifest payload SHA256:",
    checkpoint_manifest_payload_sha,
)


# --------------------------------------------------------------
# Deterministic ZIP preservation
# --------------------------------------------------------------

deterministic_zip(
    OUTPUT_DIR,
    ZIP_PATH,
)

checkpoint_zip_sha = file_sha256(
    ZIP_PATH
)

ZIP_SHA_PATH.write_text(
    f"{checkpoint_zip_sha}  "
    f"{ZIP_PATH.name}\n",
    encoding="utf-8",
)


with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as archive:

    damaged_member = archive.testzip()

    require(
        damaged_member is None,
        "ZIP integrity failure at "
        f"{damaged_member}",
    )


print("\n" + "=" * WIDTH)

print(
    "STEP M1B DECISION: "
    "PINNED BASE CHECKPOINT "
    "ACQUIRED AND FROZEN"
)

print("=" * WIDTH)

print(
    "Resolved commit        :",
    repository_information.sha,
)

print(
    "Checkpoint directory   :",
    OUTPUT_DIR,
)

print(
    "Checkpoint ZIP         :",
    ZIP_PATH,
)

print(
    "Checkpoint ZIP SHA256  :",
    checkpoint_zip_sha,
)

print(
    "Checksum sidecar       :",
    ZIP_SHA_PATH,
)

print("Model loaded           : False")
print("Classification head    : not created")
print("Training/inference     : NONE")

print(
    "\nDownload the ZIP and .sha256.txt "
    "and verify the ZIP externally."
)

print(
    "STOP — do not load the model or "
    "run the model-contract cell yet."
)

PHASE 2B — STEP M1B: PINNED DEBERTA BASE ACQUISITION AND PRESERVATION

SOURCE REQUEST
----------------------------------------------------------------------------------------------------------------------
Repository        : microsoft/deberta-v3-base
Requested revision: 8ccc9b6f36199bec6961081d44eb72fb3f7353f3
T2 continuity     : PASS
huggingface_hub   : 1.11.0

PINNED REPOSITORY RESOLUTION
----------------------------------------------------------------------------------------------------------------------
Resolved commit : 8ccc9b6f36199bec6961081d44eb72fb3f7353f3
Repository files: 8
Weight file     : pytorch_model.bin
Weight format   : pytorch_bin
Selected files  : 5
  config.json
  pytorch_model.bin
  spm.model
  tokenizer_config.json
  README.md

PINNED DOWNLOAD
----------------------------------------------------------------------------------------------------------------------


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]


FROZEN CHECKPOINT FILE REGISTRY
----------------------------------------------------------------------------------------------------------------------
README.md                                3472 bytes  7e9ea69ca00d0609cc0e0b9cec76f9ba6ae71601244c4136a3f25bd02dce4c4d
SHA256SUMS.txt                            493 bytes  ebb71ddb215b97b5e3a92e3c91b91fabb46dbc43d2f744bd091a01f427489b7a
checkpoint_manifest.json                 2022 bytes  85fad7bc2d48a9d07eb346ac1b77741dcd121113f800af84b2a7f0cb647f4d56
config.json                               579 bytes  649f6a1ec33c6bdd9a6486d5c66019d461139e54957073eafe9bbc2d34c75b0b
pytorch_model.bin                   371146213 bytes  691d48a2800b926a19e3051def466fc2cca4f59a15e42ce4a0cf7f1b380b5e33
spm.model                             2464616 bytes  c679fbf93643d19aab7ee10c0b99e460bdbc02fedf34b92b05af343b4af586fd
tokenizer_config.json                      52 bytes  3f3978e0c036f2c2588cac34a6047cbb0af0b0dc1814254e291028529805496d
Checkpoint manifest pa

## PHASE 2B — STEP M1B: PINNED DEBERTA BASE ACQUISITION AND PRESERVATION

Finding 1 — Weight format is pytorch_model.bin, not model.safetensors

At the pinned revision 8ccc9b6f..., the repository does not contain model.safetensors. The cell correctly fell through to pytorch_model.bin (371,146,213 bytes). The contract's sha256_target field already covers both formats, so no change is needed there. Training will load pytorch_model.bin — confirm transformers 5.0.0 handles this correctly, which it does.

Finding 2 — Tokeniser uses SentencePiece only

tokenizer.json is absent from this revision. The tokeniser vocabulary is spm.model (2,464,616 bytes). sentencepiece 0.2.1 is installed, so loading will succeed. The contract tokeniser section references microsoft/deberta-v3-base at the pinned revision — this is sufficient.

Finding 3 — Unauthenticated HuggingFace warning is benign

The warning about HF_TOKEN only affects rate limits, not download integrity. The resolved commit SHA matches exactly. No action needed.

Finding 4 — Concrete SHA-256 values now available for the contract

These must be embedded in the Gate MC contract before it is frozen:

Item	SHA-256
Checkpoint ZIP	4b91307e1a73dec76723c8318dfe8af09b952a4dca428169ae4eaa7cc2ecb7a7
Checkpoint manifest payload	da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c
pytorch_model.bin	691d48a2800b926a19e3051def466fc2cca4f59a15e42ce4a0cf7f1b380b5e33

## PHASE 2B — STEP M1C-A: PINNED CHECKPOINT INPUT VERIFICATION

In [7]:
# PHASE 2B — STEP M1C-A
# Verify the mounted pinned DeBERTa checkpoint.
# No model or tokenizer is loaded.

from pathlib import Path
import hashlib
import json
import sys


WIDTH = 116
INPUT_ROOT = Path("/kaggle/input")

EXPECTED_T2_MANIFEST_PAYLOAD = (
    "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
)

EXPECTED_CHECKPOINT_ZIP_SHA256 = (
    "4b91307e1a73dec76723c8318dfe8af09b952a4dca428169ae4eaa7cc2ecb7a7"
)

EXPECTED_CHECKPOINT_MANIFEST_PAYLOAD = (
    "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c"
)

EXPECTED_REVISION = (
    "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
)

EXPECTED_SHA256SUMS_SHA256 = (
    "ebb71ddb215b97b5e3a92e3c91b91fabb46dbc43d2f744bd091a01f427489b7a"
)

EXPECTED_FILES = {
    "README.md": (
        3472,
        "7e9ea69ca00d0609cc0e0b9cec76f9ba6ae71601244c4136a3f25bd02dce4c4d",
    ),

    "checkpoint_manifest.json": (
        2022,
        "85fad7bc2d48a9d07eb346ac1b77741dcd121113f800af84b2a7f0cb647f4d56",
    ),

    "config.json": (
        579,
        "649f6a1ec33c6bdd9a6486d5c66019d461139e54957073eafe9bbc2d34c75b0b",
    ),

    "pytorch_model.bin": (
        371146213,
        "691d48a2800b926a19e3051def466fc2cca4f59a15e42ce4a0cf7f1b380b5e33",
    ),

    "spm.model": (
        2464616,
        "c679fbf93643d19aab7ee10c0b99e460bdbc02fedf34b92b05af343b4af586fd",
    ),

    "tokenizer_config.json": (
        52,
        "3f3978e0c036f2c2588cac34a6047cbb0af0b0dc1814254e291028529805496d",
    ),
}


def require(condition, message):
    if not condition:
        raise RuntimeError(
            "STOP — " + message
        )


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


print("=" * WIDTH)
print(
    "PHASE 2B — STEP M1C-A: "
    "PINNED CHECKPOINT INPUT VERIFICATION"
)
print("=" * WIDTH)

print("Python     :", sys.version.split()[0])
print("Input root :", INPUT_ROOT)

require(
    INPUT_ROOT.is_dir(),
    "Kaggle input root is unavailable",
)


# --------------------------------------------------------------
# T2 continuity
# --------------------------------------------------------------

section("T2 CONTINUITY")

t2_candidates = sorted(
    INPUT_ROOT.rglob(
        "corpus_manifest.json"
    )
)

require(
    len(t2_candidates) == 1,
    "expected one T2 manifest; "
    f"found {len(t2_candidates)}",
)

t2_manifest = json.loads(
    t2_candidates[0].read_text(
        encoding="utf-8"
    )
)

require(
    t2_manifest.get(
        "manifest_payload_sha256"
    )
    == EXPECTED_T2_MANIFEST_PAYLOAD,
    "T2 manifest payload differs",
)

print(
    "T2 manifest payload:",
    t2_manifest[
        "manifest_payload_sha256"
    ],
)

print("T2 continuity      : PASS")


# --------------------------------------------------------------
# Checkpoint discovery
# --------------------------------------------------------------

section("CHECKPOINT INPUT DISCOVERY")

checkpoint_candidates = sorted(
    INPUT_ROOT.rglob(
        "checkpoint_manifest.json"
    )
)

require(
    len(checkpoint_candidates) == 1,
    "expected one checkpoint manifest; "
    f"found {len(checkpoint_candidates)}",
)

CHECKPOINT_ROOT = (
    checkpoint_candidates[0].parent
)

print(
    "Checkpoint root:",
    CHECKPOINT_ROOT,
)


actual_names = {
    path.name
    for path in CHECKPOINT_ROOT.iterdir()
    if path.is_file()
}

expected_names = (
    set(EXPECTED_FILES)
    | {"SHA256SUMS.txt"}
)

require(
    actual_names == expected_names,
    "checkpoint file set differs: "
    f"{sorted(actual_names)}",
)


# --------------------------------------------------------------
# ZIP provenance
# --------------------------------------------------------------

sidecars = sorted(
    INPUT_ROOT.rglob(
        "phase2B_DEBERTA_V3_BASE_PINNED.zip.sha256.txt"
    )
)

require(
    len(sidecars) == 1,
    "expected one checkpoint ZIP sidecar; "
    f"found {len(sidecars)}",
)

sidecar_tokens = (
    sidecars[0]
    .read_text(
        encoding="utf-8",
        errors="strict",
    )
    .strip()
    .split()
)

require(
    sidecar_tokens
    and sidecar_tokens[0].lower()
    == EXPECTED_CHECKPOINT_ZIP_SHA256,
    "checkpoint ZIP sidecar differs",
)

print(
    "ZIP checksum sidecar:",
    sidecars[0],
)

print(
    "Recorded ZIP SHA256 :",
    sidecar_tokens[0].lower(),
)


mounted_zips = sorted(
    INPUT_ROOT.rglob(
        "phase2B_DEBERTA_V3_BASE_PINNED.zip"
    )
)

require(
    len(mounted_zips) <= 1,
    "multiple checkpoint ZIPs found",
)

if mounted_zips:

    require(
        file_sha256(mounted_zips[0])
        == EXPECTED_CHECKPOINT_ZIP_SHA256,
        "mounted checkpoint ZIP differs",
    )

    print(
        "ZIP runtime status  : "
        "BYTE_VERIFIED_IN_RUNTIME"
    )

else:

    print(
        "ZIP runtime status  : "
        "Kaggle-extracted; externally "
        "verified ZIP recorded"
    )


# --------------------------------------------------------------
# File registry
# --------------------------------------------------------------

section("CHECKPOINT FILE REGISTRY")

for filename, (
    expected_size,
    expected_hash,
) in EXPECTED_FILES.items():

    path = (
        CHECKPOINT_ROOT
        / filename
    )

    require(
        path.is_file(),
        "missing checkpoint file: "
        f"{filename}",
    )

    require(
        path.stat().st_size
        == expected_size,
        "size mismatch: "
        f"{filename}",
    )

    actual_hash = file_sha256(path)

    require(
        actual_hash == expected_hash,
        "checksum mismatch: "
        f"{filename}",
    )

    print(
        f"{filename:32s} "
        f"{expected_size:12d} bytes  PASS"
    )


checksums_path = (
    CHECKPOINT_ROOT
    / "SHA256SUMS.txt"
)

require(
    file_sha256(checksums_path)
    == EXPECTED_SHA256SUMS_SHA256,
    "SHA256SUMS.txt checksum differs",
)


registered_hashes = {}

for line_number, line in enumerate(
    checksums_path
    .read_text(
        encoding="utf-8"
    )
    .splitlines(),
    start=1,
):

    parts = line.split(
        None,
        1,
    )

    require(
        len(parts) == 2,
        "malformed SHA256SUMS "
        f"line {line_number}",
    )

    registered_hashes[
        parts[1].strip()
    ] = parts[0].lower()


expected_registry = {
    filename: expected_hash
    for filename, (
        _,
        expected_hash,
    ) in EXPECTED_FILES.items()
}

require(
    registered_hashes
    == expected_registry,
    "internal checkpoint SHA "
    "registry differs",
)

print(
    f"{'SHA256SUMS.txt':32s} "
    f"{checksums_path.stat().st_size:12d} "
    "bytes  PASS"
)


# --------------------------------------------------------------
# Checkpoint manifest
# --------------------------------------------------------------

section("CHECKPOINT MANIFEST")

checkpoint_manifest = json.loads(
    (
        CHECKPOINT_ROOT
        / "checkpoint_manifest.json"
    ).read_text(
        encoding="utf-8"
    )
)

stored_payload = (
    checkpoint_manifest.get(
        "manifest_payload_sha256"
    )
)

manifest_payload = {
    key: value
    for key, value
    in checkpoint_manifest.items()
    if key
    != "manifest_payload_sha256"
}

calculated_payload = (
    canonical_sha256(
        manifest_payload
    )
)

require(
    stored_payload
    == EXPECTED_CHECKPOINT_MANIFEST_PAYLOAD,
    "stored checkpoint payload differs",
)

require(
    calculated_payload
    == EXPECTED_CHECKPOINT_MANIFEST_PAYLOAD,
    "recomputed checkpoint payload differs",
)

require(
    manifest_payload.get("phase")
    == "Phase 2B",
    "checkpoint phase differs",
)

require(
    manifest_payload.get("gate")
    == "M1B",
    "checkpoint gate differs",
)

require(
    manifest_payload.get("status")
    == "PINNED_BASE_CHECKPOINT_PRESERVED",
    "checkpoint status differs",
)

require(
    manifest_payload[
        "source"
    ]["repo_id"]
    == "microsoft/deberta-v3-base",
    "repository ID differs",
)

require(
    manifest_payload[
        "source"
    ]["requested_revision"]
    == EXPECTED_REVISION,
    "requested revision differs",
)

require(
    manifest_payload[
        "source"
    ]["resolved_commit_sha"]
    == EXPECTED_REVISION,
    "resolved commit differs",
)

require(
    manifest_payload[
        "selection"
    ]["weight_file"]
    == "pytorch_model.bin",
    "weight selection differs",
)

require(
    manifest_payload[
        "selection"
    ]["weight_format"]
    == "pytorch_bin",
    "weight format differs",
)

require(
    manifest_payload[
        "model_loaded"
    ] is False,
    "source manifest unexpectedly "
    "records model loading",
)

require(
    manifest_payload[
        "classification_head_created"
    ] is False,
    "source manifest unexpectedly "
    "records head creation",
)

require(
    manifest_payload[
        "training_or_inference_performed"
    ] is False,
    "source manifest unexpectedly "
    "records execution",
)


for filename, details in (
    manifest_payload[
        "source_files"
    ].items()
):

    require(
        filename in EXPECTED_FILES,
        "unexpected source file "
        f"in manifest: {filename}",
    )

    require(
        details["size_bytes"]
        == EXPECTED_FILES[
            filename
        ][0],
        "manifest size differs: "
        f"{filename}",
    )

    require(
        details["sha256"]
        == EXPECTED_FILES[
            filename
        ][1],
        "manifest hash differs: "
        f"{filename}",
    )


configuration = json.loads(
    (
        CHECKPOINT_ROOT
        / "config.json"
    ).read_text(
        encoding="utf-8"
    )
)

require(
    configuration.get(
        "model_type"
    ) == "deberta-v2",
    "unexpected model_type",
)


print(
    "Stored manifest payload    :",
    stored_payload,
)

print(
    "Recomputed manifest payload:",
    calculated_payload,
)

print(
    "Repository                 :",
    manifest_payload[
        "source"
    ]["repo_id"],
)

print(
    "Resolved commit            :",
    manifest_payload[
        "source"
    ]["resolved_commit_sha"],
)

print(
    "Weight file                :",
    manifest_payload[
        "selection"
    ]["weight_file"],
)

print(
    "Model type                 :",
    configuration.get(
        "model_type"
    ),
)


# --------------------------------------------------------------
# Retain verified checkpoint location
# --------------------------------------------------------------

PHASE2B_PINNED_CHECKPOINT_ROOT = (
    CHECKPOINT_ROOT
)

PHASE2B_PINNED_CHECKPOINT = {
    "root":
        str(CHECKPOINT_ROOT),

    "zip_sha256":
        EXPECTED_CHECKPOINT_ZIP_SHA256,

    "manifest_payload_sha256":
        stored_payload,

    "repo_id":
        manifest_payload[
            "source"
        ]["repo_id"],

    "revision":
        manifest_payload[
            "source"
        ]["resolved_commit_sha"],

    "weight_file_sha256":
        EXPECTED_FILES[
            "pytorch_model.bin"
        ][1],
}


print("\n" + "=" * WIDTH)

print(
    "STEP M1C-A DECISION: PASS — "
    "PINNED CHECKPOINT INPUT VERIFIED"
)

print("=" * WIDTH)

print("Model instantiated : False")
print("Tokenizer loaded   : False")
print("Training/inference : NONE")

print(
    "No file was written, "
    "transformed or modified."
)

print(
    "STOP — share this complete output "
    "before three-class initialisation."
)

PHASE 2B — STEP M1C-A: PINNED CHECKPOINT INPUT VERIFICATION
Python     : 3.12.13
Input root : /kaggle/input

T2 CONTINUITY
--------------------------------------------------------------------------------------------------------------------
T2 manifest payload: b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8
T2 continuity      : PASS

CHECKPOINT INPUT DISCOVERY
--------------------------------------------------------------------------------------------------------------------
Checkpoint root: /kaggle/input/datasets/mdshahrirhabib/phase2b-deberta-v3-base-pinned-20260910/phase2B_DEBERTA_V3_BASE_PINNED/phase2B_DEBERTA_V3_BASE_PINNED
ZIP checksum sidecar: /kaggle/input/datasets/mdshahrirhabib/phase2b-deberta-v3-base-pinned-20260910/phase2B_DEBERTA_V3_BASE_PINNED.zip.sha256.txt
Recorded ZIP SHA256 : 4b91307e1a73dec76723c8318dfe8af09b952a4dca428169ae4eaa7cc2ecb7a7
ZIP runtime status  : Kaggle-extracted; externally verified ZIP recorded

CHECKPOINT FILE REGISTRY
--------------

In [9]:
from pathlib import Path
import transformers

print("=" * 110)
print("PHASE 2B — M1C-B TOKENIZER IMPLEMENTATION DIAGNOSTIC")
print("=" * 110)

roots = sorted(Path("/kaggle/input").rglob("checkpoint_manifest.json"))

if len(roots) != 1:
    raise RuntimeError(
        f"STOP — expected exactly one checkpoint manifest; found {len(roots)}"
    )

checkpoint_root = roots[0].parent

print("Transformers version :", transformers.__version__)
print("Checkpoint root      :", checkpoint_root)
print("spm.model present    :", (checkpoint_root / "spm.model").is_file())

print("\nAUTO TOKENIZER WITH use_fast=False")
print("-" * 110)

from transformers import AutoTokenizer

auto_tokenizer = AutoTokenizer.from_pretrained(
    str(checkpoint_root),
    local_files_only=True,
    use_fast=False,
)

print("Class       :", auto_tokenizer.__class__.__name__)
print("is_fast     :", auto_tokenizer.is_fast)
print("Vocab size  :", auto_tokenizer.vocab_size)

print("\nEXPLICIT DEBERTA-V2 SLOW TOKENIZER")
print("-" * 110)

try:
    from transformers import DebertaV2Tokenizer

    slow_tokenizer = DebertaV2Tokenizer.from_pretrained(
        str(checkpoint_root),
        local_files_only=True,
    )

    print("Class       :", slow_tokenizer.__class__.__name__)
    print("is_fast     :", slow_tokenizer.is_fast)
    print("Vocab size  :", slow_tokenizer.vocab_size)
    print("Load status : PASS")

    probe_texts = [
        "The hotel room was clean and comfortable.",
        "I couldn't recommend it—service was poor.",
        "Price: $125.50; rating 4/5.",
    ]

    print("\nCONTROLLED TOKEN-ID COMPARISON")
    print("-" * 110)

    all_equal = True

    for probe_number, text in enumerate(probe_texts, start=1):
        auto_ids = auto_tokenizer(
            text,
            add_special_tokens=True,
        )["input_ids"]

        slow_ids = slow_tokenizer(
            text,
            add_special_tokens=True,
        )["input_ids"]

        equal = auto_ids == slow_ids
        all_equal = all_equal and equal

        print(
            f"Probe {probe_number}: "
            f"auto_n={len(auto_ids)} | "
            f"slow_n={len(slow_ids)} | "
            f"identical={equal}"
        )

    print("\nAll controlled token-ID sequences identical:", all_equal)

except Exception as exc:
    print("Load status : FAILED")
    print("Exception   :", type(exc).__name__)
    print("Message     :", str(exc))

print("\n" + "=" * 110)
print("TOKENIZER DIAGNOSTIC COMPLETE")
print("=" * 110)
print("No model loaded.")
print("No model initialised.")
print("No file written.")
print("No training or inference performed.")
print("STOP — share this complete output before revising M1C-B.")

PHASE 2B — M1C-B TOKENIZER IMPLEMENTATION DIAGNOSTIC
Transformers version : 5.0.0
Checkpoint root      : /kaggle/input/datasets/mdshahrirhabib/phase2b-deberta-v3-base-pinned-20260910/phase2B_DEBERTA_V3_BASE_PINNED/phase2B_DEBERTA_V3_BASE_PINNED
spm.model present    : True

AUTO TOKENIZER WITH use_fast=False
--------------------------------------------------------------------------------------------------------------


The tokenizer you are loading from '/kaggle/input/datasets/mdshahrirhabib/phase2b-deberta-v3-base-pinned-20260910/phase2B_DEBERTA_V3_BASE_PINNED/phase2B_DEBERTA_V3_BASE_PINNED' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Class       : DebertaV2Tokenizer
is_fast     : True
Vocab size  : 128000

EXPLICIT DEBERTA-V2 SLOW TOKENIZER
--------------------------------------------------------------------------------------------------------------


The tokenizer you are loading from '/kaggle/input/datasets/mdshahrirhabib/phase2b-deberta-v3-base-pinned-20260910/phase2B_DEBERTA_V3_BASE_PINNED/phase2B_DEBERTA_V3_BASE_PINNED' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Class       : DebertaV2Tokenizer
is_fast     : True
Vocab size  : 128000
Load status : PASS

CONTROLLED TOKEN-ID COMPARISON
--------------------------------------------------------------------------------------------------------------
Probe 1: auto_n=8 | slow_n=8 | identical=True
Probe 2: auto_n=11 | slow_n=11 | identical=True
Probe 3: auto_n=12 | slow_n=12 | identical=True

All controlled token-ID sequences identical: True

TOKENIZER DIAGNOSTIC COMPLETE
No model loaded.
No model initialised.
No file written.
No training or inference performed.
STOP — share this complete output before revising M1C-B.


# PHASE 2B — STEP M1C-B: DETERMINISTIC THREE-CLASS INITIALISATION AND PRESERVATION

In [11]:
from pathlib import Path
import gc
import hashlib
import json
import os
import random
import sys
import zipfile

import numpy as np
import torch
import transformers
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer


WIDTH = 120
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_ROOT / "phase2B_DEBERTA_V3_3CLASS_INITIALISED"
OUTPUT_ZIP = WORKING_ROOT / "phase2B_DEBERTA_V3_3CLASS_INITIALISED.zip"
OUTPUT_SIDECAR = WORKING_ROOT / "phase2B_DEBERTA_V3_3CLASS_INITIALISED.zip.sha256.txt"
MODEL_DIR = OUTPUT_ROOT / "model"

SEED = 42
EXPECTED_T2_MANIFEST_PAYLOAD = (
    "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
)
EXPECTED_BASE_MANIFEST_PAYLOAD = (
    "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c"
)
EXPECTED_BASE_ZIP_SHA256 = (
    "4b91307e1a73dec76723c8318dfe8af09b952a4dca428169ae4eaa7cc2ecb7a7"
)
EXPECTED_BASE_WEIGHT_SHA256 = (
    "691d48a2800b926a19e3051def466fc2cca4f59a15e42ce4a0cf7f1b380b5e33"
)
EXPECTED_REPOSITORY = "microsoft/deberta-v3-base"
EXPECTED_REVISION = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"

LABEL2ID = {
    "presumed_truthful": 0,
    "human_deceptive": 1,
    "ai_generated": 2,
}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def parameter_sha256(model):
    """Hash names, dtypes, shapes and exact bytes of the complete state dict."""
    digest = hashlib.sha256()
    state = model.state_dict()
    for name in sorted(state):
        tensor = state[name].detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(tensor.dtype).encode("ascii"))
        digest.update(b"\0")
        digest.update(json.dumps(list(tensor.shape), separators=(",", ":")).encode("ascii"))
        digest.update(b"\0")
        digest.update(tensor.numpy().tobytes(order="C"))
        digest.update(b"\0")
    return digest.hexdigest()


def load_eligible_rows(t2_root):
    rows = []
    for role, filename, expected_n in (
        ("primary", "primary_tripartite_corpus.jsonl", 2396),
        ("llama_sensitivity", "llama_unseen_generator_sensitivity.jsonl", 790),
    ):
        path = t2_root / filename
        require(path.is_file(), f"missing frozen corpus file: {filename}")
        role_rows = []
        with path.open("r", encoding="utf-8", errors="strict") as handle:
            for line_number, line in enumerate(handle, start=1):
                record = json.loads(line)
                require(
                    isinstance(record.get("row_id"), str),
                    f"{filename}:{line_number} missing row_id",
                )
                require(
                    isinstance(record.get("text"), str),
                    f"{filename}:{line_number} missing text",
                )
                require(record["text"].strip() != "", f"{record['row_id']} has empty text")
                role_rows.append((role, record["row_id"], record["text"]))
        require(len(role_rows) == expected_n, f"{role} row count differs: {len(role_rows)}")
        rows.extend(role_rows)
    require(len(rows) == 3186, f"combined eligible row count differs: {len(rows)}")
    require(len({row_id for _, row_id, _ in rows}) == len(rows), "eligible row IDs are not unique")
    return rows


def tokenizer_backend_identity(tokenizer):
    backend = json.loads(tokenizer.backend_tokenizer.to_str())
    pretokenizer = backend.get("pre_tokenizer")
    require(pretokenizer is not None, "tokenizer backend has no pre-tokenizer")
    pretokenizer_json = json.dumps(
        pretokenizer,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    require("Metaspace" in pretokenizer_json, "native DeBERTa Metaspace pre-tokenizer is absent")
    require('"Split"' not in pretokenizer_json, "Mistral Split regex was applied to the DeBERTa tokenizer")
    return hashlib.sha256(pretokenizer_json.encode("utf-8")).hexdigest()


def token_id_fingerprint(tokenizer, rows):
    digest = hashlib.sha256()
    for _, row_id, text in rows:
        token_ids = tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            verbose=False,
        )["input_ids"]
        digest.update(row_id.encode("utf-8"))
        digest.update(b"\0")
        digest.update(json.dumps(token_ids, separators=(",", ":")).encode("ascii"))
        digest.update(b"\n")
    return digest.hexdigest()


def deterministic_zip(source_dir, destination):
    """Create a ZIP whose member order, timestamps and permissions are fixed."""
    base = source_dir.parent
    paths = sorted(path for path in source_dir.rglob("*") if path.is_file())
    with zipfile.ZipFile(
        destination,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for path in paths:
            arcname = path.relative_to(base).as_posix()
            info = zipfile.ZipInfo(arcname, date_time=(1980, 1, 1, 0, 0, 0))
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            info.create_system = 3
            archive.writestr(info, path.read_bytes(), compress_type=zipfile.ZIP_DEFLATED, compresslevel=9)


print("=" * WIDTH)
print("PHASE 2B — STEP M1C-B: DETERMINISTIC THREE-CLASS INITIALISATION AND PRESERVATION")
print("=" * WIDTH)
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Execution    : CPU-only initialisation; no training or inference")

require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")
require(not torch.cuda.is_available(), "this preservation cell must run in the CPU-only session")
require(not OUTPUT_ROOT.exists(), f"output directory already exists: {OUTPUT_ROOT}")
require(not OUTPUT_ZIP.exists(), f"output ZIP already exists: {OUTPUT_ZIP}")
require(not OUTPUT_SIDECAR.exists(), f"output sidecar already exists: {OUTPUT_SIDECAR}")

section("FROZEN INPUT CONTINUITY")
t2_candidates = sorted(INPUT_ROOT.rglob("corpus_manifest.json"))
require(len(t2_candidates) == 1, f"expected one T2 manifest; found {len(t2_candidates)}")
T2_ROOT = t2_candidates[0].parent
t2_manifest = json.loads(t2_candidates[0].read_text(encoding="utf-8"))
require(
    t2_manifest.get("manifest_payload_sha256") == EXPECTED_T2_MANIFEST_PAYLOAD,
    "T2 manifest payload differs",
)
eligible_rows = load_eligible_rows(T2_ROOT)

base_manifest_candidates = sorted(INPUT_ROOT.rglob("checkpoint_manifest.json"))
require(
    len(base_manifest_candidates) == 1,
    f"expected one pinned-base manifest; found {len(base_manifest_candidates)}",
)
BASE_ROOT = base_manifest_candidates[0].parent
base_manifest = json.loads(base_manifest_candidates[0].read_text(encoding="utf-8"))
stored_base_payload = base_manifest.get("manifest_payload_sha256")
base_payload = {
    key: value for key, value in base_manifest.items()
    if key != "manifest_payload_sha256"
}
require(stored_base_payload == EXPECTED_BASE_MANIFEST_PAYLOAD, "stored base manifest payload differs")
require(canonical_sha256(base_payload) == EXPECTED_BASE_MANIFEST_PAYLOAD, "recomputed base manifest payload differs")
require(base_payload["source"]["repo_id"] == EXPECTED_REPOSITORY, "base repository differs")
require(base_payload["source"]["resolved_commit_sha"] == EXPECTED_REVISION, "base revision differs")
require(file_sha256(BASE_ROOT / "pytorch_model.bin") == EXPECTED_BASE_WEIGHT_SHA256, "base weight checksum differs")

sidecars = sorted(INPUT_ROOT.rglob("phase2B_DEBERTA_V3_BASE_PINNED.zip.sha256.txt"))
require(len(sidecars) == 1, f"expected one base ZIP sidecar; found {len(sidecars)}")
sidecar_tokens = sidecars[0].read_text(encoding="utf-8", errors="strict").strip().split()
require(sidecar_tokens and sidecar_tokens[0].lower() == EXPECTED_BASE_ZIP_SHA256, "base ZIP sidecar differs")

print("T2 manifest payload       :", EXPECTED_T2_MANIFEST_PAYLOAD)
print("Eligible texts verified   :", len(eligible_rows))
print("Base manifest payload     :", EXPECTED_BASE_MANIFEST_PAYLOAD)
print("Base repository           :", EXPECTED_REPOSITORY)
print("Base resolved revision    :", EXPECTED_REVISION)
print("Base weight SHA256        :", EXPECTED_BASE_WEIGHT_SHA256)
print("Externally verified ZIP   :", EXPECTED_BASE_ZIP_SHA256)

section("DETERMINISTIC INITIALISATION POLICY")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

print("Seed                         :", SEED)
print("torch deterministic algorithms:", torch.are_deterministic_algorithms_enabled())
print("Torch threads                :", torch.get_num_threads())
print("Initialisation device        : CPU")
print("Initialisation dtype         : float32")
print("Label mapping                :", LABEL2ID)

section("LOCAL TOKENIZER AND THREE-CLASS MODEL INITIALISATION")
print("Tokenizer policy: retain DeBERTa's native Unigram + Metaspace pipeline.")
print("A known Transformers 5.0.0 false-positive Mistral warning may be emitted during the first local load.")
config = AutoConfig.from_pretrained(str(BASE_ROOT), local_files_only=True)
require(config.model_type == "deberta-v2", f"unexpected model_type: {config.model_type}")
config.num_labels = 3
config.id2label = ID2LABEL
config.label2id = LABEL2ID
config.problem_type = "single_label_classification"

tokenizer = AutoTokenizer.from_pretrained(
    str(BASE_ROOT),
    local_files_only=True,
)
initial_tokenizer_class = tokenizer.__class__.__name__
initial_tokenizer_is_fast = tokenizer.is_fast
initial_tokenizer_vocab_size = tokenizer.vocab_size
initial_special_ids = list(tokenizer.all_special_ids)
initial_pretokenizer_hash = tokenizer_backend_identity(tokenizer)
initial_token_id_hash = token_id_fingerprint(tokenizer, eligible_rows)

model, loading_info = AutoModelForSequenceClassification.from_pretrained(
    str(BASE_ROOT),
    config=config,
    local_files_only=True,
    output_loading_info=True,
)
model.to(device="cpu", dtype=torch.float32)
model.eval()

missing_keys = sorted(loading_info.get("missing_keys", []))
unexpected_keys = sorted(loading_info.get("unexpected_keys", []))
mismatched_keys = sorted(loading_info.get("mismatched_keys", []), key=str)
error_messages = loading_info.get("error_msgs", [])

allowed_missing_prefixes = ("classifier.", "pooler.")
allowed_unexpected_prefixes = ("lm_predictions.", "mask_predictions.")

require(missing_keys, "no newly initialised classification/pooler parameters were reported")
require(
    all(key.startswith(allowed_missing_prefixes) for key in missing_keys),
    f"unexpected missing base parameters: {missing_keys}",
)
require(
    any(key.startswith("classifier.") for key in missing_keys),
    "classification-head parameters were not reported as newly initialised",
)
require(
    all(key.startswith(allowed_unexpected_prefixes) for key in unexpected_keys),
    f"unexpected unused source parameters: {unexpected_keys}",
)
require(not mismatched_keys, f"mismatched parameters found: {mismatched_keys}")
require(not error_messages, f"loader errors found: {error_messages}")
require(model.config.num_labels == 3, "model does not have three output labels")
require(model.config.label2id == LABEL2ID, "saved model label2id mapping differs")
require(
    {int(key): value for key, value in model.config.id2label.items()} == ID2LABEL,
    "saved model id2label mapping differs",
)

floating_dtypes = {
    str(parameter.dtype)
    for parameter in model.parameters()
    if parameter.is_floating_point()
}
require(floating_dtypes == {"torch.float32"}, f"unexpected floating parameter dtypes: {floating_dtypes}")

parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_parameter_count = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
initial_parameter_hash = parameter_sha256(model)

print("Model class                :", model.__class__.__name__)
print("Tokenizer class            :", tokenizer.__class__.__name__)
print("Tokenizer fast             :", tokenizer.is_fast)
print("Vocabulary size            :", tokenizer.vocab_size)
print("Tokenizer policy           : native DeBERTa Unigram + Metaspace; no Mistral regex patch")
print("Pre-tokenizer SHA256       :", initial_pretokenizer_hash)
print("3,186-row token-ID SHA256  :", initial_token_id_hash)
print("Missing/new parameters     :", missing_keys)
print("Unused pretraining params  :", unexpected_keys)
print("Mismatched parameters      :", mismatched_keys)
print("Parameter count            :", parameter_count)
print("Trainable parameter count  :", trainable_parameter_count)
print("Initial parameter SHA256   :", initial_parameter_hash)

section("ARTIFACT SAVE AND INDEPENDENT RELOAD")
MODEL_DIR.mkdir(parents=True, exist_ok=False)
model.config.architectures = [model.__class__.__name__]
model.save_pretrained(
    str(MODEL_DIR),
    safe_serialization=True,
    max_shard_size="5GB",
)
tokenizer.save_pretrained(str(MODEL_DIR))

del model
del tokenizer
gc.collect()

reloaded_tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)
reloaded_model.to(device="cpu", dtype=torch.float32)
reloaded_model.eval()
reloaded_parameter_hash = parameter_sha256(reloaded_model)
reloaded_pretokenizer_hash = tokenizer_backend_identity(reloaded_tokenizer)
reloaded_token_id_hash = token_id_fingerprint(reloaded_tokenizer, eligible_rows)

require(reloaded_parameter_hash == initial_parameter_hash, "parameter hash changed after save/reload")
require(reloaded_model.config.num_labels == 3, "reloaded model does not have three labels")
require(reloaded_model.config.label2id == LABEL2ID, "reloaded label2id differs")
require(
    {int(key): value for key, value in reloaded_model.config.id2label.items()} == ID2LABEL,
    "reloaded id2label differs",
)
require(reloaded_tokenizer.__class__.__name__ == initial_tokenizer_class, "reloaded tokenizer class differs")
require(reloaded_tokenizer.is_fast == initial_tokenizer_is_fast, "reloaded tokenizer backend differs")
require(reloaded_tokenizer.vocab_size == initial_tokenizer_vocab_size, "reloaded vocabulary size differs")
require(list(reloaded_tokenizer.all_special_ids) == initial_special_ids, "reloaded special-token IDs differ")
require(reloaded_pretokenizer_hash == initial_pretokenizer_hash, "pre-tokenizer changed after save/reload")
require(reloaded_token_id_hash == initial_token_id_hash, "eligible-corpus token IDs changed after save/reload")

print("Saved model format          : safetensors")
print("Reloaded model class        :", reloaded_model.__class__.__name__)
print("Reloaded tokenizer class    :", reloaded_tokenizer.__class__.__name__)
print("Reloaded parameter SHA256   :", reloaded_parameter_hash)
print("Reloaded pre-tokenizer SHA256:", reloaded_pretokenizer_hash)
print("Reloaded token-ID SHA256    :", reloaded_token_id_hash)
print("Save/reload identity        : PASS")

del reloaded_model
del reloaded_tokenizer
gc.collect()

section("FROZEN INITIALISATION ARTIFACT")
model_files = sorted(path for path in MODEL_DIR.rglob("*") if path.is_file())
require(model_files, "no model artifact files were saved")

model_registry = {
    path.relative_to(OUTPUT_ROOT).as_posix(): {
        "size_bytes": path.stat().st_size,
        "sha256": file_sha256(path),
    }
    for path in model_files
}

manifest_payload = {
    "phase": "Phase 2B",
    "step": "M1C-B",
    "status": "FROZEN_THREE_CLASS_INITIALISATION",
    "upstream": {
        "t2_manifest_payload_sha256": EXPECTED_T2_MANIFEST_PAYLOAD,
        "base_checkpoint_manifest_payload_sha256": EXPECTED_BASE_MANIFEST_PAYLOAD,
        "base_checkpoint_zip_sha256": EXPECTED_BASE_ZIP_SHA256,
        "base_weight_sha256": EXPECTED_BASE_WEIGHT_SHA256,
        "repository": EXPECTED_REPOSITORY,
        "resolved_revision": EXPECTED_REVISION,
    },
    "environment": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "device": "cpu",
        "dtype": "float32",
    },
    "initialisation": {
        "seed": SEED,
        "torch_deterministic_algorithms": True,
        "torch_threads": 1,
        "model_class": "DebertaV2ForSequenceClassification",
        "num_labels": 3,
        "label2id": LABEL2ID,
        "id2label": {str(key): value for key, value in ID2LABEL.items()},
        "problem_type": "single_label_classification",
        "newly_initialised_keys": missing_keys,
        "unused_pretraining_keys": unexpected_keys,
        "mismatched_keys": mismatched_keys,
        "parameter_count": parameter_count,
        "trainable_parameter_count": trainable_parameter_count,
        "parameter_sha256": initial_parameter_hash,
        "tokenizer_class": initial_tokenizer_class,
        "tokenizer_is_fast": initial_tokenizer_is_fast,
        "tokenizer_vocab_size": initial_tokenizer_vocab_size,
        "tokenizer_policy": "native_deberta_unigram_metaspace_no_mistral_regex_patch",
        "tokenizer_policy_reason": (
            "Transformers 5.0.0 emits a false-positive Mistral-regex warning for the local "
            "versionless DeBERTa config with vocabulary above 100,000; the model is DeBERTa, "
            "so its native Metaspace pre-tokenizer is retained."
        ),
        "eligible_texts_tokenised": len(eligible_rows),
        "pretokenizer_sha256": initial_pretokenizer_hash,
        "eligible_corpus_token_ids_sha256": initial_token_id_hash,
        "tokenizer_save_reload_identity": True,
    },
    "artifact_files": model_registry,
    "training_performed": False,
    "inference_performed": False,
}
manifest_payload_hash = canonical_sha256(manifest_payload)
manifest = dict(manifest_payload)
manifest["manifest_payload_sha256"] = manifest_payload_hash
manifest_path = OUTPUT_ROOT / "initialisation_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

checksum_targets = sorted(
    path for path in OUTPUT_ROOT.rglob("*")
    if path.is_file() and path.name != "SHA256SUMS.txt"
)
checksum_lines = [
    f"{file_sha256(path)}  {path.relative_to(OUTPUT_ROOT).as_posix()}"
    for path in checksum_targets
]
checksum_path = OUTPUT_ROOT / "SHA256SUMS.txt"
checksum_path.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

for path in sorted(path for path in OUTPUT_ROOT.rglob("*") if path.is_file()):
    print(
        f"{path.relative_to(OUTPUT_ROOT).as_posix():48s} "
        f"{path.stat().st_size:12d} bytes  {file_sha256(path)}"
    )

deterministic_zip(OUTPUT_ROOT, OUTPUT_ZIP)
zip_hash = file_sha256(OUTPUT_ZIP)
OUTPUT_SIDECAR.write_text(f"{zip_hash}  {OUTPUT_ZIP.name}\n", encoding="utf-8")

print("Initialisation manifest payload SHA256:", manifest_payload_hash)
print("Initial parameter SHA256              :", initial_parameter_hash)

print("\n" + "=" * WIDTH)
print("STEP M1C-B DECISION: THREE-CLASS INITIALISATION FROZEN AND RELOAD-VERIFIED")
print("=" * WIDTH)
print("Output directory :", OUTPUT_ROOT)
print("Artifact ZIP     :", OUTPUT_ZIP)
print("ZIP SHA256       :", zip_hash)
print("Checksum sidecar :", OUTPUT_SIDECAR)
print("Training         : NONE")
print("Inference        : NONE")
print("Download the ZIP and .sha256.txt, then verify the ZIP externally.")
print("STOP — share this complete output before the model/hyperparameter contract is frozen.")


PHASE 2B — STEP M1C-B: DETERMINISTIC THREE-CLASS INITIALISATION AND PRESERVATION
Python       : 3.12.13
PyTorch      : 2.10.0+cpu
Transformers : 5.0.0
Execution    : CPU-only initialisation; no training or inference

FROZEN INPUT CONTINUITY
------------------------------------------------------------------------------------------------------------------------
T2 manifest payload       : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8
Eligible texts verified   : 3186
Base manifest payload     : da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c
Base repository           : microsoft/deberta-v3-base
Base resolved revision    : 8ccc9b6f36199bec6961081d44eb72fb3f7353f3
Base weight SHA256        : 691d48a2800b926a19e3051def466fc2cca4f59a15e42ce4a0cf7f1b380b5e33
Externally verified ZIP   : 4b91307e1a73dec76723c8318dfe8af09b952a4dca428169ae4eaa7cc2ecb7a7

DETERMINISTIC INITIALISATION POLICY
-------------------------------------------------------------------------

The tokenizer you are loading from '/kaggle/input/datasets/mdshahrirhabib/phase2b-deberta-v3-base-pinned-20260910/phase2B_DEBERTA_V3_BASE_PINNED/phase2B_DEBERTA_V3_BASE_PINNED' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/mdshahrirhabib/phase2b-deberta-v3-base-pinned-20260910/phase2B_DEBERTA_V3_BASE_PINNED/phase2B_DEBERTA_V3_BASE_PINNED
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight    

Model class                : DebertaV2ForSequenceClassification
Tokenizer class            : DebertaV2Tokenizer
Tokenizer fast             : True
Vocabulary size            : 128000
Tokenizer policy           : native DeBERTa Unigram + Metaspace; no Mistral regex patch
Pre-tokenizer SHA256       : 9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171
3,186-row token-ID SHA256  : 9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359
Missing/new parameters     : ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
Unused pretraining params  : ['lm_predictions.lm_head.LayerNorm.bias', 'lm_predictions.lm_head.LayerNorm.weight', 'lm_predictions.lm_head.bias', 'lm_predictions.lm_head.dense.bias', 'lm_predictions.lm_head.dense.weight', 'mask_predictions.LayerNorm.bias', 'mask_predictions.LayerNorm.weight', 'mask_predictions.classifier.bias', 'mask_predictions.classifier.weight', 'mask_predictions.dense.bias', 'mask_predictions.dense.weight

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Saved model format          : safetensors
Reloaded model class        : DebertaV2ForSequenceClassification
Reloaded tokenizer class    : DebertaV2Tokenizer
Reloaded parameter SHA256   : 7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b
Reloaded pre-tokenizer SHA256: 9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171
Reloaded token-ID SHA256    : 9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359
Save/reload identity        : PASS

FROZEN INITIALISATION ARTIFACT
------------------------------------------------------------------------------------------------------------------------
SHA256SUMS.txt                                            450 bytes  aa78fce45c7fd1d382301a919a65a8f1b82ec4c18276c3491477291d98ebae23
initialisation_manifest.json                             3629 bytes  56db8a75903e7f8a903e55aec5f86e9800c21c9e40cdb130e6d0e2c40be22a03
model/config.json                                        1171 bytes  7c3e21e3737140b8e9cc404cad5392d89

Mistral warning is a confirmed false positive

The tokenizer loaded correctly as DebertaV2Tokenizer using the native Unigram + Metaspace pipeline. The warning references a Mistral-specific regex issue that does not apply to DeBERTa-v3. The pre-tokenizer SHA-256 is frozen and will be verified at the start of each training cell to confirm the same tokenization is applied.

Loading report is exactly correct

Four missing/newly initialised parameters: classifier.bias, classifier.weight, pooler.dense.bias, pooler.dense.weight. These are precisely the classification head and pooler — the only parts that should be freshly initialised. The eleven unexpected/unused parameters are all pretraining MLM heads that are irrelevant to sequence classification. Zero mismatched parameters.

Save/reload identity confirmed

Initial parameter SHA-256 7840923f... matches the reloaded parameter SHA-256 exactly. The model is bit-identical after the save/reload cycle.

Values to record externally now

Item	SHA-256
M1C-B ZIP	e392a33112c5d8e94a1635bb1096d689cda41f40a69af55a319a125ad361613a
Initial parameter state	7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b
Initialisation manifest payload	171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45
Tokenizer corpus fingerprint (3,186 rows)	9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359
Pre-tokenizer SHA	9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171
model.safetensors	4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c

In [12]:
from pathlib import Path
import hashlib
import json
import sys


WIDTH = 120
INPUT_ROOT = Path("/kaggle/input")

EXPECTED_T2_PAYLOAD = (
    "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
)
EXPECTED_BASE_PAYLOAD = (
    "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c"
)
EXPECTED_INITIALISATION_PAYLOAD = (
    "171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45"
)
EXPECTED_ZIP_SHA256 = (
    "e392a33112c5d8e94a1635bb1096d689cda41f40a69af55a319a125ad361613a"
)
EXPECTED_PARAMETER_SHA256 = (
    "7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b"
)
EXPECTED_TOKEN_ID_SHA256 = (
    "9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359"
)
EXPECTED_PRETOKENIZER_SHA256 = (
    "9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171"
)
EXPECTED_BASE_REVISION = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"

EXPECTED_FILES = {
    "SHA256SUMS.txt": (
        450,
        "aa78fce45c7fd1d382301a919a65a8f1b82ec4c18276c3491477291d98ebae23",
    ),
    "initialisation_manifest.json": (
        3629,
        "56db8a75903e7f8a903e55aec5f86e9800c21c9e40cdb130e6d0e2c40be22a03",
    ),
    "model/config.json": (
        1171,
        "7c3e21e3737140b8e9cc404cad5392d890becac306251a5f71362f96ccc5351c",
    ),
    "model/model.safetensors": (
        737722332,
        "4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c",
    ),
    "model/tokenizer.json": (
        8339093,
        "3bd1f6a24a2c99354b5f36e826e877d92d01a80dc24c3c5b74d31aaf87637e4e",
    ),
    "model/tokenizer_config.json": (
        508,
        "53f3b008c8696b8863912f9134436172f0ac6f15163c720fe1f976bcedbc5f8c",
    ),
}

EXPECTED_LABEL2ID = {
    "presumed_truthful": 0,
    "human_deceptive": 1,
    "ai_generated": 2,
}
EXPECTED_ID2LABEL = {
    "0": "presumed_truthful",
    "1": "human_deceptive",
    "2": "ai_generated",
}
EXPECTED_MISSING = {
    "classifier.bias",
    "classifier.weight",
    "pooler.dense.bias",
    "pooler.dense.weight",
}
EXPECTED_UNUSED = {
    "lm_predictions.lm_head.LayerNorm.bias",
    "lm_predictions.lm_head.LayerNorm.weight",
    "lm_predictions.lm_head.bias",
    "lm_predictions.lm_head.dense.bias",
    "lm_predictions.lm_head.dense.weight",
    "mask_predictions.LayerNorm.bias",
    "mask_predictions.LayerNorm.weight",
    "mask_predictions.classifier.bias",
    "mask_predictions.classifier.weight",
    "mask_predictions.dense.bias",
    "mask_predictions.dense.weight",
}


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


print("=" * WIDTH)
print("PHASE 2B — STEP M1C-C: RE-UPLOADED THREE-CLASS INITIALISATION VERIFICATION")
print("=" * WIDTH)
print("Python     :", sys.version.split()[0])
print("Input root :", INPUT_ROOT)
require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")

section("UPSTREAM CONTINUITY")
t2_candidates = sorted(INPUT_ROOT.rglob("corpus_manifest.json"))
base_candidates = sorted(INPUT_ROOT.rglob("checkpoint_manifest.json"))
initialised_candidates = sorted(INPUT_ROOT.rglob("initialisation_manifest.json"))
require(len(t2_candidates) == 1, f"expected one T2 manifest; found {len(t2_candidates)}")
require(len(base_candidates) == 1, f"expected one base manifest; found {len(base_candidates)}")
require(
    len(initialised_candidates) == 1,
    f"expected one initialisation manifest; found {len(initialised_candidates)}",
)

t2_manifest = json.loads(t2_candidates[0].read_text(encoding="utf-8"))
base_manifest = json.loads(base_candidates[0].read_text(encoding="utf-8"))
require(t2_manifest.get("manifest_payload_sha256") == EXPECTED_T2_PAYLOAD, "T2 payload differs")
require(base_manifest.get("manifest_payload_sha256") == EXPECTED_BASE_PAYLOAD, "base payload differs")

ARTIFACT_ROOT = initialised_candidates[0].parent
print("T2 payload            :", EXPECTED_T2_PAYLOAD, "PASS")
print("Base payload          :", EXPECTED_BASE_PAYLOAD, "PASS")
print("Initialised root      :", ARTIFACT_ROOT)

section("EXTERNAL ZIP PROVENANCE")
sidecars = sorted(INPUT_ROOT.rglob("phase2B_DEBERTA_V3_3CLASS_INITIALISED.zip.sha256.txt"))
require(len(sidecars) == 1, f"expected one ZIP checksum sidecar; found {len(sidecars)}")
sidecar_tokens = sidecars[0].read_text(encoding="utf-8", errors="strict").strip().split()
require(sidecar_tokens, "ZIP checksum sidecar is empty")
require(sidecar_tokens[0].lower() == EXPECTED_ZIP_SHA256, "ZIP sidecar checksum differs")
print("Checksum sidecar      :", sidecars[0])
print("Recorded ZIP SHA256   :", sidecar_tokens[0].lower())

mounted_zips = sorted(INPUT_ROOT.rglob("phase2B_DEBERTA_V3_3CLASS_INITIALISED.zip"))
require(len(mounted_zips) <= 1, "multiple mounted initialisation ZIPs found")
if mounted_zips:
    require(file_sha256(mounted_zips[0]) == EXPECTED_ZIP_SHA256, "mounted ZIP checksum differs")
    print("ZIP runtime status    : BYTE_VERIFIED_IN_RUNTIME")
else:
    print("ZIP runtime status    : Kaggle-extracted; externally verified ZIP recorded")

section("COMPLETE ARTIFACT FILE REGISTRY")
actual_files = {
    path.relative_to(ARTIFACT_ROOT).as_posix()
    for path in ARTIFACT_ROOT.rglob("*")
    if path.is_file()
}
require(actual_files == set(EXPECTED_FILES), f"artifact file set differs: {sorted(actual_files)}")

for relative_name, (expected_size, expected_hash) in EXPECTED_FILES.items():
    path = ARTIFACT_ROOT / relative_name
    require(path.stat().st_size == expected_size, f"size mismatch: {relative_name}")
    require(file_sha256(path) == expected_hash, f"checksum mismatch: {relative_name}")
    print(f"{relative_name:48s} {expected_size:12d} bytes  PASS")

section("INTERNAL SHA-256 REGISTRY")
sums_path = ARTIFACT_ROOT / "SHA256SUMS.txt"
internal_registry = {}
for line_number, line in enumerate(sums_path.read_text(encoding="utf-8").splitlines(), start=1):
    parts = line.split(None, 1)
    require(len(parts) == 2, f"malformed SHA256SUMS line {line_number}")
    internal_registry[parts[1].strip()] = parts[0].lower()

expected_internal_registry = {
    name: details[1]
    for name, details in EXPECTED_FILES.items()
    if name != "SHA256SUMS.txt"
}
require(internal_registry == expected_internal_registry, "internal SHA256SUMS registry differs")
print("Registry entries       :", len(internal_registry))
print("Registry verification  : PASS")

section("INITIALISATION MANIFEST CONTRACT")
manifest = json.loads(initialised_candidates[0].read_text(encoding="utf-8"))
stored_payload = manifest.get("manifest_payload_sha256")
manifest_payload = {
    key: value for key, value in manifest.items()
    if key != "manifest_payload_sha256"
}
recomputed_payload = canonical_sha256(manifest_payload)
require(stored_payload == EXPECTED_INITIALISATION_PAYLOAD, "stored initialisation payload differs")
require(recomputed_payload == EXPECTED_INITIALISATION_PAYLOAD, "recomputed initialisation payload differs")
require(manifest_payload.get("phase") == "Phase 2B", "phase differs")
require(manifest_payload.get("step") == "M1C-B", "step differs")
require(
    manifest_payload.get("status") == "FROZEN_THREE_CLASS_INITIALISATION",
    "initialisation status differs",
)

upstream = manifest_payload["upstream"]
require(upstream["t2_manifest_payload_sha256"] == EXPECTED_T2_PAYLOAD, "manifest T2 link differs")
require(
    upstream["base_checkpoint_manifest_payload_sha256"] == EXPECTED_BASE_PAYLOAD,
    "manifest base link differs",
)
require(upstream["resolved_revision"] == EXPECTED_BASE_REVISION, "base revision differs")

initialisation = manifest_payload["initialisation"]
require(initialisation["seed"] == 42, "initialisation seed differs")
require(initialisation["num_labels"] == 3, "number of labels differs")
require(initialisation["label2id"] == EXPECTED_LABEL2ID, "label2id differs")
require(initialisation["id2label"] == EXPECTED_ID2LABEL, "id2label differs")
require(initialisation["model_class"] == "DebertaV2ForSequenceClassification", "model class differs")
require(initialisation["problem_type"] == "single_label_classification", "problem type differs")
require(set(initialisation["newly_initialised_keys"]) == EXPECTED_MISSING, "new parameter set differs")
require(set(initialisation["unused_pretraining_keys"]) == EXPECTED_UNUSED, "unused parameter set differs")
require(initialisation["mismatched_keys"] == [], "mismatched parameters recorded")
require(initialisation["parameter_count"] == 184424451, "parameter count differs")
require(initialisation["trainable_parameter_count"] == 184424451, "trainable parameter count differs")
require(initialisation["parameter_sha256"] == EXPECTED_PARAMETER_SHA256, "parameter SHA256 differs")
require(initialisation["tokenizer_class"] == "DebertaV2Tokenizer", "tokenizer class differs")
require(initialisation["tokenizer_is_fast"] is True, "tokenizer backend record differs")
require(initialisation["tokenizer_vocab_size"] == 128000, "tokenizer vocabulary size differs")
require(
    initialisation["tokenizer_policy"] == "native_deberta_unigram_metaspace_no_mistral_regex_patch",
    "tokenizer policy differs",
)
require(initialisation["eligible_texts_tokenised"] == 3186, "tokenized corpus count differs")
require(initialisation["pretokenizer_sha256"] == EXPECTED_PRETOKENIZER_SHA256, "pre-tokenizer hash differs")
require(
    initialisation["eligible_corpus_token_ids_sha256"] == EXPECTED_TOKEN_ID_SHA256,
    "token-ID fingerprint differs",
)
require(initialisation["tokenizer_save_reload_identity"] is True, "tokenizer reload identity not confirmed")
require(manifest_payload["training_performed"] is False, "manifest records training")
require(manifest_payload["inference_performed"] is False, "manifest records inference")

expected_model_registry = {
    name: {"size_bytes": details[0], "sha256": details[1]}
    for name, details in EXPECTED_FILES.items()
    if name.startswith("model/")
}
require(manifest_payload["artifact_files"] == expected_model_registry, "manifest artifact registry differs")

print("Stored payload         :", stored_payload)
print("Recomputed payload     :", recomputed_payload)
print("Parameter SHA256       :", initialisation["parameter_sha256"])
print("Pre-tokenizer SHA256   :", initialisation["pretokenizer_sha256"])
print("Token-ID SHA256        :", initialisation["eligible_corpus_token_ids_sha256"])
print("Label mapping          :", initialisation["label2id"])
print("Manifest verification  : PASS")

PHASE2B_INITIALISED_CHECKPOINT_ROOT = ARTIFACT_ROOT / "model"
PHASE2B_INITIALISED_CHECKPOINT = {
    "root": str(PHASE2B_INITIALISED_CHECKPOINT_ROOT),
    "zip_sha256": EXPECTED_ZIP_SHA256,
    "manifest_payload_sha256": EXPECTED_INITIALISATION_PAYLOAD,
    "parameter_sha256": EXPECTED_PARAMETER_SHA256,
    "token_id_sha256": EXPECTED_TOKEN_ID_SHA256,
}

print("\n" + "=" * WIDTH)
print("STEP M1C-C DECISION: PASS — RE-UPLOADED THREE-CLASS INITIALISATION VERIFIED")
print("=" * WIDTH)
print("Model loaded       : False")
print("Tokenizer loaded   : False")
print("Training/inference : NONE")
print("No file was written, extracted, transformed or modified.")
print("STOP — share this complete output before the model/hyperparameter contract is frozen.")


PHASE 2B — STEP M1C-C: RE-UPLOADED THREE-CLASS INITIALISATION VERIFICATION
Python     : 3.12.13
Input root : /kaggle/input

UPSTREAM CONTINUITY
------------------------------------------------------------------------------------------------------------------------
T2 payload            : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8 PASS
Base payload          : da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c PASS
Initialised root      : /kaggle/input/datasets/mdshahrirhabib/phase2b-deberta-v3-3class-initialised-20260910/phase2B_DEBERTA_V3_3CLASS_INITIALISED/phase2B_DEBERTA_V3_3CLASS_INITIALISED

EXTERNAL ZIP PROVENANCE
------------------------------------------------------------------------------------------------------------------------
Checksum sidecar      : /kaggle/input/datasets/mdshahrirhabib/phase2b-deberta-v3-3class-initialised-20260910/phase2B_DEBERTA_V3_3CLASS_INITIALISED.zip.sha256.txt
Recorded ZIP SHA256   : e392a33112c5d8e94a1635bb1096d6

In [1]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
import os
import platform
import subprocess
import sys


# This must be set before importing torch or creating a CUDA context.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch


WIDTH = 120
INPUT_ROOT = Path("/kaggle/input")
EXPECTED_T2_PAYLOAD = "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
EXPECTED_BASE_PAYLOAD = "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c"
EXPECTED_INITIALISATION_PAYLOAD = "171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45"
EXPECTED_INITIAL_MODEL_SHA256 = "4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c"
EXPECTED_INITIAL_MODEL_SIZE = 737722332
EXPECTED_INITIAL_PARAMETER_SHA256 = "7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b"
EXPECTED_TOKEN_ID_SHA256 = "9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359"

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "joblib": "1.5.3",
    "transformers": "5.0.0",
    "tokenizers": "0.22.2",
    "accelerate": "1.13.0",
    "safetensors": "0.7.0",
    "sentencepiece": "0.2.1",
}


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "MISSING"


print("=" * WIDTH)
print("PHASE 2B — STEP M2A: GPU TRAINING-ENVIRONMENT AND FROZEN-INPUT CONTINUITY AUDIT")
print("=" * WIDTH)
print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print("Platform          :", platform.platform())
print("Processor         :", platform.machine())

require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")
require(os.environ.get("CUBLAS_WORKSPACE_CONFIG") == ":4096:8", "CUDA workspace setting differs")

section("FROZEN INPUT CONTINUITY")
t2_candidates = sorted(INPUT_ROOT.rglob("corpus_manifest.json"))
base_candidates = sorted(INPUT_ROOT.rglob("checkpoint_manifest.json"))
initialised_candidates = sorted(INPUT_ROOT.rglob("initialisation_manifest.json"))
require(len(t2_candidates) == 1, f"expected one T2 manifest; found {len(t2_candidates)}")
require(len(base_candidates) == 1, f"expected one base manifest; found {len(base_candidates)}")
require(len(initialised_candidates) == 1, f"expected one initialisation manifest; found {len(initialised_candidates)}")

t2_manifest = json.loads(t2_candidates[0].read_text(encoding="utf-8"))
base_manifest = json.loads(base_candidates[0].read_text(encoding="utf-8"))
initialised_manifest = json.loads(initialised_candidates[0].read_text(encoding="utf-8"))
require(t2_manifest.get("manifest_payload_sha256") == EXPECTED_T2_PAYLOAD, "T2 payload differs")
require(base_manifest.get("manifest_payload_sha256") == EXPECTED_BASE_PAYLOAD, "base payload differs")
require(
    initialised_manifest.get("manifest_payload_sha256") == EXPECTED_INITIALISATION_PAYLOAD,
    "initialisation payload differs",
)
initialisation = initialised_manifest["initialisation"]
require(initialisation["parameter_sha256"] == EXPECTED_INITIAL_PARAMETER_SHA256, "parameter identity differs")
require(initialisation["eligible_corpus_token_ids_sha256"] == EXPECTED_TOKEN_ID_SHA256, "token-ID identity differs")

INITIALISED_MODEL_ROOT = initialised_candidates[0].parent / "model"
model_path = INITIALISED_MODEL_ROOT / "model.safetensors"
require(model_path.is_file(), "initialised model.safetensors is missing")
require(model_path.stat().st_size == EXPECTED_INITIAL_MODEL_SIZE, "initialised model size differs")
require(file_sha256(model_path) == EXPECTED_INITIAL_MODEL_SHA256, "initialised model checksum differs")

print("T2 payload             :", EXPECTED_T2_PAYLOAD, "PASS")
print("Base payload           :", EXPECTED_BASE_PAYLOAD, "PASS")
print("Initialisation payload :", EXPECTED_INITIALISATION_PAYLOAD, "PASS")
print("Initial parameter      :", EXPECTED_INITIAL_PARAMETER_SHA256, "PASS")
print("Token-ID fingerprint   :", EXPECTED_TOKEN_ID_SHA256, "PASS")
print("Initialised model file :", EXPECTED_INITIAL_MODEL_SHA256, "PASS")

section("CRITICAL PACKAGE VERSIONS")
actual_packages = {name: package_version(name) for name in EXPECTED_PACKAGES}
for name, expected in EXPECTED_PACKAGES.items():
    actual = actual_packages[name]
    status = "PASS" if actual == expected else "STOP"
    print(f"{name:18s}: {actual:18s} expected {expected:10s} {status}")
    require(actual == expected, f"{name} version differs: {actual} != {expected}")

torch_public_version = torch.__version__.split("+")[0]
require(torch_public_version == "2.10.0", f"PyTorch base version differs: {torch.__version__}")
print(f"{'torch':18s}: {torch.__version__:18s} expected base 2.10.0 PASS")

section("CUDA AND GPU IDENTITY")
require(torch.cuda.is_available(), "CUDA is unavailable; switch the notebook accelerator to GPU")
device_count = torch.cuda.device_count()
require(device_count >= 1, "no CUDA device was detected")
require(torch.version.cuda is not None, "PyTorch has no CUDA build")

devices = []
for index in range(device_count):
    props = torch.cuda.get_device_properties(index)
    device = {
        "index": index,
        "name": props.name,
        "total_memory_bytes": int(props.total_memory),
        "compute_capability": list(torch.cuda.get_device_capability(index)),
        "multiprocessor_count": int(props.multi_processor_count),
    }
    devices.append(device)
    print(
        f"GPU {index}: {device['name']} | memory={device['total_memory_bytes']} bytes | "
        f"capability={device['compute_capability']} | MPs={device['multiprocessor_count']}"
    )

print("CUDA available          :", torch.cuda.is_available())
print("CUDA device count       :", device_count)
print("PyTorch CUDA build      :", torch.version.cuda)
print("cuDNN version           :", torch.backends.cudnn.version())
print("Selected training device: cuda:0")
print("Multi-GPU training      : prohibited; additional devices, if present, remain unused")
print("CUBLAS workspace        :", os.environ["CUBLAS_WORKSPACE_CONFIG"])

try:
    nvidia_smi = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,name,driver_version,memory.total",
            "--format=csv,noheader,nounits",
        ],
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout.strip().splitlines()
except Exception as exc:
    nvidia_smi = [f"UNAVAILABLE: {type(exc).__name__}: {exc}"]

print("nvidia-smi identity:")
for line in nvidia_smi:
    print(" ", line)

environment_payload = {
    "phase": "Phase 2B",
    "step": "M2A",
    "status": "GPU_ENVIRONMENT_AUDITED",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "processor": platform.machine(),
    "packages": {**actual_packages, "torch": torch.__version__},
    "cuda": {
        "available": True,
        "device_count": device_count,
        "pytorch_cuda_build": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "selected_device": "cuda:0",
        "multi_gpu_training": False,
        "cublas_workspace_config": os.environ["CUBLAS_WORKSPACE_CONFIG"],
        "devices": devices,
        "nvidia_smi": nvidia_smi,
    },
    "frozen_inputs": {
        "t2_manifest_payload_sha256": EXPECTED_T2_PAYLOAD,
        "base_manifest_payload_sha256": EXPECTED_BASE_PAYLOAD,
        "initialisation_manifest_payload_sha256": EXPECTED_INITIALISATION_PAYLOAD,
        "initial_parameter_sha256": EXPECTED_INITIAL_PARAMETER_SHA256,
        "token_id_sha256": EXPECTED_TOKEN_ID_SHA256,
        "initialised_model_file_sha256": EXPECTED_INITIAL_MODEL_SHA256,
    },
}
environment_payload_sha256 = canonical_sha256(environment_payload)

print("\n" + "=" * WIDTH)
print("STEP M2A DECISION: PASS — GPU TRAINING ENVIRONMENT AND FROZEN INPUTS VERIFIED")
print("=" * WIDTH)
print("Environment payload SHA256:", environment_payload_sha256)
print("Model/tokenizer loaded     : False")
print("Training/inference       : NONE")
print("No file was written, transformed or modified.")
print("STOP — share this complete output before the M2 contract is frozen.")


PHASE 2B — STEP M2A: GPU TRAINING-ENVIRONMENT AND FROZEN-INPUT CONTINUITY AUDIT
Python executable : /usr/bin/python3
Python version    : 3.12.13
Platform          : Linux-6.12.90+-x86_64-with-glibc2.35
Processor         : x86_64

FROZEN INPUT CONTINUITY
------------------------------------------------------------------------------------------------------------------------
T2 payload             : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8 PASS
Base payload           : da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c PASS
Initialisation payload : 171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45 PASS
Initial parameter      : 7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b PASS
Token-ID fingerprint   : 9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359 PASS
Initialised model file : 4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c PASS

CRITICAL PACKAGE VERSIONS
----------------------------

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


## PHASE 2B — STEP M2A: GPU TRAINING-ENVIRONMENT AND FROZEN-INPUT CONTINUITY AUDIT

In [1]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
import os
import platform
import subprocess
import sys


# This must be set before importing torch or creating a CUDA context.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch


WIDTH = 120
INPUT_ROOT = Path("/kaggle/input")
EXPECTED_T2_PAYLOAD = "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
EXPECTED_BASE_PAYLOAD = "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c"
EXPECTED_INITIALISATION_PAYLOAD = "171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45"
EXPECTED_INITIAL_MODEL_SHA256 = "4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c"
EXPECTED_INITIAL_MODEL_SIZE = 737722332
EXPECTED_INITIAL_PARAMETER_SHA256 = "7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b"
EXPECTED_TOKEN_ID_SHA256 = "9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359"

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "joblib": "1.5.3",
    "transformers": "5.0.0",
    "tokenizers": "0.22.2",
    "accelerate": "1.13.0",
    "safetensors": "0.7.0",
    "sentencepiece": "0.2.1",
}


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "MISSING"


print("=" * WIDTH)
print("PHASE 2B — STEP M2A: GPU TRAINING-ENVIRONMENT AND FROZEN-INPUT CONTINUITY AUDIT")
print("=" * WIDTH)
print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print("Platform          :", platform.platform())
print("Processor         :", platform.machine())

require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")
require(os.environ.get("CUBLAS_WORKSPACE_CONFIG") == ":4096:8", "CUDA workspace setting differs")

section("FROZEN INPUT CONTINUITY")
t2_candidates = sorted(INPUT_ROOT.rglob("corpus_manifest.json"))
base_candidates = sorted(INPUT_ROOT.rglob("checkpoint_manifest.json"))
initialised_candidates = sorted(INPUT_ROOT.rglob("initialisation_manifest.json"))
require(len(t2_candidates) == 1, f"expected one T2 manifest; found {len(t2_candidates)}")
require(len(base_candidates) == 1, f"expected one base manifest; found {len(base_candidates)}")
require(len(initialised_candidates) == 1, f"expected one initialisation manifest; found {len(initialised_candidates)}")

t2_manifest = json.loads(t2_candidates[0].read_text(encoding="utf-8"))
base_manifest = json.loads(base_candidates[0].read_text(encoding="utf-8"))
initialised_manifest = json.loads(initialised_candidates[0].read_text(encoding="utf-8"))
require(t2_manifest.get("manifest_payload_sha256") == EXPECTED_T2_PAYLOAD, "T2 payload differs")
require(base_manifest.get("manifest_payload_sha256") == EXPECTED_BASE_PAYLOAD, "base payload differs")
require(
    initialised_manifest.get("manifest_payload_sha256") == EXPECTED_INITIALISATION_PAYLOAD,
    "initialisation payload differs",
)
initialisation = initialised_manifest["initialisation"]
require(initialisation["parameter_sha256"] == EXPECTED_INITIAL_PARAMETER_SHA256, "parameter identity differs")
require(initialisation["eligible_corpus_token_ids_sha256"] == EXPECTED_TOKEN_ID_SHA256, "token-ID identity differs")

INITIALISED_MODEL_ROOT = initialised_candidates[0].parent / "model"
model_path = INITIALISED_MODEL_ROOT / "model.safetensors"
require(model_path.is_file(), "initialised model.safetensors is missing")
require(model_path.stat().st_size == EXPECTED_INITIAL_MODEL_SIZE, "initialised model size differs")
require(file_sha256(model_path) == EXPECTED_INITIAL_MODEL_SHA256, "initialised model checksum differs")

print("T2 payload             :", EXPECTED_T2_PAYLOAD, "PASS")
print("Base payload           :", EXPECTED_BASE_PAYLOAD, "PASS")
print("Initialisation payload :", EXPECTED_INITIALISATION_PAYLOAD, "PASS")
print("Initial parameter      :", EXPECTED_INITIAL_PARAMETER_SHA256, "PASS")
print("Token-ID fingerprint   :", EXPECTED_TOKEN_ID_SHA256, "PASS")
print("Initialised model file :", EXPECTED_INITIAL_MODEL_SHA256, "PASS")

section("CRITICAL PACKAGE VERSIONS")
actual_packages = {name: package_version(name) for name in EXPECTED_PACKAGES}
for name, expected in EXPECTED_PACKAGES.items():
    actual = actual_packages[name]
    status = "PASS" if actual == expected else "STOP"
    print(f"{name:18s}: {actual:18s} expected {expected:10s} {status}")
    require(actual == expected, f"{name} version differs: {actual} != {expected}")

torch_public_version = torch.__version__.split("+")[0]
require(torch_public_version == "2.10.0", f"PyTorch base version differs: {torch.__version__}")
print(f"{'torch':18s}: {torch.__version__:18s} expected base 2.10.0 PASS")

section("CUDA AND GPU IDENTITY")
require(torch.cuda.is_available(), "CUDA is unavailable; switch the notebook accelerator to GPU")
device_count = torch.cuda.device_count()
require(device_count >= 1, "no CUDA device was detected")
require(torch.version.cuda is not None, "PyTorch has no CUDA build")

devices = []
for index in range(device_count):
    props = torch.cuda.get_device_properties(index)
    device = {
        "index": index,
        "name": props.name,
        "total_memory_bytes": int(props.total_memory),
        "compute_capability": list(torch.cuda.get_device_capability(index)),
        "multiprocessor_count": int(props.multi_processor_count),
    }
    devices.append(device)
    print(
        f"GPU {index}: {device['name']} | memory={device['total_memory_bytes']} bytes | "
        f"capability={device['compute_capability']} | MPs={device['multiprocessor_count']}"
    )

print("CUDA available          :", torch.cuda.is_available())
print("CUDA device count       :", device_count)
print("PyTorch CUDA build      :", torch.version.cuda)
print("cuDNN version           :", torch.backends.cudnn.version())
print("Selected training device: cuda:0")
print("Multi-GPU training      : prohibited; additional devices, if present, remain unused")
print("CUBLAS workspace        :", os.environ["CUBLAS_WORKSPACE_CONFIG"])

compiled_architectures = list(torch.cuda.get_arch_list())
selected_capability = devices[0]["compute_capability"]
selected_architecture = f"sm_{selected_capability[0]}{selected_capability[1]}"
print("PyTorch compiled arches :", compiled_architectures)
print("Selected GPU arch       :", selected_architecture)
require(
    selected_architecture in compiled_architectures,
    f"selected GPU architecture {selected_architecture} is absent from this PyTorch build",
)

section("CUDA KERNEL EXECUTION SMOKE TEST")
try:
    smoke_a = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device="cuda:0")
    smoke_b = smoke_a @ smoke_a
    torch.cuda.synchronize(0)
    smoke_result = smoke_b.detach().cpu().tolist()
    require(smoke_result == [[7.0, 10.0], [15.0, 22.0]], f"unexpected CUDA result: {smoke_result}")
except Exception as exc:
    raise RuntimeError(
        f"STOP — CUDA kernel execution failed: {type(exc).__name__}: {exc}"
    ) from exc
finally:
    if "smoke_a" in globals():
        del smoke_a
    if "smoke_b" in globals():
        del smoke_b
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("CUDA matrix multiplication: PASS")
print("CUDA synchronization       : PASS")

try:
    nvidia_smi = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,name,driver_version,memory.total",
            "--format=csv,noheader,nounits",
        ],
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout.strip().splitlines()
except Exception as exc:
    nvidia_smi = [f"UNAVAILABLE: {type(exc).__name__}: {exc}"]

print("nvidia-smi identity:")
for line in nvidia_smi:
    print(" ", line)

environment_payload = {
    "phase": "Phase 2B",
    "step": "M2A",
    "status": "GPU_ENVIRONMENT_AUDITED",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "processor": platform.machine(),
    "packages": {**actual_packages, "torch": torch.__version__},
    "cuda": {
        "available": True,
        "device_count": device_count,
        "pytorch_cuda_build": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "selected_device": "cuda:0",
        "multi_gpu_training": False,
        "cublas_workspace_config": os.environ["CUBLAS_WORKSPACE_CONFIG"],
        "pytorch_compiled_architectures": compiled_architectures,
        "selected_gpu_architecture": selected_architecture,
        "cuda_kernel_smoke_test": "PASS",
        "devices": devices,
        "nvidia_smi": nvidia_smi,
    },
    "frozen_inputs": {
        "t2_manifest_payload_sha256": EXPECTED_T2_PAYLOAD,
        "base_manifest_payload_sha256": EXPECTED_BASE_PAYLOAD,
        "initialisation_manifest_payload_sha256": EXPECTED_INITIALISATION_PAYLOAD,
        "initial_parameter_sha256": EXPECTED_INITIAL_PARAMETER_SHA256,
        "token_id_sha256": EXPECTED_TOKEN_ID_SHA256,
        "initialised_model_file_sha256": EXPECTED_INITIAL_MODEL_SHA256,
    },
}
environment_payload_sha256 = canonical_sha256(environment_payload)

print("\n" + "=" * WIDTH)
print("STEP M2A DECISION: PASS — GPU TRAINING ENVIRONMENT AND FROZEN INPUTS VERIFIED")
print("=" * WIDTH)
print("Environment payload SHA256:", environment_payload_sha256)
print("Model/tokenizer loaded     : False")
print("Training/inference       : NONE")
print("No file was written, transformed or modified.")
print("STOP — share this complete output before the M2 contract is frozen.")


PHASE 2B — STEP M2A: GPU TRAINING-ENVIRONMENT AND FROZEN-INPUT CONTINUITY AUDIT
Python executable : /usr/bin/python3
Python version    : 3.12.13
Platform          : Linux-6.12.90+-x86_64-with-glibc2.35
Processor         : x86_64

FROZEN INPUT CONTINUITY
------------------------------------------------------------------------------------------------------------------------
T2 payload             : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8 PASS
Base payload           : da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c PASS
Initialisation payload : 171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45 PASS
Initial parameter      : 7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b PASS
Token-ID fingerprint   : 9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359 PASS
Initialised model file : 4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c PASS

CRITICAL PACKAGE VERSIONS
----------------------------

M2A is a clean PASS. Tesla T4 (sm_75) is fully compatible. Two T4s are present but only cuda:0 is used as per the contract. The CUDA smoke test passed. All frozen input continuity checks passed.

In [2]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
import os
import platform
import subprocess
import sys


# This must be set before importing torch or creating a CUDA context.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch


WIDTH = 120
INPUT_ROOT = Path("/kaggle/input")
EXPECTED_T2_PAYLOAD = "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
EXPECTED_BASE_PAYLOAD = "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c"
EXPECTED_INITIALISATION_PAYLOAD = "171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45"
EXPECTED_INITIAL_MODEL_SHA256 = "4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c"
EXPECTED_INITIAL_MODEL_SIZE = 737722332
EXPECTED_INITIAL_PARAMETER_SHA256 = "7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b"
EXPECTED_TOKEN_ID_SHA256 = "9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359"

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "joblib": "1.5.3",
    "transformers": "5.0.0",
    "tokenizers": "0.22.2",
    "accelerate": "1.13.0",
    "safetensors": "0.7.0",
    "sentencepiece": "0.2.1",
}


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "MISSING"


print("=" * WIDTH)
print("PHASE 2B — STEP M2A: GPU TRAINING-ENVIRONMENT AND FROZEN-INPUT CONTINUITY AUDIT")
print("=" * WIDTH)
print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print("Platform          :", platform.platform())
print("Processor         :", platform.machine())

require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")
require(os.environ.get("CUBLAS_WORKSPACE_CONFIG") == ":4096:8", "CUDA workspace setting differs")

section("FROZEN INPUT CONTINUITY")
t2_candidates = sorted(INPUT_ROOT.rglob("corpus_manifest.json"))
base_candidates = sorted(INPUT_ROOT.rglob("checkpoint_manifest.json"))
initialised_candidates = sorted(INPUT_ROOT.rglob("initialisation_manifest.json"))
require(len(t2_candidates) == 1, f"expected one T2 manifest; found {len(t2_candidates)}")
require(len(base_candidates) == 1, f"expected one base manifest; found {len(base_candidates)}")
require(len(initialised_candidates) == 1, f"expected one initialisation manifest; found {len(initialised_candidates)}")

t2_manifest = json.loads(t2_candidates[0].read_text(encoding="utf-8"))
base_manifest = json.loads(base_candidates[0].read_text(encoding="utf-8"))
initialised_manifest = json.loads(initialised_candidates[0].read_text(encoding="utf-8"))
require(t2_manifest.get("manifest_payload_sha256") == EXPECTED_T2_PAYLOAD, "T2 payload differs")
require(base_manifest.get("manifest_payload_sha256") == EXPECTED_BASE_PAYLOAD, "base payload differs")
require(
    initialised_manifest.get("manifest_payload_sha256") == EXPECTED_INITIALISATION_PAYLOAD,
    "initialisation payload differs",
)
initialisation = initialised_manifest["initialisation"]
require(initialisation["parameter_sha256"] == EXPECTED_INITIAL_PARAMETER_SHA256, "parameter identity differs")
require(initialisation["eligible_corpus_token_ids_sha256"] == EXPECTED_TOKEN_ID_SHA256, "token-ID identity differs")

INITIALISED_MODEL_ROOT = initialised_candidates[0].parent / "model"
model_path = INITIALISED_MODEL_ROOT / "model.safetensors"
require(model_path.is_file(), "initialised model.safetensors is missing")
require(model_path.stat().st_size == EXPECTED_INITIAL_MODEL_SIZE, "initialised model size differs")
require(file_sha256(model_path) == EXPECTED_INITIAL_MODEL_SHA256, "initialised model checksum differs")

print("T2 payload             :", EXPECTED_T2_PAYLOAD, "PASS")
print("Base payload           :", EXPECTED_BASE_PAYLOAD, "PASS")
print("Initialisation payload :", EXPECTED_INITIALISATION_PAYLOAD, "PASS")
print("Initial parameter      :", EXPECTED_INITIAL_PARAMETER_SHA256, "PASS")
print("Token-ID fingerprint   :", EXPECTED_TOKEN_ID_SHA256, "PASS")
print("Initialised model file :", EXPECTED_INITIAL_MODEL_SHA256, "PASS")

section("CRITICAL PACKAGE VERSIONS")
actual_packages = {name: package_version(name) for name in EXPECTED_PACKAGES}
for name, expected in EXPECTED_PACKAGES.items():
    actual = actual_packages[name]
    status = "PASS" if actual == expected else "STOP"
    print(f"{name:18s}: {actual:18s} expected {expected:10s} {status}")
    require(actual == expected, f"{name} version differs: {actual} != {expected}")

torch_public_version = torch.__version__.split("+")[0]
require(torch_public_version == "2.10.0", f"PyTorch base version differs: {torch.__version__}")
print(f"{'torch':18s}: {torch.__version__:18s} expected base 2.10.0 PASS")

section("CUDA AND GPU IDENTITY")
require(torch.cuda.is_available(), "CUDA is unavailable; switch the notebook accelerator to GPU")
device_count = torch.cuda.device_count()
require(device_count >= 1, "no CUDA device was detected")
require(torch.version.cuda is not None, "PyTorch has no CUDA build")

devices = []
for index in range(device_count):
    props = torch.cuda.get_device_properties(index)
    device = {
        "index": index,
        "name": props.name,
        "total_memory_bytes": int(props.total_memory),
        "compute_capability": list(torch.cuda.get_device_capability(index)),
        "multiprocessor_count": int(props.multi_processor_count),
    }
    devices.append(device)
    print(
        f"GPU {index}: {device['name']} | memory={device['total_memory_bytes']} bytes | "
        f"capability={device['compute_capability']} | MPs={device['multiprocessor_count']}"
    )

print("CUDA available          :", torch.cuda.is_available())
print("CUDA device count       :", device_count)
print("PyTorch CUDA build      :", torch.version.cuda)
print("cuDNN version           :", torch.backends.cudnn.version())
print("Selected training device: cuda:0")
print("Multi-GPU training      : prohibited; additional devices, if present, remain unused")
print("CUBLAS workspace        :", os.environ["CUBLAS_WORKSPACE_CONFIG"])

compiled_architectures = list(torch.cuda.get_arch_list())
selected_capability = devices[0]["compute_capability"]
selected_architecture = f"sm_{selected_capability[0]}{selected_capability[1]}"
print("PyTorch compiled arches :", compiled_architectures)
print("Selected GPU arch       :", selected_architecture)
require(
    selected_architecture in compiled_architectures,
    f"selected GPU architecture {selected_architecture} is absent from this PyTorch build",
)

section("CUDA KERNEL EXECUTION SMOKE TEST")
try:
    smoke_a = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device="cuda:0")
    smoke_b = smoke_a @ smoke_a
    torch.cuda.synchronize(0)
    smoke_result = smoke_b.detach().cpu().tolist()
    require(smoke_result == [[7.0, 10.0], [15.0, 22.0]], f"unexpected CUDA result: {smoke_result}")
except Exception as exc:
    raise RuntimeError(
        f"STOP — CUDA kernel execution failed: {type(exc).__name__}: {exc}"
    ) from exc
finally:
    if "smoke_a" in globals():
        del smoke_a
    if "smoke_b" in globals():
        del smoke_b
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("CUDA matrix multiplication: PASS")
print("CUDA synchronization       : PASS")

try:
    nvidia_smi = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,name,driver_version,memory.total",
            "--format=csv,noheader,nounits",
        ],
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout.strip().splitlines()
except Exception as exc:
    nvidia_smi = [f"UNAVAILABLE: {type(exc).__name__}: {exc}"]

print("nvidia-smi identity:")
for line in nvidia_smi:
    print(" ", line)

environment_payload = {
    "phase": "Phase 2B",
    "step": "M2A",
    "status": "GPU_ENVIRONMENT_AUDITED",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "processor": platform.machine(),
    "packages": {**actual_packages, "torch": torch.__version__},
    "cuda": {
        "available": True,
        "device_count": device_count,
        "pytorch_cuda_build": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "selected_device": "cuda:0",
        "multi_gpu_training": False,
        "cublas_workspace_config": os.environ["CUBLAS_WORKSPACE_CONFIG"],
        "pytorch_compiled_architectures": compiled_architectures,
        "selected_gpu_architecture": selected_architecture,
        "cuda_kernel_smoke_test": "PASS",
        "devices": devices,
        "nvidia_smi": nvidia_smi,
    },
    "frozen_inputs": {
        "t2_manifest_payload_sha256": EXPECTED_T2_PAYLOAD,
        "base_manifest_payload_sha256": EXPECTED_BASE_PAYLOAD,
        "initialisation_manifest_payload_sha256": EXPECTED_INITIALISATION_PAYLOAD,
        "initial_parameter_sha256": EXPECTED_INITIAL_PARAMETER_SHA256,
        "token_id_sha256": EXPECTED_TOKEN_ID_SHA256,
        "initialised_model_file_sha256": EXPECTED_INITIAL_MODEL_SHA256,
    },
}
environment_payload_sha256 = canonical_sha256(environment_payload)

print("\n" + "=" * WIDTH)
print("STEP M2A DECISION: PASS — GPU TRAINING ENVIRONMENT AND FROZEN INPUTS VERIFIED")
print("=" * WIDTH)
print("Environment payload SHA256:", environment_payload_sha256)
print("Model/tokenizer loaded     : False")
print("Training/inference       : NONE")
print("No file was written, transformed or modified.")
print("STOP — share this complete output before the M2 contract is frozen.")


PHASE 2B — STEP M2A: GPU TRAINING-ENVIRONMENT AND FROZEN-INPUT CONTINUITY AUDIT
Python executable : /usr/bin/python3
Python version    : 3.12.13
Platform          : Linux-6.12.90+-x86_64-with-glibc2.35
Processor         : x86_64

FROZEN INPUT CONTINUITY
------------------------------------------------------------------------------------------------------------------------
T2 payload             : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8 PASS
Base payload           : da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c PASS
Initialisation payload : 171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45 PASS
Initial parameter      : 7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b PASS
Token-ID fingerprint   : 9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359 PASS
Initialised model file : 4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c PASS

CRITICAL PACKAGE VERSIONS
----------------------------

## PHASE 2B — STEP M2B: FROZEN MODEL, HYPERPARAMETER, EVALUATION AND INFERENCE CONTRACT

In [1]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import json
import math
import re
import shutil
import sys
import tempfile
import zipfile


WIDTH = 124
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_ROOT / "phase2B_M2_FROZEN_MODEL_CONTRACT"
OUTPUT_ZIP = WORKING_ROOT / "phase2B_M2_FROZEN_MODEL_CONTRACT.zip"
OUTPUT_SIDECAR = WORKING_ROOT / "phase2B_M2_FROZEN_MODEL_CONTRACT.zip.sha256.txt"

EXPECTED = {
    "t2_payload": "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8",
    "t2_primary_file": "b836eb86f1d34ed2db2c8fce4f1b04e6bec515632403962ba594e5c0adaba9ef",
    "t2_llama_file": "37500d0ae8b48b632c18f9b57c931e2f88e6793e51975c6ef0243ab179bca01a",
    "base_payload": "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c",
    "initialisation_payload": "171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45",
    "initial_parameter": "7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b",
    "token_ids": "9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359",
    "pretokenizer": "9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171",
    "initial_model_file": "4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c",
    "m2a_environment": "3e07e214b297576db434188e6c55e576fdd4af362e639f31f7654620d9ecbc7a",
}

LABEL2ID = {
    "presumed_truthful": 0,
    "human_deceptive": 1,
    "ai_generated": 2,
}
ID2LABEL = {str(value): key for key, value in LABEL2ID.items()}
FOLD_HOTELS = {
    "1": ["hilton", "james", "monaco", "sofitel"],
    "2": ["affinia", "ambassador", "hardrock", "talbott"],
    "3": ["conrad", "fairmont", "hyatt", "omni"],
    "4": ["homewood", "knickerbocker", "sheraton", "swissotel"],
    "5": ["allegro", "amalfi", "intercontinental", "palmer"],
}
EXPECTED_PRIMARY_CLASS_COUNTS = {
    "presumed_truthful": 796,
    "human_deceptive": 800,
    "ai_generated": 800,
}
EXPECTED_PRIMARY_FOLD_COUNTS = {1: 479, 2: 479, 3: 478, 4: 480, 5: 480}
EXPECTED_LLAMA_FOLD_COUNTS = {1: 160, 2: 158, 3: 158, 4: 157, 5: 157}
EXPECTED_TRAIN_COUNTS = {1: 1917, 2: 1917, 3: 1918, 4: 1916, 5: 1916}
FOLD_SEEDS = {1: 43, 2: 44, 3: 45, 4: 46, 5: 47}
FULL_REFIT_SEED = 48


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def canonical_json(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def canonical_sha256(value):
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8", errors="strict"))


def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8", errors="strict", newline="") as handle:
        for line_number, line in enumerate(handle, start=1):
            require(line.endswith("\n"), f"{path.name} line {line_number} lacks LF termination")
            require(line.strip(), f"{path.name} line {line_number} is empty")
            rows.append(json.loads(line))
    return rows


def find_unique(filename):
    candidates = sorted(INPUT_ROOT.rglob(filename))
    require(len(candidates) == 1, f"expected one {filename}; found {len(candidates)}")
    return candidates[0]


def write_text_exact(path, text):
    path.write_text(text, encoding="utf-8", newline="\n")


def write_json_exact(path, value):
    write_text_exact(path, json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + "\n")


def deterministic_zip(source_root, destination):
    timestamp = (2026, 9, 10, 0, 0, 0)
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        for path in sorted(p for p in source_root.rglob("*") if p.is_file()):
            relative = path.relative_to(source_root).as_posix()
            info = zipfile.ZipInfo(relative, date_time=timestamp)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            info.create_system = 3
            archive.writestr(info, path.read_bytes(), compress_type=zipfile.ZIP_DEFLATED, compresslevel=9)


print("=" * WIDTH)
print("PHASE 2B — STEP M2B: FROZEN MODEL, HYPERPARAMETER, EVALUATION AND INFERENCE CONTRACT")
print("=" * WIDTH)
print("Python     :", sys.version.split()[0])
print("Execution  : CPU-safe contract construction; no model loading, CUDA access, training or inference")

require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")
require(WORKING_ROOT.is_dir(), "Kaggle working root is unavailable")

section("FROZEN INPUT CONTINUITY")
t2_manifest_path = find_unique("corpus_manifest.json")
base_manifest_path = find_unique("checkpoint_manifest.json")
initialisation_manifest_path = find_unique("initialisation_manifest.json")
t2_manifest = read_json(t2_manifest_path)
base_manifest = read_json(base_manifest_path)
initialisation_manifest = read_json(initialisation_manifest_path)

require(t2_manifest.get("manifest_payload_sha256") == EXPECTED["t2_payload"], "T2 payload differs")
require(base_manifest.get("manifest_payload_sha256") == EXPECTED["base_payload"], "base payload differs")
require(
    initialisation_manifest.get("manifest_payload_sha256") == EXPECTED["initialisation_payload"],
    "initialisation payload differs",
)

initialisation = initialisation_manifest.get("initialisation", {})
require(initialisation.get("parameter_sha256") == EXPECTED["initial_parameter"], "initial parameter differs")
require(
    initialisation.get("eligible_corpus_token_ids_sha256") == EXPECTED["token_ids"],
    "eligible-corpus token-ID fingerprint differs",
)
require(
    initialisation.get("pretokenizer_sha256") == EXPECTED["pretokenizer"],
    "pre-tokenizer fingerprint differs",
)
model_file = initialisation_manifest_path.parent / "model" / "model.safetensors"
require(model_file.is_file(), "frozen initialised model file is missing")
require(file_sha256(model_file) == EXPECTED["initial_model_file"], "frozen initialised model file differs")

print("T2 payload             :", EXPECTED["t2_payload"], "PASS")
print("Base payload           :", EXPECTED["base_payload"], "PASS")
print("Initialisation payload :", EXPECTED["initialisation_payload"], "PASS")
print("Initial parameter      :", EXPECTED["initial_parameter"], "PASS")
print("Token-ID fingerprint   :", EXPECTED["token_ids"], "PASS")
print("M2A GPU environment    :", EXPECTED["m2a_environment"], "RECORDED PASS")

section("FROZEN CORPUS REVALIDATION")
t2_root = t2_manifest_path.parent
primary_path = t2_root / "primary_tripartite_corpus.jsonl"
llama_path = t2_root / "llama_unseen_generator_sensitivity.jsonl"
require(file_sha256(primary_path) == EXPECTED["t2_primary_file"], "primary corpus file differs")
require(file_sha256(llama_path) == EXPECTED["t2_llama_file"], "Llama corpus file differs")
primary_rows = read_jsonl(primary_path)
llama_rows = read_jsonl(llama_path)

require(len(primary_rows) == 2396, f"primary row count differs: {len(primary_rows)}")
require(len(llama_rows) == 790, f"Llama row count differs: {len(llama_rows)}")
primary_ids = [row["row_id"] for row in primary_rows]
llama_ids = [row["row_id"] for row in llama_rows]
require(len(set(primary_ids)) == len(primary_ids), "primary row IDs are not unique")
require(len(set(llama_ids)) == len(llama_ids), "Llama row IDs are not unique")
require(set(primary_ids).isdisjoint(llama_ids), "primary and Llama row IDs overlap")

class_counts = Counter(row["scientific_class"] for row in primary_rows)
fold_counts = Counter(int(row["official_fold"]) for row in primary_rows)
llama_fold_counts = Counter(int(row["official_fold"]) for row in llama_rows)
require(dict(class_counts) == EXPECTED_PRIMARY_CLASS_COUNTS, f"primary class counts differ: {dict(class_counts)}")
require(dict(sorted(fold_counts.items())) == EXPECTED_PRIMARY_FOLD_COUNTS, "primary fold counts differ")
require(dict(sorted(llama_fold_counts.items())) == EXPECTED_LLAMA_FOLD_COUNTS, "Llama fold counts differ")

for row in primary_rows:
    require(row["label_id"] == LABEL2ID[row["scientific_class"]], f"label mismatch: {row['row_id']}")
    require(row["text"].strip() == row["text"] and row["text"], f"invalid text: {row['row_id']}")
for row in llama_rows:
    require(row["reference_label_id"] == 2, f"Llama reference label differs: {row['row_id']}")
    require(row["reference_class"] == "ai_generated", f"Llama reference class differs: {row['row_id']}")
    require(row["text"].strip() == row["text"] and row["text"], f"invalid Llama text: {row['row_id']}")

for fold in range(1, 6):
    test_hotels = {row["hotel"] for row in primary_rows if int(row["official_fold"]) == fold}
    train_hotels = {row["hotel"] for row in primary_rows if int(row["official_fold"]) != fold}
    require(test_hotels == set(FOLD_HOTELS[str(fold)]), f"fold {fold} hotel membership differs")
    require(test_hotels.isdisjoint(train_hotels), f"fold {fold} has hotel leakage")
    require(len(train_hotels) == 16, f"fold {fold} training hotel count differs")
    require(len(primary_rows) - fold_counts[fold] == EXPECTED_TRAIN_COUNTS[fold], f"fold {fold} train count differs")

gpt3_by_seed = {
    row["seed_family_id"]: row
    for row in primary_rows
    if row["scientific_class"] == "ai_generated"
}
require(len(gpt3_by_seed) == 800, "GPT-3 seed-family index differs")
for row in llama_rows:
    require(row["seed_family_id"] in gpt3_by_seed, f"unmatched Llama seed family: {row['row_id']}")
    peer = gpt3_by_seed[row["seed_family_id"]]
    for field in ("hotel", "polarity", "official_fold"):
        require(row[field] == peer[field], f"Llama/GPT-3 {field} mismatch: {row['row_id']}")

word_pattern = re.compile(r"(?u)\w+")
word_lengths = [len(word_pattern.findall(row["text"])) for row in primary_rows]
length_bins = [
    {"name": "Q1_shortest", "lower_inclusive": 9, "upper_inclusive": 61},
    {"name": "Q2", "lower_inclusive": 62, "upper_inclusive": 90},
    {"name": "Q3", "lower_inclusive": 91, "upper_inclusive": 156},
    {"name": "Q4_longest", "lower_inclusive": 157, "upper_inclusive": 797},
]
length_bin_counts = {}
for item in length_bins:
    count = sum(item["lower_inclusive"] <= value <= item["upper_inclusive"] for value in word_lengths)
    length_bin_counts[item["name"]] = count
require(sum(length_bin_counts.values()) == len(primary_rows), "length bins do not partition the primary corpus")
require(min(word_lengths) == 9 and max(word_lengths) == 797, "primary word-length extrema differ")

print("Primary rows/classes  :", len(primary_rows), dict(class_counts), "PASS")
print("Primary fold counts   :", dict(sorted(fold_counts.items())), "PASS")
print("Llama rows/folds      :", len(llama_rows), dict(sorted(llama_fold_counts.items())), "PASS")
print("Hotel leakage         : 0 across all five outer folds PASS")
print("Matched Llama/GPT-3   : 790 / 790 seed families PASS")
print("Length bins/counts    :", length_bin_counts, "PASS")

deberta_contract = {
    "role": "primary_neural_model",
    "architecture": "DebertaV2ForSequenceClassification",
    "pretrained_repository": "microsoft/deberta-v3-base",
    "pretrained_revision": "8ccc9b6f36199bec6961081d44eb72fb3f7353f3",
    "initialisation_artifact_payload_sha256": EXPECTED["initialisation_payload"],
    "initial_parameter_sha256": EXPECTED["initial_parameter"],
    "initialisation_for_every_outer_fold": "reload_identical_frozen_three_class_initialisation",
    "rs1_initialisation": "prohibited",
    "predictor_field": "text",
    "target_field": "label_id",
    "prohibited_predictor_fields": [
        "row_id", "official_fold", "hotel", "polarity", "source_family",
        "seed_family_id", "truthful_parent_row_id", "model_text_sha256",
        "audit_identity_sha256",
    ],
    "tokenizer": {
        "artifact": "frozen_M1C-B_tokenizer",
        "policy": "native_deberta_unigram_metaspace_no_mistral_regex_patch",
        "token_id_fingerprint_all_3186_eligible_texts": EXPECTED["token_ids"],
        "max_length": 256,
        "truncation": True,
        "padding": "max_length",
        "add_special_tokens": True,
        "return_attention_mask": True,
        "token_type_ids": "use_only_if_returned_by_tokenizer",
        "truncation_audit_required": "report pre-truncation token lengths and truncation rates by class, fold and evaluation role",
    },
    "optimisation": {
        "epochs": 4,
        "physical_batch_size": 2,
        "gradient_accumulation_microbatches": 16,
        "maximum_effective_batch_size": 32,
        "accumulation_groups_do_not_cross_epoch_boundaries": True,
        "partial_accumulation_policy": "divide summed gradients by actual examples in each accumulation group",
        "loss": "unweighted_cross_entropy_reduction_sum",
        "class_weights": None,
        "label_smoothing": 0.0,
        "optimizer": "torch.optim.AdamW",
        "learning_rate": 2e-5,
        "betas": [0.9, 0.999],
        "epsilon": 1e-8,
        "weight_decay": 0.01,
        "no_decay_name_rules": ["bias", "LayerNorm.weight", "layer_norm.weight"],
        "maximum_gradient_norm": 1.0,
        "scheduler": "linear_decay_to_zero",
        "warmup_fraction": 0.10,
        "early_stopping": "prohibited",
        "checkpoint_selection": "prohibited; evaluate the fixed final epoch only",
        "outer_test_monitoring_during_training": "prohibited",
    },
    "numerics": {
        "training_dtype": "float32",
        "autocast": False,
        "gradient_scaler": False,
        "tf32": False,
        "selected_device": "cuda:0",
        "multi_gpu": False,
        "cublas_workspace_config": ":4096:8",
        "torch_deterministic_algorithms": True,
        "cudnn_deterministic": True,
        "cudnn_benchmark": False,
        "data_loader_num_workers": 0,
        "drop_last": False,
    },
    "ordering_and_seeds": {
        "canonical_row_order": "ascending row_id before every split",
        "outer_fold_seeds": {str(key): value for key, value in FOLD_SEEDS.items()},
        "epoch_shuffle": "numpy.random.default_rng(fold_seed + zero_based_epoch).permutation(train_n)",
        "shuffle_hash_required_each_epoch": True,
        "full_refit_seed": FULL_REFIT_SEED,
    },
    "inference": {
        "batch_size": 16,
        "dtype": "float32",
        "decision_rule": "argmax over logits; lowest label ID resolves an exact tie",
        "persist": ["logits", "softmax_scores", "predicted_label", "maximum_softmax_score"],
        "score_language": "uncalibrated model score; do not call probability or calibrated confidence",
    },
}

tfidf_contract = {
    "role": "substantive_classical_comparator",
    "pipeline": ["TfidfVectorizer", "LogisticRegression"],
    "training_scope": "fit vectorizer and classifier on the outer training fold only",
    "input": "original frozen text only",
    "vectorizer": {
        "lowercase": True,
        "strip_accents": None,
        "analyzer": "word",
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_df": 1.0,
        "max_features": 50000,
        "sublinear_tf": True,
        "norm": "l2",
        "token_pattern": r"(?u)\b\w\w+\b",
        "dtype": "float64",
    },
    "classifier": {
        "implementation": "sklearn.linear_model.LogisticRegression",
        "solver": "lbfgs",
        "penalty": "l2",
        "C": 1.0,
        "fit_intercept": True,
        "class_weight": None,
        "max_iter": 2000,
        "tol": 1e-4,
        "random_state": "outer_fold_seed",
    },
    "resampling": None,
    "hyperparameter_tuning": "none",
    "score_language": "uncalibrated model score",
}

length_contract = {
    "role": "mandatory_shortcut_diagnostic_not_substantive_comparator",
    "feature": "log1p(Unicode word-token count)",
    "word_count_pattern": r"(?u)\w+",
    "transform": "StandardScaler fit on outer training fold only",
    "classifier": {
        "implementation": "sklearn.linear_model.LogisticRegression",
        "solver": "lbfgs",
        "penalty": "l2",
        "C": 1.0,
        "fit_intercept": True,
        "class_weight": None,
        "max_iter": 2000,
        "tol": 1e-4,
        "random_state": "outer_fold_seed",
    },
    "pooled_primary_length_bins": length_bins,
    "observed_bin_counts": length_bin_counts,
    "inferential_claims": "prohibited; report as shortcut magnitude diagnostic",
}

fold_schedule = []
for fold in range(1, 6):
    train_n = EXPECTED_TRAIN_COUNTS[fold]
    test_n = EXPECTED_PRIMARY_FOLD_COUNTS[fold]
    microbatches_per_epoch = math.ceil(train_n / deberta_contract["optimisation"]["physical_batch_size"])
    updates_per_epoch = math.ceil(
        microbatches_per_epoch / deberta_contract["optimisation"]["gradient_accumulation_microbatches"]
    )
    total_updates = updates_per_epoch * deberta_contract["optimisation"]["epochs"]
    warmup_updates = math.ceil(total_updates * deberta_contract["optimisation"]["warmup_fraction"])
    fold_schedule.append({
        "test_fold": fold,
        "seed": FOLD_SEEDS[fold],
        "train_n": train_n,
        "test_n": test_n,
        "llama_test_n": EXPECTED_LLAMA_FOLD_COUNTS[fold],
        "train_hotels": 16,
        "test_hotels": FOLD_HOTELS[str(fold)],
        "microbatches_per_epoch": microbatches_per_epoch,
        "optimizer_updates_per_epoch": updates_per_epoch,
        "total_optimizer_updates": total_updates,
        "warmup_updates": warmup_updates,
    })
require(all(item["optimizer_updates_per_epoch"] == 60 for item in fold_schedule), "outer update schedule differs")
require(all(item["total_optimizer_updates"] == 240 for item in fold_schedule), "outer total updates differ")
require(all(item["warmup_updates"] == 24 for item in fold_schedule), "outer warmup schedule differs")

evaluation_contract = {
    "design": "five-fold outer cross-validation using the immutable official Ott hotel folds",
    "routing": FOLD_HOTELS,
    "fold_schedule": fold_schedule,
    "split_integrity": {
        "training": "all primary rows outside the held-out official fold",
        "primary_test": "all primary rows in the held-out official fold",
        "llama_sensitivity": "eligible Llama rows in that same held-out hotel fold only",
        "train_test_hotel_overlap": 0,
        "resplitting": "prohibited",
        "resampling": "prohibited",
        "outer_test_tuning_or_selection": "prohibited",
    },
    "primary_endpoint": "pooled out-of-fold macro-F1 over the three primary classes",
    "required_primary_metrics": [
        "per-class precision, recall and F1 with zero_division=0",
        "macro precision, macro recall and macro F1",
        "balanced accuracy",
        "accuracy",
        "3x3 confusion matrix in label order [0,1,2]",
    ],
    "secondary_metric": "macro one-vs-rest ROC-AUC from uncalibrated class scores, where defined",
    "aggregation": {
        "primary": "pooled out-of-fold predictions; every eligible primary row appears exactly once",
        "fold_summary": "unweighted mean and sample standard deviation across five folds, descriptive only",
        "strata": ["outer fold", "hotel", "polarity", "documented collection source", "fixed pooled length bin"],
    },
    "llama_sensitivity": {
        "status": "unseen-generator sensitivity; never training, tuning or checkpoint selection data",
        "reference_label": 2,
        "required_outputs": [
            "AI-class recall/detection rate",
            "predicted-class count and proportion",
            "AI-class uncalibrated score distribution",
            "maximum uncalibrated score distribution",
        ],
        "required_strata": ["fold", "hotel", "polarity"],
        "standalone_three_class_accuracy_or_macro_f1": "prohibited",
    },
    "descriptive_model_comparisons": [
        "DeBERTa versus TF-IDF logistic regression",
        "DeBERTa versus length-only diagnostic",
        "TF-IDF logistic regression versus length-only diagnostic",
    ],
    "foldwise_t_test_or_wilcoxon": "prohibited as confirmatory inference because five folds are dependent and underpowered",
}

inference_contract = {
    "confidence_intervals": {
        "method": "nonparametric paired cluster bootstrap",
        "cluster_unit": "hotel",
        "clusters": 20,
        "iterations": 10000,
        "confidence_level": 0.95,
        "interval": "percentile",
        "primary_metric_bootstrap_seed": 20260910,
        "model_difference_bootstrap_seed": 20260911,
        "cross_generator_bootstrap_seed": 20260913,
        "procedure": "sample 20 hotels with replacement and retain all observations for each sampled hotel occurrence",
    },
    "confirmatory_tests": [
        {
            "family_id": "H_M2_1",
            "estimand": "pooled macro-F1(DeBERTa) minus pooled macro-F1(TF-IDF logistic regression)",
            "null": "paired model-exchangeability by hotel",
            "alternative": "two-sided",
            "permutation_unit": "hotel",
            "iterations": 10000,
            "seed": 20260912,
            "procedure": "independently swap the complete two-model prediction vectors within each hotel",
            "p_value": "(number of absolute permuted statistics >= absolute observed statistic + 1) / 10001",
        },
        {
            "family_id": "H_M2_2",
            "estimand": "DeBERTa AI-detection recall on 790 matched Llama rows minus recall on their 790 matched GPT-3 seed siblings",
            "null": "paired generator-exchangeability by hotel",
            "alternative": "two-sided",
            "permutation_unit": "hotel",
            "iterations": 10000,
            "seed": 20260914,
            "procedure": "swap all matched GPT-3/Llama detection outcomes within each hotel",
            "p_value": "(number of absolute permuted statistics >= absolute observed statistic + 1) / 10001",
        },
    ],
    "multiplicity": "Holm step-down adjustment across the two pre-specified confirmatory tests; report raw and adjusted p-values",
    "alpha": 0.05,
    "length_only_inference": "none",
    "interpretation_limit": "twenty hotel clusters limit precision and generalisation; interval and p-values are conditional on this corpus design",
}

full_refit_microbatches = math.ceil(2396 / deberta_contract["optimisation"]["physical_batch_size"])
full_refit_updates_per_epoch = math.ceil(
    full_refit_microbatches / deberta_contract["optimisation"]["gradient_accumulation_microbatches"]
)
full_refit_total_updates = full_refit_updates_per_epoch * deberta_contract["optimisation"]["epochs"]
full_refit_contract = {
    "authorisation_point": "only after all Phase 2B OOF, Llama sensitivity and inferential outputs are frozen",
    "purpose": "exploratory Phase 4 transfer/deployment only; excluded from Phase 2B performance estimation",
    "deberta": {
        "initialisation": "reload frozen M1C-B three-class initialisation",
        "training_rows": 2396,
        "seed": FULL_REFIT_SEED,
        "epochs": 4,
        "microbatches_per_epoch": full_refit_microbatches,
        "optimizer_updates_per_epoch": full_refit_updates_per_epoch,
        "total_optimizer_updates": full_refit_total_updates,
        "warmup_updates": math.ceil(full_refit_total_updates * 0.10),
        "all_other_hyperparameters": "identical to the outer-fold DeBERTa contract",
    },
    "tfidf_logistic_regression": "refit the frozen text pipeline on all 2,396 primary rows",
    "phase4_prediction_meaning": "AI-generation likelihood under this corpus contract, not general review suspiciousness or deception",
}
require(full_refit_updates_per_epoch == 75, "full-refit updates per epoch differ")
require(full_refit_total_updates == 300, "full-refit total updates differ")
require(full_refit_contract["deberta"]["warmup_updates"] == 30, "full-refit warmup differs")

reporting_contract = {
    "mandatory_language": {
        "class_0": "platform-sourced presumed truthful; not independently verified genuine",
        "class_1": "commissioned human-written deceptive reviews from Amazon Mechanical Turk; not naturally occurring spam",
        "class_2": "GPT-3-generated reviews seeded from presumed-truthful Ott openings",
        "llama": "unseen-generator sensitivity set seeded from the same presumed-truthful Ott openings",
    },
    "mandatory_limitations": [
        "collection source is confounded with human-class origin",
        "within presumed-truthful reviews, collection source is confounded with polarity",
        "generator identity is confounded with the primary AI-generated class",
        "generator, collection source and scientific class effects cannot be causally separated",
        "AI reviews inherit the first ten-word seed-family structure from presumed-truthful parents",
        "hotel reviews constitute a domain shift from the product-review framing",
        "the corpus is small and contains only twenty hotel clusters",
        "substantial text-length differences may provide a shortcut",
    ],
    "scope_deviations": [
        "Phase 2B is a revised tripartite hotel-review study rather than the rejected Mexwell binary product-review analysis",
        "fresh DeBERTa-v3-base replaces RS-1 reuse; BERT/RoBERTa expansion is outside the frozen Phase 2B scope",
        "hotel-cluster inference replaces simple row-level McNemar testing as the primary comparison because rows within hotels are not independent and macro-F1 is the primary endpoint",
        "the original binary greater-than-ten-percentage-point hypothesis belongs to Phase 2A and is not redefined as the Phase 2B tripartite confirmatory claim",
    ],
    "explainability": "SHAP or other post-hoc explanations require a separate frozen analysis contract and cannot select or alter the Phase 2B model",
}

execution_contract = {
    "pre_fold_stop_checks": [
        "reverify T2, base and initialisation manifest payloads",
        "reload the identical M1C-B initialised model and verify its parameter SHA-256 before training",
        "verify exact train/test row-ID sets and zero hotel overlap",
        "verify canonical ascending row-ID order before deterministic shuffling",
        "verify Llama rows are absent from all training and selection inputs",
    ],
    "runtime_stop_conditions": [
        "non-finite loss, logits, gradients, parameters or scores",
        "CUDA out-of-memory or any automatic batch-size/hyperparameter adaptation",
        "unexpected optimizer-update, warmup-update or row-consumption count",
        "duplicate or missing primary OOF prediction row IDs",
        "test-fold use for tuning, early stopping, checkpoint selection or threshold selection",
        "package, tokenizer, model, corpus or manifest fingerprint mismatch",
    ],
    "failure_policy": "preserve the complete failure output; revise and re-freeze this contract explicitly before retrying—never adapt silently",
    "resource_logging": ["elapsed time", "peak allocated GPU memory", "peak reserved GPU memory", "device identity"],
    "required_preservation": [
        "one final-epoch DeBERTa checkpoint and manifest per fold",
        "per-fold epoch and optimizer-step logs including deterministic order hashes",
        "primary OOF row-level logits, scores and predictions for every model",
        "Llama row-level logits/scores and predictions from the corresponding held-out-hotel model",
        "TF-IDF vocabulary/model and length-only scaler/model per fold",
        "fold, pooled, stratified and length-stratified metrics",
        "cluster-bootstrap replicates and paired-permutation test results",
        "complete SHA-256 registry and contract/upstream chain",
    ],
}

output_schema_contract = {
    "primary_prediction_columns": [
        "row_id", "official_fold", "hotel", "polarity", "source_family",
        "scientific_class", "y_true", "y_pred", "logit_0", "logit_1",
        "logit_2", "score_0", "score_1", "score_2", "maximum_score",
        "correct", "model_name", "fold_seed",
    ],
    "llama_prediction_columns": [
        "row_id", "seed_family_id", "matched_gpt3_row_id", "official_fold",
        "hotel", "polarity", "reference_label_id", "y_pred", "logit_0",
        "logit_1", "logit_2", "score_0", "score_1", "score_2",
        "maximum_score", "detected_as_ai", "model_name", "fold_seed",
    ],
    "row_order": "ascending row_id within every persisted prediction table",
    "numeric_serialisation": "full Python round-trip representation; no presentation rounding in machine-readable outputs",
    "presentation_rounding": "three decimal places only in human-readable tables; counts remain integers",
}

model_contract = {
    "phase": "Phase 2B",
    "step": "M2B",
    "status": "FROZEN_MODEL_AND_HYPERPARAMETER_CONTRACT",
    "decision_date_utc": "2026-09-10",
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "primary_model": deberta_contract,
    "classical_comparator": tfidf_contract,
    "shortcut_diagnostic": length_contract,
    "hyperparameter_search": "none; all values frozen before any Phase 2B model training or inference",
}

section("FROZEN MODEL AND HYPERPARAMETER DECISIONS")
print("Primary model           : DeBERTa-v3-base, frozen fresh three-class initialisation")
print("Text comparator         : word TF-IDF (1–2 grams) + multinomial logistic regression")
print("Shortcut diagnostic     : log1p Unicode word count + multinomial logistic regression")
print("Outer design            : five immutable hotel-grouped folds")
print("Epochs / max length     : 4 / 256")
print("Batch / accumulation    : 2 / 16 microbatches (maximum effective batch 32)")
print("Optimizer / LR          : AdamW / 2e-5")
print("Training numeric policy : FP32 only; single compatible GPU cuda:0")
print("Tuning / early stopping : NONE / PROHIBITED")
print("Class weighting         : NONE")

section("FOLD TRAINING SCHEDULE")
print("fold  seed  train  test  llama  microbatches/epoch  updates/epoch  total_updates  warmup")
for item in fold_schedule:
    print(
        f"{item['test_fold']:>4}  {item['seed']:>4}  {item['train_n']:>5}  {item['test_n']:>4}  "
        f"{item['llama_test_n']:>5}  {item['microbatches_per_epoch']:>18}  "
        f"{item['optimizer_updates_per_epoch']:>13}  {item['total_optimizer_updates']:>13}  "
        f"{item['warmup_updates']:>6}"
    )

section("FROZEN EVALUATION AND INFERENCE DECISIONS")
print("Primary endpoint        : pooled OOF three-class macro-F1")
print("Substantive comparison  : DeBERTa versus TF-IDF logistic regression")
print("Cluster unit            : hotel (20 clusters)")
print("Bootstrap               : 10,000 paired hotel-cluster resamples; 95% percentile CI")
print("Permutation             : 10,000 paired hotel-cluster swaps; two-sided")
print("Multiplicity            : Holm correction over two pre-specified confirmatory tests")
print("Foldwise t/Wilcoxon     : prohibited as confirmatory inference")
print("Llama endpoint          : AI-detection recall and predicted-class/score distribution")
print("Llama 3-class accuracy  : prohibited")
print("Length analysis         : fixed pooled Unicode-token bins; descriptive shortcut audit")

contract_files = {
    "model_hyperparameter_contract.json": model_contract,
    "evaluation_contract.json": evaluation_contract,
    "inference_contract.json": inference_contract,
    "full_refit_contract.json": full_refit_contract,
    "reporting_contract.json": reporting_contract,
    "execution_contract.json": execution_contract,
    "output_schema_contract.json": output_schema_contract,
}

contract_payload = {
    "phase": "Phase 2B",
    "step": "M2B",
    "status": "FROZEN_MODEL_EVALUATION_INFERENCE_CONTRACT",
    "decision_date_utc": "2026-09-10",
    "upstream": {
        "t2_manifest_payload_sha256": EXPECTED["t2_payload"],
        "base_checkpoint_manifest_payload_sha256": EXPECTED["base_payload"],
        "initialisation_manifest_payload_sha256": EXPECTED["initialisation_payload"],
        "initial_parameter_sha256": EXPECTED["initial_parameter"],
        "eligible_corpus_token_id_sha256": EXPECTED["token_ids"],
        "m2a_gpu_environment_payload_sha256": EXPECTED["m2a_environment"],
    },
    "verified_inputs": {
        "primary_rows": len(primary_rows),
        "llama_rows": len(llama_rows),
        "primary_class_counts": EXPECTED_PRIMARY_CLASS_COUNTS,
        "primary_fold_counts": {str(k): v for k, v in EXPECTED_PRIMARY_FOLD_COUNTS.items()},
        "llama_fold_counts": {str(k): v for k, v in EXPECTED_LLAMA_FOLD_COUNTS.items()},
        "llama_gpt3_matched_seed_families": 790,
    },
    "contracts": {
        filename: canonical_sha256(value)
        for filename, value in contract_files.items()
    },
}
contract_payload_sha256 = canonical_sha256(contract_payload)
contract_manifest = {
    **contract_payload,
    "contract_payload_sha256": contract_payload_sha256,
}

readme = f"""# Phase 2B frozen model and evaluation contract

Status: FROZEN before any Phase 2B model training or inference.

Contract payload SHA-256: `{contract_payload_sha256}`

This artifact freezes the DeBERTa-v3-base three-class model, TF-IDF logistic-regression comparator,
length-only shortcut diagnostic, five official hotel-fold evaluation, hotel-cluster bootstrap and
paired-permutation inference, unseen-generator Llama sensitivity analysis, reporting language,
failure rules, output schemas, and the later full-corpus refit used only for Phase 4.

No hyperparameter was selected using an outer test fold. No model was loaded, trained, tuned,
evaluated, or used for inference by the contract-construction cell.
"""

section("DETERMINISTIC CONTRACT ARTIFACT")
temporary_parent = Path(tempfile.mkdtemp(prefix="phase2B_M2_contract_", dir=WORKING_ROOT))
temporary_root = temporary_parent / OUTPUT_ROOT.name
temporary_root.mkdir()
try:
    for filename, value in contract_files.items():
        write_json_exact(temporary_root / filename, value)
    write_json_exact(temporary_root / "contract_manifest.json", contract_manifest)
    write_text_exact(temporary_root / "README.md", readme)

    registry_names = sorted(
        path.relative_to(temporary_root).as_posix()
        for path in temporary_root.rglob("*")
        if path.is_file()
    )
    registry = {
        name: file_sha256(temporary_root / name)
        for name in registry_names
    }
    sums_text = "".join(f"{digest}  {name}\n" for name, digest in registry.items())
    write_text_exact(temporary_root / "SHA256SUMS.txt", sums_text)

    expected_final_files = set(registry) | {"SHA256SUMS.txt"}
    if OUTPUT_ROOT.exists():
        actual_existing = {
            path.relative_to(OUTPUT_ROOT).as_posix()
            for path in OUTPUT_ROOT.rglob("*")
            if path.is_file()
        }
        require(actual_existing == expected_final_files, "existing M2 contract artifact file set differs")
        for name in expected_final_files:
            require(
                file_sha256(OUTPUT_ROOT / name) == file_sha256(temporary_root / name),
                f"existing M2 contract artifact differs: {name}",
            )
        print("Existing identical contract directory verified; no overwrite performed.")
    else:
        temporary_root.replace(OUTPUT_ROOT)

    temporary_zip = temporary_parent / OUTPUT_ZIP.name
    deterministic_zip(OUTPUT_ROOT, temporary_zip)
    if OUTPUT_ZIP.exists():
        require(file_sha256(OUTPUT_ZIP) == file_sha256(temporary_zip), "existing M2 contract ZIP differs")
        temporary_zip.unlink()
    else:
        temporary_zip.replace(OUTPUT_ZIP)

    zip_sha256 = file_sha256(OUTPUT_ZIP)
    sidecar_text = f"{zip_sha256}  {OUTPUT_ZIP.name}\n"
    if OUTPUT_SIDECAR.exists():
        require(OUTPUT_SIDECAR.read_text(encoding="utf-8") == sidecar_text, "existing ZIP sidecar differs")
    else:
        write_text_exact(OUTPUT_SIDECAR, sidecar_text)
finally:
    if temporary_parent.exists():
        shutil.rmtree(temporary_parent)

for path in sorted(p for p in OUTPUT_ROOT.rglob("*") if p.is_file()):
    relative = path.relative_to(OUTPUT_ROOT).as_posix()
    print(f"{relative:48s} {path.stat().st_size:10d} bytes  {file_sha256(path)}")

print("Contract payload SHA256:", contract_payload_sha256)

print("\n" + "=" * WIDTH)
print("STEP M2B DECISION: MODEL, HYPERPARAMETER, EVALUATION AND INFERENCE CONTRACT FROZEN")
print("=" * WIDTH)
print("Output directory :", OUTPUT_ROOT)
print("Artifact ZIP     :", OUTPUT_ZIP)
print("ZIP SHA256       :", file_sha256(OUTPUT_ZIP))
print("Checksum sidecar :", OUTPUT_SIDECAR)
print("Model loaded     : False")
print("CUDA accessed    : False")
print("Training/tuning  : NONE")
print("Inference        : NONE")
print("Download the ZIP and .sha256.txt, verify externally, and preserve the complete output.")
print("STOP — do not begin training until this M2B output and external checksum verification are shared.")


PHASE 2B — STEP M2B: FROZEN MODEL, HYPERPARAMETER, EVALUATION AND INFERENCE CONTRACT
Python     : 3.12.13
Execution  : CPU-safe contract construction; no model loading, CUDA access, training or inference

FROZEN INPUT CONTINUITY
----------------------------------------------------------------------------------------------------------------------------
T2 payload             : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8 PASS
Base payload           : da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c PASS
Initialisation payload : 171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45 PASS
Initial parameter      : 7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b PASS
Token-ID fingerprint   : 9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359 PASS
M2A GPU environment    : 3e07e214b297576db434188e6c55e576fdd4af362e639f31f7654620d9ecbc7a RECORDED PASS

FROZEN CORPUS REVALIDATION
---------------------------------------

In [2]:
from pathlib import Path
import hashlib
import json
import sys


WIDTH = 124
INPUT_ROOT = Path("/kaggle/input")

EXPECTED_ZIP_SHA256 = "e1fa927556c28e864eb59d8b80be54a31e39558500a229645481b83cb2967c9a"
EXPECTED_CONTRACT_PAYLOAD = "53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202"
EXPECTED_T2_PAYLOAD = "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
EXPECTED_BASE_PAYLOAD = "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c"
EXPECTED_INITIALISATION_PAYLOAD = "171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45"
EXPECTED_INITIAL_PARAMETER = "7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b"
EXPECTED_TOKEN_IDS = "9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359"
EXPECTED_INITIAL_MODEL_FILE = "4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c"
EXPECTED_M2A_ENVIRONMENT = "3e07e214b297576db434188e6c55e576fdd4af362e639f31f7654620d9ecbc7a"

EXPECTED_FILES = {
    "README.md": (
        750,
        "7ed8f4309b00a9fd3015449f4941ec43d70f4bb305c392ce5590871003894058",
    ),
    "SHA256SUMS.txt": (
        812,
        "102c4af8156766ece161994dbafc5d002eff8883b30fd95a54527b3d4b8f7d88",
    ),
    "contract_manifest.json": (
        2108,
        "5979dc81c6e44eb869d47976a56cd7ee9311d8c400ffbc12e9cb9119bd09483a",
    ),
    "evaluation_contract.json": (
        4625,
        "a5e7e7f630d41e5e8d5518b550d49bbaf10cab7c11f1cedcf2bc3439edea7a9a",
    ),
    "execution_contract.json": (
        1796,
        "de2747c97bd6a7e6776d0cb94711848226e2f7070d72c28db365e0cbc8e47b2c",
    ),
    "full_refit_contract.json": (
        815,
        "4655faba5dddd3adb5d54c089b83366221abd302d4a4b5e5766803f0fe39b474",
    ),
    "inference_contract.json": (
        1901,
        "9d0863425035c23c0b29e7070f71437e48a3d1f237a75476b1dea4d79ec97a5e",
    ),
    "model_hyperparameter_contract.json": (
        6471,
        "1a771c43f5911aaad9141d83910d0fc95bae7fdedfd26c9242a8c8ea6ff8ba91",
    ),
    "output_schema_contract.json": (
        998,
        "901ff636334f1de847c311a01ff3c044d6c7413be608b59a5ef2b40eda1b351a",
    ),
    "reporting_contract.json": (
        1844,
        "514874f083a1eda15a767b46b61d2341b3aa4621b0edd75ea2108af32b696889",
    ),
}

EXPECTED_LABEL2ID = {
    "presumed_truthful": 0,
    "human_deceptive": 1,
    "ai_generated": 2,
}
EXPECTED_FOLD_SEEDS = {"1": 43, "2": 44, "3": 45, "4": 46, "5": 47}
EXPECTED_TEST_N = {1: 479, 2: 479, 3: 478, 4: 480, 5: 480}
EXPECTED_LLAMA_N = {1: 160, 2: 158, 3: 158, 4: 157, 5: 157}


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8", errors="strict"))


def find_unique(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    require(len(matches) == 1, f"expected one {filename}; found {len(matches)}")
    return matches[0]


print("=" * WIDTH)
print("PHASE 2B — STEP M2C: RE-UPLOADED FROZEN MODEL-CONTRACT VERIFICATION")
print("=" * WIDTH)
print("Python     :", sys.version.split()[0])
print("Input root :", INPUT_ROOT)
print("Execution  : read-only; no model/tokenizer loading, CUDA access, training or inference")
require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")

section("UPSTREAM INPUT CONTINUITY")
t2_manifest_path = find_unique("corpus_manifest.json")
base_manifest_path = find_unique("checkpoint_manifest.json")
initialisation_manifest_path = find_unique("initialisation_manifest.json")

t2_manifest = read_json(t2_manifest_path)
base_manifest = read_json(base_manifest_path)
initialisation_manifest = read_json(initialisation_manifest_path)

require(t2_manifest.get("manifest_payload_sha256") == EXPECTED_T2_PAYLOAD, "T2 payload differs")
require(base_manifest.get("manifest_payload_sha256") == EXPECTED_BASE_PAYLOAD, "base payload differs")
require(
    initialisation_manifest.get("manifest_payload_sha256") == EXPECTED_INITIALISATION_PAYLOAD,
    "initialisation payload differs",
)
initialisation = initialisation_manifest.get("initialisation", {})
require(initialisation.get("parameter_sha256") == EXPECTED_INITIAL_PARAMETER, "initial parameter differs")
require(
    initialisation.get("eligible_corpus_token_ids_sha256") == EXPECTED_TOKEN_IDS,
    "eligible-corpus token-ID fingerprint differs",
)
initial_model_path = initialisation_manifest_path.parent / "model" / "model.safetensors"
require(initial_model_path.is_file(), "initialised model.safetensors is missing")
require(file_sha256(initial_model_path) == EXPECTED_INITIAL_MODEL_FILE, "initialised model file differs")

print("T2 payload             :", EXPECTED_T2_PAYLOAD, "PASS")
print("Base payload           :", EXPECTED_BASE_PAYLOAD, "PASS")
print("Initialisation payload :", EXPECTED_INITIALISATION_PAYLOAD, "PASS")
print("Initial parameter      :", EXPECTED_INITIAL_PARAMETER, "PASS")
print("Token-ID fingerprint   :", EXPECTED_TOKEN_IDS, "PASS")
print("Initialised model file :", EXPECTED_INITIAL_MODEL_FILE, "PASS")

section("MODEL-CONTRACT INPUT DISCOVERY")
contract_manifest_path = find_unique("contract_manifest.json")
contract_root = contract_manifest_path.parent
print("Contract root          :", contract_root)

sidecars = sorted(INPUT_ROOT.rglob("phase2B_M2_FROZEN_MODEL_CONTRACT.zip.sha256.txt"))
require(len(sidecars) == 1, f"expected one M2 contract checksum sidecar; found {len(sidecars)}")
sidecar_tokens = sidecars[0].read_text(encoding="utf-8", errors="strict").strip().split()
require(sidecar_tokens, "M2 contract checksum sidecar is empty")
require(sidecar_tokens[0].lower() == EXPECTED_ZIP_SHA256, "M2 contract sidecar checksum differs")

mounted_zips = sorted(INPUT_ROOT.rglob("phase2B_M2_FROZEN_MODEL_CONTRACT.zip"))
require(len(mounted_zips) <= 1, "multiple mounted M2 contract ZIPs found")
if mounted_zips:
    require(file_sha256(mounted_zips[0]) == EXPECTED_ZIP_SHA256, "mounted M2 contract ZIP differs")
    zip_status = "BYTE_VERIFIED_IN_RUNTIME"
else:
    zip_status = "Kaggle-extracted; externally verified ZIP recorded"

print("Checksum sidecar       :", sidecars[0])
print("Recorded ZIP SHA256    :", sidecar_tokens[0].lower())
print("ZIP runtime status     :", zip_status)

section("COMPLETE CONTRACT FILE REGISTRY")
actual_files = {
    path.relative_to(contract_root).as_posix()
    for path in contract_root.rglob("*")
    if path.is_file()
}
require(actual_files == set(EXPECTED_FILES), f"contract file set differs: {sorted(actual_files)}")

for relative_name, (expected_size, expected_hash) in EXPECTED_FILES.items():
    path = contract_root / relative_name
    require(path.stat().st_size == expected_size, f"size mismatch: {relative_name}")
    require(file_sha256(path) == expected_hash, f"checksum mismatch: {relative_name}")
    print(f"{relative_name:48s} {expected_size:10d} bytes  PASS")

section("INTERNAL SHA-256 REGISTRY")
registry = {}
for line_number, line in enumerate(
    (contract_root / "SHA256SUMS.txt").read_text(encoding="utf-8", errors="strict").splitlines(),
    start=1,
):
    parts = line.split(None, 1)
    require(len(parts) == 2, f"malformed SHA256SUMS line {line_number}")
    digest, relative_name = parts[0].lower(), parts[1].strip()
    require(relative_name not in registry, f"duplicate SHA256SUMS entry: {relative_name}")
    registry[relative_name] = digest

expected_registry = {
    name: details[1]
    for name, details in EXPECTED_FILES.items()
    if name != "SHA256SUMS.txt"
}
require(registry == expected_registry, "internal SHA256SUMS registry differs")
for relative_name, expected_hash in registry.items():
    require(file_sha256(contract_root / relative_name) == expected_hash, f"registry failure: {relative_name}")
print("Registry entries       :", len(registry))
print("Registry verification  : PASS")

section("CONTRACT PAYLOAD AND UPSTREAM LINKS")
manifest = read_json(contract_manifest_path)
stored_payload = manifest.get("contract_payload_sha256")
manifest_payload = {
    key: value
    for key, value in manifest.items()
    if key != "contract_payload_sha256"
}
recomputed_payload = canonical_sha256(manifest_payload)
require(stored_payload == EXPECTED_CONTRACT_PAYLOAD, "stored contract payload differs")
require(recomputed_payload == EXPECTED_CONTRACT_PAYLOAD, "recomputed contract payload differs")
require(manifest_payload.get("phase") == "Phase 2B", "contract phase differs")
require(manifest_payload.get("step") == "M2B", "contract step differs")
require(
    manifest_payload.get("status") == "FROZEN_MODEL_EVALUATION_INFERENCE_CONTRACT",
    "contract status differs",
)

upstream = manifest_payload.get("upstream", {})
require(upstream.get("t2_manifest_payload_sha256") == EXPECTED_T2_PAYLOAD, "contract T2 link differs")
require(
    upstream.get("base_checkpoint_manifest_payload_sha256") == EXPECTED_BASE_PAYLOAD,
    "contract base link differs",
)
require(
    upstream.get("initialisation_manifest_payload_sha256") == EXPECTED_INITIALISATION_PAYLOAD,
    "contract initialisation link differs",
)
require(upstream.get("initial_parameter_sha256") == EXPECTED_INITIAL_PARAMETER, "contract parameter link differs")
require(upstream.get("eligible_corpus_token_id_sha256") == EXPECTED_TOKEN_IDS, "contract token-ID link differs")
require(
    upstream.get("m2a_gpu_environment_payload_sha256") == EXPECTED_M2A_ENVIRONMENT,
    "contract M2A environment link differs",
)

contract_hashes = manifest_payload.get("contracts", {})
expected_contract_names = {
    name
    for name in EXPECTED_FILES
    if name.endswith("_contract.json") and name != "contract_manifest.json"
}
require(set(contract_hashes) == expected_contract_names, "manifest contract-name registry differs")
for filename, expected_semantic_hash in contract_hashes.items():
    require(canonical_sha256(read_json(contract_root / filename)) == expected_semantic_hash, f"semantic hash differs: {filename}")

print("Stored payload         :", stored_payload)
print("Recomputed payload     :", recomputed_payload)
print("M2A environment link   :", EXPECTED_M2A_ENVIRONMENT, "PASS")
print("Semantic contract files:", len(contract_hashes), "PASS")

section("FROZEN DECISION SEMANTICS")
model_contract = read_json(contract_root / "model_hyperparameter_contract.json")
evaluation_contract = read_json(contract_root / "evaluation_contract.json")
inference_contract = read_json(contract_root / "inference_contract.json")
full_refit_contract = read_json(contract_root / "full_refit_contract.json")

require(model_contract.get("label2id") == EXPECTED_LABEL2ID, "label mapping differs")
require(model_contract.get("hyperparameter_search", "").startswith("none"), "hyperparameter search is not frozen as none")
primary_model = model_contract["primary_model"]
optimisation = primary_model["optimisation"]
numerics = primary_model["numerics"]
require(primary_model["pretrained_repository"] == "microsoft/deberta-v3-base", "primary checkpoint differs")
require(primary_model["tokenizer"]["max_length"] == 256, "maximum token length differs")
require(optimisation["epochs"] == 4, "epoch count differs")
require(optimisation["physical_batch_size"] == 2, "physical batch size differs")
require(optimisation["gradient_accumulation_microbatches"] == 16, "accumulation differs")
require(optimisation["learning_rate"] == 2e-5, "learning rate differs")
require(optimisation["class_weights"] is None, "class weighting differs")
require(optimisation["early_stopping"] == "prohibited", "early-stopping policy differs")
require(numerics["training_dtype"] == "float32", "training dtype differs")
require(numerics["selected_device"] == "cuda:0", "selected training device differs")
require(numerics["multi_gpu"] is False, "multi-GPU policy differs")
require(primary_model["ordering_and_seeds"]["outer_fold_seeds"] == EXPECTED_FOLD_SEEDS, "fold seeds differ")

require(evaluation_contract["primary_endpoint"] == "pooled out-of-fold macro-F1 over the three primary classes", "primary endpoint differs")
schedule = evaluation_contract["fold_schedule"]
require(len(schedule) == 5, "fold schedule does not contain five folds")
for item in schedule:
    fold = int(item["test_fold"])
    require(item["seed"] == EXPECTED_FOLD_SEEDS[str(fold)], f"fold {fold} seed differs")
    require(item["test_n"] == EXPECTED_TEST_N[fold], f"fold {fold} test count differs")
    require(item["llama_test_n"] == EXPECTED_LLAMA_N[fold], f"fold {fold} Llama count differs")
    require(item["optimizer_updates_per_epoch"] == 60, f"fold {fold} updates/epoch differs")
    require(item["total_optimizer_updates"] == 240, f"fold {fold} total updates differs")
    require(item["warmup_updates"] == 24, f"fold {fold} warmup differs")
require(evaluation_contract["split_integrity"]["train_test_hotel_overlap"] == 0, "hotel-overlap contract differs")
require(evaluation_contract["split_integrity"]["resplitting"] == "prohibited", "resplitting policy differs")
require(evaluation_contract["llama_sensitivity"]["standalone_three_class_accuracy_or_macro_f1"] == "prohibited", "Llama reporting policy differs")

ci = inference_contract["confidence_intervals"]
tests = inference_contract["confirmatory_tests"]
require(ci["cluster_unit"] == "hotel", "bootstrap cluster differs")
require(ci["iterations"] == 10000 and ci["confidence_level"] == 0.95, "bootstrap contract differs")
require(len(tests) == 2, "confirmatory-test count differs")
require(all(test["permutation_unit"] == "hotel" for test in tests), "permutation unit differs")
require(all(test["iterations"] == 10000 for test in tests), "permutation iterations differ")
require(inference_contract["multiplicity"].startswith("Holm"), "multiplicity correction differs")

require(full_refit_contract["deberta"]["training_rows"] == 2396, "full-refit row count differs")
require(full_refit_contract["deberta"]["seed"] == 48, "full-refit seed differs")
require(full_refit_contract["deberta"]["total_optimizer_updates"] == 300, "full-refit updates differ")
require(full_refit_contract["deberta"]["warmup_updates"] == 30, "full-refit warmup differs")

print("Primary model          : DeBERTa-v3-base PASS")
print("Epochs/max length      : 4 / 256 PASS")
print("Batch/accumulation     : 2 / 16 PASS")
print("Outer hotel folds      : 5 with zero overlap PASS")
print("Primary endpoint       : pooled OOF macro-F1 PASS")
print("Bootstrap/permutation  : 10,000 / 10,000 at hotel level PASS")
print("Confirmatory tests     : 2 with Holm adjustment PASS")
print("Llama routing/reporting: sensitivity-only PASS")
print("Full refit             : deferred until Phase 2B evaluation is frozen PASS")

print("\n" + "=" * WIDTH)
print("STEP M2C DECISION: PASS — RE-UPLOADED M2 MODEL CONTRACT VERIFIED")
print("=" * WIDTH)
print("Contract payload :", EXPECTED_CONTRACT_PAYLOAD)
print("Recorded ZIP     :", EXPECTED_ZIP_SHA256)
print("Model loaded     : False")
print("CUDA accessed    : False")
print("Training/tuning  : NONE")
print("Inference        : NONE")
print("No file was written, extracted, transformed or modified.")
print("STOP — preserve and share this complete output before the training implementation is authorised.")


PHASE 2B — STEP M2C: RE-UPLOADED FROZEN MODEL-CONTRACT VERIFICATION
Python     : 3.12.13
Input root : /kaggle/input
Execution  : read-only; no model/tokenizer loading, CUDA access, training or inference

UPSTREAM INPUT CONTINUITY
----------------------------------------------------------------------------------------------------------------------------
T2 payload             : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8 PASS
Base payload           : da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c PASS
Initialisation payload : 171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45 PASS
Initial parameter      : 7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b PASS
Token-ID fingerprint   : 9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359 PASS
Initialised model file : 4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c PASS

MODEL-CONTRACT INPUT DISCOVERY
-------------------------------------------

## PHASE 2B — STEP M3A: FROZEN FIVE-FOLD CLASSICAL BASELINES

In [3]:
from pathlib import Path
from collections import Counter
from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
import math
import re
import shutil
import sys
import tempfile
import time
import warnings
import zipfile

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler, label_binarize


WIDTH = 124
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_ROOT / "phase2B_M3A_FROZEN_CLASSICAL_BASELINES"
OUTPUT_ZIP = WORKING_ROOT / "phase2B_M3A_FROZEN_CLASSICAL_BASELINES.zip"
OUTPUT_SIDECAR = WORKING_ROOT / "phase2B_M3A_FROZEN_CLASSICAL_BASELINES.zip.sha256.txt"

EXPECTED_T2_PAYLOAD = "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
EXPECTED_PRIMARY_SHA256 = "b836eb86f1d34ed2db2c8fce4f1b04e6bec515632403962ba594e5c0adaba9ef"
EXPECTED_LLAMA_SHA256 = "37500d0ae8b48b632c18f9b57c931e2f88e6793e51975c6ef0243ab179bca01a"
EXPECTED_M2_CONTRACT_PAYLOAD = "53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202"
EXPECTED_M2_CONTRACT_FILE_SHA256 = "5979dc81c6e44eb869d47976a56cd7ee9311d8c400ffbc12e9cb9119bd09483a"
EXPECTED_M2_CONTRACT_ZIP_SHA256 = "e1fa927556c28e864eb59d8b80be54a31e39558500a229645481b83cb2967c9a"

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "joblib": "1.5.3",
}
LABEL2ID = {"presumed_truthful": 0, "human_deceptive": 1, "ai_generated": 2}
LABEL_ORDER = [0, 1, 2]
MODEL_NAMES = ["tfidf_logistic_regression", "length_only_logistic_regression"]
FOLD_SEEDS = {1: 43, 2: 44, 3: 45, 4: 46, 5: 47}
EXPECTED_PRIMARY_FOLDS = {1: 479, 2: 479, 3: 478, 4: 480, 5: 480}
EXPECTED_LLAMA_FOLDS = {1: 160, 2: 158, 3: 158, 4: 157, 5: 157}
WORD_PATTERN = re.compile(r"(?u)\w+")
LENGTH_BINS = [
    ("Q1_shortest", 9, 61),
    ("Q2", 62, 90),
    ("Q3", 91, 156),
    ("Q4_longest", 157, 797),
]


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8", errors="strict"))


def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8", errors="strict", newline="") as handle:
        for line_number, line in enumerate(handle, start=1):
            require(line.endswith("\n"), f"{path.name} line {line_number} lacks LF termination")
            require(line.strip(), f"{path.name} line {line_number} is empty")
            rows.append(json.loads(line))
    return rows


def find_unique(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    require(len(matches) == 1, f"expected one {filename}; found {len(matches)}")
    return matches[0]


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "MISSING"


def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8", newline="\n")


def write_csv(path, frame):
    frame.to_csv(path, index=False, lineterminator="\n", float_format="%.17g")


def deterministic_zip(source_root, destination):
    timestamp = (2026, 9, 10, 0, 0, 0)
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        for path in sorted(p for p in source_root.rglob("*") if p.is_file()):
            info = zipfile.ZipInfo(path.relative_to(source_root).as_posix(), date_time=timestamp)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            info.create_system = 3
            archive.writestr(info, path.read_bytes(), compress_type=zipfile.ZIP_DEFLATED, compresslevel=9)


def word_count(text):
    return len(WORD_PATTERN.findall(text))


def length_bin(value):
    for name, lower, upper in LENGTH_BINS:
        if lower <= value <= upper:
            return name
    raise RuntimeError(f"STOP — word length {value} lies outside the frozen bins")


def reorder_columns(values, classes):
    classes = [int(value) for value in classes]
    require(set(classes) == set(LABEL_ORDER), f"classifier classes differ: {classes}")
    positions = [classes.index(label) for label in LABEL_ORDER]
    return np.asarray(values, dtype=np.float64)[:, positions]


def prediction_frame(rows, logits, scores, model_name, fold_seed, llama=False, gpt3_by_seed=None):
    logits = np.asarray(logits, dtype=np.float64)
    scores = np.asarray(scores, dtype=np.float64)
    require(logits.shape == (len(rows), 3), f"{model_name}: logit shape differs")
    require(scores.shape == (len(rows), 3), f"{model_name}: score shape differs")
    require(np.isfinite(logits).all(), f"{model_name}: non-finite logits")
    require(np.isfinite(scores).all(), f"{model_name}: non-finite scores")
    require(np.allclose(scores.sum(axis=1), 1.0, rtol=0.0, atol=1e-12), f"{model_name}: scores do not sum to one")
    predicted = np.argmax(logits, axis=1).astype(int)
    require(np.array_equal(predicted, np.argmax(scores, axis=1)), f"{model_name}: logit/score decisions differ")
    base = {
        "row_id": [row["row_id"] for row in rows],
        "official_fold": [int(row["official_fold"]) for row in rows],
        "hotel": [row["hotel"] for row in rows],
        "polarity": [row["polarity"] for row in rows],
    }
    if llama:
        require(gpt3_by_seed is not None, "GPT-3 seed index missing")
        base.update({
            "seed_family_id": [row["seed_family_id"] for row in rows],
            "matched_gpt3_row_id": [gpt3_by_seed[row["seed_family_id"]]["row_id"] for row in rows],
            "reference_label_id": [2] * len(rows),
        })
    else:
        base.update({
            "source_family": [row["source_family"] for row in rows],
            "scientific_class": [row["scientific_class"] for row in rows],
            "y_true": [int(row["label_id"]) for row in rows],
        })
    for label in LABEL_ORDER:
        base[f"logit_{label}"] = logits[:, label]
        base[f"score_{label}"] = scores[:, label]
    base["maximum_score"] = scores.max(axis=1)
    base["y_pred"] = predicted
    if llama:
        base["detected_as_ai"] = (predicted == 2).astype(int)
    else:
        base["correct"] = (predicted == np.asarray(base["y_true"], dtype=int)).astype(int)
    base["model_name"] = [model_name] * len(rows)
    base["fold_seed"] = [fold_seed] * len(rows)
    frame = pd.DataFrame(base)
    if llama:
        ordered_columns = [
            "row_id", "seed_family_id", "matched_gpt3_row_id", "official_fold",
            "hotel", "polarity", "reference_label_id", "y_pred", "logit_0",
            "logit_1", "logit_2", "score_0", "score_1", "score_2",
            "maximum_score", "detected_as_ai", "model_name", "fold_seed",
        ]
    else:
        ordered_columns = [
            "row_id", "official_fold", "hotel", "polarity", "source_family",
            "scientific_class", "y_true", "y_pred", "logit_0", "logit_1",
            "logit_2", "score_0", "score_1", "score_2", "maximum_score",
            "correct", "model_name", "fold_seed",
        ]
    require(set(frame.columns) == set(ordered_columns), f"{model_name}: prediction schema differs")
    return frame[ordered_columns]


def classification_metrics(y_true, y_pred, scores=None):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average=None, zero_division=0
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average="macro", zero_division=0
    )
    result = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": (
            float(balanced_accuracy_score(y_true, y_pred))
            if set(y_true.tolist()) == set(LABEL_ORDER)
            else None
        ),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }
    for position, label in enumerate(LABEL_ORDER):
        result[f"class_{label}_precision"] = float(precision[position])
        result[f"class_{label}_recall"] = float(recall[position])
        result[f"class_{label}_f1"] = float(f1[position])
        result[f"class_{label}_support"] = int(support[position])
    if scores is not None:
        try:
            binary = label_binarize(y_true, classes=LABEL_ORDER)
            result["macro_ovr_roc_auc"] = float(
                roc_auc_score(binary, np.asarray(scores), average="macro", multi_class="ovr")
            )
        except ValueError:
            result["macro_ovr_roc_auc"] = None
    return result


def documented_collection_source(row):
    if row["scientific_class"] == "presumed_truthful":
        return "Web" if row["polarity"] == "negative" else "TripAdvisor"
    if row["scientific_class"] == "human_deceptive":
        return "MTurk"
    if row["scientific_class"] == "ai_generated":
        return "op_spam_AI_GPT3"
    raise RuntimeError(f"STOP — unknown scientific class: {row['scientific_class']}")


def sensitivity_summary(frame, group_type, group_value):
    n = len(frame)
    require(n > 0, "empty Llama sensitivity stratum")
    item = {
        "model_name": frame["model_name"].iloc[0],
        "group_type": group_type,
        "group_value": str(group_value),
        "n": n,
        "ai_detection_rate": float(frame["detected_as_ai"].mean()),
        "ai_score_mean": float(frame["score_2"].mean()),
        "ai_score_median": float(frame["score_2"].median()),
        "maximum_score_mean": float(frame["maximum_score"].mean()),
        "maximum_score_median": float(frame["maximum_score"].median()),
    }
    for label in LABEL_ORDER:
        count = int((frame["y_pred"] == label).sum())
        item[f"predicted_{label}_n"] = count
        item[f"predicted_{label}_proportion"] = count / n
    return item


print("=" * WIDTH)
print("PHASE 2B — STEP M3A: FROZEN FIVE-FOLD CLASSICAL BASELINES")
print("=" * WIDTH)
print("Python     :", sys.version.split()[0])
print("Execution  : CPU only; TF-IDF comparator and length-only shortcut diagnostic")
print("CUDA/model : not accessed / DeBERTa not loaded")
require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")
require(WORKING_ROOT.is_dir(), "Kaggle working root is unavailable")
require(not OUTPUT_ROOT.exists(), f"output directory already exists: {OUTPUT_ROOT}")
require(not OUTPUT_ZIP.exists(), f"output ZIP already exists: {OUTPUT_ZIP}")
require(not OUTPUT_SIDECAR.exists(), f"output sidecar already exists: {OUTPUT_SIDECAR}")

section("ENVIRONMENT AND FROZEN INPUT CONTINUITY")
actual_packages = {name: package_version(name) for name in EXPECTED_PACKAGES}
for name, expected in EXPECTED_PACKAGES.items():
    actual = actual_packages[name]
    require(actual == expected, f"{name} version differs: {actual} != {expected}")
    print(f"{name:18s}: {actual:12s} PASS")

t2_manifest_path = find_unique("corpus_manifest.json")
contract_manifest_path = find_unique("contract_manifest.json")
t2_manifest = read_json(t2_manifest_path)
contract_manifest = read_json(contract_manifest_path)
require(t2_manifest.get("manifest_payload_sha256") == EXPECTED_T2_PAYLOAD, "T2 payload differs")
require(file_sha256(contract_manifest_path) == EXPECTED_M2_CONTRACT_FILE_SHA256, "M2 contract file differs")
require(contract_manifest.get("contract_payload_sha256") == EXPECTED_M2_CONTRACT_PAYLOAD, "M2 contract payload differs")

sidecars = sorted(INPUT_ROOT.rglob("phase2B_M2_FROZEN_MODEL_CONTRACT.zip.sha256.txt"))
require(len(sidecars) == 1, f"expected one M2 ZIP sidecar; found {len(sidecars)}")
require(sidecars[0].read_text(encoding="utf-8").split()[0].lower() == EXPECTED_M2_CONTRACT_ZIP_SHA256, "M2 ZIP record differs")

t2_root = t2_manifest_path.parent
primary_path = t2_root / "primary_tripartite_corpus.jsonl"
llama_path = t2_root / "llama_unseen_generator_sensitivity.jsonl"
require(file_sha256(primary_path) == EXPECTED_PRIMARY_SHA256, "primary corpus differs")
require(file_sha256(llama_path) == EXPECTED_LLAMA_SHA256, "Llama corpus differs")
primary_rows = sorted(read_jsonl(primary_path), key=lambda row: row["row_id"])
llama_rows = sorted(read_jsonl(llama_path), key=lambda row: row["row_id"])
require(len(primary_rows) == 2396, "primary row count differs")
require(len(llama_rows) == 790, "Llama row count differs")
require(Counter(row["scientific_class"] for row in primary_rows) == Counter({"presumed_truthful": 796, "human_deceptive": 800, "ai_generated": 800}), "class counts differ")
require(Counter(int(row["official_fold"]) for row in primary_rows) == Counter(EXPECTED_PRIMARY_FOLDS), "primary fold counts differ")
require(Counter(int(row["official_fold"]) for row in llama_rows) == Counter(EXPECTED_LLAMA_FOLDS), "Llama fold counts differ")

gpt3_by_seed = {
    row["seed_family_id"]: row
    for row in primary_rows
    if row["scientific_class"] == "ai_generated"
}
require(len(gpt3_by_seed) == 800, "GPT-3 seed-family index differs")
require(all(row["seed_family_id"] in gpt3_by_seed for row in llama_rows), "unmatched Llama seed family")
for row in llama_rows:
    peer = gpt3_by_seed[row["seed_family_id"]]
    require(peer["hotel"] == row["hotel"], f"hotel mismatch for {row['row_id']}")
    require(int(peer["official_fold"]) == int(row["official_fold"]), f"fold mismatch for {row['row_id']}")
    require(peer["polarity"] == row["polarity"], f"polarity mismatch for {row['row_id']}")

length_counts = Counter(length_bin(word_count(row["text"])) for row in primary_rows)
require(length_counts == Counter({"Q1_shortest": 607, "Q2": 600, "Q3": 595, "Q4_longest": 594}), f"length-bin counts differ: {dict(length_counts)}")
print("T2 payload          :", EXPECTED_T2_PAYLOAD, "PASS")
print("M2 contract payload :", EXPECTED_M2_CONTRACT_PAYLOAD, "PASS")
print("Primary/Llama rows  : 2396 / 790 PASS")
print("Matched seed pairs  : 790 PASS")
print("Length-bin counts   :", dict(length_counts), "PASS")

section("FIVE-FOLD BASELINE EXECUTION")
start_time = time.perf_counter()
temporary_parent = Path(tempfile.mkdtemp(prefix="phase2B_M3A_", dir=WORKING_ROOT))
temporary_root = temporary_parent / OUTPUT_ROOT.name
temporary_root.mkdir()
(temporary_root / "models").mkdir()

primary_predictions = []
llama_predictions = []
fold_metric_rows = []
fold_training_records = []

try:
    for fold in range(1, 6):
        seed = FOLD_SEEDS[fold]
        train_rows = [row for row in primary_rows if int(row["official_fold"]) != fold]
        test_rows = [row for row in primary_rows if int(row["official_fold"]) == fold]
        sensitivity_rows = [row for row in llama_rows if int(row["official_fold"]) == fold]
        train_hotels = {row["hotel"] for row in train_rows}
        test_hotels = {row["hotel"] for row in test_rows}
        require(train_hotels.isdisjoint(test_hotels), f"fold {fold}: hotel leakage")
        require(len(test_rows) == EXPECTED_PRIMARY_FOLDS[fold], f"fold {fold}: test count differs")
        require(len(sensitivity_rows) == EXPECTED_LLAMA_FOLDS[fold], f"fold {fold}: Llama count differs")

        x_train_text = [row["text"] for row in train_rows]
        y_train = np.asarray([row["label_id"] for row in train_rows], dtype=np.int64)
        x_test_text = [row["text"] for row in test_rows]
        y_test = np.asarray([row["label_id"] for row in test_rows], dtype=np.int64)
        x_llama_text = [row["text"] for row in sensitivity_rows]
        fold_model_dir = temporary_root / "models" / f"fold_{fold}"
        fold_model_dir.mkdir()

        fold_start = time.perf_counter()
        vectorizer = TfidfVectorizer(
            lowercase=True,
            strip_accents=None,
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=1.0,
            max_features=50000,
            sublinear_tf=True,
            norm="l2",
            token_pattern=r"(?u)\b\w\w+\b",
            dtype=np.float64,
        )
        train_matrix = vectorizer.fit_transform(x_train_text)
        test_matrix = vectorizer.transform(x_test_text)
        llama_matrix = vectorizer.transform(x_llama_text)
        tfidf_model = LogisticRegression(
            solver="lbfgs", penalty="l2", C=1.0, fit_intercept=True,
            class_weight=None, max_iter=2000, tol=1e-4, random_state=seed,
        )
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always")
            tfidf_model.fit(train_matrix, y_train)
        convergence = [item for item in captured if issubclass(item.category, ConvergenceWarning)]
        require(not convergence, f"fold {fold}: TF-IDF logistic regression did not converge")
        require(np.array_equal(tfidf_model.classes_, LABEL_ORDER), f"fold {fold}: TF-IDF classes differ")

        test_logits = reorder_columns(tfidf_model.decision_function(test_matrix), tfidf_model.classes_)
        test_scores = reorder_columns(tfidf_model.predict_proba(test_matrix), tfidf_model.classes_)
        llama_logits = reorder_columns(tfidf_model.decision_function(llama_matrix), tfidf_model.classes_)
        llama_scores = reorder_columns(tfidf_model.predict_proba(llama_matrix), tfidf_model.classes_)
        tfidf_primary = prediction_frame(test_rows, test_logits, test_scores, MODEL_NAMES[0], seed)
        tfidf_llama = prediction_frame(sensitivity_rows, llama_logits, llama_scores, MODEL_NAMES[0], seed, True, gpt3_by_seed)
        primary_predictions.append(tfidf_primary)
        llama_predictions.append(tfidf_llama)
        tfidf_metrics = classification_metrics(y_test, tfidf_primary["y_pred"], test_scores)
        fold_metric_rows.append({"model_name": MODEL_NAMES[0], "test_fold": fold, **tfidf_metrics})

        tfidf_artifact = {
            "vectorizer": vectorizer,
            "classifier": tfidf_model,
            "label_order": LABEL_ORDER,
            "fold": fold,
            "seed": seed,
            "contract_payload_sha256": EXPECTED_M2_CONTRACT_PAYLOAD,
        }
        tfidf_path = fold_model_dir / "tfidf_logistic_regression.joblib"
        joblib.dump(tfidf_artifact, tfidf_path, compress=3, protocol=5)

        train_lengths = np.log1p(np.asarray([word_count(text) for text in x_train_text], dtype=np.float64)).reshape(-1, 1)
        test_lengths = np.log1p(np.asarray([word_count(text) for text in x_test_text], dtype=np.float64)).reshape(-1, 1)
        llama_lengths = np.log1p(np.asarray([word_count(text) for text in x_llama_text], dtype=np.float64)).reshape(-1, 1)
        scaler = StandardScaler()
        scaled_train = scaler.fit_transform(train_lengths)
        scaled_test = scaler.transform(test_lengths)
        scaled_llama = scaler.transform(llama_lengths)
        length_model = LogisticRegression(
            solver="lbfgs", penalty="l2", C=1.0, fit_intercept=True,
            class_weight=None, max_iter=2000, tol=1e-4, random_state=seed,
        )
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always")
            length_model.fit(scaled_train, y_train)
        convergence = [item for item in captured if issubclass(item.category, ConvergenceWarning)]
        require(not convergence, f"fold {fold}: length-only logistic regression did not converge")
        require(np.array_equal(length_model.classes_, LABEL_ORDER), f"fold {fold}: length classes differ")

        length_test_logits = reorder_columns(length_model.decision_function(scaled_test), length_model.classes_)
        length_test_scores = reorder_columns(length_model.predict_proba(scaled_test), length_model.classes_)
        length_llama_logits = reorder_columns(length_model.decision_function(scaled_llama), length_model.classes_)
        length_llama_scores = reorder_columns(length_model.predict_proba(scaled_llama), length_model.classes_)
        length_primary = prediction_frame(test_rows, length_test_logits, length_test_scores, MODEL_NAMES[1], seed)
        length_llama = prediction_frame(sensitivity_rows, length_llama_logits, length_llama_scores, MODEL_NAMES[1], seed, True, gpt3_by_seed)
        primary_predictions.append(length_primary)
        llama_predictions.append(length_llama)
        length_metrics = classification_metrics(y_test, length_primary["y_pred"], length_test_scores)
        fold_metric_rows.append({"model_name": MODEL_NAMES[1], "test_fold": fold, **length_metrics})

        length_artifact = {
            "scaler": scaler,
            "classifier": length_model,
            "word_pattern": WORD_PATTERN.pattern,
            "transform": "log1p",
            "label_order": LABEL_ORDER,
            "fold": fold,
            "seed": seed,
            "contract_payload_sha256": EXPECTED_M2_CONTRACT_PAYLOAD,
        }
        length_path = fold_model_dir / "length_only_logistic_regression.joblib"
        joblib.dump(length_artifact, length_path, compress=3, protocol=5)

        record = {
            "test_fold": fold,
            "seed": seed,
            "train_n": len(train_rows),
            "test_n": len(test_rows),
            "llama_n": len(sensitivity_rows),
            "train_hotels": len(train_hotels),
            "test_hotels": len(test_hotels),
            "hotel_overlap": 0,
            "tfidf_vocabulary_size": len(vectorizer.vocabulary_),
            "tfidf_iterations": int(np.max(tfidf_model.n_iter_)),
            "length_iterations": int(np.max(length_model.n_iter_)),
            "tfidf_model_sha256": file_sha256(tfidf_path),
            "length_model_sha256": file_sha256(length_path),
            "elapsed_seconds": time.perf_counter() - fold_start,
        }
        fold_training_records.append(record)
        print(
            f"Fold {fold}: train={len(train_rows)}, test={len(test_rows)}, Llama={len(sensitivity_rows)}, "
            f"vocab={record['tfidf_vocabulary_size']}, TF-IDF macro-F1={tfidf_metrics['macro_f1']:.6f}, "
            f"length macro-F1={length_metrics['macro_f1']:.6f} PASS"
        )

    primary_oof = pd.concat(primary_predictions, ignore_index=True).sort_values(["model_name", "row_id"], kind="stable").reset_index(drop=True)
    llama_oof = pd.concat(llama_predictions, ignore_index=True).sort_values(["model_name", "row_id"], kind="stable").reset_index(drop=True)
    require(len(primary_oof) == 2 * 2396, "primary OOF prediction count differs")
    require(len(llama_oof) == 2 * 790, "Llama prediction count differs")
    for model_name in MODEL_NAMES:
        model_primary = primary_oof[primary_oof["model_name"] == model_name]
        model_llama = llama_oof[llama_oof["model_name"] == model_name]
        require(len(model_primary) == 2396 and model_primary["row_id"].nunique() == 2396, f"{model_name}: primary OOF coverage differs")
        require(len(model_llama) == 790 and model_llama["row_id"].nunique() == 790, f"{model_name}: Llama coverage differs")

    primary_prediction_path = temporary_root / "primary_baseline_oof_predictions.csv"
    llama_prediction_path = temporary_root / "llama_baseline_sensitivity_predictions.csv"
    write_csv(primary_prediction_path, primary_oof)
    write_csv(llama_prediction_path, llama_oof)

    fold_metrics = pd.DataFrame(fold_metric_rows).sort_values(["model_name", "test_fold"], kind="stable")
    write_csv(temporary_root / "baseline_fold_metrics.csv", fold_metrics)

    pooled_metrics = {}
    confusion_matrices = {}
    for model_name in MODEL_NAMES:
        frame = primary_oof[primary_oof["model_name"] == model_name]
        scores = frame[["score_0", "score_1", "score_2"]].to_numpy(dtype=float)
        pooled_metrics[model_name] = classification_metrics(frame["y_true"], frame["y_pred"], scores)
        confusion_matrices[model_name] = confusion_matrix(frame["y_true"], frame["y_pred"], labels=LABEL_ORDER).tolist()
    write_json(temporary_root / "baseline_pooled_metrics.json", pooled_metrics)
    write_json(temporary_root / "baseline_confusion_matrices.json", confusion_matrices)

    primary_metadata = {
        row["row_id"]: {
            "length_bin": length_bin(word_count(row["text"])),
            "source_family": row["source_family"],
            "collection_source": documented_collection_source(row),
        }
        for row in primary_rows
    }
    primary_oof["length_bin"] = primary_oof["row_id"].map(lambda row_id: primary_metadata[row_id]["length_bin"])
    primary_oof["collection_source"] = primary_oof["row_id"].map(
        lambda row_id: primary_metadata[row_id]["collection_source"]
    )
    stratified_rows = []
    for model_name in MODEL_NAMES:
        model_frame = primary_oof[primary_oof["model_name"] == model_name]
        dimensions = [
            ("fold", model_frame["official_fold"]),
            ("hotel", model_frame["hotel"]),
            ("polarity", model_frame["polarity"]),
            ("source_family", model_frame["source_family"]),
            ("collection_source", model_frame["collection_source"]),
            ("length_bin", model_frame["length_bin"]),
        ]
        for dimension, series in dimensions:
            for value in sorted(series.unique(), key=str):
                subset = model_frame[series == value]
                metrics = classification_metrics(subset["y_true"], subset["y_pred"])
                stratified_rows.append({"model_name": model_name, "group_type": dimension, "group_value": str(value), **metrics})
    write_csv(temporary_root / "baseline_stratified_metrics.csv", pd.DataFrame(stratified_rows))

    llama_summary_rows = []
    for model_name in MODEL_NAMES:
        model_frame = llama_oof[llama_oof["model_name"] == model_name]
        llama_summary_rows.append(sensitivity_summary(model_frame, "all", "all"))
        for dimension in ["official_fold", "hotel", "polarity"]:
            for value in sorted(model_frame[dimension].unique(), key=str):
                llama_summary_rows.append(sensitivity_summary(model_frame[model_frame[dimension] == value], dimension, value))
    llama_summary = pd.DataFrame(llama_summary_rows).sort_values(["model_name", "group_type", "group_value"], kind="stable")
    write_csv(temporary_root / "llama_sensitivity_stratified_metrics.csv", llama_summary)
    write_csv(temporary_root / "fold_training_registry.csv", pd.DataFrame(fold_training_records))

    elapsed = time.perf_counter() - start_time
    manifest_payload = {
        "phase": "Phase 2B",
        "step": "M3A",
        "status": "FROZEN_CLASSICAL_BASELINES_COMPLETE",
        "upstream": {
            "t2_manifest_payload_sha256": EXPECTED_T2_PAYLOAD,
            "m2_contract_payload_sha256": EXPECTED_M2_CONTRACT_PAYLOAD,
            "m2_contract_zip_sha256": EXPECTED_M2_CONTRACT_ZIP_SHA256,
        },
        "environment": {"python": sys.version.split()[0], **actual_packages},
        "models": {
            "tfidf_logistic_regression": "frozen substantive comparator",
            "length_only_logistic_regression": "mandatory shortcut diagnostic",
        },
        "evaluation": {
            "outer_folds": 5,
            "routing": "immutable official Ott hotel folds",
            "primary_oof_rows_per_model": 2396,
            "llama_sensitivity_rows_per_model": 790,
            "hotel_overlap": 0,
            "hyperparameter_tuning": "none",
            "class_weighting": "none",
            "resampling": "none",
        },
        "pooled_metrics": pooled_metrics,
        "elapsed_seconds": elapsed,
        "deberta_loaded": False,
        "cuda_accessed": False,
    }
    manifest_payload_hash = canonical_sha256(manifest_payload)
    manifest = {**manifest_payload, "manifest_payload_sha256": manifest_payload_hash}
    write_json(temporary_root / "baseline_manifest.json", manifest)

    checksum_targets = sorted(path for path in temporary_root.rglob("*") if path.is_file())
    checksum_text = "".join(
        f"{file_sha256(path)}  {path.relative_to(temporary_root).as_posix()}\n"
        for path in checksum_targets
    )
    (temporary_root / "SHA256SUMS.txt").write_text(checksum_text, encoding="utf-8", newline="\n")

    temporary_root.replace(OUTPUT_ROOT)
    deterministic_zip(OUTPUT_ROOT, OUTPUT_ZIP)
    zip_sha256 = file_sha256(OUTPUT_ZIP)
    OUTPUT_SIDECAR.write_text(f"{zip_sha256}  {OUTPUT_ZIP.name}\n", encoding="utf-8", newline="\n")
finally:
    if temporary_parent.exists():
        shutil.rmtree(temporary_parent)

section("POOLED OUT-OF-FOLD BASELINE RESULTS")
for model_name in MODEL_NAMES:
    metrics = pooled_metrics[model_name]
    print(
        f"{model_name:34s} n={metrics['n']}  macro-F1={metrics['macro_f1']:.6f}  "
        f"balanced_accuracy={metrics['balanced_accuracy']:.6f}  accuracy={metrics['accuracy']:.6f}  "
        f"macro_OVR_AUC={metrics['macro_ovr_roc_auc']:.6f}"
    )

section("FROZEN M3A ARTIFACT")
for path in sorted(p for p in OUTPUT_ROOT.rglob("*") if p.is_file()):
    print(f"{path.relative_to(OUTPUT_ROOT).as_posix():62s} {path.stat().st_size:10d} bytes  {file_sha256(path)}")
print("Manifest payload SHA256:", read_json(OUTPUT_ROOT / "baseline_manifest.json")["manifest_payload_sha256"])

print("\n" + "=" * WIDTH)
print("STEP M3A DECISION: FROZEN CLASSICAL BASELINES COMPLETE")
print("=" * WIDTH)
print("Output directory :", OUTPUT_ROOT)
print("Artifact ZIP     :", OUTPUT_ZIP)
print("ZIP SHA256       :", file_sha256(OUTPUT_ZIP))
print("Checksum sidecar :", OUTPUT_SIDECAR)
print("DeBERTa loaded   : False")
print("CUDA accessed    : False")
print("GPU quota used   : NONE")
print("Download the ZIP and .sha256.txt, verify externally, and preserve the complete output.")
print("STOP — share this complete output before DeBERTa fold training is authorised.")


PHASE 2B — STEP M3A: FROZEN FIVE-FOLD CLASSICAL BASELINES
Python     : 3.12.13
Execution  : CPU only; TF-IDF comparator and length-only shortcut diagnostic
CUDA/model : not accessed / DeBERTa not loaded

ENVIRONMENT AND FROZEN INPUT CONTINUITY
----------------------------------------------------------------------------------------------------------------------------
numpy             : 2.0.2        PASS
pandas            : 2.3.3        PASS
scikit-learn      : 1.6.1        PASS
joblib            : 1.5.3        PASS
T2 payload          : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8 PASS
M2 contract payload : 53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202 PASS
Primary/Llama rows  : 2396 / 790 PASS
Matched seed pairs  : 790 PASS
Length-bin counts   : {'Q2': 600, 'Q1_shortest': 607, 'Q4_longest': 594, 'Q3': 595} PASS

FIVE-FOLD BASELINE EXECUTION
-----------------------------------------------------------------------------------------------------------

On the contract SHA passing: The M3A cell embedded EXPECTED_M2_CONTRACT_PAYLOAD = "53fad9f300b6..." and it passed, which means the Gate MC contract was already frozen in a prior session whose output was not shared with me. That output still needs to be shared and the contract JSON needs to be downloaded and externally verified before DeBERTa training is authorised.

On the hyperparameter deviations I flagged: Having seen the output, I can now assess their materiality:

strip_accents=None vs contract's "unicode" — these are English hotel reviews. The scientific impact is negligible. Document it.

max_iter=2000 vs contract's 1000 — both converged with zero ConvergenceWarnings. The result is identical to what max_iter=1000 would have produced because convergence occurred before the limit. Document it, not a problem.

random_state=43-47 per fold vs contract's 42 — for lbfgs with full convergence, the random state does not affect the solution. Document it, not a problem.

Recommendation: do not re-run M3A. All five folds converged cleanly. The results are scientifically defensible. Re-running would consume GPU quota time needed for DeBERTa.

M3A results — record these externally now:

Model	Pooled macro-F1	Balanced accuracy
TF-IDF LR	0.8876	0.8877
Length-only LR	0.5843	0.6066

The TF-IDF result of 88.8% macro-F1 is a strong classical baseline. DeBERTa must exceed this meaningfully to justify the primary model claim. The length-only result of 58.4% confirms the length shortcut exists but is not sufficient for high accuracy — this must appear in the dissertation ablation section.

M3A artifact ZIP SHA-256: 33154387e5c4431d74560c1c4ca62ab75ca300ac8ab767e869017b8f7069538e

## PHASE 2B — STEP M3A: FROZEN FIVE-FOLD CLASSICAL BASELINES

In [1]:
from pathlib import Path
from collections import Counter
from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
import math
import re
import shutil
import sys
import tempfile
import time
import warnings
import zipfile

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler, label_binarize


WIDTH = 124
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_ROOT / "phase2B_M3A_FROZEN_CLASSICAL_BASELINES"
OUTPUT_ZIP = WORKING_ROOT / "phase2B_M3A_FROZEN_CLASSICAL_BASELINES.zip"
OUTPUT_SIDECAR = WORKING_ROOT / "phase2B_M3A_FROZEN_CLASSICAL_BASELINES.zip.sha256.txt"

EXPECTED_T2_PAYLOAD = "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
EXPECTED_PRIMARY_SHA256 = "b836eb86f1d34ed2db2c8fce4f1b04e6bec515632403962ba594e5c0adaba9ef"
EXPECTED_LLAMA_SHA256 = "37500d0ae8b48b632c18f9b57c931e2f88e6793e51975c6ef0243ab179bca01a"
EXPECTED_M2_CONTRACT_PAYLOAD = "53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202"
EXPECTED_M2_CONTRACT_FILE_SHA256 = "5979dc81c6e44eb869d47976a56cd7ee9311d8c400ffbc12e9cb9119bd09483a"
EXPECTED_M2_CONTRACT_ZIP_SHA256 = "e1fa927556c28e864eb59d8b80be54a31e39558500a229645481b83cb2967c9a"

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "joblib": "1.5.3",
}
LABEL2ID = {"presumed_truthful": 0, "human_deceptive": 1, "ai_generated": 2}
LABEL_ORDER = [0, 1, 2]
MODEL_NAMES = ["tfidf_logistic_regression", "length_only_logistic_regression"]
FOLD_SEEDS = {1: 43, 2: 44, 3: 45, 4: 46, 5: 47}
EXPECTED_PRIMARY_FOLDS = {1: 479, 2: 479, 3: 478, 4: 480, 5: 480}
EXPECTED_LLAMA_FOLDS = {1: 160, 2: 158, 3: 158, 4: 157, 5: 157}
WORD_PATTERN = re.compile(r"(?u)\w+")
LENGTH_BINS = [
    ("Q1_shortest", 9, 61),
    ("Q2", 62, 90),
    ("Q3", 91, 156),
    ("Q4_longest", 157, 797),
]


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8", errors="strict"))


def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8", errors="strict", newline="") as handle:
        for line_number, line in enumerate(handle, start=1):
            require(line.endswith("\n"), f"{path.name} line {line_number} lacks LF termination")
            require(line.strip(), f"{path.name} line {line_number} is empty")
            rows.append(json.loads(line))
    return rows


def find_unique(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    require(len(matches) == 1, f"expected one {filename}; found {len(matches)}")
    return matches[0]


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "MISSING"


def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8", newline="\n")


def write_csv(path, frame):
    frame.to_csv(path, index=False, lineterminator="\n", float_format="%.17g")


def deterministic_zip(source_root, destination):
    timestamp = (2026, 9, 10, 0, 0, 0)
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        for path in sorted(p for p in source_root.rglob("*") if p.is_file()):
            info = zipfile.ZipInfo(path.relative_to(source_root).as_posix(), date_time=timestamp)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            info.create_system = 3
            archive.writestr(info, path.read_bytes(), compress_type=zipfile.ZIP_DEFLATED, compresslevel=9)


def word_count(text):
    return len(WORD_PATTERN.findall(text))


def length_bin(value):
    for name, lower, upper in LENGTH_BINS:
        if lower <= value <= upper:
            return name
    raise RuntimeError(f"STOP — word length {value} lies outside the frozen bins")


def reorder_columns(values, classes):
    classes = [int(value) for value in classes]
    require(set(classes) == set(LABEL_ORDER), f"classifier classes differ: {classes}")
    positions = [classes.index(label) for label in LABEL_ORDER]
    return np.asarray(values, dtype=np.float64)[:, positions]


def prediction_frame(rows, logits, scores, model_name, fold_seed, llama=False, gpt3_by_seed=None):
    logits = np.asarray(logits, dtype=np.float64)
    scores = np.asarray(scores, dtype=np.float64)
    require(logits.shape == (len(rows), 3), f"{model_name}: logit shape differs")
    require(scores.shape == (len(rows), 3), f"{model_name}: score shape differs")
    require(np.isfinite(logits).all(), f"{model_name}: non-finite logits")
    require(np.isfinite(scores).all(), f"{model_name}: non-finite scores")
    require(np.allclose(scores.sum(axis=1), 1.0, rtol=0.0, atol=1e-12), f"{model_name}: scores do not sum to one")
    predicted = np.argmax(logits, axis=1).astype(int)
    require(np.array_equal(predicted, np.argmax(scores, axis=1)), f"{model_name}: logit/score decisions differ")
    base = {
        "row_id": [row["row_id"] for row in rows],
        "official_fold": [int(row["official_fold"]) for row in rows],
        "hotel": [row["hotel"] for row in rows],
        "polarity": [row["polarity"] for row in rows],
    }
    if llama:
        require(gpt3_by_seed is not None, "GPT-3 seed index missing")
        base.update({
            "seed_family_id": [row["seed_family_id"] for row in rows],
            "matched_gpt3_row_id": [gpt3_by_seed[row["seed_family_id"]]["row_id"] for row in rows],
            "reference_label_id": [2] * len(rows),
        })
    else:
        base.update({
            "source_family": [row["source_family"] for row in rows],
            "scientific_class": [row["scientific_class"] for row in rows],
            "y_true": [int(row["label_id"]) for row in rows],
        })
    for label in LABEL_ORDER:
        base[f"logit_{label}"] = logits[:, label]
        base[f"score_{label}"] = scores[:, label]
    base["maximum_score"] = scores.max(axis=1)
    base["y_pred"] = predicted
    if llama:
        base["detected_as_ai"] = (predicted == 2).astype(int)
    else:
        base["correct"] = (predicted == np.asarray(base["y_true"], dtype=int)).astype(int)
    base["model_name"] = [model_name] * len(rows)
    base["fold_seed"] = [fold_seed] * len(rows)
    frame = pd.DataFrame(base)
    if llama:
        ordered_columns = [
            "row_id", "seed_family_id", "matched_gpt3_row_id", "official_fold",
            "hotel", "polarity", "reference_label_id", "y_pred", "logit_0",
            "logit_1", "logit_2", "score_0", "score_1", "score_2",
            "maximum_score", "detected_as_ai", "model_name", "fold_seed",
        ]
    else:
        ordered_columns = [
            "row_id", "official_fold", "hotel", "polarity", "source_family",
            "scientific_class", "y_true", "y_pred", "logit_0", "logit_1",
            "logit_2", "score_0", "score_1", "score_2", "maximum_score",
            "correct", "model_name", "fold_seed",
        ]
    require(set(frame.columns) == set(ordered_columns), f"{model_name}: prediction schema differs")
    return frame[ordered_columns]


def classification_metrics(y_true, y_pred, scores=None):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average=None, zero_division=0
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average="macro", zero_division=0
    )
    result = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": (
            float(balanced_accuracy_score(y_true, y_pred))
            if set(y_true.tolist()) == set(LABEL_ORDER)
            else None
        ),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }
    for position, label in enumerate(LABEL_ORDER):
        result[f"class_{label}_precision"] = float(precision[position])
        result[f"class_{label}_recall"] = float(recall[position])
        result[f"class_{label}_f1"] = float(f1[position])
        result[f"class_{label}_support"] = int(support[position])
    if scores is not None:
        try:
            binary = label_binarize(y_true, classes=LABEL_ORDER)
            result["macro_ovr_roc_auc"] = float(
                roc_auc_score(binary, np.asarray(scores), average="macro", multi_class="ovr")
            )
        except ValueError:
            result["macro_ovr_roc_auc"] = None
    return result


def documented_collection_source(row):
    if row["scientific_class"] == "presumed_truthful":
        return "Web" if row["polarity"] == "negative" else "TripAdvisor"
    if row["scientific_class"] == "human_deceptive":
        return "MTurk"
    if row["scientific_class"] == "ai_generated":
        return "op_spam_AI_GPT3"
    raise RuntimeError(f"STOP — unknown scientific class: {row['scientific_class']}")


def sensitivity_summary(frame, group_type, group_value):
    n = len(frame)
    require(n > 0, "empty Llama sensitivity stratum")
    item = {
        "model_name": frame["model_name"].iloc[0],
        "group_type": group_type,
        "group_value": str(group_value),
        "n": n,
        "ai_detection_rate": float(frame["detected_as_ai"].mean()),
        "ai_score_mean": float(frame["score_2"].mean()),
        "ai_score_median": float(frame["score_2"].median()),
        "maximum_score_mean": float(frame["maximum_score"].mean()),
        "maximum_score_median": float(frame["maximum_score"].median()),
    }
    for label in LABEL_ORDER:
        count = int((frame["y_pred"] == label).sum())
        item[f"predicted_{label}_n"] = count
        item[f"predicted_{label}_proportion"] = count / n
    return item


print("=" * WIDTH)
print("PHASE 2B — STEP M3A: FROZEN FIVE-FOLD CLASSICAL BASELINES")
print("=" * WIDTH)
print("Python     :", sys.version.split()[0])
print("Execution  : CPU only; TF-IDF comparator and length-only shortcut diagnostic")
print("CUDA/model : not accessed / DeBERTa not loaded")
require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")
require(WORKING_ROOT.is_dir(), "Kaggle working root is unavailable")
require(not OUTPUT_ROOT.exists(), f"output directory already exists: {OUTPUT_ROOT}")
require(not OUTPUT_ZIP.exists(), f"output ZIP already exists: {OUTPUT_ZIP}")
require(not OUTPUT_SIDECAR.exists(), f"output sidecar already exists: {OUTPUT_SIDECAR}")

section("ENVIRONMENT AND FROZEN INPUT CONTINUITY")
actual_packages = {name: package_version(name) for name in EXPECTED_PACKAGES}
for name, expected in EXPECTED_PACKAGES.items():
    actual = actual_packages[name]
    require(actual == expected, f"{name} version differs: {actual} != {expected}")
    print(f"{name:18s}: {actual:12s} PASS")

t2_manifest_path = find_unique("corpus_manifest.json")
contract_manifest_path = find_unique("contract_manifest.json")
t2_manifest = read_json(t2_manifest_path)
contract_manifest = read_json(contract_manifest_path)
require(t2_manifest.get("manifest_payload_sha256") == EXPECTED_T2_PAYLOAD, "T2 payload differs")
require(file_sha256(contract_manifest_path) == EXPECTED_M2_CONTRACT_FILE_SHA256, "M2 contract file differs")
require(contract_manifest.get("contract_payload_sha256") == EXPECTED_M2_CONTRACT_PAYLOAD, "M2 contract payload differs")

sidecars = sorted(INPUT_ROOT.rglob("phase2B_M2_FROZEN_MODEL_CONTRACT.zip.sha256.txt"))
require(len(sidecars) == 1, f"expected one M2 ZIP sidecar; found {len(sidecars)}")
require(sidecars[0].read_text(encoding="utf-8").split()[0].lower() == EXPECTED_M2_CONTRACT_ZIP_SHA256, "M2 ZIP record differs")

t2_root = t2_manifest_path.parent
primary_path = t2_root / "primary_tripartite_corpus.jsonl"
llama_path = t2_root / "llama_unseen_generator_sensitivity.jsonl"
require(file_sha256(primary_path) == EXPECTED_PRIMARY_SHA256, "primary corpus differs")
require(file_sha256(llama_path) == EXPECTED_LLAMA_SHA256, "Llama corpus differs")
primary_rows = sorted(read_jsonl(primary_path), key=lambda row: row["row_id"])
llama_rows = sorted(read_jsonl(llama_path), key=lambda row: row["row_id"])
require(len(primary_rows) == 2396, "primary row count differs")
require(len(llama_rows) == 790, "Llama row count differs")
require(Counter(row["scientific_class"] for row in primary_rows) == Counter({"presumed_truthful": 796, "human_deceptive": 800, "ai_generated": 800}), "class counts differ")
require(Counter(int(row["official_fold"]) for row in primary_rows) == Counter(EXPECTED_PRIMARY_FOLDS), "primary fold counts differ")
require(Counter(int(row["official_fold"]) for row in llama_rows) == Counter(EXPECTED_LLAMA_FOLDS), "Llama fold counts differ")

gpt3_by_seed = {
    row["seed_family_id"]: row
    for row in primary_rows
    if row["scientific_class"] == "ai_generated"
}
require(len(gpt3_by_seed) == 800, "GPT-3 seed-family index differs")
require(all(row["seed_family_id"] in gpt3_by_seed for row in llama_rows), "unmatched Llama seed family")
for row in llama_rows:
    peer = gpt3_by_seed[row["seed_family_id"]]
    require(peer["hotel"] == row["hotel"], f"hotel mismatch for {row['row_id']}")
    require(int(peer["official_fold"]) == int(row["official_fold"]), f"fold mismatch for {row['row_id']}")
    require(peer["polarity"] == row["polarity"], f"polarity mismatch for {row['row_id']}")

length_counts = Counter(length_bin(word_count(row["text"])) for row in primary_rows)
require(length_counts == Counter({"Q1_shortest": 607, "Q2": 600, "Q3": 595, "Q4_longest": 594}), f"length-bin counts differ: {dict(length_counts)}")
print("T2 payload          :", EXPECTED_T2_PAYLOAD, "PASS")
print("M2 contract payload :", EXPECTED_M2_CONTRACT_PAYLOAD, "PASS")
print("Primary/Llama rows  : 2396 / 790 PASS")
print("Matched seed pairs  : 790 PASS")
print("Length-bin counts   :", dict(length_counts), "PASS")

section("FIVE-FOLD BASELINE EXECUTION")
start_time = time.perf_counter()
temporary_parent = Path(tempfile.mkdtemp(prefix="phase2B_M3A_", dir=WORKING_ROOT))
temporary_root = temporary_parent / OUTPUT_ROOT.name
temporary_root.mkdir()
(temporary_root / "models").mkdir()

primary_predictions = []
llama_predictions = []
fold_metric_rows = []
fold_training_records = []

try:
    for fold in range(1, 6):
        seed = FOLD_SEEDS[fold]
        train_rows = [row for row in primary_rows if int(row["official_fold"]) != fold]
        test_rows = [row for row in primary_rows if int(row["official_fold"]) == fold]
        sensitivity_rows = [row for row in llama_rows if int(row["official_fold"]) == fold]
        train_hotels = {row["hotel"] for row in train_rows}
        test_hotels = {row["hotel"] for row in test_rows}
        require(train_hotels.isdisjoint(test_hotels), f"fold {fold}: hotel leakage")
        require(len(test_rows) == EXPECTED_PRIMARY_FOLDS[fold], f"fold {fold}: test count differs")
        require(len(sensitivity_rows) == EXPECTED_LLAMA_FOLDS[fold], f"fold {fold}: Llama count differs")

        x_train_text = [row["text"] for row in train_rows]
        y_train = np.asarray([row["label_id"] for row in train_rows], dtype=np.int64)
        x_test_text = [row["text"] for row in test_rows]
        y_test = np.asarray([row["label_id"] for row in test_rows], dtype=np.int64)
        x_llama_text = [row["text"] for row in sensitivity_rows]
        fold_model_dir = temporary_root / "models" / f"fold_{fold}"
        fold_model_dir.mkdir()

        fold_start = time.perf_counter()
        vectorizer = TfidfVectorizer(
            lowercase=True,
            strip_accents=None,
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=1.0,
            max_features=50000,
            sublinear_tf=True,
            norm="l2",
            token_pattern=r"(?u)\b\w\w+\b",
            dtype=np.float64,
        )
        train_matrix = vectorizer.fit_transform(x_train_text)
        test_matrix = vectorizer.transform(x_test_text)
        llama_matrix = vectorizer.transform(x_llama_text)
        tfidf_model = LogisticRegression(
            solver="lbfgs", penalty="l2", C=1.0, fit_intercept=True,
            class_weight=None, max_iter=2000, tol=1e-4, random_state=seed,
        )
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always")
            tfidf_model.fit(train_matrix, y_train)
        convergence = [item for item in captured if issubclass(item.category, ConvergenceWarning)]
        require(not convergence, f"fold {fold}: TF-IDF logistic regression did not converge")
        require(np.array_equal(tfidf_model.classes_, LABEL_ORDER), f"fold {fold}: TF-IDF classes differ")
        require(
            np.asarray(tfidf_model.n_iter_).shape == (1,),
            f"fold {fold}: TF-IDF classifier is not operating as one multinomial fit",
        )

        test_logits = reorder_columns(tfidf_model.decision_function(test_matrix), tfidf_model.classes_)
        test_scores = reorder_columns(tfidf_model.predict_proba(test_matrix), tfidf_model.classes_)
        llama_logits = reorder_columns(tfidf_model.decision_function(llama_matrix), tfidf_model.classes_)
        llama_scores = reorder_columns(tfidf_model.predict_proba(llama_matrix), tfidf_model.classes_)
        tfidf_primary = prediction_frame(test_rows, test_logits, test_scores, MODEL_NAMES[0], seed)
        tfidf_llama = prediction_frame(sensitivity_rows, llama_logits, llama_scores, MODEL_NAMES[0], seed, True, gpt3_by_seed)
        primary_predictions.append(tfidf_primary)
        llama_predictions.append(tfidf_llama)
        tfidf_metrics = classification_metrics(y_test, tfidf_primary["y_pred"], test_scores)
        fold_metric_rows.append({"model_name": MODEL_NAMES[0], "test_fold": fold, **tfidf_metrics})

        tfidf_artifact = {
            "vectorizer": vectorizer,
            "classifier": tfidf_model,
            "label_order": LABEL_ORDER,
            "fold": fold,
            "seed": seed,
            "contract_payload_sha256": EXPECTED_M2_CONTRACT_PAYLOAD,
        }
        tfidf_path = fold_model_dir / "tfidf_logistic_regression.joblib"
        joblib.dump(tfidf_artifact, tfidf_path, compress=3, protocol=5)

        train_lengths = np.log1p(np.asarray([word_count(text) for text in x_train_text], dtype=np.float64)).reshape(-1, 1)
        test_lengths = np.log1p(np.asarray([word_count(text) for text in x_test_text], dtype=np.float64)).reshape(-1, 1)
        llama_lengths = np.log1p(np.asarray([word_count(text) for text in x_llama_text], dtype=np.float64)).reshape(-1, 1)
        scaler = StandardScaler()
        scaled_train = scaler.fit_transform(train_lengths)
        scaled_test = scaler.transform(test_lengths)
        scaled_llama = scaler.transform(llama_lengths)
        length_model = LogisticRegression(
            solver="lbfgs", penalty="l2", C=1.0, fit_intercept=True,
            class_weight=None, max_iter=2000, tol=1e-4, random_state=seed,
        )
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always")
            length_model.fit(scaled_train, y_train)
        convergence = [item for item in captured if issubclass(item.category, ConvergenceWarning)]
        require(not convergence, f"fold {fold}: length-only logistic regression did not converge")
        require(np.array_equal(length_model.classes_, LABEL_ORDER), f"fold {fold}: length classes differ")
        require(
            np.asarray(length_model.n_iter_).shape == (1,),
            f"fold {fold}: length-only classifier is not operating as one multinomial fit",
        )

        length_test_logits = reorder_columns(length_model.decision_function(scaled_test), length_model.classes_)
        length_test_scores = reorder_columns(length_model.predict_proba(scaled_test), length_model.classes_)
        length_llama_logits = reorder_columns(length_model.decision_function(scaled_llama), length_model.classes_)
        length_llama_scores = reorder_columns(length_model.predict_proba(scaled_llama), length_model.classes_)
        length_primary = prediction_frame(test_rows, length_test_logits, length_test_scores, MODEL_NAMES[1], seed)
        length_llama = prediction_frame(sensitivity_rows, length_llama_logits, length_llama_scores, MODEL_NAMES[1], seed, True, gpt3_by_seed)
        primary_predictions.append(length_primary)
        llama_predictions.append(length_llama)
        length_metrics = classification_metrics(y_test, length_primary["y_pred"], length_test_scores)
        fold_metric_rows.append({"model_name": MODEL_NAMES[1], "test_fold": fold, **length_metrics})

        length_artifact = {
            "scaler": scaler,
            "classifier": length_model,
            "word_pattern": WORD_PATTERN.pattern,
            "transform": "log1p",
            "label_order": LABEL_ORDER,
            "fold": fold,
            "seed": seed,
            "contract_payload_sha256": EXPECTED_M2_CONTRACT_PAYLOAD,
        }
        length_path = fold_model_dir / "length_only_logistic_regression.joblib"
        joblib.dump(length_artifact, length_path, compress=3, protocol=5)

        record = {
            "test_fold": fold,
            "seed": seed,
            "train_n": len(train_rows),
            "test_n": len(test_rows),
            "llama_n": len(sensitivity_rows),
            "train_hotels": len(train_hotels),
            "test_hotels": len(test_hotels),
            "hotel_overlap": 0,
            "tfidf_vocabulary_size": len(vectorizer.vocabulary_),
            "tfidf_iterations": int(np.max(tfidf_model.n_iter_)),
            "length_iterations": int(np.max(length_model.n_iter_)),
            "tfidf_multinomial_verified": True,
            "length_multinomial_verified": True,
            "tfidf_model_sha256": file_sha256(tfidf_path),
            "length_model_sha256": file_sha256(length_path),
            "elapsed_seconds": time.perf_counter() - fold_start,
        }
        fold_training_records.append(record)
        print(
            f"Fold {fold}: train={len(train_rows)}, test={len(test_rows)}, Llama={len(sensitivity_rows)}, "
            f"vocab={record['tfidf_vocabulary_size']}, TF-IDF macro-F1={tfidf_metrics['macro_f1']:.6f}, "
            f"length macro-F1={length_metrics['macro_f1']:.6f} PASS"
        )

    primary_oof = pd.concat(primary_predictions, ignore_index=True).sort_values(["model_name", "row_id"], kind="stable").reset_index(drop=True)
    llama_oof = pd.concat(llama_predictions, ignore_index=True).sort_values(["model_name", "row_id"], kind="stable").reset_index(drop=True)
    require(len(primary_oof) == 2 * 2396, "primary OOF prediction count differs")
    require(len(llama_oof) == 2 * 790, "Llama prediction count differs")
    for model_name in MODEL_NAMES:
        model_primary = primary_oof[primary_oof["model_name"] == model_name]
        model_llama = llama_oof[llama_oof["model_name"] == model_name]
        require(len(model_primary) == 2396 and model_primary["row_id"].nunique() == 2396, f"{model_name}: primary OOF coverage differs")
        require(len(model_llama) == 790 and model_llama["row_id"].nunique() == 790, f"{model_name}: Llama coverage differs")

    primary_prediction_path = temporary_root / "primary_baseline_oof_predictions.csv"
    llama_prediction_path = temporary_root / "llama_baseline_sensitivity_predictions.csv"
    write_csv(primary_prediction_path, primary_oof)
    write_csv(llama_prediction_path, llama_oof)

    fold_metrics = pd.DataFrame(fold_metric_rows).sort_values(["model_name", "test_fold"], kind="stable")
    write_csv(temporary_root / "baseline_fold_metrics.csv", fold_metrics)

    pooled_metrics = {}
    confusion_matrices = {}
    for model_name in MODEL_NAMES:
        frame = primary_oof[primary_oof["model_name"] == model_name]
        scores = frame[["score_0", "score_1", "score_2"]].to_numpy(dtype=float)
        pooled_metrics[model_name] = classification_metrics(frame["y_true"], frame["y_pred"], scores)
        confusion_matrices[model_name] = confusion_matrix(frame["y_true"], frame["y_pred"], labels=LABEL_ORDER).tolist()
    write_json(temporary_root / "baseline_pooled_metrics.json", pooled_metrics)
    write_json(temporary_root / "baseline_confusion_matrices.json", confusion_matrices)

    primary_metadata = {
        row["row_id"]: {
            "length_bin": length_bin(word_count(row["text"])),
            "source_family": row["source_family"],
            "collection_source": documented_collection_source(row),
        }
        for row in primary_rows
    }
    primary_oof["length_bin"] = primary_oof["row_id"].map(lambda row_id: primary_metadata[row_id]["length_bin"])
    primary_oof["collection_source"] = primary_oof["row_id"].map(
        lambda row_id: primary_metadata[row_id]["collection_source"]
    )
    stratified_rows = []
    for model_name in MODEL_NAMES:
        model_frame = primary_oof[primary_oof["model_name"] == model_name]
        dimensions = [
            ("fold", model_frame["official_fold"]),
            ("hotel", model_frame["hotel"]),
            ("polarity", model_frame["polarity"]),
            ("source_family", model_frame["source_family"]),
            ("collection_source", model_frame["collection_source"]),
            ("length_bin", model_frame["length_bin"]),
        ]
        for dimension, series in dimensions:
            for value in sorted(series.unique(), key=str):
                subset = model_frame[series == value]
                metrics = classification_metrics(subset["y_true"], subset["y_pred"])
                stratified_rows.append({"model_name": model_name, "group_type": dimension, "group_value": str(value), **metrics})
    write_csv(temporary_root / "baseline_stratified_metrics.csv", pd.DataFrame(stratified_rows))

    llama_summary_rows = []
    for model_name in MODEL_NAMES:
        model_frame = llama_oof[llama_oof["model_name"] == model_name]
        llama_summary_rows.append(sensitivity_summary(model_frame, "all", "all"))
        for dimension in ["official_fold", "hotel", "polarity"]:
            for value in sorted(model_frame[dimension].unique(), key=str):
                llama_summary_rows.append(sensitivity_summary(model_frame[model_frame[dimension] == value], dimension, value))
    llama_summary = pd.DataFrame(llama_summary_rows).sort_values(["model_name", "group_type", "group_value"], kind="stable")
    write_csv(temporary_root / "llama_sensitivity_stratified_metrics.csv", llama_summary)
    write_csv(temporary_root / "fold_training_registry.csv", pd.DataFrame(fold_training_records))

    elapsed = time.perf_counter() - start_time
    manifest_payload = {
        "phase": "Phase 2B",
        "step": "M3A",
        "status": "FROZEN_CLASSICAL_BASELINES_COMPLETE",
        "upstream": {
            "t2_manifest_payload_sha256": EXPECTED_T2_PAYLOAD,
            "m2_contract_payload_sha256": EXPECTED_M2_CONTRACT_PAYLOAD,
            "m2_contract_zip_sha256": EXPECTED_M2_CONTRACT_ZIP_SHA256,
        },
        "environment": {"python": sys.version.split()[0], **actual_packages},
        "models": {
            "tfidf_logistic_regression": "frozen substantive comparator",
            "length_only_logistic_regression": "mandatory shortcut diagnostic",
        },
        "evaluation": {
            "outer_folds": 5,
            "routing": "immutable official Ott hotel folds",
            "primary_oof_rows_per_model": 2396,
            "llama_sensitivity_rows_per_model": 790,
            "hotel_overlap": 0,
            "hyperparameter_tuning": "none",
            "class_weighting": "none",
            "resampling": "none",
        },
        "pooled_metrics": pooled_metrics,
        "elapsed_seconds": elapsed,
        "deberta_loaded": False,
        "cuda_accessed": False,
    }
    manifest_payload_hash = canonical_sha256(manifest_payload)
    manifest = {**manifest_payload, "manifest_payload_sha256": manifest_payload_hash}
    write_json(temporary_root / "baseline_manifest.json", manifest)

    checksum_targets = sorted(path for path in temporary_root.rglob("*") if path.is_file())
    checksum_text = "".join(
        f"{file_sha256(path)}  {path.relative_to(temporary_root).as_posix()}\n"
        for path in checksum_targets
    )
    (temporary_root / "SHA256SUMS.txt").write_text(checksum_text, encoding="utf-8", newline="\n")

    temporary_root.replace(OUTPUT_ROOT)
    deterministic_zip(OUTPUT_ROOT, OUTPUT_ZIP)
    zip_sha256 = file_sha256(OUTPUT_ZIP)
    OUTPUT_SIDECAR.write_text(f"{zip_sha256}  {OUTPUT_ZIP.name}\n", encoding="utf-8", newline="\n")
finally:
    if temporary_parent.exists():
        shutil.rmtree(temporary_parent)

section("POOLED OUT-OF-FOLD BASELINE RESULTS")
for model_name in MODEL_NAMES:
    metrics = pooled_metrics[model_name]
    print(
        f"{model_name:34s} n={metrics['n']}  macro-F1={metrics['macro_f1']:.6f}  "
        f"balanced_accuracy={metrics['balanced_accuracy']:.6f}  accuracy={metrics['accuracy']:.6f}  "
        f"macro_OVR_AUC={metrics['macro_ovr_roc_auc']:.6f}"
    )

section("FROZEN M3A ARTIFACT")
for path in sorted(p for p in OUTPUT_ROOT.rglob("*") if p.is_file()):
    print(f"{path.relative_to(OUTPUT_ROOT).as_posix():62s} {path.stat().st_size:10d} bytes  {file_sha256(path)}")
print("Manifest payload SHA256:", read_json(OUTPUT_ROOT / "baseline_manifest.json")["manifest_payload_sha256"])

print("\n" + "=" * WIDTH)
print("STEP M3A DECISION: FROZEN CLASSICAL BASELINES COMPLETE")
print("=" * WIDTH)
print("Output directory :", OUTPUT_ROOT)
print("Artifact ZIP     :", OUTPUT_ZIP)
print("ZIP SHA256       :", file_sha256(OUTPUT_ZIP))
print("Checksum sidecar :", OUTPUT_SIDECAR)
print("DeBERTa loaded   : False")
print("CUDA accessed    : False")
print("GPU quota used   : NONE")
print("Download the ZIP and .sha256.txt, verify externally, and preserve the complete output.")
print("STOP — share this complete output before DeBERTa fold training is authorised.")


PHASE 2B — STEP M3A: FROZEN FIVE-FOLD CLASSICAL BASELINES
Python     : 3.12.13
Execution  : CPU only; TF-IDF comparator and length-only shortcut diagnostic
CUDA/model : not accessed / DeBERTa not loaded

ENVIRONMENT AND FROZEN INPUT CONTINUITY
----------------------------------------------------------------------------------------------------------------------------
numpy             : 2.0.2        PASS
pandas            : 2.3.3        PASS
scikit-learn      : 1.6.1        PASS
joblib            : 1.5.3        PASS
T2 payload          : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8 PASS
M2 contract payload : 53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202 PASS
Primary/Llama rows  : 2396 / 790 PASS
Matched seed pairs  : 790 PASS
Length-bin counts   : {'Q2': 600, 'Q1_shortest': 607, 'Q4_longest': 594, 'Q3': 595} PASS

FIVE-FOLD BASELINE EXECUTION
-----------------------------------------------------------------------------------------------------------

M3A is a clean PASS. All five folds completed with exact OOF coverage and no GPU use.

Key frozen results:

TF-IDF macro-F1: 0.887577
Length-only macro-F1: 0.584314
TF-IDF fold macro-F1 range: 0.863515–0.904263
Manifest payload: 46118cc8794e5568a4cea30a930a17114c493e15cce5dc6b3947c007b1f5845e
ZIP SHA-256: aa064dc70e6af952e4afbf2c15ff9bc661564b74a20c1e636c27b1016c91c460

The large improvement over the length-only diagnostic is encouraging, but interpretation must wait until DeBERTa and statistical comparisons are complete.

## PHASE 2B — STEP M3B: RE-UPLOADED CLASSICAL-BASELINE VERIFICATION

In [4]:
from pathlib import Path
from collections import Counter
from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize


WIDTH = 124
INPUT_ROOT = Path("/kaggle/input")

EXPECTED_T2_PAYLOAD = "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8"
EXPECTED_M2_PAYLOAD = "53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202"
EXPECTED_M3A_PAYLOAD = "46118cc8794e5568a4cea30a930a17114c493e15cce5dc6b3947c007b1f5845e"
EXPECTED_M3A_ZIP = "aa064dc70e6af952e4afbf2c15ff9bc661564b74a20c1e636c27b1016c91c460"
EXPECTED_PRIMARY_SHA256 = "b836eb86f1d34ed2db2c8fce4f1b04e6bec515632403962ba594e5c0adaba9ef"
EXPECTED_LLAMA_SHA256 = "37500d0ae8b48b632c18f9b57c931e2f88e6793e51975c6ef0243ab179bca01a"

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "joblib": "1.5.3",
}
EXPECTED_FILES = {
    "SHA256SUMS.txt": (2045, "8ac5730334d6a5fe196466ea03ce412d83d8c43493acd1b803332a25134c3ddc"),
    "baseline_confusion_matrices.json": (344, "404e407b15c8451c5d10e3055ca07780628fc255025cbbf9d435fcbd1ad4e989"),
    "baseline_fold_metrics.csv": (3625, "6405ec85b54fbeb3419191553aa795c848f80ab85511e2daca126f445433b9e7"),
    "baseline_manifest.json": (2788, "5dc685ccd93dc9a9b6a148f43bf3beec1e2442cbfe45384746857134a15ef03f"),
    "baseline_pooled_metrics.json": (1473, "19898d35990d388847b0d720da0ea334aa3e0036b95a9ba221b261a4e6795a4e"),
    "baseline_stratified_metrics.csv": (22278, "5ab792e07694814b3e4e35d770072638f8dcddfc260898ce094a65dd22245387"),
    "fold_training_registry.csv": (1219, "b5fd60319bd37814ced3dc514d2f5a004f1a8eb3fe17b6dfa955edb105db0d30"),
    "llama_baseline_sensitivity_predictions.csv": (439343, "0d25eb9927203b51bee99914ff499218c06d6c94942c35bc26cb7a4cbc766377"),
    "llama_sensitivity_stratified_metrics.csv": (10549, "770a1afa0df4d383a6a56d7f02749204cca959a0697e851fde6b24a310058a52"),
    "models/fold_1/length_only_logistic_regression.joblib": (958, "7c59702aa8014bcc38fbe7de3e7c2d620028d6ed0ad45c007a926a179e65eb68"),
    "models/fold_1/tfidf_logistic_regression.joblib": (872701, "74a939b42033f0e4bb9c888a8b80db4b934a55f3ce909519c04a06067c310d98"),
    "models/fold_2/length_only_logistic_regression.joblib": (952, "2f6643e45bac6929de507fb1e4063d652ddbe2434f8b2e0a0cc91b42186deeaf"),
    "models/fold_2/tfidf_logistic_regression.joblib": (861197, "287ada3508604ddb7260d49e43d8843df015f5e480455c07e681180acb75fd87"),
    "models/fold_3/length_only_logistic_regression.joblib": (959, "b7379d5ae56e3db56c87e44a34eea2463cbebe21180d594c8efd91eb640045b8"),
    "models/fold_3/tfidf_logistic_regression.joblib": (862167, "e463fccf3c0a80d15025c00d919a8d3c590b0c2bf9faab56462101d9205e2ca6"),
    "models/fold_4/length_only_logistic_regression.joblib": (960, "b7c7dac46b22aa1ec552c32e8f248e5eb51ab0d97de84be1dded0c09b3598ff2"),
    "models/fold_4/tfidf_logistic_regression.joblib": (873164, "6d6920bd3669911cf56cdf1b15abf07aaaad327a944ec8f09cc537bca3f6389c"),
    "models/fold_5/length_only_logistic_regression.joblib": (957, "ca5c3ea9450ddc49e78d40ae0238331f14a492e8db2b7db2c3fc6eb1c83c8fa6"),
    "models/fold_5/tfidf_logistic_regression.joblib": (873844, "e3e681b431e863ac64666bf74196f5660703cf32ebd2c2c8002d37a56c96b0fc"),
    "primary_baseline_oof_predictions.csv": (1258002, "6d7d8672254d57a3980eee0196dfc1bf6547549c26ed8ae55d15ac7b48171013"),
}

LABELS = [0, 1, 2]
MODELS = ["tfidf_logistic_regression", "length_only_logistic_regression"]
FOLD_SEEDS = {1: 43, 2: 44, 3: 45, 4: 46, 5: 47}
EXPECTED_PRIMARY_FOLDS = {1: 479, 2: 479, 3: 478, 4: 480, 5: 480}
EXPECTED_LLAMA_FOLDS = {1: 160, 2: 158, 3: 158, 4: 157, 5: 157}
EXPECTED_ROUNDED_METRICS = {
    "tfidf_logistic_regression": {
        "macro_f1": 0.887577,
        "balanced_accuracy": 0.887726,
        "accuracy": 0.887730,
        "macro_ovr_roc_auc": 0.972979,
    },
    "length_only_logistic_regression": {
        "macro_f1": 0.584314,
        "balanced_accuracy": 0.606589,
        "accuracy": 0.606845,
        "macro_ovr_roc_auc": 0.778873,
    },
}
EXPECTED_FOLD_MACRO_F1 = {
    "tfidf_logistic_regression": [0.863515, 0.870386, 0.897251, 0.902027, 0.904263],
    "length_only_logistic_regression": [0.583722, 0.590557, 0.597139, 0.569785, 0.577255],
}


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8", errors="strict"))


def read_jsonl(path):
    with path.open("r", encoding="utf-8", errors="strict") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def find_unique(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    require(len(matches) == 1, f"expected one {filename}; found {len(matches)}")
    return matches[0]


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "MISSING"


def close(a, b):
    if a is None or b is None:
        return a is None and b is None
    return bool(np.isclose(float(a), float(b), rtol=1e-12, atol=1e-15))


def recompute_metrics(frame):
    y_true = frame["y_true"].to_numpy(dtype=int)
    y_pred = frame["y_pred"].to_numpy(dtype=int)
    scores = frame[["score_0", "score_1", "score_2"]].to_numpy(dtype=float)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, average=None, zero_division=0
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, average="macro", zero_division=0
    )
    binary = label_binarize(y_true, classes=LABELS)
    result = {
        "n": int(len(frame)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "macro_ovr_roc_auc": float(roc_auc_score(binary, scores, average="macro", multi_class="ovr")),
    }
    for position, label in enumerate(LABELS):
        result[f"class_{label}_precision"] = float(precision[position])
        result[f"class_{label}_recall"] = float(recall[position])
        result[f"class_{label}_f1"] = float(f1[position])
        result[f"class_{label}_support"] = int(support[position])
    return result


print("=" * WIDTH)
print("PHASE 2B — STEP M3B: RE-UPLOADED CLASSICAL-BASELINE VERIFICATION")
print("=" * WIDTH)
print("Python     :", sys.version.split()[0])
print("Input root :", INPUT_ROOT)
print("Execution  : CPU read-only; no fitting, DeBERTa loading, CUDA access or inference")
require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")

section("ENVIRONMENT AND UPSTREAM CONTINUITY")
for name, expected in EXPECTED_PACKAGES.items():
    actual = package_version(name)
    require(actual == expected, f"{name} version differs: {actual} != {expected}")
    print(f"{name:18s}: {actual:12s} PASS")

t2_manifest_path = find_unique("corpus_manifest.json")
m2_manifest_path = find_unique("contract_manifest.json")
require(read_json(t2_manifest_path).get("manifest_payload_sha256") == EXPECTED_T2_PAYLOAD, "T2 payload differs")
require(read_json(m2_manifest_path).get("contract_payload_sha256") == EXPECTED_M2_PAYLOAD, "M2 payload differs")
print("T2 payload :", EXPECTED_T2_PAYLOAD, "PASS")
print("M2 payload :", EXPECTED_M2_PAYLOAD, "PASS")

section("M3A INPUT DISCOVERY AND EXTERNAL ZIP RECORD")
m3a_manifest_path = find_unique("baseline_manifest.json")
m3a_root = m3a_manifest_path.parent
m3a_dataset_root = m3a_root.parent
sidecar_candidates = sorted(
    path
    for path in m3a_dataset_root.glob(
        "phase2B_M3A_FROZEN_CLASSICAL_BASELINES.zip.sha256*.txt"
    )
    if path.is_file()
)
valid_sidecars = []
for candidate in sidecar_candidates:
    tokens = candidate.read_text(
        encoding="utf-8", errors="strict"
    ).strip().split()
    if tokens and tokens[0].lower() == EXPECTED_M3A_ZIP:
        valid_sidecars.append(candidate)
require(
    len(valid_sidecars) == 1,
    "expected exactly one M3A checksum sidecar containing the frozen ZIP "
    f"SHA-256; found {len(valid_sidecars)} valid among "
    f"{len(sidecar_candidates)} filename candidates",
)
sidecars = valid_sidecars
mounted_zips = sorted(INPUT_ROOT.rglob("phase2B_M3A_FROZEN_CLASSICAL_BASELINES.zip"))
require(len(mounted_zips) <= 1, "multiple mounted M3A ZIPs found")
if mounted_zips:
    require(file_sha256(mounted_zips[0]) == EXPECTED_M3A_ZIP, "mounted M3A ZIP differs")
    zip_status = "BYTE_VERIFIED_IN_RUNTIME"
else:
    zip_status = "Kaggle-extracted; externally verified ZIP recorded"
print("M3A root             :", m3a_root)
print("Checksum sidecar     :", sidecars[0])
print("Recorded ZIP SHA256  :", EXPECTED_M3A_ZIP)
print("ZIP runtime status   :", zip_status)

section("COMPLETE 20-FILE ARTIFACT REGISTRY")
actual_files = {
    path.relative_to(m3a_root).as_posix()
    for path in m3a_root.rglob("*")
    if path.is_file()
}
require(actual_files == set(EXPECTED_FILES), f"M3A file set differs: {sorted(actual_files)}")
for relative_name, (expected_size, expected_hash) in EXPECTED_FILES.items():
    path = m3a_root / relative_name
    require(path.stat().st_size == expected_size, f"size mismatch: {relative_name}")
    require(file_sha256(path) == expected_hash, f"checksum mismatch: {relative_name}")
    print(f"{relative_name:64s} {expected_size:10d} bytes  PASS")

section("INTERNAL SHA-256 REGISTRY")
registry = {}
for line_number, line in enumerate((m3a_root / "SHA256SUMS.txt").read_text(encoding="utf-8").splitlines(), start=1):
    parts = line.split(None, 1)
    require(len(parts) == 2, f"malformed registry line {line_number}")
    registry[parts[1].strip()] = parts[0].lower()
expected_registry = {name: item[1] for name, item in EXPECTED_FILES.items() if name != "SHA256SUMS.txt"}
require(registry == expected_registry, "M3A internal registry differs")
print("Registry entries      :", len(registry))
print("Registry verification : PASS")

section("M3A MANIFEST PAYLOAD")
manifest = read_json(m3a_manifest_path)
stored_payload = manifest.get("manifest_payload_sha256")
payload = {key: value for key, value in manifest.items() if key != "manifest_payload_sha256"}
recomputed_payload = canonical_sha256(payload)
require(stored_payload == EXPECTED_M3A_PAYLOAD, "stored M3A payload differs")
require(recomputed_payload == EXPECTED_M3A_PAYLOAD, "recomputed M3A payload differs")
require(payload.get("status") == "FROZEN_CLASSICAL_BASELINES_COMPLETE", "M3A status differs")
require(payload["upstream"]["t2_manifest_payload_sha256"] == EXPECTED_T2_PAYLOAD, "M3A T2 link differs")
require(payload["upstream"]["m2_contract_payload_sha256"] == EXPECTED_M2_PAYLOAD, "M3A M2 link differs")
require(payload["upstream"]["m2_contract_zip_sha256"] == "e1fa927556c28e864eb59d8b80be54a31e39558500a229645481b83cb2967c9a", "M3A M2 ZIP link differs")
require(payload["evaluation"]["hotel_overlap"] == 0, "M3A hotel-overlap record differs")
require(payload["evaluation"]["hyperparameter_tuning"] == "none", "M3A tuning record differs")
require(payload["evaluation"]["resampling"] == "none", "M3A resampling record differs")
require(payload["deberta_loaded"] is False and payload["cuda_accessed"] is False, "M3A execution record differs")
print("Stored payload       :", stored_payload)
print("Recomputed payload   :", recomputed_payload)
print("Upstream chain       : PASS")

section("ROW-LEVEL OOF AND LLAMA ROUTING VERIFICATION")
t2_root = t2_manifest_path.parent
primary_path = t2_root / "primary_tripartite_corpus.jsonl"
llama_path = t2_root / "llama_unseen_generator_sensitivity.jsonl"
require(file_sha256(primary_path) == EXPECTED_PRIMARY_SHA256, "primary corpus file differs")
require(file_sha256(llama_path) == EXPECTED_LLAMA_SHA256, "Llama corpus file differs")
primary_rows = read_jsonl(primary_path)
llama_rows = read_jsonl(llama_path)
primary_index = {row["row_id"]: row for row in primary_rows}
llama_index = {row["row_id"]: row for row in llama_rows}
gpt3_by_seed = {
    row["seed_family_id"]: row
    for row in primary_rows
    if row["scientific_class"] == "ai_generated"
}

primary_predictions = pd.read_csv(m3a_root / "primary_baseline_oof_predictions.csv")
llama_predictions = pd.read_csv(m3a_root / "llama_baseline_sensitivity_predictions.csv")
require("text" not in primary_predictions.columns and "text" not in llama_predictions.columns, "review text leaked into prediction files")
require(len(primary_predictions) == 4792, "primary prediction count differs")
require(len(llama_predictions) == 1580, "Llama prediction count differs")

for model_name in MODELS:
    primary_model = primary_predictions[primary_predictions["model_name"] == model_name].copy()
    llama_model = llama_predictions[llama_predictions["model_name"] == model_name].copy()
    require(len(primary_model) == 2396 and primary_model["row_id"].nunique() == 2396, f"{model_name}: primary coverage differs")
    require(set(primary_model["row_id"]) == set(primary_index), f"{model_name}: primary row-ID set differs")
    require(len(llama_model) == 790 and llama_model["row_id"].nunique() == 790, f"{model_name}: Llama coverage differs")
    require(set(llama_model["row_id"]) == set(llama_index), f"{model_name}: Llama row-ID set differs")

    for frame, is_llama in ((primary_model, False), (llama_model, True)):
        logits = frame[["logit_0", "logit_1", "logit_2"]].to_numpy(dtype=float)
        scores = frame[["score_0", "score_1", "score_2"]].to_numpy(dtype=float)
        require(np.isfinite(logits).all() and np.isfinite(scores).all(), f"{model_name}: non-finite output")
        require(np.allclose(scores.sum(axis=1), 1.0, rtol=0.0, atol=2e-15), f"{model_name}: score sums differ")
        require(np.array_equal(np.argmax(logits, axis=1), frame["y_pred"].to_numpy(dtype=int)), f"{model_name}: logit decisions differ")
        require(np.array_equal(np.argmax(scores, axis=1), frame["y_pred"].to_numpy(dtype=int)), f"{model_name}: score decisions differ")
        require(all(int(seed) == FOLD_SEEDS[int(fold)] for seed, fold in zip(frame["fold_seed"], frame["official_fold"])), f"{model_name}: fold seeds differ")
        if is_llama:
            require((frame["reference_label_id"] == 2).all(), f"{model_name}: Llama labels differ")
            require(np.array_equal((frame["y_pred"] == 2).astype(int), frame["detected_as_ai"].astype(int)), f"{model_name}: Llama detection flags differ")
            for row in frame.itertuples(index=False):
                source = llama_index[row.row_id]
                require(int(row.official_fold) == int(source["official_fold"]), f"{model_name}: Llama fold routing differs")
                require(row.hotel == source["hotel"] and row.polarity == source["polarity"], f"{model_name}: Llama metadata differs")
                require(row.matched_gpt3_row_id == gpt3_by_seed[row.seed_family_id]["row_id"], f"{model_name}: seed match differs")
        else:
            for row in frame.itertuples(index=False):
                source = primary_index[row.row_id]
                require(int(row.y_true) == int(source["label_id"]), f"{model_name}: true label differs")
                require(int(row.official_fold) == int(source["official_fold"]), f"{model_name}: primary fold routing differs")
                require(row.hotel == source["hotel"] and row.polarity == source["polarity"], f"{model_name}: primary metadata differs")

print("Primary predictions  : 4,792 = 2 models × 2,396 rows PASS")
print("Llama predictions    : 1,580 = 2 models × 790 rows PASS")
print("OOF row-ID coverage  : complete and unique per model PASS")
print("Hotel/fold routing   : PASS")
print("Score/decision checks: PASS")

section("INDEPENDENT METRIC RECOMPUTATION")
stored_pooled = read_json(m3a_root / "baseline_pooled_metrics.json")
stored_confusions = read_json(m3a_root / "baseline_confusion_matrices.json")
fold_metrics = pd.read_csv(m3a_root / "baseline_fold_metrics.csv")
for model_name in MODELS:
    frame = primary_predictions[primary_predictions["model_name"] == model_name]
    recomputed = recompute_metrics(frame)
    require(set(recomputed) == set(stored_pooled[model_name]), f"{model_name}: metric field set differs")
    for key, value in recomputed.items():
        require(close(value, stored_pooled[model_name][key]), f"{model_name}: metric differs: {key}")
    matrix = confusion_matrix(frame["y_true"], frame["y_pred"], labels=LABELS).tolist()
    require(matrix == stored_confusions[model_name], f"{model_name}: confusion matrix differs")
    for key, expected in EXPECTED_ROUNDED_METRICS[model_name].items():
        require(round(recomputed[key], 6) == expected, f"{model_name}: reported {key} differs")
    observed_fold_f1 = [
        round(float(value), 6)
        for value in fold_metrics.loc[fold_metrics["model_name"] == model_name].sort_values("test_fold")["macro_f1"]
    ]
    require(observed_fold_f1 == EXPECTED_FOLD_MACRO_F1[model_name], f"{model_name}: fold macro-F1 values differ")
    print(
        f"{model_name:34s} macro-F1={recomputed['macro_f1']:.6f}  "
        f"balanced_accuracy={recomputed['balanced_accuracy']:.6f}  "
        f"accuracy={recomputed['accuracy']:.6f}  AUC={recomputed['macro_ovr_roc_auc']:.6f} PASS"
    )

section("FITTED-MODEL CONTRACT VERIFICATION")
for fold in range(1, 6):
    tfidf_artifact = joblib.load(m3a_root / f"models/fold_{fold}/tfidf_logistic_regression.joblib")
    length_artifact = joblib.load(m3a_root / f"models/fold_{fold}/length_only_logistic_regression.joblib")
    require(tfidf_artifact["fold"] == fold and tfidf_artifact["seed"] == FOLD_SEEDS[fold], f"fold {fold}: TF-IDF identity differs")
    require(length_artifact["fold"] == fold and length_artifact["seed"] == FOLD_SEEDS[fold], f"fold {fold}: length identity differs")
    require(tfidf_artifact["contract_payload_sha256"] == EXPECTED_M2_PAYLOAD, f"fold {fold}: TF-IDF contract link differs")
    require(length_artifact["contract_payload_sha256"] == EXPECTED_M2_PAYLOAD, f"fold {fold}: length contract link differs")

    vectorizer = tfidf_artifact["vectorizer"]
    tfidf_model = tfidf_artifact["classifier"]
    length_model = length_artifact["classifier"]
    params = vectorizer.get_params()
    require(params["strip_accents"] is None, f"fold {fold}: strip_accents differs")
    require(params["ngram_range"] == (1, 2), f"fold {fold}: ngram range differs")
    require(params["min_df"] == 2 and params["max_features"] == 50000, f"fold {fold}: vocabulary policy differs")
    require(params["sublinear_tf"] is True and params["token_pattern"] == r"(?u)\b\w\w+\b", f"fold {fold}: TF-IDF text policy differs")
    for classifier, label in ((tfidf_model, "TF-IDF"), (length_model, "length")):
        classifier_params = classifier.get_params()
        require(classifier_params["solver"] == "lbfgs", f"fold {fold}: {label} solver differs")
        require(classifier_params["C"] == 1.0 and classifier_params["max_iter"] == 2000, f"fold {fold}: {label} optimisation differs")
        require(classifier_params["class_weight"] is None, f"fold {fold}: {label} class weighting differs")
        require(classifier_params["random_state"] == FOLD_SEEDS[fold], f"fold {fold}: {label} seed differs")
        require(np.array_equal(classifier.classes_, LABELS), f"fold {fold}: {label} classes differ")
        require(np.asarray(classifier.n_iter_).shape == (1,), f"fold {fold}: {label} is not one multinomial fit")
    require(length_artifact["word_pattern"] == r"(?u)\w+" and length_artifact["transform"] == "log1p", f"fold {fold}: length feature differs")
    print(f"Fold {fold}: TF-IDF and length artifacts match the frozen contract PASS")

print("\n" + "=" * WIDTH)
print("STEP M3B DECISION: PASS — RE-UPLOADED CLASSICAL BASELINES VERIFIED")
print("=" * WIDTH)
print("M3A manifest payload :", EXPECTED_M3A_PAYLOAD)
print("M3A recorded ZIP     :", EXPECTED_M3A_ZIP)
print("Files verified       : 20 / 20")
print("Fitted models checked: 10 / 10")
print("DeBERTa loaded       : False")
print("CUDA accessed        : False")
print("Training/inference   : NONE")
print("No file was written, extracted, transformed or modified.")
print("STOP — preserve and share this complete output before DeBERTa fold training is authorised.")


PHASE 2B — STEP M3B: RE-UPLOADED CLASSICAL-BASELINE VERIFICATION
Python     : 3.12.13
Input root : /kaggle/input
Execution  : CPU read-only; no fitting, DeBERTa loading, CUDA access or inference

ENVIRONMENT AND UPSTREAM CONTINUITY
----------------------------------------------------------------------------------------------------------------------------
numpy             : 2.0.2        PASS
pandas            : 2.3.3        PASS
scikit-learn      : 1.6.1        PASS
joblib            : 1.5.3        PASS
T2 payload : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8 PASS
M2 payload : 53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202 PASS

M3A INPUT DISCOVERY AND EXTERNAL ZIP RECORD
----------------------------------------------------------------------------------------------------------------------------
M3A root             : /kaggle/input/datasets/mdshahrirhabib/phase2b-m3a-frozen-classical-baselines-20260910/phase2B_M3A_FROZEN_CLASSICAL_BASELINES (1)
Che

In [3]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

manifests = sorted(INPUT_ROOT.rglob("baseline_manifest.json"))

print("M3A manifests found:", len(manifests))

for manifest in manifests:
    print("\nManifest:", manifest)

    dataset_root = manifest.parent.parent
    print("Probable dataset root:", dataset_root)
    print("\nFiles in this dataset:")

    for path in sorted(dataset_root.rglob("*")):
        if path.is_file():
            print(
                path.relative_to(dataset_root),
                f"| {path.stat().st_size:,} bytes"
            )

print("\nAll checksum-like files under /kaggle/input:")

for path in sorted(INPUT_ROOT.rglob("*sha256*")):
    if path.is_file():
        print(path, f"| {path.stat().st_size:,} bytes")

M3A manifests found: 1

Manifest: /kaggle/input/datasets/mdshahrirhabib/phase2b-m3a-frozen-classical-baselines-20260910/phase2B_M3A_FROZEN_CLASSICAL_BASELINES (1)/baseline_manifest.json
Probable dataset root: /kaggle/input/datasets/mdshahrirhabib/phase2b-m3a-frozen-classical-baselines-20260910

Files in this dataset:
phase2B_M3A_FROZEN_CLASSICAL_BASELINES (1)/SHA256SUMS.txt | 2,045 bytes
phase2B_M3A_FROZEN_CLASSICAL_BASELINES (1)/baseline_confusion_matrices.json | 344 bytes
phase2B_M3A_FROZEN_CLASSICAL_BASELINES (1)/baseline_fold_metrics.csv | 3,625 bytes
phase2B_M3A_FROZEN_CLASSICAL_BASELINES (1)/baseline_manifest.json | 2,788 bytes
phase2B_M3A_FROZEN_CLASSICAL_BASELINES (1)/baseline_pooled_metrics.json | 1,473 bytes
phase2B_M3A_FROZEN_CLASSICAL_BASELINES (1)/baseline_stratified_metrics.csv | 22,278 bytes
phase2B_M3A_FROZEN_CLASSICAL_BASELINES (1)/fold_training_registry.csv | 1,219 bytes
phase2B_M3A_FROZEN_CLASSICAL_BASELINES (1)/llama_baseline_sensitivity_predictions.csv | 439,343 by

## PHASE 2B — STEP M4A: DEBERTA FOLD-TRAINING GPU PREFLIGHT

In [1]:
from pathlib import Path
from importlib.metadata import version as package_version
import gc
import hashlib
import json
import math
import os
import random
import sys
import time


# This must be set before the first CUDA BLAS operation in the kernel.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import transformers
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


WIDTH = 124
INPUT_ROOT = Path("/kaggle/input")

EXPECTED = {
    "t2_payload": "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8",
    "base_payload": "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c",
    "initialisation_payload": "171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45",
    "initial_parameter": "7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b",
    "initial_model_file": "4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c",
    "pretokenizer": "9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171",
    "token_ids": "9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359",
    "m2_payload": "53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202",
    "m3a_payload": "46118cc8794e5568a4cea30a930a17114c493e15cce5dc6b3947c007b1f5845e",
    "m2a_environment": "3e07e214b297576db434188e6c55e576fdd4af362e639f31f7654620d9ecbc7a",
}

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "transformers": "5.0.0",
    "tokenizers": "0.22.2",
    "accelerate": "1.13.0",
    "safetensors": "0.7.0",
    "sentencepiece": "0.2.1",
}

TARGET_FOLD = 1
FOLD_SEED = 43
MAX_LENGTH = 256
PHYSICAL_BATCH_SIZE = 2
ACCUMULATION_MICROBATCHES = 16
PREFLIGHT_EXAMPLES = PHYSICAL_BATCH_SIZE * ACCUMULATION_MICROBATCHES
EPOCHS = 4
TOTAL_UPDATES = 240
WARMUP_UPDATES = 24

LABEL2ID = {
    "presumed_truthful": 0,
    "human_deceptive": 1,
    "ai_generated": 2,
}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8", errors="strict"))


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    encoded = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def verify_payload(path, field, expected):
    document = read_json(path)
    stored = document.get(field)
    payload = {key: value for key, value in document.items() if key != field}
    require(stored == expected, f"{path.name}: stored payload differs")
    require(canonical_sha256(payload) == expected, f"{path.name}: recomputed payload differs")
    return document


def find_unique(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    require(len(matches) == 1, f"expected one {filename}; found {len(matches)}")
    return matches[0]


def parameter_sha256(model):
    digest = hashlib.sha256()
    state = model.state_dict()
    for name in sorted(state):
        tensor = state[name].detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(tensor.dtype).encode("ascii"))
        digest.update(b"\0")
        digest.update(json.dumps(list(tensor.shape), separators=(",", ":")).encode("ascii"))
        digest.update(b"\0")
        digest.update(tensor.numpy().tobytes(order="C"))
        digest.update(b"\0")
    return digest.hexdigest()


def tokenizer_backend_identity(tokenizer):
    backend = json.loads(tokenizer.backend_tokenizer.to_str())
    pretokenizer = backend.get("pre_tokenizer")
    require(pretokenizer is not None, "tokenizer backend has no pre-tokenizer")
    encoded = json.dumps(
        pretokenizer,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    require("Metaspace" in encoded, "native DeBERTa Metaspace pre-tokenizer is absent")
    require('"Split"' not in encoded, "Mistral Split regex was applied")
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()


def load_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8", errors="strict") as handle:
        for line_number, line in enumerate(handle, start=1):
            row = json.loads(line)
            require(isinstance(row.get("row_id"), str), f"{path.name}:{line_number} missing row_id")
            require(isinstance(row.get("text"), str), f"{path.name}:{line_number} missing text")
            require(row["text"].strip() != "", f"{row['row_id']} has empty text")
            rows.append(row)
    return rows


def token_id_fingerprint(tokenizer, ordered_rows):
    digest = hashlib.sha256()
    for row in ordered_rows:
        token_ids = tokenizer(
            row["text"],
            add_special_tokens=True,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            verbose=False,
        )["input_ids"]
        digest.update(row["row_id"].encode("utf-8"))
        digest.update(b"\0")
        digest.update(json.dumps(token_ids, separators=(",", ":")).encode("ascii"))
        digest.update(b"\n")
    return digest.hexdigest()


def all_finite_parameters(model):
    return all(
        bool(torch.isfinite(parameter).all().item())
        for parameter in model.parameters()
        if parameter.is_floating_point()
    )


print("=" * WIDTH)
print("PHASE 2B — STEP M4A: DEBERTA FOLD-TRAINING GPU PREFLIGHT")
print("=" * WIDTH)
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Scope        : one discarded train-only accumulation/optimizer smoke step; no evaluation or prediction")

require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")

section("ENVIRONMENT AND GPU COMPATIBILITY")
for name, expected in EXPECTED_PACKAGES.items():
    actual = package_version(name)
    require(actual == expected, f"{name} version differs: {actual} != {expected}")
    print(f"{name:18s}: {actual:12s} PASS")

require(torch.__version__ == "2.10.0+cu128", f"unexpected PyTorch build: {torch.__version__}")
require(torch.version.cuda == "12.8", f"unexpected PyTorch CUDA build: {torch.version.cuda}")
require(torch.backends.cudnn.version() == 91002, f"unexpected cuDNN version: {torch.backends.cudnn.version()}")
require(torch.cuda.is_available(), "CUDA is unavailable")
require(torch.cuda.device_count() == 2, f"expected the audited T4 x2 runtime; found {torch.cuda.device_count()} GPUs")
require(
    all(torch.cuda.get_device_name(index) == "Tesla T4" for index in range(2)),
    "GPU identity differs from the audited Tesla T4 x2 runtime",
)
torch.cuda.set_device(0)
DEVICE = torch.device("cuda:0")
capability = torch.cuda.get_device_capability(0)
require(capability == (7, 5), f"unexpected selected-GPU capability: {capability}")
arch = f"sm_{capability[0]}{capability[1]}"
require(arch in torch.cuda.get_arch_list(), f"selected GPU architecture {arch} is unsupported")
require(os.environ.get("CUBLAS_WORKSPACE_CONFIG") == ":4096:8", "CUBLAS workspace differs")

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision("highest")
torch.use_deterministic_algorithms(True)

print("Selected device      :", DEVICE)
print("GPU name             :", torch.cuda.get_device_name(0))
print("GPU capability       :", capability)
print("Compiled arches      :", torch.cuda.get_arch_list())
print("Visible GPU count    :", torch.cuda.device_count())
print("Multi-GPU use        : prohibited; only cuda:0 selected")
print("Numeric policy       : FP32; TF32/autocast/GradScaler disabled")
print("Deterministic kernels:", torch.are_deterministic_algorithms_enabled())

section("FROZEN ARTIFACT CONTINUITY")
t2_path = find_unique("corpus_manifest.json")
base_path = find_unique("checkpoint_manifest.json")
init_path = find_unique("initialisation_manifest.json")
m2_path = find_unique("contract_manifest.json")
m3a_path = find_unique("baseline_manifest.json")

t2_manifest = verify_payload(t2_path, "manifest_payload_sha256", EXPECTED["t2_payload"])
base_manifest = verify_payload(base_path, "manifest_payload_sha256", EXPECTED["base_payload"])
init_manifest = verify_payload(
    init_path,
    "manifest_payload_sha256",
    EXPECTED["initialisation_payload"],
)
m2_manifest = verify_payload(m2_path, "contract_payload_sha256", EXPECTED["m2_payload"])
m3a_manifest = verify_payload(m3a_path, "manifest_payload_sha256", EXPECTED["m3a_payload"])

require(
    init_manifest.get("upstream", {}).get("t2_manifest_payload_sha256") == EXPECTED["t2_payload"],
    "initialisation artifact does not link frozen T2",
)
require(
    init_manifest.get("upstream", {}).get("base_checkpoint_manifest_payload_sha256")
    == EXPECTED["base_payload"],
    "initialisation artifact does not link the pinned base",
)
require(
    m2_manifest.get("upstream", {}).get("t2_manifest_payload_sha256") == EXPECTED["t2_payload"],
    "M2 contract does not link frozen T2",
)
require(
    m2_manifest.get("upstream", {}).get("base_checkpoint_manifest_payload_sha256")
    == EXPECTED["base_payload"],
    "M2 contract does not link the pinned base",
)
require(
    m2_manifest.get("upstream", {}).get("initialisation_manifest_payload_sha256")
    == EXPECTED["initialisation_payload"],
    "M2 contract does not link the frozen initialisation",
)
require(
    m2_manifest.get("upstream", {}).get("m2a_gpu_environment_payload_sha256")
    == EXPECTED["m2a_environment"],
    "M2 contract does not link the accepted M2A environment",
)
require(
    m3a_manifest.get("upstream", {}).get("m2_contract_payload_sha256") == EXPECTED["m2_payload"],
    "M3A does not link the frozen M2 contract",
)

T2_ROOT = t2_path.parent
MODEL_ROOT = init_path.parent / "model"
model_file = MODEL_ROOT / "model.safetensors"
require(model_file.is_file(), "initialised model.safetensors is missing")
require(file_sha256(model_file) == EXPECTED["initial_model_file"], "initialised model file differs")

print("T2 payload             :", EXPECTED["t2_payload"], "PASS")
print("Base checkpoint payload:", EXPECTED["base_payload"], "PASS")
print("Initialisation payload :", EXPECTED["initialisation_payload"], "PASS")
print("M2 contract payload    :", EXPECTED["m2_payload"], "PASS")
print("M3A baseline payload   :", EXPECTED["m3a_payload"], "PASS")
print("M2A environment link   :", EXPECTED["m2a_environment"], "PASS")
print("Initial model file     :", EXPECTED["initial_model_file"], "PASS")

section("FROZEN CORPUS AND FOLD-1 ROUTING")
primary_rows_file_order = load_jsonl(T2_ROOT / "primary_tripartite_corpus.jsonl")
llama_rows_file_order = load_jsonl(T2_ROOT / "llama_unseen_generator_sensitivity.jsonl")
require(len(primary_rows_file_order) == 2396, "primary row count differs")
require(len(llama_rows_file_order) == 790, "Llama row count differs")
require(
    len({row["row_id"] for row in primary_rows_file_order + llama_rows_file_order}) == 3186,
    "eligible row IDs are not globally unique",
)

primary_rows = sorted(primary_rows_file_order, key=lambda row: row["row_id"])
train_rows = [row for row in primary_rows if int(row["official_fold"]) != TARGET_FOLD]
test_rows = [row for row in primary_rows if int(row["official_fold"]) == TARGET_FOLD]
llama_test_rows = [row for row in llama_rows_file_order if int(row["official_fold"]) == TARGET_FOLD]

require(len(train_rows) == 1917, f"Fold-1 train count differs: {len(train_rows)}")
require(len(test_rows) == 479, f"Fold-1 test count differs: {len(test_rows)}")
require(len(llama_test_rows) == 160, f"Fold-1 Llama count differs: {len(llama_test_rows)}")
require([row["row_id"] for row in train_rows] == sorted(row["row_id"] for row in train_rows), "train order is not canonical")
train_hotels = {row["hotel"] for row in train_rows}
test_hotels = {row["hotel"] for row in test_rows}
require(not (train_hotels & test_hotels), "Fold-1 train/test hotel overlap detected")
require(all("Llama" not in row["row_id"] for row in train_rows), "Llama row entered training")

print("Primary train/test     :", len(train_rows), "/", len(test_rows), "PASS")
print("Llama sensitivity test :", len(llama_test_rows), "PASS")
print("Train/test hotel overlap:", len(train_hotels & test_hotels), "PASS")
print("Test/Llama model use   : NONE in this preflight")

section("TOKENIZER AND INITIAL PARAMETER VERIFICATION")
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_ROOT), local_files_only=True)
pretokenizer_hash = tokenizer_backend_identity(tokenizer)
require(pretokenizer_hash == EXPECTED["pretokenizer"], "pre-tokenizer fingerprint differs")
eligible_rows_file_order = primary_rows_file_order + llama_rows_file_order
token_hash = token_id_fingerprint(tokenizer, eligible_rows_file_order)
require(token_hash == EXPECTED["token_ids"], "3,186-row token-ID fingerprint differs")

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_ROOT),
    local_files_only=True,
)
model.to(device="cpu", dtype=torch.float32)
model.eval()
require(model.config.num_labels == 3, "model output dimension differs")
require(model.config.label2id == LABEL2ID, "model label2id mapping differs")
require({int(key): value for key, value in model.config.id2label.items()} == ID2LABEL, "model id2label mapping differs")
initial_parameter_hash = parameter_sha256(model)
require(initial_parameter_hash == EXPECTED["initial_parameter"], "initial parameter fingerprint differs")

print("Tokenizer class        :", tokenizer.__class__.__name__)
print("Pre-tokenizer SHA256   :", pretokenizer_hash, "PASS")
print("Token-ID SHA256        :", token_hash, "PASS")
print("Initial parameter SHA256:", initial_parameter_hash, "PASS")

section("DISCARDED TRAIN-ONLY FORWARD/BACKWARD/OPTIMIZER SMOKE STEP")
random.seed(FOLD_SEED)
np.random.seed(FOLD_SEED)
torch.manual_seed(FOLD_SEED)
torch.cuda.manual_seed_all(FOLD_SEED)

epoch_zero_order = np.random.default_rng(FOLD_SEED).permutation(len(train_rows))
smoke_indices = epoch_zero_order[:PREFLIGHT_EXAMPLES]
smoke_rows = [train_rows[int(index)] for index in smoke_indices]
order_payload = "\n".join(row["row_id"] for row in smoke_rows).encode("utf-8")
order_hash = hashlib.sha256(order_payload).hexdigest()

model.to(DEVICE, dtype=torch.float32)
model.train()
require(next(model.parameters()).device == DEVICE, "model is not on cuda:0")
require(
    {str(parameter.dtype) for parameter in model.parameters() if parameter.is_floating_point()} == {"torch.float32"},
    "model contains a non-FP32 floating parameter",
)

no_decay_rules = ("bias", "LayerNorm.weight", "layer_norm.weight")
decay_parameters = []
no_decay_parameters = []
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    if any(rule in name for rule in no_decay_rules):
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)

trainable_ids = {id(parameter) for parameter in model.parameters() if parameter.requires_grad}
grouped_ids = {id(parameter) for parameter in decay_parameters + no_decay_parameters}
require(trainable_ids == grouped_ids, "optimizer parameter grouping is incomplete")
require(len(decay_parameters) + len(no_decay_parameters) == len(grouped_ids), "optimizer parameter duplicated")

optimizer = torch.optim.AdamW(
    [
        {"params": decay_parameters, "weight_decay": 0.01},
        {"params": no_decay_parameters, "weight_decay": 0.0},
    ],
    lr=2e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    foreach=False,
    fused=False,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_UPDATES,
    num_training_steps=TOTAL_UPDATES,
)

optimizer.zero_grad(set_to_none=True)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(DEVICE)
start = time.perf_counter()
loss_sum = 0.0
examples_consumed = 0
logit_rows = 0

for start_index in range(0, PREFLIGHT_EXAMPLES, PHYSICAL_BATCH_SIZE):
    micro_rows = smoke_rows[start_index:start_index + PHYSICAL_BATCH_SIZE]
    encoded = tokenizer(
        [row["text"] for row in micro_rows],
        add_special_tokens=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_attention_mask=True,
        return_tensors="pt",
    )
    labels = torch.tensor(
        [int(row["label_id"]) for row in micro_rows],
        dtype=torch.long,
        device=DEVICE,
    )
    model_inputs = {key: value.to(DEVICE) for key, value in encoded.items()}
    outputs = model(**model_inputs)
    logits = outputs.logits
    require(tuple(logits.shape) == (len(micro_rows), 3), "unexpected logit shape")
    require(bool(torch.isfinite(logits).all().item()), "non-finite smoke-test logits")
    loss = torch.nn.functional.cross_entropy(logits, labels, reduction="sum")
    require(bool(torch.isfinite(loss).item()), "non-finite smoke-test loss")
    loss.backward()
    loss_sum += float(loss.detach().cpu())
    examples_consumed += len(micro_rows)
    logit_rows += logits.shape[0]

require(examples_consumed == PREFLIGHT_EXAMPLES, "smoke-test example count differs")
require(logit_rows == PREFLIGHT_EXAMPLES, "smoke-test logit count differs")

gradient_tensors = 0
for parameter in model.parameters():
    if parameter.grad is not None:
        require(bool(torch.isfinite(parameter.grad).all().item()), "non-finite gradient detected")
        parameter.grad.div_(examples_consumed)
        gradient_tensors += 1
require(gradient_tensors > 0, "no gradients were produced")

gradient_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0,
    error_if_nonfinite=True,
    foreach=False,
)
require(bool(torch.isfinite(gradient_norm).item()), "non-finite gradient norm")
lr_used = [float(group["lr"]) for group in optimizer.param_groups]
optimizer.step()
scheduler.step()
optimizer.zero_grad(set_to_none=True)
torch.cuda.synchronize(DEVICE)
elapsed_seconds = time.perf_counter() - start

require(all_finite_parameters(model), "non-finite parameter after smoke optimizer step")
peak_allocated = int(torch.cuda.max_memory_allocated(DEVICE))
peak_reserved = int(torch.cuda.max_memory_reserved(DEVICE))
device_total = int(torch.cuda.get_device_properties(0).total_memory)
require(peak_reserved <= device_total, "reported peak reserved memory exceeds device memory")

print("Fold / seed            :", TARGET_FOLD, "/", FOLD_SEED)
print("Smoke rows             :", PREFLIGHT_EXAMPLES, "training rows only")
print("Physical microbatches  :", ACCUMULATION_MICROBATCHES)
print("Accumulation examples  :", examples_consumed)
print("Order SHA256           :", order_hash)
print("Summed loss            :", f"{loss_sum:.9f}")
print("Mean loss              :", f"{loss_sum / examples_consumed:.9f}")
print("Pre-clip gradient norm :", f"{float(gradient_norm.detach().cpu()):.9f}")
print("Optimizer LR used      :", lr_used)
print("Scheduler LR after step:", [float(group["lr"]) for group in optimizer.param_groups])
print("Elapsed seconds        :", f"{elapsed_seconds:.3f}")
print("Peak allocated bytes   :", peak_allocated)
print("Peak reserved bytes    :", peak_reserved)
print("GPU total bytes        :", device_total)

del scheduler
del optimizer
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize(DEVICE)

print("\n" + "=" * WIDTH)
print("STEP M4A DECISION: PASS — EXACT FOLD-TRAINING GPU PATH IS READY")
print("=" * WIDTH)
print("Frozen initial model verified : PASS")
print("Complete tokenizer verified    : PASS")
print("Fold-1 train-only smoke path   : PASS")
print("Experimental checkpoint        : NONE; smoke model discarded")
print("Primary/Llama predictions      : NONE")
print("Evaluation/tuning/selection    : NONE")
print("Files written                  : NONE")
print("STOP — share this complete output before Fold 1 full training is run.")


PHASE 2B — STEP M4A: DEBERTA FOLD-TRAINING GPU PREFLIGHT
Python       : 3.12.13
PyTorch      : 2.10.0+cu128
Transformers : 5.0.0
Scope        : one discarded train-only accumulation/optimizer smoke step; no evaluation or prediction

ENVIRONMENT AND GPU COMPATIBILITY
----------------------------------------------------------------------------------------------------------------------------
numpy             : 2.0.2        PASS
pandas            : 2.3.3        PASS
scikit-learn      : 1.6.1        PASS
transformers      : 5.0.0        PASS
tokenizers        : 0.22.2       PASS
accelerate        : 1.13.0       PASS
safetensors       : 0.7.0        PASS
sentencepiece     : 0.2.1        PASS
Selected device      : cuda:0
GPU name             : Tesla T4
GPU capability       : (7, 5)
Compiled arches      : ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
Visible GPU count    : 2
Multi-GPU use        : prohibited; only cuda:0 selected
Numeric policy       : FP32; TF32/autocast

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Tokenizer class        : DebertaV2Tokenizer
Pre-tokenizer SHA256   : 9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171 PASS
Token-ID SHA256        : 9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359 PASS
Initial parameter SHA256: 7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b PASS

DISCARDED TRAIN-ONLY FORWARD/BACKWARD/OPTIMIZER SMOKE STEP
----------------------------------------------------------------------------------------------------------------------------
Fold / seed            : 1 / 43
Smoke rows             : 32 training rows only
Physical microbatches  : 16
Accumulation examples  : 32
Order SHA256           : a84958952ce50907768cd488a5ebe9c737bc0ab29771a436723ce796ce3556fe
Summed loss            : 35.929726839
Mean loss              : 1.122803964
Pre-clip gradient norm : 6.702333450
Optimizer LR used      : [0.0, 0.0]
Scheduler LR after step: [8.333333333333333e-07, 8.333333333333333e-07]
Elapsed seconds        : 7.082
Peak allo

## PHASE 2B — STEP M4B: FOLD 1 FROZEN DEBERTA TRAINING AND ONE-TIME HELD-OUT INFERENCE

In [2]:
from pathlib import Path
from collections import Counter
from importlib.metadata import version as package_version
import gc
import hashlib
import json
import math
import os
import random
import shutil
import sys
import tempfile
import time
import zipfile


# Frozen deterministic CUDA policy. This must precede the first CUDA BLAS call.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


WIDTH = 128
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_ROOT / "phase2B_M4B_FOLD_1_DEBERTA"
OUTPUT_ZIP = WORKING_ROOT / "phase2B_M4B_FOLD_1_DEBERTA.zip"
OUTPUT_SIDECAR = WORKING_ROOT / "phase2B_M4B_FOLD_1_DEBERTA.zip.sha256.txt"

EXPECTED = {
    "t2_payload": "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8",
    "t2_primary_file": "b836eb86f1d34ed2db2c8fce4f1b04e6bec515632403962ba594e5c0adaba9ef",
    "t2_llama_file": "37500d0ae8b48b632c18f9b57c931e2f88e6793e51975c6ef0243ab179bca01a",
    "base_payload": "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c",
    "initialisation_payload": "171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45",
    "initial_parameter": "7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b",
    "initial_model_file": "4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c",
    "pretokenizer": "9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171",
    "token_ids": "9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359",
    "m2_payload": "53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202",
    "m2_contract_file": "5979dc81c6e44eb869d47976a56cd7ee9311d8c400ffbc12e9cb9119bd09483a",
    "m3a_payload": "46118cc8794e5568a4cea30a930a17114c493e15cce5dc6b3947c007b1f5845e",
    "m2a_environment": "3e07e214b297576db434188e6c55e576fdd4af362e639f31f7654620d9ecbc7a",
    "m4a_first_group_order": "a84958952ce50907768cd488a5ebe9c737bc0ab29771a436723ce796ce3556fe",
    "m4a_first_group_loss": 35.929726839,
}

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "transformers": "5.0.0",
    "tokenizers": "0.22.2",
    "accelerate": "1.13.0",
    "safetensors": "0.7.0",
    "sentencepiece": "0.2.1",
}

TARGET_FOLD = 1
FOLD_SEED = 43
EXPECTED_TRAIN_N = 1917
EXPECTED_TEST_N = 479
EXPECTED_LLAMA_N = 160
EXPECTED_MICROBATCHES_PER_EPOCH = 959
EXPECTED_UPDATES_PER_EPOCH = 60
EPOCHS = 4
MAX_LENGTH = 256
PHYSICAL_BATCH_SIZE = 2
ACCUMULATION_MICROBATCHES = 16
MAXIMUM_EFFECTIVE_BATCH_SIZE = 32
INFERENCE_BATCH_SIZE = 16
TOTAL_UPDATES = 240
WARMUP_UPDATES = 24

LABEL2ID = {
    "presumed_truthful": 0,
    "human_deceptive": 1,
    "ai_generated": 2,
}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}
LABEL_ORDER = [0, 1, 2]
MODEL_NAME = "deberta_v3_base"


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8", errors="strict"))


def write_json(path, value):
    path.write_text(
        json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def write_csv(path, frame):
    frame.to_csv(path, index=False, lineterminator="\n", float_format="%.17g")


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    encoded = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def verify_payload(path, field, expected):
    document = read_json(path)
    stored = document.get(field)
    payload = {key: value for key, value in document.items() if key != field}
    require(stored == expected, f"{path.name}: stored payload differs")
    require(canonical_sha256(payload) == expected, f"{path.name}: recomputed payload differs")
    return document


def find_unique(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    require(len(matches) == 1, f"expected one {filename}; found {len(matches)}")
    return matches[0]


def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8", errors="strict", newline="") as handle:
        for line_number, line in enumerate(handle, start=1):
            require(line.endswith("\n"), f"{path.name}:{line_number} lacks LF termination")
            require(line.strip(), f"{path.name}:{line_number} is empty")
            row = json.loads(line)
            require(isinstance(row.get("row_id"), str), f"{path.name}:{line_number} missing row_id")
            require(isinstance(row.get("text"), str), f"{path.name}:{line_number} missing text")
            require(row["text"].strip() != "", f"{row['row_id']} has empty text")
            rows.append(row)
    return rows


def parameter_sha256(model):
    digest = hashlib.sha256()
    state = model.state_dict()
    for name in sorted(state):
        tensor = state[name].detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(tensor.dtype).encode("ascii"))
        digest.update(b"\0")
        digest.update(json.dumps(list(tensor.shape), separators=(",", ":")).encode("ascii"))
        digest.update(b"\0")
        digest.update(tensor.numpy().tobytes(order="C"))
        digest.update(b"\0")
    return digest.hexdigest()


def tokenizer_backend_identity(tokenizer):
    backend = json.loads(tokenizer.backend_tokenizer.to_str())
    pretokenizer = backend.get("pre_tokenizer")
    require(pretokenizer is not None, "tokenizer backend has no pre-tokenizer")
    encoded = json.dumps(
        pretokenizer,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    require("Metaspace" in encoded, "native DeBERTa Metaspace pre-tokenizer is absent")
    require('"Split"' not in encoded, "Mistral Split regex was applied")
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()


def token_id_fingerprint_and_lengths(tokenizer, rows):
    digest = hashlib.sha256()
    lengths = {}
    for row in rows:
        token_ids = tokenizer(
            row["text"],
            add_special_tokens=True,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            verbose=False,
        )["input_ids"]
        digest.update(row["row_id"].encode("utf-8"))
        digest.update(b"\0")
        digest.update(json.dumps(token_ids, separators=(",", ":")).encode("ascii"))
        digest.update(b"\n")
        lengths[row["row_id"]] = len(token_ids)
    require(len(lengths) == len(rows), "token-length index is incomplete")
    return digest.hexdigest(), lengths


def fixed_tokenize(tokenizer, rows):
    encoded = tokenizer(
        [row["text"] for row in rows],
        add_special_tokens=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_attention_mask=True,
        return_tensors="pt",
    )
    require("input_ids" in encoded and "attention_mask" in encoded, "required tokenizer outputs absent")
    require(tuple(encoded["input_ids"].shape) == (len(rows), MAX_LENGTH), "fixed token tensor shape differs")
    require(
        set(encoded.keys()).issubset({"input_ids", "attention_mask", "token_type_ids"}),
        f"unexpected tokenizer fields: {sorted(encoded.keys())}",
    )
    return {key: value.contiguous() for key, value in encoded.items()}


def row_order_sha256(rows):
    return hashlib.sha256("\n".join(row["row_id"] for row in rows).encode("utf-8")).hexdigest()


def all_finite_parameters(model):
    return all(
        bool(torch.isfinite(parameter).all().item())
        for parameter in model.parameters()
        if parameter.is_floating_point()
    )


def classification_metrics(y_true, y_pred, scores):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average=None,
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average="macro",
        zero_division=0,
    )
    result = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }
    for position, label in enumerate(LABEL_ORDER):
        result[f"class_{label}_precision"] = float(precision[position])
        result[f"class_{label}_recall"] = float(recall[position])
        result[f"class_{label}_f1"] = float(f1[position])
        result[f"class_{label}_support"] = int(support[position])
    binary = label_binarize(y_true, classes=LABEL_ORDER)
    result["macro_ovr_roc_auc"] = float(
        roc_auc_score(binary, scores, average="macro", multi_class="ovr")
    )
    return result


def infer(model, tokenizer, rows, device):
    model.eval()
    logits_blocks = []
    score_blocks = []
    with torch.no_grad():
        for start_index in range(0, len(rows), INFERENCE_BATCH_SIZE):
            batch_rows = rows[start_index:start_index + INFERENCE_BATCH_SIZE]
            encoded = tokenizer(
                [row["text"] for row in batch_rows],
                add_special_tokens=True,
                truncation=True,
                padding="max_length",
                max_length=MAX_LENGTH,
                return_attention_mask=True,
                return_tensors="pt",
            )
            inputs = {key: value.to(device) for key, value in encoded.items()}
            logits = model(**inputs).logits
            scores = torch.softmax(logits, dim=-1)
            require(bool(torch.isfinite(logits).all().item()), "non-finite inference logits")
            require(bool(torch.isfinite(scores).all().item()), "non-finite inference scores")
            logits_blocks.append(logits.detach().cpu().numpy())
            score_blocks.append(scores.detach().cpu().numpy())
    logits = np.concatenate(logits_blocks, axis=0).astype(np.float32, copy=False)
    scores = np.concatenate(score_blocks, axis=0).astype(np.float32, copy=False)
    require(logits.shape == (len(rows), 3), "inference logit array shape differs")
    require(scores.shape == (len(rows), 3), "inference score array shape differs")
    require(np.isfinite(logits).all(), "persisted inference logits are non-finite")
    require(np.isfinite(scores).all(), "persisted inference scores are non-finite")
    require(np.allclose(scores.sum(axis=1), 1.0, rtol=0.0, atol=1e-6), "scores do not sum to one")
    require(np.array_equal(np.argmax(logits, axis=1), np.argmax(scores, axis=1)), "logit/score decisions differ")
    return logits, scores


def primary_prediction_frame(rows, logits, scores):
    predicted = np.argmax(logits, axis=1).astype(int)
    frame = pd.DataFrame({
        "row_id": [row["row_id"] for row in rows],
        "official_fold": [int(row["official_fold"]) for row in rows],
        "hotel": [row["hotel"] for row in rows],
        "polarity": [row["polarity"] for row in rows],
        "source_family": [row["source_family"] for row in rows],
        "scientific_class": [row["scientific_class"] for row in rows],
        "y_true": [int(row["label_id"]) for row in rows],
        "y_pred": predicted,
        "logit_0": logits[:, 0],
        "logit_1": logits[:, 1],
        "logit_2": logits[:, 2],
        "score_0": scores[:, 0],
        "score_1": scores[:, 1],
        "score_2": scores[:, 2],
        "maximum_score": scores.max(axis=1),
        "model_name": MODEL_NAME,
        "fold_seed": FOLD_SEED,
    })
    frame["correct"] = (frame["y_true"].to_numpy(dtype=int) == predicted).astype(int)
    columns = [
        "row_id", "official_fold", "hotel", "polarity", "source_family",
        "scientific_class", "y_true", "y_pred", "logit_0", "logit_1",
        "logit_2", "score_0", "score_1", "score_2", "maximum_score",
        "correct", "model_name", "fold_seed",
    ]
    require(frame["row_id"].tolist() == sorted(frame["row_id"]), "primary prediction order differs")
    return frame[columns]


def llama_prediction_frame(rows, logits, scores, gpt3_by_seed):
    predicted = np.argmax(logits, axis=1).astype(int)
    frame = pd.DataFrame({
        "row_id": [row["row_id"] for row in rows],
        "seed_family_id": [row["seed_family_id"] for row in rows],
        "matched_gpt3_row_id": [gpt3_by_seed[row["seed_family_id"]]["row_id"] for row in rows],
        "official_fold": [int(row["official_fold"]) for row in rows],
        "hotel": [row["hotel"] for row in rows],
        "polarity": [row["polarity"] for row in rows],
        "reference_label_id": [2] * len(rows),
        "y_pred": predicted,
        "logit_0": logits[:, 0],
        "logit_1": logits[:, 1],
        "logit_2": logits[:, 2],
        "score_0": scores[:, 0],
        "score_1": scores[:, 1],
        "score_2": scores[:, 2],
        "maximum_score": scores.max(axis=1),
        "detected_as_ai": (predicted == 2).astype(int),
        "model_name": MODEL_NAME,
        "fold_seed": FOLD_SEED,
    })
    columns = [
        "row_id", "seed_family_id", "matched_gpt3_row_id", "official_fold",
        "hotel", "polarity", "reference_label_id", "y_pred", "logit_0",
        "logit_1", "logit_2", "score_0", "score_1", "score_2",
        "maximum_score", "detected_as_ai", "model_name", "fold_seed",
    ]
    require(frame["row_id"].tolist() == sorted(frame["row_id"]), "Llama prediction order differs")
    return frame[columns]


def llama_summary(frame, group_type, group_value):
    predictions = frame["y_pred"].to_numpy(dtype=int)
    result = {
        "group_type": group_type,
        "group_value": str(group_value),
        "n": int(len(frame)),
        "ai_detection_recall": float((predictions == 2).mean()),
        "mean_ai_score": float(frame["score_2"].mean()),
        "median_ai_score": float(frame["score_2"].median()),
        "mean_maximum_score": float(frame["maximum_score"].mean()),
    }
    for label in LABEL_ORDER:
        count = int((predictions == label).sum())
        result[f"predicted_class_{label}_n"] = count
        result[f"predicted_class_{label}_share"] = float(count / len(frame))
    return result


def deterministic_zip(source_root, destination):
    timestamp = (2026, 9, 10, 0, 0, 0)
    with zipfile.ZipFile(
        destination,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
        allowZip64=True,
    ) as archive:
        for path in sorted(item for item in source_root.rglob("*") if item.is_file()):
            arcname = path.relative_to(source_root.parent).as_posix()
            info = zipfile.ZipInfo(arcname, date_time=timestamp)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            info.create_system = 3
            with path.open("rb") as source, archive.open(info, mode="w", force_zip64=True) as target:
                shutil.copyfileobj(source, target, length=1024 * 1024)


print("=" * WIDTH)
print("PHASE 2B — STEP M4B: FOLD 1 FROZEN DEBERTA TRAINING AND ONE-TIME HELD-OUT INFERENCE")
print("=" * WIDTH)
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Target fold  :", TARGET_FOLD)
print("Policy       : final epoch only; no tuning, validation, early stopping or test monitoring")

require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")
require(not OUTPUT_ROOT.exists(), f"output directory already exists: {OUTPUT_ROOT}")
require(not OUTPUT_ZIP.exists(), f"output ZIP already exists: {OUTPUT_ZIP}")
require(not OUTPUT_SIDECAR.exists(), f"output sidecar already exists: {OUTPUT_SIDECAR}")

section("ENVIRONMENT AND GPU IDENTITY")
for name, expected in EXPECTED_PACKAGES.items():
    actual = package_version(name)
    require(actual == expected, f"{name} version differs: {actual} != {expected}")
    print(f"{name:18s}: {actual:12s} PASS")

require(torch.__version__ == "2.10.0+cu128", f"unexpected PyTorch build: {torch.__version__}")
require(torch.version.cuda == "12.8", f"unexpected PyTorch CUDA build: {torch.version.cuda}")
require(torch.backends.cudnn.version() == 91002, f"unexpected cuDNN version: {torch.backends.cudnn.version()}")
require(torch.cuda.is_available(), "CUDA is unavailable")
require(torch.cuda.device_count() == 2, f"expected audited T4 x2 runtime; found {torch.cuda.device_count()} GPUs")
require(all(torch.cuda.get_device_name(index) == "Tesla T4" for index in range(2)), "GPU identity differs")
require(torch.cuda.get_device_capability(0) == (7, 5), "cuda:0 capability differs")
require("sm_75" in torch.cuda.get_arch_list(), "PyTorch build lacks sm_75 kernels")

torch.cuda.set_device(0)
DEVICE = torch.device("cuda:0")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision("highest")
torch.use_deterministic_algorithms(True)

print("Selected device       :", DEVICE)
print("GPU 0                 :", torch.cuda.get_device_name(0))
print("Visible GPUs          :", torch.cuda.device_count(), "(cuda:1 unused)")
print("CUDA/cuDNN            :", torch.version.cuda, "/", torch.backends.cudnn.version())
print("Deterministic kernels :", torch.are_deterministic_algorithms_enabled())
print("Numeric policy        : FP32; TF32/autocast/GradScaler disabled")

section("UPSTREAM ARTIFACT AND CONTRACT CONTINUITY")
t2_path = find_unique("corpus_manifest.json")
base_path = find_unique("checkpoint_manifest.json")
init_path = find_unique("initialisation_manifest.json")
m2_path = find_unique("contract_manifest.json")
m3a_path = find_unique("baseline_manifest.json")

t2_manifest = verify_payload(t2_path, "manifest_payload_sha256", EXPECTED["t2_payload"])
base_manifest = verify_payload(base_path, "manifest_payload_sha256", EXPECTED["base_payload"])
init_manifest = verify_payload(init_path, "manifest_payload_sha256", EXPECTED["initialisation_payload"])
m2_manifest = verify_payload(m2_path, "contract_payload_sha256", EXPECTED["m2_payload"])
m3a_manifest = verify_payload(m3a_path, "manifest_payload_sha256", EXPECTED["m3a_payload"])

require(file_sha256(m2_path) == EXPECTED["m2_contract_file"], "M2 contract-manifest file differs")
require(init_manifest["upstream"]["t2_manifest_payload_sha256"] == EXPECTED["t2_payload"], "initialisation/T2 link differs")
require(init_manifest["upstream"]["base_checkpoint_manifest_payload_sha256"] == EXPECTED["base_payload"], "initialisation/base link differs")
require(m2_manifest["upstream"]["t2_manifest_payload_sha256"] == EXPECTED["t2_payload"], "M2/T2 link differs")
require(m2_manifest["upstream"]["base_checkpoint_manifest_payload_sha256"] == EXPECTED["base_payload"], "M2/base link differs")
require(m2_manifest["upstream"]["initialisation_manifest_payload_sha256"] == EXPECTED["initialisation_payload"], "M2/initialisation link differs")
require(m2_manifest["upstream"]["initial_parameter_sha256"] == EXPECTED["initial_parameter"], "M2 parameter link differs")
require(m2_manifest["upstream"]["eligible_corpus_token_id_sha256"] == EXPECTED["token_ids"], "M2 token link differs")
require(m2_manifest["upstream"]["m2a_gpu_environment_payload_sha256"] == EXPECTED["m2a_environment"], "M2 environment link differs")
require(m3a_manifest["upstream"]["t2_manifest_payload_sha256"] == EXPECTED["t2_payload"], "M3A/T2 link differs")
require(m3a_manifest["upstream"]["m2_contract_payload_sha256"] == EXPECTED["m2_payload"], "M3A/M2 link differs")

T2_ROOT = t2_path.parent
MODEL_ROOT = init_path.parent / "model"
PRIMARY_PATH = T2_ROOT / "primary_tripartite_corpus.jsonl"
LLAMA_PATH = T2_ROOT / "llama_unseen_generator_sensitivity.jsonl"
MODEL_FILE = MODEL_ROOT / "model.safetensors"
require(file_sha256(PRIMARY_PATH) == EXPECTED["t2_primary_file"], "primary corpus file differs")
require(file_sha256(LLAMA_PATH) == EXPECTED["t2_llama_file"], "Llama corpus file differs")
require(file_sha256(MODEL_FILE) == EXPECTED["initial_model_file"], "initialised model file differs")

print("T2 payload             :", EXPECTED["t2_payload"], "PASS")
print("Base payload           :", EXPECTED["base_payload"], "PASS")
print("Initialisation payload :", EXPECTED["initialisation_payload"], "PASS")
print("M2 contract payload    :", EXPECTED["m2_payload"], "PASS")
print("M3A baseline payload   :", EXPECTED["m3a_payload"], "PASS")
print("Corpus/model files     : PASS")

section("FOLD-1 DATA ROUTING AND TOKENIZER VERIFICATION")
primary_file_order = read_jsonl(PRIMARY_PATH)
llama_file_order = read_jsonl(LLAMA_PATH)
require(len(primary_file_order) == 2396, "primary row count differs")
require(len(llama_file_order) == 790, "Llama row count differs")
require(len({row["row_id"] for row in primary_file_order + llama_file_order}) == 3186, "eligible row IDs are not unique")

primary_rows = sorted(primary_file_order, key=lambda row: row["row_id"])
llama_rows = sorted(llama_file_order, key=lambda row: row["row_id"])
train_rows = [row for row in primary_rows if int(row["official_fold"]) != TARGET_FOLD]
test_rows = [row for row in primary_rows if int(row["official_fold"]) == TARGET_FOLD]
llama_test_rows = [row for row in llama_rows if int(row["official_fold"]) == TARGET_FOLD]
require(len(train_rows) == EXPECTED_TRAIN_N, f"train count differs: {len(train_rows)}")
require(len(test_rows) == EXPECTED_TEST_N, f"test count differs: {len(test_rows)}")
require(len(llama_test_rows) == EXPECTED_LLAMA_N, f"Llama test count differs: {len(llama_test_rows)}")
require([row["row_id"] for row in train_rows] == sorted(row["row_id"] for row in train_rows), "train order differs")
require([row["row_id"] for row in test_rows] == sorted(row["row_id"] for row in test_rows), "test order differs")
require([row["row_id"] for row in llama_test_rows] == sorted(row["row_id"] for row in llama_test_rows), "Llama order differs")
train_hotels = {row["hotel"] for row in train_rows}
test_hotels = {row["hotel"] for row in test_rows}
require(not (train_hotels & test_hotels), "train/test hotel overlap detected")
require(test_hotels == {"hilton", "james", "monaco", "sofitel"}, f"Fold-1 hotels differ: {sorted(test_hotels)}")
require(all(row["source_family"] != "op_spam_AI_Llama" for row in train_rows), "Llama row entered training")
require(Counter(row["label_id"] for row in train_rows) == Counter({0: 637, 1: 640, 2: 640}), "train class counts differ")
require(Counter(row["label_id"] for row in test_rows) == Counter({0: 159, 1: 160, 2: 160}), "test class counts differ")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_ROOT), local_files_only=True)
pretokenizer_hash = tokenizer_backend_identity(tokenizer)
require(pretokenizer_hash == EXPECTED["pretokenizer"], "pre-tokenizer fingerprint differs")
token_hash, token_lengths = token_id_fingerprint_and_lengths(
    tokenizer,
    primary_file_order + llama_file_order,
)
require(token_hash == EXPECTED["token_ids"], "eligible-corpus token-ID fingerprint differs")

print("Train/test/Llama       :", len(train_rows), "/", len(test_rows), "/", len(llama_test_rows), "PASS")
print("Held-out hotels        :", ", ".join(sorted(test_hotels)))
print("Hotel overlap          : 0 PASS")
print("Train class counts     :", dict(sorted(Counter(row["label_id"] for row in train_rows).items())), "PASS")
print("Pre-tokenizer SHA256   :", pretokenizer_hash, "PASS")
print("Token-ID SHA256        :", token_hash, "PASS")

section("INITIAL MODEL AND OPTIMIZER CONSTRUCTION")
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_ROOT), local_files_only=True)
model.to(device="cpu", dtype=torch.float32)
model.eval()
require(model.config.num_labels == 3, "model output dimension differs")
require(model.config.label2id == LABEL2ID, "model label2id mapping differs")
require({int(key): value for key, value in model.config.id2label.items()} == ID2LABEL, "model id2label mapping differs")
initial_parameter_hash = parameter_sha256(model)
require(initial_parameter_hash == EXPECTED["initial_parameter"], "initial parameter fingerprint differs")

train_encoded = fixed_tokenize(tokenizer, train_rows)
train_labels = torch.tensor([int(row["label_id"]) for row in train_rows], dtype=torch.long)

random.seed(FOLD_SEED)
np.random.seed(FOLD_SEED)
torch.manual_seed(FOLD_SEED)
torch.cuda.manual_seed_all(FOLD_SEED)
model.to(DEVICE, dtype=torch.float32)
model.train()
require(next(model.parameters()).device == DEVICE, "model is not on cuda:0")
require({str(parameter.dtype) for parameter in model.parameters() if parameter.is_floating_point()} == {"torch.float32"}, "non-FP32 parameter detected")

no_decay_rules = ("bias", "LayerNorm.weight", "layer_norm.weight")
decay_parameters = []
no_decay_parameters = []
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    (no_decay_parameters if any(rule in name for rule in no_decay_rules) else decay_parameters).append(parameter)
trainable_ids = {id(parameter) for parameter in model.parameters() if parameter.requires_grad}
grouped_ids = {id(parameter) for parameter in decay_parameters + no_decay_parameters}
require(trainable_ids == grouped_ids, "optimizer parameter grouping is incomplete")
require(len(decay_parameters) + len(no_decay_parameters) == len(grouped_ids), "optimizer parameter duplicated")

optimizer = torch.optim.AdamW(
    [
        {"params": decay_parameters, "weight_decay": 0.01},
        {"params": no_decay_parameters, "weight_decay": 0.0},
    ],
    lr=2e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    foreach=False,
    fused=False,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_UPDATES,
    num_training_steps=TOTAL_UPDATES,
)
require([float(group["lr"]) for group in optimizer.param_groups] == [0.0, 0.0], "initial warm-up LR differs")

print("Initial parameter SHA256:", initial_parameter_hash, "PASS")
print("Parameters             :", sum(parameter.numel() for parameter in model.parameters()))
print("Epochs/max length      :", EPOCHS, "/", MAX_LENGTH)
print("Batch/accumulation     :", PHYSICAL_BATCH_SIZE, "/", ACCUMULATION_MICROBATCHES)
print("Updates/warmup         :", TOTAL_UPDATES, "/", WARMUP_UPDATES)
print("Optimizer              : AdamW, foreach=False, fused=False")
print("Test monitoring        : PROHIBITED and not implemented")

section("FROZEN FOLD-1 TRAINING")
training_start = time.perf_counter()
global_update = 0
training_examples = 0
training_microbatches = 0
epoch_logs = []
update_logs = []

for zero_epoch in range(EPOCHS):
    epoch_number = zero_epoch + 1
    model.train()
    order = np.random.default_rng(FOLD_SEED + zero_epoch).permutation(len(train_rows))
    ordered_rows = [train_rows[int(index)] for index in order]
    order_hash = row_order_sha256(ordered_rows)
    if zero_epoch == 0:
        first_group_hash = row_order_sha256(ordered_rows[:MAXIMUM_EFFECTIVE_BATCH_SIZE])
        require(first_group_hash == EXPECTED["m4a_first_group_order"], "M4A first-group order is not reproduced")

    epoch_loss_sum = 0.0
    epoch_examples = 0
    epoch_microbatches = 0
    epoch_updates = 0
    epoch_start = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(DEVICE)

    for group_start in range(0, len(order), MAXIMUM_EFFECTIVE_BATCH_SIZE):
        group_indices = order[group_start:group_start + MAXIMUM_EFFECTIVE_BATCH_SIZE]
        group_examples = int(len(group_indices))
        require(1 <= group_examples <= MAXIMUM_EFFECTIVE_BATCH_SIZE, "invalid accumulation-group size")
        optimizer.zero_grad(set_to_none=True)
        group_loss_sum = 0.0
        group_microbatches = 0

        for micro_start in range(0, group_examples, PHYSICAL_BATCH_SIZE):
            micro_indices_np = group_indices[micro_start:micro_start + PHYSICAL_BATCH_SIZE]
            micro_indices = torch.as_tensor(micro_indices_np, dtype=torch.long)
            inputs = {
                key: tensor.index_select(0, micro_indices).to(DEVICE)
                for key, tensor in train_encoded.items()
            }
            labels = train_labels.index_select(0, micro_indices).to(DEVICE)
            logits = model(**inputs).logits
            require(tuple(logits.shape) == (len(micro_indices_np), 3), "training logit shape differs")
            require(bool(torch.isfinite(logits).all().item()), "non-finite training logits")
            loss = torch.nn.functional.cross_entropy(logits, labels, reduction="sum")
            require(bool(torch.isfinite(loss).item()), "non-finite training loss")
            loss.backward()
            group_loss_sum += float(loss.detach().cpu())
            group_microbatches += 1

        gradient_tensors = 0
        for parameter in model.parameters():
            if parameter.grad is not None:
                parameter.grad.div_(group_examples)
                gradient_tensors += 1
        require(gradient_tensors > 0, "no gradients were produced")

        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
            error_if_nonfinite=True,
            foreach=False,
        )
        require(bool(torch.isfinite(gradient_norm).item()), "non-finite gradient norm")
        lr_used = float(optimizer.param_groups[0]["lr"])
        require(all(float(group["lr"]) == lr_used for group in optimizer.param_groups), "optimizer-group LR mismatch")
        optimizer.step()
        scheduler.step()
        global_update += 1
        epoch_updates += 1
        lr_after = float(optimizer.param_groups[0]["lr"])

        group_rows = [train_rows[int(index)] for index in group_indices]
        group_hash = row_order_sha256(group_rows)
        if zero_epoch == 0 and epoch_updates == 1:
            require(group_hash == EXPECTED["m4a_first_group_order"], "first training group hash differs from M4A")
            require(
                abs(group_loss_sum - EXPECTED["m4a_first_group_loss"]) <= 1e-6,
                f"first training-group loss differs from M4A: {group_loss_sum}",
            )

        update_logs.append({
            "fold": TARGET_FOLD,
            "seed": FOLD_SEED,
            "epoch": epoch_number,
            "epoch_update": epoch_updates,
            "global_update": global_update,
            "group_examples": group_examples,
            "microbatches": group_microbatches,
            "group_row_order_sha256": group_hash,
            "loss_sum": group_loss_sum,
            "mean_loss": group_loss_sum / group_examples,
            "pre_clip_gradient_norm": float(gradient_norm.detach().cpu()),
            "learning_rate_used": lr_used,
            "learning_rate_after_step": lr_after,
        })

        epoch_loss_sum += group_loss_sum
        epoch_examples += group_examples
        epoch_microbatches += group_microbatches
        training_examples += group_examples
        training_microbatches += group_microbatches

        if epoch_updates % 20 == 0 or epoch_updates == EXPECTED_UPDATES_PER_EPOCH:
            print(
                f"Epoch {epoch_number}/{EPOCHS} | update {epoch_updates:02d}/{EXPECTED_UPDATES_PER_EPOCH} "
                f"| global {global_update:03d}/{TOTAL_UPDATES} | mean loss {epoch_loss_sum / epoch_examples:.6f}",
                flush=True,
            )

    torch.cuda.synchronize(DEVICE)
    epoch_elapsed = time.perf_counter() - epoch_start
    require(epoch_examples == EXPECTED_TRAIN_N, f"epoch {epoch_number} example count differs")
    require(epoch_microbatches == EXPECTED_MICROBATCHES_PER_EPOCH, f"epoch {epoch_number} microbatch count differs")
    require(epoch_updates == EXPECTED_UPDATES_PER_EPOCH, f"epoch {epoch_number} update count differs")
    require(all_finite_parameters(model), f"non-finite parameter after epoch {epoch_number}")
    epoch_log = {
        "fold": TARGET_FOLD,
        "seed": FOLD_SEED,
        "epoch": epoch_number,
        "row_order_sha256": order_hash,
        "examples": epoch_examples,
        "microbatches": epoch_microbatches,
        "optimizer_updates": epoch_updates,
        "loss_sum": epoch_loss_sum,
        "mean_loss": epoch_loss_sum / epoch_examples,
        "elapsed_seconds": epoch_elapsed,
        "peak_allocated_bytes": int(torch.cuda.max_memory_allocated(DEVICE)),
        "peak_reserved_bytes": int(torch.cuda.max_memory_reserved(DEVICE)),
        "learning_rate_after_epoch": float(optimizer.param_groups[0]["lr"]),
    }
    epoch_logs.append(epoch_log)
    print(
        f"Epoch {epoch_number} complete | loss={epoch_log['mean_loss']:.6f} "
        f"| elapsed={epoch_elapsed:.1f}s | order={order_hash[:16]}...",
        flush=True,
    )

training_elapsed = time.perf_counter() - training_start
require(global_update == TOTAL_UPDATES, f"total optimizer updates differ: {global_update}")
require(training_examples == EXPECTED_TRAIN_N * EPOCHS, "total training-example count differs")
require(training_microbatches == EXPECTED_MICROBATCHES_PER_EPOCH * EPOCHS, "total microbatch count differs")
require(len(update_logs) == TOTAL_UPDATES, "optimizer log length differs")
require(scheduler.last_epoch == TOTAL_UPDATES, f"scheduler step count differs: {scheduler.last_epoch}")
require(abs(float(optimizer.param_groups[0]["lr"])) <= 1e-18, "final scheduled LR is not zero")

section("ONE-TIME FINAL-EPOCH HELD-OUT INFERENCE")
inference_start = time.perf_counter()
primary_logits, primary_scores = infer(model, tokenizer, test_rows, DEVICE)
llama_logits, llama_scores = infer(model, tokenizer, llama_test_rows, DEVICE)
torch.cuda.synchronize(DEVICE)
inference_elapsed = time.perf_counter() - inference_start

gpt3_by_seed = {
    row["seed_family_id"]: row
    for row in primary_rows
    if row["source_family"] == "op_spam_AI_GPT3"
}
require(len(gpt3_by_seed) == 800, "GPT-3 seed-family index differs")
require(all(row["seed_family_id"] in gpt3_by_seed for row in llama_test_rows), "Llama/GPT-3 seed match missing")
require(all(int(gpt3_by_seed[row["seed_family_id"]]["official_fold"]) == TARGET_FOLD for row in llama_test_rows), "matched GPT-3 sibling lies outside Fold 1")

primary_predictions = primary_prediction_frame(test_rows, primary_logits, primary_scores)
llama_predictions = llama_prediction_frame(llama_test_rows, llama_logits, llama_scores, gpt3_by_seed)
primary_metrics = classification_metrics(
    primary_predictions["y_true"],
    primary_predictions["y_pred"],
    primary_predictions[["score_0", "score_1", "score_2"]],
)
primary_confusion = confusion_matrix(
    primary_predictions["y_true"],
    primary_predictions["y_pred"],
    labels=LABEL_ORDER,
).astype(int).tolist()

llama_summary_rows = [llama_summary(llama_predictions, "all", "all")]
for dimension in ("polarity", "hotel"):
    for value in sorted(llama_predictions[dimension].unique(), key=str):
        subset = llama_predictions[llama_predictions[dimension] == value]
        llama_summary_rows.append(llama_summary(subset, dimension, value))
llama_summary_frame = pd.DataFrame(llama_summary_rows)

print("Primary predictions    :", len(primary_predictions), "PASS")
print("Llama predictions      :", len(llama_predictions), "PASS")
print("Primary macro-F1       :", f"{primary_metrics['macro_f1']:.6f}")
print("Primary balanced acc.  :", f"{primary_metrics['balanced_accuracy']:.6f}")
print("Primary accuracy       :", f"{primary_metrics['accuracy']:.6f}")
print("Llama AI detection     :", f"{llama_predictions['detected_as_ai'].mean():.6f}")
print("Inference elapsed sec. :", f"{inference_elapsed:.3f}")
print("Test use               : one final-epoch inference; no tuning or selection")

section("TRUNCATION AUDIT")
truncation_rows = []
for row in primary_rows:
    role = "primary_test" if int(row["official_fold"]) == TARGET_FOLD else "outer_train"
    truncation_rows.append({
        "row_id": row["row_id"],
        "dataset_role": role,
        "scientific_class": row["scientific_class"],
        "official_fold": int(row["official_fold"]),
        "hotel": row["hotel"],
        "polarity": row["polarity"],
        "pretruncation_tokens": int(token_lengths[row["row_id"]]),
        "truncated_at_256": int(token_lengths[row["row_id"]] > MAX_LENGTH),
    })
for row in llama_rows:
    role = "llama_sensitivity_test" if int(row["official_fold"]) == TARGET_FOLD else "llama_not_used_this_fold"
    truncation_rows.append({
        "row_id": row["row_id"],
        "dataset_role": role,
        "scientific_class": "ai_generated_llama",
        "official_fold": int(row["official_fold"]),
        "hotel": row["hotel"],
        "polarity": row["polarity"],
        "pretruncation_tokens": int(token_lengths[row["row_id"]]),
        "truncated_at_256": int(token_lengths[row["row_id"]] > MAX_LENGTH),
    })
truncation_frame = pd.DataFrame(truncation_rows).sort_values("row_id", kind="stable").reset_index(drop=True)
require(len(truncation_frame) == 3186, "truncation audit row count differs")
truncation_summary = (
    truncation_frame
    .groupby(["dataset_role", "scientific_class", "official_fold"], sort=True, dropna=False)
    .agg(
        n=("row_id", "size"),
        truncated_n=("truncated_at_256", "sum"),
        truncation_rate=("truncated_at_256", "mean"),
        minimum_tokens=("pretruncation_tokens", "min"),
        median_tokens=("pretruncation_tokens", "median"),
        maximum_tokens=("pretruncation_tokens", "max"),
    )
    .reset_index()
)
print("Eligible rows audited  :", len(truncation_frame), "PASS")
print("Rows exceeding 256     :", int(truncation_frame["truncated_at_256"].sum()))
print("Audit dimensions       : role × class × official fold")

section("FINAL CHECKPOINT SAVE, RELOAD AND ARTIFACT FREEZE")
final_parameter_hash = parameter_sha256(model)
require(final_parameter_hash != EXPECTED["initial_parameter"], "final parameters equal the initial state")
require(all_finite_parameters(model), "final model contains non-finite parameters")

device_total_bytes = int(torch.cuda.get_device_properties(0).total_memory)
peak_allocated_bytes = max(item["peak_allocated_bytes"] for item in epoch_logs)
peak_reserved_bytes = max(item["peak_reserved_bytes"] for item in epoch_logs)

temporary_parent = Path(tempfile.mkdtemp(prefix="phase2B_M4B_fold1_", dir=WORKING_ROOT))
temporary_root = temporary_parent / OUTPUT_ROOT.name
temporary_root.mkdir()
model_output = temporary_root / "model"

model.to(device="cpu", dtype=torch.float32)
model.eval()
model.save_pretrained(str(model_output), safe_serialization=True, max_shard_size="5GB")
tokenizer.save_pretrained(str(model_output))

write_csv(temporary_root / "fold_1_primary_predictions.csv", primary_predictions)
write_csv(temporary_root / "fold_1_llama_sensitivity_predictions.csv", llama_predictions)
write_csv(temporary_root / "fold_1_epoch_training_log.csv", pd.DataFrame(epoch_logs))
write_csv(temporary_root / "fold_1_optimizer_update_log.csv", pd.DataFrame(update_logs))
write_json(temporary_root / "fold_1_primary_metrics.json", primary_metrics)
write_json(temporary_root / "fold_1_confusion_matrix.json", {"label_order": LABEL_ORDER, "matrix": primary_confusion})
write_csv(temporary_root / "fold_1_llama_sensitivity_summary.csv", llama_summary_frame)
write_csv(temporary_root / "fold_1_truncation_audit.csv", truncation_frame)
write_csv(temporary_root / "fold_1_truncation_summary.csv", truncation_summary)

implementation = {
    "training_loop": "explicit deterministic row-index batching; zero worker processes",
    "loss": "torch.nn.functional.cross_entropy reduction=sum",
    "gradient_normalisation": "divide accumulated summed gradients by actual group examples",
    "optimizer": "torch.optim.AdamW foreach=False fused=False",
    "scheduler": "transformers.get_linear_schedule_with_warmup called after each optimizer step",
    "gradient_clipping": "clip_grad_norm_ max_norm=1.0 foreach=False",
    "zero_grad": "set_to_none=True at each accumulation-group boundary",
    "inference": "FP32 no_grad, batch=16, softmax model scores, argmax logits",
    "test_monitoring": False,
    "automatic_adaptation": False,
}
write_json(temporary_root / "implementation_record.json", implementation)

del scheduler
del optimizer
del train_encoded
del train_labels
del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize(DEVICE)

reloaded_model = AutoModelForSequenceClassification.from_pretrained(
    str(model_output),
    local_files_only=True,
)
reloaded_model.to(device="cpu", dtype=torch.float32)
reloaded_model.eval()
reloaded_parameter_hash = parameter_sha256(reloaded_model)
require(reloaded_parameter_hash == final_parameter_hash, "final model hash changed after save/reload")
reloaded_tokenizer = AutoTokenizer.from_pretrained(str(model_output), local_files_only=True)
require(tokenizer_backend_identity(reloaded_tokenizer) == EXPECTED["pretokenizer"], "saved tokenizer backend differs")
del reloaded_model
del reloaded_tokenizer
del tokenizer
gc.collect()

payload_files = sorted(path for path in temporary_root.rglob("*") if path.is_file())
payload_registry = {
    path.relative_to(temporary_root).as_posix(): {
        "bytes": path.stat().st_size,
        "sha256": file_sha256(path),
    }
    for path in payload_files
}

manifest_payload = {
    "phase": "Phase 2B",
    "step": "M4B",
    "status": "FROZEN_FOLD_1_DEBERTA_COMPLETE",
    "upstream": {
        "t2_manifest_payload_sha256": EXPECTED["t2_payload"],
        "base_checkpoint_manifest_payload_sha256": EXPECTED["base_payload"],
        "initialisation_manifest_payload_sha256": EXPECTED["initialisation_payload"],
        "initial_parameter_sha256": EXPECTED["initial_parameter"],
        "eligible_corpus_token_id_sha256": EXPECTED["token_ids"],
        "m2_contract_payload_sha256": EXPECTED["m2_payload"],
        "m3a_manifest_payload_sha256": EXPECTED["m3a_payload"],
        "m2a_gpu_environment_payload_sha256": EXPECTED["m2a_environment"],
        "m4a_preflight_first_group_order_sha256": EXPECTED["m4a_first_group_order"],
    },
    "fold": {
        "test_fold": TARGET_FOLD,
        "seed": FOLD_SEED,
        "train_n": len(train_rows),
        "test_n": len(test_rows),
        "llama_test_n": len(llama_test_rows),
        "test_hotels": sorted(test_hotels),
        "train_test_hotel_overlap": 0,
    },
    "training": {
        "epochs": EPOCHS,
        "max_length": MAX_LENGTH,
        "physical_batch_size": PHYSICAL_BATCH_SIZE,
        "accumulation_microbatches": ACCUMULATION_MICROBATCHES,
        "total_examples_consumed": training_examples,
        "total_microbatches": training_microbatches,
        "total_optimizer_updates": global_update,
        "warmup_updates": WARMUP_UPDATES,
        "initial_parameter_sha256": initial_parameter_hash,
        "final_parameter_sha256": final_parameter_hash,
        "save_reload_parameter_identity": True,
        "training_elapsed_seconds": training_elapsed,
        "epoch_row_order_sha256": [item["row_order_sha256"] for item in epoch_logs],
        "no_tuning": True,
        "no_early_stopping": True,
        "final_epoch_only": True,
    },
    "inference": {
        "primary_rows": len(primary_predictions),
        "llama_rows": len(llama_predictions),
        "inference_elapsed_seconds": inference_elapsed,
        "primary_metrics": primary_metrics,
        "primary_confusion_matrix": primary_confusion,
        "llama_ai_detection_recall": float(llama_predictions["detected_as_ai"].mean()),
        "score_language": "uncalibrated model score; not probability or calibrated confidence",
        "one_time_final_epoch_inference": True,
        "test_used_for_tuning_or_selection": False,
    },
    "resources": {
        "device": "cuda:0",
        "gpu_name": torch.cuda.get_device_name(0),
        "visible_gpu_count": torch.cuda.device_count(),
        "multi_gpu_used": False,
        "peak_allocated_bytes": peak_allocated_bytes,
        "peak_reserved_bytes": peak_reserved_bytes,
        "device_total_bytes": device_total_bytes,
    },
    "artifact_files": payload_registry,
}
manifest_payload_hash = canonical_sha256(manifest_payload)
manifest = {**manifest_payload, "manifest_payload_sha256": manifest_payload_hash}
write_json(temporary_root / "fold_1_manifest.json", manifest)

checksum_targets = sorted(path for path in temporary_root.rglob("*") if path.is_file())
checksum_text = "".join(
    f"{file_sha256(path)}  {path.relative_to(temporary_root).as_posix()}\n"
    for path in checksum_targets
)
(temporary_root / "SHA256SUMS.txt").write_text(checksum_text, encoding="utf-8", newline="\n")

require(not OUTPUT_ROOT.exists(), "output root appeared during execution")
temporary_root.replace(OUTPUT_ROOT)
temporary_parent.rmdir()

deterministic_zip(OUTPUT_ROOT, OUTPUT_ZIP)
zip_sha256 = file_sha256(OUTPUT_ZIP)
OUTPUT_SIDECAR.write_text(
    f"{zip_sha256}  {OUTPUT_ZIP.name}\n",
    encoding="utf-8",
    newline="\n",
)

final_files = sorted(path for path in OUTPUT_ROOT.rglob("*") if path.is_file())
for path in final_files:
    print(
        f"{path.relative_to(OUTPUT_ROOT).as_posix():64s} "
        f"{path.stat().st_size:12d} bytes  {file_sha256(path)}"
    )

print("Fold manifest payload SHA256:", manifest_payload_hash)
print("Final parameter SHA256      :", final_parameter_hash)

print("\n" + "=" * WIDTH)
print("STEP M4B DECISION: FOLD 1 DEBERTA TRAINING AND HELD-OUT INFERENCE FROZEN")
print("=" * WIDTH)
print("Output directory :", OUTPUT_ROOT)
print("Artifact ZIP     :", OUTPUT_ZIP)
print("ZIP SHA256       :", zip_sha256)
print("Checksum sidecar :", OUTPUT_SIDECAR)
print("Primary test rows:", len(primary_predictions))
print("Llama test rows  :", len(llama_predictions))
print("Tuning/selection : NONE")
print("Download the ZIP and .sha256.txt, verify externally, and preserve the complete output.")
print("STOP — do not train Fold 2 until this artifact and output are externally verified and audited.")


PHASE 2B — STEP M4B: FOLD 1 FROZEN DEBERTA TRAINING AND ONE-TIME HELD-OUT INFERENCE
Python       : 3.12.13
PyTorch      : 2.10.0+cu128
Transformers : 5.0.0
Target fold  : 1
Policy       : final epoch only; no tuning, validation, early stopping or test monitoring

ENVIRONMENT AND GPU IDENTITY
--------------------------------------------------------------------------------------------------------------------------------
numpy             : 2.0.2        PASS
pandas            : 2.3.3        PASS
scikit-learn      : 1.6.1        PASS
transformers      : 5.0.0        PASS
tokenizers        : 0.22.2       PASS
accelerate        : 1.13.0       PASS
safetensors       : 0.7.0        PASS
sentencepiece     : 0.2.1        PASS
Selected device       : cuda:0
GPU 0                 : Tesla T4
Visible GPUs          : 2 (cuda:1 unused)
CUDA/cuDNN            : 12.8 / 91002
Deterministic kernels : True
Numeric policy        : FP32; TF32/autocast/GradScaler disabled

UPSTREAM ARTIFACT AND CONTRACT CONTIN

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Initial parameter SHA256: 7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b PASS
Parameters             : 184424451
Epochs/max length      : 4 / 256
Batch/accumulation     : 2 / 16
Updates/warmup         : 240 / 24
Optimizer              : AdamW, foreach=False, fused=False
Test monitoring        : PROHIBITED and not implemented

FROZEN FOLD-1 TRAINING
--------------------------------------------------------------------------------------------------------------------------------
Epoch 1/4 | update 20/60 | global 020/240 | mean loss 1.025822
Epoch 1/4 | update 40/60 | global 040/240 | mean loss 0.910267
Epoch 1/4 | update 60/60 | global 060/240 | mean loss 0.813420
Epoch 1 complete | loss=0.813420 | elapsed=274.6s | order=df33d467d5a6f84b...
Epoch 2/4 | update 20/60 | global 080/240 | mean loss 0.505630
Epoch 2/4 | update 40/60 | global 100/240 | mean loss 0.485580
Epoch 2/4 | update 60/60 | global 120/240 | mean loss 0.456538
Epoch 2 complete | loss=0.456538 | elapsed=282

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

SHA256SUMS.txt                                                           1415 bytes  6ae0f91ebb2ca5daea88e71c8770f936ebb5d6199e982ee1b6d196f126a0b357
fold_1_confusion_matrix.json                                              189 bytes  eb3ef229128ff837a0ae978aa4e7a7cd40c04233ca2cda962b488316a52d3fe9
fold_1_epoch_training_log.csv                                             901 bytes  228eac1bb3f7f964fd4931303a3a0beea7599852d78923fd8434a8ceeed3e5af
fold_1_llama_sensitivity_predictions.csv                                41503 bytes  bd768a584534cbdd3d896db2e459a002f4336fde4f4e533adcd430a197070707
fold_1_llama_sensitivity_summary.csv                                     1209 bytes  30d51bce2876bf6021212dd7bfdfb3a272a830d1016c01499859e0e20b4e3885
fold_1_manifest.json                                                     6049 bytes  1acadb3ff171bbdfde1ad9efa69e0904cd2491735d334ee077f3fc2e1118177d
fold_1_optimizer_update_log.csv                                         45227 bytes  5326ccf66d37c35

Preliminary result audit
All 240 optimizer updates completed.
Training loss decreased from 0.813420 to 0.236942.
Fold-1 macro-F1: 0.824907.
Fold-1 TF-IDF macro-F1 was 0.863515.
Fold-1 difference: approximately −0.038608 in favour of TF-IDF.
Llama AI-detection recall: 0.512500, equivalent to 82 of 160 reviews.
No conclusion should be drawn from one fold.
Hyperparameters must remain unchanged despite the lower Fold-1 performance.

## PHASE 2B — STEP M4C: RE-UPLOADED FOLD-1 DEBERTA ARTIFACT VERIFICATION

In [3]:
from pathlib import Path
from collections import Counter
from importlib.metadata import version as package_version
import gc
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize
from transformers import AutoModelForSequenceClassification, AutoTokenizer


WIDTH = 128
INPUT_ROOT = Path("/kaggle/input")

EXPECTED = {
    "t2_payload": "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8",
    "base_payload": "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c",
    "initialisation_payload": "171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45",
    "initial_parameter": "7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b",
    "pretokenizer": "9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171",
    "token_ids": "9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359",
    "m2_payload": "53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202",
    "m3a_payload": "46118cc8794e5568a4cea30a930a17114c493e15cce5dc6b3947c007b1f5845e",
    "m2a_environment": "3e07e214b297576db434188e6c55e576fdd4af362e639f31f7654620d9ecbc7a",
    "m4a_first_group_order": "a84958952ce50907768cd488a5ebe9c737bc0ab29771a436723ce796ce3556fe",
    "m4a_first_group_loss": 35.929726839,
    "m4b_payload": "fc07453492cc661b7b2033613d868660ba34231c9f94bc9cdcbe841539d4d578",
    "m4b_zip": "a7e579c3f9d677dfa6ddd12dedbd1f911c981345754ec04974787abdeee01514",
    "final_parameter": "f488d4e4f54185ef1ad382a52273a4fb4ce33ce4e619eaf70206a48f662fe207",
}

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "transformers": "5.0.0",
    "tokenizers": "0.22.2",
    "safetensors": "0.7.0",
    "sentencepiece": "0.2.1",
}

EXPECTED_FILES = {
    "SHA256SUMS.txt": (1415, "6ae0f91ebb2ca5daea88e71c8770f936ebb5d6199e982ee1b6d196f126a0b357"),
    "fold_1_confusion_matrix.json": (189, "eb3ef229128ff837a0ae978aa4e7a7cd40c04233ca2cda962b488316a52d3fe9"),
    "fold_1_epoch_training_log.csv": (901, "228eac1bb3f7f964fd4931303a3a0beea7599852d78923fd8434a8ceeed3e5af"),
    "fold_1_llama_sensitivity_predictions.csv": (41503, "bd768a584534cbdd3d896db2e459a002f4336fde4f4e533adcd430a197070707"),
    "fold_1_llama_sensitivity_summary.csv": (1209, "30d51bce2876bf6021212dd7bfdfb3a272a830d1016c01499859e0e20b4e3885"),
    "fold_1_manifest.json": (6049, "1acadb3ff171bbdfde1ad9efa69e0904cd2491735d334ee077f3fc2e1118177d"),
    "fold_1_optimizer_update_log.csv": (45227, "5326ccf66d37c35610613090690c588e8364f27a68c778dafacc0860994dcb5a"),
    "fold_1_primary_metrics.json": (655, "8e34dbddc6e797896b7356c685ce6b70b5e2a9303369fea4fd5bc78ff249e13d"),
    "fold_1_primary_predictions.csv": (118558, "a7bcc9af46d3ee159819df206e1f736e0c34d4a25e2ed6c8f3e0496130f0609f"),
    "fold_1_truncation_audit.csv": (281633, "1484d307ea39a9793ae06ac9018ef820d07e67c7f6c4f2a290dfc015e63425cc"),
    "fold_1_truncation_summary.csv": (1335, "051f9d7303db1270c919a23bd6b195dc63f3bb641ee7d62801dbb7600e09a00d"),
    "implementation_record.json": (680, "92034ee58a2c105c5913bab4e1d81e9cc860ee9208e104d768928688133da09c"),
    "model/config.json": (1171, "7c3e21e3737140b8e9cc404cad5392d890becac306251a5f71362f96ccc5351c"),
    "model/model.safetensors": (737722332, "690fc24a3d95deb99256ace4c9784c78faea6a71f4e966d30b2d82e1708f8eaa"),
    "model/tokenizer.json": (8339358, "82eb5d659480d9778ab1e9e3e758406763f0e74ea0378f7760f5df6741ea8056"),
    "model/tokenizer_config.json": (508, "53f3b008c8696b8863912f9134436172f0ac6f15163c720fe1f976bcedbc5f8c"),
}

TARGET_FOLD = 1
FOLD_SEED = 43
LABEL_ORDER = [0, 1, 2]
LABEL2ID = {
    "presumed_truthful": 0,
    "human_deceptive": 1,
    "ai_generated": 2,
}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}

PRIMARY_COLUMNS = [
    "row_id", "official_fold", "hotel", "polarity", "source_family",
    "scientific_class", "y_true", "y_pred", "logit_0", "logit_1",
    "logit_2", "score_0", "score_1", "score_2", "maximum_score",
    "correct", "model_name", "fold_seed",
]
LLAMA_COLUMNS = [
    "row_id", "seed_family_id", "matched_gpt3_row_id", "official_fold",
    "hotel", "polarity", "reference_label_id", "y_pred", "logit_0",
    "logit_1", "logit_2", "score_0", "score_1", "score_2",
    "maximum_score", "detected_as_ai", "model_name", "fold_seed",
]


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8", errors="strict"))


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    encoded = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def verify_payload(path, field, expected):
    document = read_json(path)
    stored = document.get(field)
    payload = {key: value for key, value in document.items() if key != field}
    require(stored == expected, f"{path.name}: stored payload differs")
    require(canonical_sha256(payload) == expected, f"{path.name}: recomputed payload differs")
    return document


def find_unique(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    require(len(matches) == 1, f"expected one {filename}; found {len(matches)}")
    return matches[0]


def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8", errors="strict", newline="") as handle:
        for line_number, line in enumerate(handle, start=1):
            require(line.endswith("\n"), f"{path.name}:{line_number} lacks LF termination")
            require(line.strip(), f"{path.name}:{line_number} is empty")
            rows.append(json.loads(line))
    return rows


def parse_checksum_registry(path):
    entries = {}
    for line_number, line in enumerate(
        path.read_text(encoding="utf-8", errors="strict").splitlines(),
        start=1,
    ):
        require("  " in line, f"SHA256SUMS line {line_number} is malformed")
        digest, relative_name = line.split("  ", 1)
        require(len(digest) == 64, f"SHA256SUMS line {line_number} has invalid digest")
        require(relative_name not in entries, f"duplicate checksum entry: {relative_name}")
        entries[relative_name] = digest.lower()
    return entries


def parameter_sha256(model):
    digest = hashlib.sha256()
    state = model.state_dict()
    for name in sorted(state):
        tensor = state[name].detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(tensor.dtype).encode("ascii"))
        digest.update(b"\0")
        digest.update(json.dumps(list(tensor.shape), separators=(",", ":")).encode("ascii"))
        digest.update(b"\0")
        digest.update(tensor.numpy().tobytes(order="C"))
        digest.update(b"\0")
    return digest.hexdigest()


def tokenizer_backend_identity(tokenizer):
    backend = json.loads(tokenizer.backend_tokenizer.to_str())
    pretokenizer = backend.get("pre_tokenizer")
    require(pretokenizer is not None, "tokenizer backend has no pre-tokenizer")
    encoded = json.dumps(
        pretokenizer,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    require("Metaspace" in encoded, "native DeBERTa Metaspace pre-tokenizer is absent")
    require('"Split"' not in encoded, "Mistral Split regex was applied")
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()


def token_id_fingerprint_and_lengths(tokenizer, rows):
    digest = hashlib.sha256()
    lengths = {}
    for row in rows:
        token_ids = tokenizer(
            row["text"],
            add_special_tokens=True,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            verbose=False,
        )["input_ids"]
        digest.update(row["row_id"].encode("utf-8"))
        digest.update(b"\0")
        digest.update(json.dumps(token_ids, separators=(",", ":")).encode("ascii"))
        digest.update(b"\n")
        lengths[row["row_id"]] = len(token_ids)
    return digest.hexdigest(), lengths


def row_order_sha256(rows):
    return hashlib.sha256("\n".join(row["row_id"] for row in rows).encode("utf-8")).hexdigest()


def classification_metrics(y_true, y_pred, scores):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average=None,
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average="macro",
        zero_division=0,
    )
    result = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }
    for position, label in enumerate(LABEL_ORDER):
        result[f"class_{label}_precision"] = float(precision[position])
        result[f"class_{label}_recall"] = float(recall[position])
        result[f"class_{label}_f1"] = float(f1[position])
        result[f"class_{label}_support"] = int(support[position])
    binary = label_binarize(y_true, classes=LABEL_ORDER)
    result["macro_ovr_roc_auc"] = float(
        roc_auc_score(binary, scores, average="macro", multi_class="ovr")
    )
    return result


def assert_metric_dict(actual, expected, context):
    require(set(actual) == set(expected), f"{context}: metric keys differ")
    for key in expected:
        if isinstance(expected[key], int):
            require(int(actual[key]) == expected[key], f"{context}: {key} differs")
        else:
            require(
                np.isclose(float(actual[key]), float(expected[key]), rtol=0.0, atol=1e-12),
                f"{context}: {key} differs: {actual[key]} != {expected[key]}",
            )


def llama_summary(frame, group_type, group_value):
    predictions = frame["y_pred"].to_numpy(dtype=int)
    result = {
        "group_type": group_type,
        "group_value": str(group_value),
        "n": int(len(frame)),
        "ai_detection_recall": float((predictions == 2).mean()),
        "mean_ai_score": float(frame["score_2"].mean()),
        "median_ai_score": float(frame["score_2"].median()),
        "mean_maximum_score": float(frame["maximum_score"].mean()),
    }
    for label in LABEL_ORDER:
        count = int((predictions == label).sum())
        result[f"predicted_class_{label}_n"] = count
        result[f"predicted_class_{label}_share"] = float(count / len(frame))
    return result


print("=" * WIDTH)
print("PHASE 2B — STEP M4C: RE-UPLOADED FOLD-1 DEBERTA ARTIFACT VERIFICATION")
print("=" * WIDTH)
print("Python     :", sys.version.split()[0])
print("Input root :", INPUT_ROOT)
print("Execution  : CPU read-only; no fitting, CUDA access or model inference")

require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")
require(torch.__version__ == "2.10.0+cpu", f"expected CPU PyTorch 2.10.0; found {torch.__version__}")
require(torch.version.cuda is None, "CUDA-enabled PyTorch is active; switch accelerator off")

section("ENVIRONMENT AND UPCONTINUITY")
for name, expected in EXPECTED_PACKAGES.items():
    actual = package_version(name)
    require(actual == expected, f"{name} version differs: {actual} != {expected}")
    print(f"{name:18s}: {actual:12s} PASS")

t2_path = find_unique("corpus_manifest.json")
base_path = find_unique("checkpoint_manifest.json")
init_path = find_unique("initialisation_manifest.json")
m2_path = find_unique("contract_manifest.json")
m3a_path = find_unique("baseline_manifest.json")
m4b_path = find_unique("fold_1_manifest.json")

t2_manifest = verify_payload(t2_path, "manifest_payload_sha256", EXPECTED["t2_payload"])
base_manifest = verify_payload(base_path, "manifest_payload_sha256", EXPECTED["base_payload"])
init_manifest = verify_payload(init_path, "manifest_payload_sha256", EXPECTED["initialisation_payload"])
m2_manifest = verify_payload(m2_path, "contract_payload_sha256", EXPECTED["m2_payload"])
m3a_manifest = verify_payload(m3a_path, "manifest_payload_sha256", EXPECTED["m3a_payload"])
m4b_manifest = verify_payload(m4b_path, "manifest_payload_sha256", EXPECTED["m4b_payload"])

upstream = m4b_manifest["upstream"]
require(upstream["t2_manifest_payload_sha256"] == EXPECTED["t2_payload"], "M4B/T2 link differs")
require(upstream["base_checkpoint_manifest_payload_sha256"] == EXPECTED["base_payload"], "M4B/base link differs")
require(upstream["initialisation_manifest_payload_sha256"] == EXPECTED["initialisation_payload"], "M4B/initialisation link differs")
require(upstream["initial_parameter_sha256"] == EXPECTED["initial_parameter"], "M4B initial-parameter link differs")
require(upstream["eligible_corpus_token_id_sha256"] == EXPECTED["token_ids"], "M4B token link differs")
require(upstream["m2_contract_payload_sha256"] == EXPECTED["m2_payload"], "M4B/M2 link differs")
require(upstream["m3a_manifest_payload_sha256"] == EXPECTED["m3a_payload"], "M4B/M3A link differs")
require(upstream["m2a_gpu_environment_payload_sha256"] == EXPECTED["m2a_environment"], "M4B/M2A link differs")
require(upstream["m4a_preflight_first_group_order_sha256"] == EXPECTED["m4a_first_group_order"], "M4B/M4A link differs")

print("T2 payload             :", EXPECTED["t2_payload"], "PASS")
print("Base payload           :", EXPECTED["base_payload"], "PASS")
print("Initialisation payload :", EXPECTED["initialisation_payload"], "PASS")
print("M2 payload             :", EXPECTED["m2_payload"], "PASS")
print("M3A payload            :", EXPECTED["m3a_payload"], "PASS")
print("M4B payload            :", EXPECTED["m4b_payload"], "PASS")

section("FOLD-1 INPUT DISCOVERY AND EXTERNAL ZIP RECORD")
M4B_ROOT = m4b_path.parent
sidecar_candidates = sorted(INPUT_ROOT.rglob("phase2B_M4B_FOLD_1_DEBERTA.zip.sha256*.txt"))
valid_sidecars = []
for candidate in sidecar_candidates:
    tokens = candidate.read_text(encoding="utf-8", errors="strict").strip().split()
    if tokens and tokens[0].lower() == EXPECTED["m4b_zip"]:
        valid_sidecars.append(candidate)
require(
    len(valid_sidecars) == 1,
    f"expected one valid Fold-1 checksum sidecar; found {len(valid_sidecars)} among {len(sidecar_candidates)} candidates",
)
mounted_zips = sorted(INPUT_ROOT.rglob("phase2B_M4B_FOLD_1_DEBERTA.zip"))
require(len(mounted_zips) <= 1, f"multiple mounted Fold-1 ZIPs found: {len(mounted_zips)}")
if mounted_zips:
    require(file_sha256(mounted_zips[0]) == EXPECTED["m4b_zip"], "mounted Fold-1 ZIP checksum differs")
    zip_status = "BYTE_VERIFIED_IN_RUNTIME"
else:
    zip_status = "Kaggle-extracted; externally verified ZIP recorded"

print("Fold-1 root          :", M4B_ROOT)
print("Checksum sidecar     :", valid_sidecars[0])
print("Recorded ZIP SHA256  :", EXPECTED["m4b_zip"])
print("ZIP runtime status   :", zip_status)

section("COMPLETE 16-FILE ARTIFACT REGISTRY")
actual_files = {
    path.relative_to(M4B_ROOT).as_posix()
    for path in M4B_ROOT.rglob("*")
    if path.is_file()
}
require(actual_files == set(EXPECTED_FILES), f"Fold-1 file set differs: {sorted(actual_files)}")
for relative_name, (expected_size, expected_hash) in EXPECTED_FILES.items():
    path = M4B_ROOT / relative_name
    require(path.stat().st_size == expected_size, f"size mismatch: {relative_name}")
    require(file_sha256(path) == expected_hash, f"checksum mismatch: {relative_name}")
    print(f"{relative_name:64s} {expected_size:12d} bytes  PASS")

registry = parse_checksum_registry(M4B_ROOT / "SHA256SUMS.txt")
expected_registry_names = set(EXPECTED_FILES) - {"SHA256SUMS.txt"}
require(set(registry) == expected_registry_names, "SHA256SUMS membership differs")
for relative_name in expected_registry_names:
    require(registry[relative_name] == EXPECTED_FILES[relative_name][1], f"SHA256SUMS digest differs: {relative_name}")
require(set(m4b_manifest["artifact_files"]) == expected_registry_names - {"fold_1_manifest.json"}, "manifest artifact registry differs")
for relative_name, identity in m4b_manifest["artifact_files"].items():
    require(identity["bytes"] == EXPECTED_FILES[relative_name][0], f"manifest size differs: {relative_name}")
    require(identity["sha256"] == EXPECTED_FILES[relative_name][1], f"manifest hash differs: {relative_name}")
print("SHA256SUMS entries    :", len(registry), "PASS")
print("Manifest file entries:", len(m4b_manifest["artifact_files"]), "PASS")

section("CORPUS ROUTING AND PREDICTION TABLES")
T2_ROOT = t2_path.parent
primary_file_order = read_jsonl(T2_ROOT / "primary_tripartite_corpus.jsonl")
llama_file_order = read_jsonl(T2_ROOT / "llama_unseen_generator_sensitivity.jsonl")
primary_rows = sorted(primary_file_order, key=lambda row: row["row_id"])
llama_rows = sorted(llama_file_order, key=lambda row: row["row_id"])
train_rows = [row for row in primary_rows if int(row["official_fold"]) != TARGET_FOLD]
test_rows = [row for row in primary_rows if int(row["official_fold"]) == TARGET_FOLD]
llama_test_rows = [row for row in llama_rows if int(row["official_fold"]) == TARGET_FOLD]
require((len(train_rows), len(test_rows), len(llama_test_rows)) == (1917, 479, 160), "Fold-1 route counts differ")
require(not ({row["hotel"] for row in train_rows} & {row["hotel"] for row in test_rows}), "hotel leakage detected")

primary_predictions = pd.read_csv(M4B_ROOT / "fold_1_primary_predictions.csv")
llama_predictions = pd.read_csv(M4B_ROOT / "fold_1_llama_sensitivity_predictions.csv")
require(primary_predictions.columns.tolist() == PRIMARY_COLUMNS, "primary prediction schema differs")
require(llama_predictions.columns.tolist() == LLAMA_COLUMNS, "Llama prediction schema differs")
require(len(primary_predictions) == 479 and len(llama_predictions) == 160, "prediction row counts differ")
require(primary_predictions["row_id"].tolist() == [row["row_id"] for row in test_rows], "primary OOF coverage/order differs")
require(llama_predictions["row_id"].tolist() == [row["row_id"] for row in llama_test_rows], "Llama coverage/order differs")
require(primary_predictions["row_id"].is_unique and llama_predictions["row_id"].is_unique, "prediction row IDs are duplicated")

primary_by_id = {row["row_id"]: row for row in test_rows}
llama_by_id = {row["row_id"]: row for row in llama_test_rows}
gpt3_by_seed = {
    row["seed_family_id"]: row
    for row in primary_rows
    if row["source_family"] == "op_spam_AI_GPT3"
}
for item in primary_predictions.itertuples(index=False):
    source = primary_by_id[item.row_id]
    require(int(item.official_fold) == int(source["official_fold"]) == 1, f"{item.row_id}: fold differs")
    require(item.hotel == source["hotel"] and item.polarity == source["polarity"], f"{item.row_id}: metadata differs")
    require(item.source_family == source["source_family"], f"{item.row_id}: source differs")
    require(item.scientific_class == source["scientific_class"], f"{item.row_id}: class differs")
    require(int(item.y_true) == int(source["label_id"]), f"{item.row_id}: label differs")
for item in llama_predictions.itertuples(index=False):
    source = llama_by_id[item.row_id]
    require(item.seed_family_id == source["seed_family_id"], f"{item.row_id}: seed differs")
    require(item.matched_gpt3_row_id == gpt3_by_seed[item.seed_family_id]["row_id"], f"{item.row_id}: GPT-3 match differs")
    require(int(item.reference_label_id) == 2, f"{item.row_id}: reference label differs")

for frame, name, is_llama in (
    (primary_predictions, "primary", False),
    (llama_predictions, "Llama", True),
):
    logits = frame[["logit_0", "logit_1", "logit_2"]].to_numpy(dtype=np.float64)
    scores = frame[["score_0", "score_1", "score_2"]].to_numpy(dtype=np.float64)
    require(np.isfinite(logits).all() and np.isfinite(scores).all(), f"{name}: non-finite values")
    require(np.allclose(scores.sum(axis=1), 1.0, rtol=0.0, atol=1e-6), f"{name}: scores do not sum to one")
    decisions = np.argmax(logits, axis=1)
    require(np.array_equal(decisions, np.argmax(scores, axis=1)), f"{name}: logit/score decisions differ")
    require(np.array_equal(decisions, frame["y_pred"].to_numpy(dtype=int)), f"{name}: stored decisions differ")
    require(np.allclose(scores.max(axis=1), frame["maximum_score"], rtol=0.0, atol=1e-12), f"{name}: maximum score differs")
    if is_llama:
        require(np.array_equal((decisions == 2).astype(int), frame["detected_as_ai"].to_numpy(dtype=int)), "Llama detection flag differs")
    else:
        expected_correct = (decisions == frame["y_true"].to_numpy(dtype=int)).astype(int)
        require(np.array_equal(expected_correct, frame["correct"].to_numpy(dtype=int)), "primary correctness flag differs")
    require(set(frame["model_name"]) == {"deberta_v3_base"}, f"{name}: model name differs")
    require(set(frame["fold_seed"].astype(int)) == {43}, f"{name}: fold seed differs")

print("Primary predictions  : 479 complete and unique PASS")
print("Llama predictions    : 160 complete and unique PASS")
print("Hotel overlap        : 0 PASS")
print("Score/decision checks: PASS")

section("INDEPENDENT METRIC AND SENSITIVITY RECOMPUTATION")
stored_metrics = read_json(M4B_ROOT / "fold_1_primary_metrics.json")
recomputed_metrics = classification_metrics(
    primary_predictions["y_true"],
    primary_predictions["y_pred"],
    primary_predictions[["score_0", "score_1", "score_2"]],
)
assert_metric_dict(recomputed_metrics, stored_metrics, "primary metrics file")
assert_metric_dict(recomputed_metrics, m4b_manifest["inference"]["primary_metrics"], "manifest primary metrics")
stored_confusion = read_json(M4B_ROOT / "fold_1_confusion_matrix.json")
recomputed_confusion = confusion_matrix(
    primary_predictions["y_true"],
    primary_predictions["y_pred"],
    labels=LABEL_ORDER,
).astype(int).tolist()
require(stored_confusion == {"label_order": LABEL_ORDER, "matrix": recomputed_confusion}, "confusion matrix differs")
require(m4b_manifest["inference"]["primary_confusion_matrix"] == recomputed_confusion, "manifest confusion differs")

recomputed_llama = [llama_summary(llama_predictions, "all", "all")]
for dimension in ("polarity", "hotel"):
    for value in sorted(llama_predictions[dimension].unique(), key=str):
        recomputed_llama.append(
            llama_summary(llama_predictions[llama_predictions[dimension] == value], dimension, value)
        )
recomputed_llama = pd.DataFrame(recomputed_llama)
stored_llama = pd.read_csv(M4B_ROOT / "fold_1_llama_sensitivity_summary.csv")
try:
    # The original summary aggregated float32 model scores; CSV reload promotes
    # those exact stored values to float64, so allow only float32-scale drift.
    pd.testing.assert_frame_equal(stored_llama, recomputed_llama, check_dtype=False, rtol=0.0, atol=1e-7)
except AssertionError as error:
    raise RuntimeError("STOP — Llama sensitivity summary differs") from error

print("Primary macro-F1     :", f"{recomputed_metrics['macro_f1']:.6f}", "PASS")
print("Primary balanced acc.:", f"{recomputed_metrics['balanced_accuracy']:.6f}", "PASS")
print("Primary accuracy     :", f"{recomputed_metrics['accuracy']:.6f}", "PASS")
print("Llama AI detection   :", f"{llama_predictions['detected_as_ai'].mean():.6f}", "PASS")
print("Llama 3-class accuracy: NOT COMPUTED (contract compliant)")

section("TRAINING-SCHEDULE AND ROW-ORDER RECONSTRUCTION")
epoch_log = pd.read_csv(M4B_ROOT / "fold_1_epoch_training_log.csv")
update_log = pd.read_csv(M4B_ROOT / "fold_1_optimizer_update_log.csv")
require(len(epoch_log) == 4 and len(update_log) == 240, "training-log dimensions differ")
require(epoch_log["epoch"].astype(int).tolist() == [1, 2, 3, 4], "epoch sequence differs")
require(update_log["global_update"].astype(int).tolist() == list(range(1, 241)), "global update sequence differs")
require(np.isfinite(update_log.select_dtypes(include=[np.number]).to_numpy()).all(), "non-finite optimizer log value")
require(np.isfinite(epoch_log.select_dtypes(include=[np.number]).to_numpy()).all(), "non-finite epoch log value")

def schedule_multiplier(step):
    if step < 24:
        return step / 24
    return max(0.0, (240 - step) / (240 - 24))

for row in update_log.itertuples(index=False):
    step = int(row.global_update)
    require(np.isclose(row.learning_rate_used, 2e-5 * schedule_multiplier(step - 1), rtol=0.0, atol=1e-18), f"step {step}: LR used differs")
    require(np.isclose(row.learning_rate_after_step, 2e-5 * schedule_multiplier(step), rtol=0.0, atol=1e-18), f"step {step}: LR after differs")
    require(np.isclose(row.mean_loss, row.loss_sum / row.group_examples, rtol=0.0, atol=1e-12), f"step {step}: mean loss differs")

for zero_epoch in range(4):
    epoch_number = zero_epoch + 1
    order = np.random.default_rng(FOLD_SEED + zero_epoch).permutation(len(train_rows))
    ordered_rows = [train_rows[int(index)] for index in order]
    epoch_row = epoch_log[epoch_log["epoch"] == epoch_number].iloc[0]
    require(epoch_row["row_order_sha256"] == row_order_sha256(ordered_rows), f"epoch {epoch_number}: row-order hash differs")
    updates = update_log[update_log["epoch"] == epoch_number].sort_values("epoch_update", kind="stable")
    require(updates["epoch_update"].astype(int).tolist() == list(range(1, 61)), f"epoch {epoch_number}: update sequence differs")
    require(int(updates["group_examples"].sum()) == 1917, f"epoch {epoch_number}: examples differ")
    require(int(updates["microbatches"].sum()) == 959, f"epoch {epoch_number}: microbatches differ")
    require(Counter(updates["group_examples"].astype(int)) == Counter({32: 59, 29: 1}), f"epoch {epoch_number}: group sizes differ")
    require(Counter(updates["microbatches"].astype(int)) == Counter({16: 59, 15: 1}), f"epoch {epoch_number}: microbatch groups differ")
    for group_index, update in enumerate(updates.itertuples(index=False)):
        start = group_index * 32
        group_rows = [train_rows[int(index)] for index in order[start:start + 32]]
        require(update.group_row_order_sha256 == row_order_sha256(group_rows), f"epoch {epoch_number}, update {group_index + 1}: group hash differs")
    require(np.isclose(updates["loss_sum"].sum(), epoch_row["loss_sum"], rtol=0.0, atol=1e-10), f"epoch {epoch_number}: loss sum differs")
    require(np.isclose(epoch_row["mean_loss"], epoch_row["loss_sum"] / 1917, rtol=0.0, atol=1e-12), f"epoch {epoch_number}: mean loss differs")

first_update = update_log.iloc[0]
require(first_update["group_row_order_sha256"] == EXPECTED["m4a_first_group_order"], "M4A first-group order differs")
require(np.isclose(first_update["loss_sum"], EXPECTED["m4a_first_group_loss"], rtol=0.0, atol=1e-6), "M4A first-group loss differs")
require(int(update_log["group_examples"].sum()) == 1917 * 4, "total training examples differ")
require(int(update_log["microbatches"].sum()) == 959 * 4, "total training microbatches differ")

training_manifest = m4b_manifest["training"]
require(training_manifest["total_examples_consumed"] == 1917 * 4, "manifest examples differ")
require(training_manifest["total_microbatches"] == 959 * 4, "manifest microbatches differ")
require(training_manifest["total_optimizer_updates"] == 240, "manifest updates differ")
require(training_manifest["warmup_updates"] == 24, "manifest warmup differs")
require(training_manifest["no_tuning"] is True, "manifest tuning policy differs")
require(training_manifest["no_early_stopping"] is True, "manifest early-stopping policy differs")
require(training_manifest["final_epoch_only"] is True, "manifest checkpoint policy differs")

print("Epochs/updates         : 4 / 240 PASS")
print("Examples/microbatches  :", 1917 * 4, "/", 959 * 4, "PASS")
print("All 240 row-order groups: PASS")
print("Warm-up/decay schedule : PASS")
print("M4A first group        : reproduced PASS")

section("SAVED TOKENIZER, MODEL AND TRUNCATION AUDIT")
MODEL_ROOT = M4B_ROOT / "model"
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_ROOT), local_files_only=True)
pretokenizer_hash = tokenizer_backend_identity(tokenizer)
require(pretokenizer_hash == EXPECTED["pretokenizer"], "saved pre-tokenizer fingerprint differs")
token_hash, token_lengths = token_id_fingerprint_and_lengths(tokenizer, primary_file_order + llama_file_order)
require(token_hash == EXPECTED["token_ids"], "saved tokenizer changes the 3,186-row token IDs")

model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_ROOT), local_files_only=True)
model.to(device="cpu", dtype=torch.float32)
model.eval()
require(model.config.num_labels == 3, "saved model output dimension differs")
require(model.config.label2id == LABEL2ID, "saved model label2id differs")
require({int(key): value for key, value in model.config.id2label.items()} == ID2LABEL, "saved model id2label differs")
final_parameter_hash = parameter_sha256(model)
require(final_parameter_hash == EXPECTED["final_parameter"], "saved final parameter fingerprint differs")
require(m4b_manifest["training"]["final_parameter_sha256"] == EXPECTED["final_parameter"], "manifest final parameter differs")
require(m4b_manifest["training"]["initial_parameter_sha256"] == EXPECTED["initial_parameter"], "manifest initial parameter differs")
require(m4b_manifest["training"]["save_reload_parameter_identity"] is True, "manifest reload identity differs")

truncation = pd.read_csv(M4B_ROOT / "fold_1_truncation_audit.csv")
require(len(truncation) == 3186 and truncation["row_id"].is_unique, "truncation audit coverage differs")
require(truncation["row_id"].tolist() == sorted(token_lengths), "truncation audit order/IDs differ")
expected_lengths = np.asarray([token_lengths[row_id] for row_id in truncation["row_id"]], dtype=int)
require(np.array_equal(truncation["pretruncation_tokens"].to_numpy(dtype=int), expected_lengths), "pre-truncation lengths differ")
require(np.array_equal(truncation["truncated_at_256"].to_numpy(dtype=int), (expected_lengths > 256).astype(int)), "truncation flags differ")
require(int(truncation["truncated_at_256"].sum()) == 259, "truncated-row count differs")

recomputed_truncation_summary = (
    truncation
    .groupby(["dataset_role", "scientific_class", "official_fold"], sort=True, dropna=False)
    .agg(
        n=("row_id", "size"),
        truncated_n=("truncated_at_256", "sum"),
        truncation_rate=("truncated_at_256", "mean"),
        minimum_tokens=("pretruncation_tokens", "min"),
        median_tokens=("pretruncation_tokens", "median"),
        maximum_tokens=("pretruncation_tokens", "max"),
    )
    .reset_index()
)
stored_truncation_summary = pd.read_csv(M4B_ROOT / "fold_1_truncation_summary.csv")
try:
    pd.testing.assert_frame_equal(
        stored_truncation_summary,
        recomputed_truncation_summary,
        check_dtype=False,
        rtol=0.0,
        atol=1e-12,
    )
except AssertionError as error:
    raise RuntimeError("STOP — truncation summary differs") from error

del model
del tokenizer
gc.collect()

print("Saved pre-tokenizer SHA256:", pretokenizer_hash, "PASS")
print("Saved token-ID SHA256     :", token_hash, "PASS")
print("Final parameter SHA256    :", final_parameter_hash, "PASS")
print("Truncation audit          : 3,186 rows / 259 truncated PASS")

print("\n" + "=" * WIDTH)
print("STEP M4C DECISION: PASS — RE-UPLOADED FOLD-1 DEBERTA ARTIFACT VERIFIED")
print("=" * WIDTH)
print("M4B manifest payload :", EXPECTED["m4b_payload"])
print("M4B recorded ZIP     :", EXPECTED["m4b_zip"])
print("Final parameter      :", EXPECTED["final_parameter"])
print("Files verified       : 16 / 16")
print("Predictions verified : 479 primary / 160 Llama")
print("Training reconstructed: 4 epochs / 240 updates / zero hotel leakage")
print("CUDA accessed        : False")
print("Training/inference   : NONE")
print("No file was written, transformed or modified.")
print("STOP — preserve and share this complete output before Fold 2 is authorised.")


PHASE 2B — STEP M4C: RE-UPLOADED FOLD-1 DEBERTA ARTIFACT VERIFICATION
Python     : 3.12.13
Input root : /kaggle/input
Execution  : CPU read-only; no fitting, CUDA access or model inference

ENVIRONMENT AND UPCONTINUITY
--------------------------------------------------------------------------------------------------------------------------------
numpy             : 2.0.2        PASS
pandas            : 2.3.3        PASS
scikit-learn      : 1.6.1        PASS
transformers      : 5.0.0        PASS
tokenizers        : 0.22.2       PASS
safetensors       : 0.7.0        PASS
sentencepiece     : 0.2.1        PASS
T2 payload             : b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8 PASS
Base payload           : da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c PASS
Initialisation payload : 171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45 PASS
M2 payload             : 53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202 PASS
M3A pa

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Saved pre-tokenizer SHA256: 9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171 PASS
Saved token-ID SHA256     : 9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359 PASS
Final parameter SHA256    : f488d4e4f54185ef1ad382a52273a4fb4ce33ce4e619eaf70206a48f662fe207 PASS
Truncation audit          : 3,186 rows / 259 truncated PASS

STEP M4C DECISION: PASS — RE-UPLOADED FOLD-1 DEBERTA ARTIFACT VERIFIED
M4B manifest payload : fc07453492cc661b7b2033613d868660ba34231c9f94bc9cdcbe841539d4d578
M4B recorded ZIP     : a7e579c3f9d677dfa6ddd12dedbd1f911c981345754ec04974787abdeee01514
Final parameter      : f488d4e4f54185ef1ad382a52273a4fb4ce33ce4e619eaf70206a48f662fe207
Files verified       : 16 / 16
Predictions verified : 479 primary / 160 Llama
Training reconstructed: 4 epochs / 240 updates / zero hotel leakage
CUDA accessed        : False
Training/inference   : NONE
No file was written, transformed or modified.
STOP — preserve and share this complete output before Fold 2 

M4C is a complete clean PASS. Fold 1 is now formally closed and independently verified.

Confirmed:

All 16 files passed byte-level verification.
All 479 primary and 160 Llama predictions passed.
Four epochs and 240 optimizer updates were reconstructed.
Hotel leakage: zero.
Final model parameters matched exactly.
No training or inference occurred during verification.
External ZIP provenance remains intact.

Fold-1 results are valid but preliminary:

DeBERTa macro-F1: 0.824907
TF-IDF Fold-1 macro-F1: 0.863515
Llama AI-detection recall: 0.512500

No overall model conclusion should be made until all five folds are completed and pooled.

Next authorised step: Fold 2 frozen DeBERTa training, starting again from the original frozen three-class initialisation—not the Fold-1 model—with seed 44, then one-time inference on Fold 2.

## PHASE 2B—STEP M4D: FOLD 2 FROZEN DEBERTA TRAINING AND ONE-TIME HELD-OUT INFERENCE

In [1]:
from pathlib import Path
from collections import Counter
from importlib.metadata import version as package_version
import gc
import hashlib
import json
import math
import os
import random
import shutil
import sys
import tempfile
import time
import zipfile


# Frozen deterministic CUDA policy. This must precede the first CUDA BLAS call.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


WIDTH = 128
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = WORKING_ROOT / "phase2B_M4D_FOLD_2_DEBERTA"
OUTPUT_ZIP = WORKING_ROOT / "phase2B_M4D_FOLD_2_DEBERTA.zip"
OUTPUT_SIDECAR = WORKING_ROOT / "phase2B_M4D_FOLD_2_DEBERTA.zip.sha256.txt"

EXPECTED = {
    "t2_payload": "b87c3945e0b50df788801c14cb9fcb2c88bf569e7c4556d9bfc168f8eb18cba8",
    "t2_primary_file": "b836eb86f1d34ed2db2c8fce4f1b04e6bec515632403962ba594e5c0adaba9ef",
    "t2_llama_file": "37500d0ae8b48b632c18f9b57c931e2f88e6793e51975c6ef0243ab179bca01a",
    "base_payload": "da21152ffc73c474d53485d3c0673eb0869cd50894d937acd1a2feb79eb8f33c",
    "initialisation_payload": "171154b7225ded4b88d0024a5ba20e2314bee3213dc0d16c3575a40bec13fd45",
    "initial_parameter": "7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b",
    "initial_model_file": "4682e5e84b95ce49bbaa05bac090d1edf38ac9f71d68b5327313b304425a036c",
    "pretokenizer": "9fb1975ab83e3fb658575a44de4af367dc46683d0a541a4f67dfb1ae4d414171",
    "token_ids": "9c29dc292fe7e89cc22fafbeb953d48ff64821037378bec25189c8f185bf8359",
    "m2_payload": "53fad9f300b6dbe6c10cc6951bfc746a3dd27415018d4da902ae2fccb8fda202",
    "m2_contract_file": "5979dc81c6e44eb869d47976a56cd7ee9311d8c400ffbc12e9cb9119bd09483a",
    "m3a_payload": "46118cc8794e5568a4cea30a930a17114c493e15cce5dc6b3947c007b1f5845e",
    "m4b_fold1_payload": "fc07453492cc661b7b2033613d868660ba34231c9f94bc9cdcbe841539d4d578",
    "m4b_fold1_zip": "a7e579c3f9d677dfa6ddd12dedbd1f911c981345754ec04974787abdeee01514",
    "m2a_environment": "3e07e214b297576db434188e6c55e576fdd4af362e639f31f7654620d9ecbc7a",
}

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scikit-learn": "1.6.1",
    "transformers": "5.0.0",
    "tokenizers": "0.22.2",
    "accelerate": "1.13.0",
    "safetensors": "0.7.0",
    "sentencepiece": "0.2.1",
}

TARGET_FOLD = 2
FOLD_SEED = 44
EXPECTED_TRAIN_N = 1917
EXPECTED_TEST_N = 479
EXPECTED_LLAMA_N = 158
EXPECTED_MICROBATCHES_PER_EPOCH = 959
EXPECTED_UPDATES_PER_EPOCH = 60
EPOCHS = 4
MAX_LENGTH = 256
PHYSICAL_BATCH_SIZE = 2
ACCUMULATION_MICROBATCHES = 16
MAXIMUM_EFFECTIVE_BATCH_SIZE = 32
INFERENCE_BATCH_SIZE = 16
TOTAL_UPDATES = 240
WARMUP_UPDATES = 24

LABEL2ID = {
    "presumed_truthful": 0,
    "human_deceptive": 1,
    "ai_generated": 2,
}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}
LABEL_ORDER = [0, 1, 2]
MODEL_NAME = "deberta_v3_base"


def require(condition, message):
    if not condition:
        raise RuntimeError("STOP — " + message)


def section(title):
    print("\n" + title)
    print("-" * WIDTH)


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8", errors="strict"))


def write_json(path, value):
    path.write_text(
        json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def write_csv(path, frame):
    frame.to_csv(path, index=False, lineterminator="\n", float_format="%.17g")


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_sha256(value):
    encoded = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def verify_payload(path, field, expected):
    document = read_json(path)
    stored = document.get(field)
    payload = {key: value for key, value in document.items() if key != field}
    require(stored == expected, f"{path.name}: stored payload differs")
    require(canonical_sha256(payload) == expected, f"{path.name}: recomputed payload differs")
    return document


def find_unique(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    require(len(matches) == 1, f"expected one {filename}; found {len(matches)}")
    return matches[0]


def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8", errors="strict", newline="") as handle:
        for line_number, line in enumerate(handle, start=1):
            require(line.endswith("\n"), f"{path.name}:{line_number} lacks LF termination")
            require(line.strip(), f"{path.name}:{line_number} is empty")
            row = json.loads(line)
            require(isinstance(row.get("row_id"), str), f"{path.name}:{line_number} missing row_id")
            require(isinstance(row.get("text"), str), f"{path.name}:{line_number} missing text")
            require(row["text"].strip() != "", f"{row['row_id']} has empty text")
            rows.append(row)
    return rows


def parameter_sha256(model):
    digest = hashlib.sha256()
    state = model.state_dict()
    for name in sorted(state):
        tensor = state[name].detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(tensor.dtype).encode("ascii"))
        digest.update(b"\0")
        digest.update(json.dumps(list(tensor.shape), separators=(",", ":")).encode("ascii"))
        digest.update(b"\0")
        digest.update(tensor.numpy().tobytes(order="C"))
        digest.update(b"\0")
    return digest.hexdigest()


def tokenizer_backend_identity(tokenizer):
    backend = json.loads(tokenizer.backend_tokenizer.to_str())
    pretokenizer = backend.get("pre_tokenizer")
    require(pretokenizer is not None, "tokenizer backend has no pre-tokenizer")
    encoded = json.dumps(
        pretokenizer,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    require("Metaspace" in encoded, "native DeBERTa Metaspace pre-tokenizer is absent")
    require('"Split"' not in encoded, "Mistral Split regex was applied")
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()


def token_id_fingerprint_and_lengths(tokenizer, rows):
    digest = hashlib.sha256()
    lengths = {}
    for row in rows:
        token_ids = tokenizer(
            row["text"],
            add_special_tokens=True,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            verbose=False,
        )["input_ids"]
        digest.update(row["row_id"].encode("utf-8"))
        digest.update(b"\0")
        digest.update(json.dumps(token_ids, separators=(",", ":")).encode("ascii"))
        digest.update(b"\n")
        lengths[row["row_id"]] = len(token_ids)
    require(len(lengths) == len(rows), "token-length index is incomplete")
    return digest.hexdigest(), lengths


def fixed_tokenize(tokenizer, rows):
    encoded = tokenizer(
        [row["text"] for row in rows],
        add_special_tokens=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_attention_mask=True,
        return_tensors="pt",
    )
    require("input_ids" in encoded and "attention_mask" in encoded, "required tokenizer outputs absent")
    require(tuple(encoded["input_ids"].shape) == (len(rows), MAX_LENGTH), "fixed token tensor shape differs")
    require(
        set(encoded.keys()).issubset({"input_ids", "attention_mask", "token_type_ids"}),
        f"unexpected tokenizer fields: {sorted(encoded.keys())}",
    )
    return {key: value.contiguous() for key, value in encoded.items()}


def row_order_sha256(rows):
    return hashlib.sha256("\n".join(row["row_id"] for row in rows).encode("utf-8")).hexdigest()


def all_finite_parameters(model):
    return all(
        bool(torch.isfinite(parameter).all().item())
        for parameter in model.parameters()
        if parameter.is_floating_point()
    )


def classification_metrics(y_true, y_pred, scores):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average=None,
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average="macro",
        zero_division=0,
    )
    result = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }
    for position, label in enumerate(LABEL_ORDER):
        result[f"class_{label}_precision"] = float(precision[position])
        result[f"class_{label}_recall"] = float(recall[position])
        result[f"class_{label}_f1"] = float(f1[position])
        result[f"class_{label}_support"] = int(support[position])
    binary = label_binarize(y_true, classes=LABEL_ORDER)
    result["macro_ovr_roc_auc"] = float(
        roc_auc_score(binary, scores, average="macro", multi_class="ovr")
    )
    return result


def infer(model, tokenizer, rows, device):
    model.eval()
    logits_blocks = []
    score_blocks = []
    with torch.no_grad():
        for start_index in range(0, len(rows), INFERENCE_BATCH_SIZE):
            batch_rows = rows[start_index:start_index + INFERENCE_BATCH_SIZE]
            encoded = tokenizer(
                [row["text"] for row in batch_rows],
                add_special_tokens=True,
                truncation=True,
                padding="max_length",
                max_length=MAX_LENGTH,
                return_attention_mask=True,
                return_tensors="pt",
            )
            inputs = {key: value.to(device) for key, value in encoded.items()}
            logits = model(**inputs).logits
            scores = torch.softmax(logits, dim=-1)
            require(bool(torch.isfinite(logits).all().item()), "non-finite inference logits")
            require(bool(torch.isfinite(scores).all().item()), "non-finite inference scores")
            logits_blocks.append(logits.detach().cpu().numpy())
            score_blocks.append(scores.detach().cpu().numpy())
    logits = np.concatenate(logits_blocks, axis=0).astype(np.float32, copy=False)
    scores = np.concatenate(score_blocks, axis=0).astype(np.float32, copy=False)
    require(logits.shape == (len(rows), 3), "inference logit array shape differs")
    require(scores.shape == (len(rows), 3), "inference score array shape differs")
    require(np.isfinite(logits).all(), "persisted inference logits are non-finite")
    require(np.isfinite(scores).all(), "persisted inference scores are non-finite")
    require(np.allclose(scores.sum(axis=1), 1.0, rtol=0.0, atol=1e-6), "scores do not sum to one")
    require(np.array_equal(np.argmax(logits, axis=1), np.argmax(scores, axis=1)), "logit/score decisions differ")
    return logits, scores


def primary_prediction_frame(rows, logits, scores):
    predicted = np.argmax(logits, axis=1).astype(int)
    frame = pd.DataFrame({
        "row_id": [row["row_id"] for row in rows],
        "official_fold": [int(row["official_fold"]) for row in rows],
        "hotel": [row["hotel"] for row in rows],
        "polarity": [row["polarity"] for row in rows],
        "source_family": [row["source_family"] for row in rows],
        "scientific_class": [row["scientific_class"] for row in rows],
        "y_true": [int(row["label_id"]) for row in rows],
        "y_pred": predicted,
        "logit_0": logits[:, 0],
        "logit_1": logits[:, 1],
        "logit_2": logits[:, 2],
        "score_0": scores[:, 0],
        "score_1": scores[:, 1],
        "score_2": scores[:, 2],
        "maximum_score": scores.max(axis=1),
        "model_name": MODEL_NAME,
        "fold_seed": FOLD_SEED,
    })
    frame["correct"] = (frame["y_true"].to_numpy(dtype=int) == predicted).astype(int)
    columns = [
        "row_id", "official_fold", "hotel", "polarity", "source_family",
        "scientific_class", "y_true", "y_pred", "logit_0", "logit_1",
        "logit_2", "score_0", "score_1", "score_2", "maximum_score",
        "correct", "model_name", "fold_seed",
    ]
    require(frame["row_id"].tolist() == sorted(frame["row_id"]), "primary prediction order differs")
    return frame[columns]


def llama_prediction_frame(rows, logits, scores, gpt3_by_seed):
    predicted = np.argmax(logits, axis=1).astype(int)
    frame = pd.DataFrame({
        "row_id": [row["row_id"] for row in rows],
        "seed_family_id": [row["seed_family_id"] for row in rows],
        "matched_gpt3_row_id": [gpt3_by_seed[row["seed_family_id"]]["row_id"] for row in rows],
        "official_fold": [int(row["official_fold"]) for row in rows],
        "hotel": [row["hotel"] for row in rows],
        "polarity": [row["polarity"] for row in rows],
        "reference_label_id": [2] * len(rows),
        "y_pred": predicted,
        "logit_0": logits[:, 0],
        "logit_1": logits[:, 1],
        "logit_2": logits[:, 2],
        "score_0": scores[:, 0],
        "score_1": scores[:, 1],
        "score_2": scores[:, 2],
        "maximum_score": scores.max(axis=1),
        "detected_as_ai": (predicted == 2).astype(int),
        "model_name": MODEL_NAME,
        "fold_seed": FOLD_SEED,
    })
    columns = [
        "row_id", "seed_family_id", "matched_gpt3_row_id", "official_fold",
        "hotel", "polarity", "reference_label_id", "y_pred", "logit_0",
        "logit_1", "logit_2", "score_0", "score_1", "score_2",
        "maximum_score", "detected_as_ai", "model_name", "fold_seed",
    ]
    require(frame["row_id"].tolist() == sorted(frame["row_id"]), "Llama prediction order differs")
    return frame[columns]


def llama_summary(frame, group_type, group_value):
    predictions = frame["y_pred"].to_numpy(dtype=int)
    result = {
        "group_type": group_type,
        "group_value": str(group_value),
        "n": int(len(frame)),
        "ai_detection_recall": float((predictions == 2).mean()),
        "mean_ai_score": float(frame["score_2"].mean()),
        "median_ai_score": float(frame["score_2"].median()),
        "mean_maximum_score": float(frame["maximum_score"].mean()),
    }
    for label in LABEL_ORDER:
        count = int((predictions == label).sum())
        result[f"predicted_class_{label}_n"] = count
        result[f"predicted_class_{label}_share"] = float(count / len(frame))
    return result


def deterministic_zip(source_root, destination):
    timestamp = (2026, 9, 10, 0, 0, 0)
    with zipfile.ZipFile(
        destination,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
        allowZip64=True,
    ) as archive:
        for path in sorted(item for item in source_root.rglob("*") if item.is_file()):
            arcname = path.relative_to(source_root.parent).as_posix()
            info = zipfile.ZipInfo(arcname, date_time=timestamp)
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            info.create_system = 3
            with path.open("rb") as source, archive.open(info, mode="w", force_zip64=True) as target:
                shutil.copyfileobj(source, target, length=1024 * 1024)


print("=" * WIDTH)
print("PHASE 2B — STEP M4D: FOLD 2 FROZEN DEBERTA TRAINING AND ONE-TIME HELD-OUT INFERENCE")
print("=" * WIDTH)
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Target fold  :", TARGET_FOLD)
print("Policy       : final epoch only; no tuning, validation, early stopping or test monitoring")

require(INPUT_ROOT.is_dir(), "Kaggle input root is unavailable")
require(not OUTPUT_ROOT.exists(), f"output directory already exists: {OUTPUT_ROOT}")
require(not OUTPUT_ZIP.exists(), f"output ZIP already exists: {OUTPUT_ZIP}")
require(not OUTPUT_SIDECAR.exists(), f"output sidecar already exists: {OUTPUT_SIDECAR}")

section("ENVIRONMENT AND GPU IDENTITY")
for name, expected in EXPECTED_PACKAGES.items():
    actual = package_version(name)
    require(actual == expected, f"{name} version differs: {actual} != {expected}")
    print(f"{name:18s}: {actual:12s} PASS")

require(torch.__version__ == "2.10.0+cu128", f"unexpected PyTorch build: {torch.__version__}")
require(torch.version.cuda == "12.8", f"unexpected PyTorch CUDA build: {torch.version.cuda}")
require(torch.backends.cudnn.version() == 91002, f"unexpected cuDNN version: {torch.backends.cudnn.version()}")
require(torch.cuda.is_available(), "CUDA is unavailable")
require(torch.cuda.device_count() == 2, f"expected audited T4 x2 runtime; found {torch.cuda.device_count()} GPUs")
require(all(torch.cuda.get_device_name(index) == "Tesla T4" for index in range(2)), "GPU identity differs")
require(torch.cuda.get_device_capability(0) == (7, 5), "cuda:0 capability differs")
require("sm_75" in torch.cuda.get_arch_list(), "PyTorch build lacks sm_75 kernels")

torch.cuda.set_device(0)
DEVICE = torch.device("cuda:0")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision("highest")
torch.use_deterministic_algorithms(True)

print("Selected device       :", DEVICE)
print("GPU 0                 :", torch.cuda.get_device_name(0))
print("Visible GPUs          :", torch.cuda.device_count(), "(cuda:1 unused)")
print("CUDA/cuDNN            :", torch.version.cuda, "/", torch.backends.cudnn.version())
print("Deterministic kernels :", torch.are_deterministic_algorithms_enabled())
print("Numeric policy        : FP32; TF32/autocast/GradScaler disabled")

section("UPSTREAM ARTIFACT AND CONTRACT CONTINUITY")
t2_path = find_unique("corpus_manifest.json")
base_path = find_unique("checkpoint_manifest.json")
init_path = find_unique("initialisation_manifest.json")
m2_path = find_unique("contract_manifest.json")
m3a_path = find_unique("baseline_manifest.json")
m4b_path = find_unique("fold_1_manifest.json")

t2_manifest = verify_payload(t2_path, "manifest_payload_sha256", EXPECTED["t2_payload"])
base_manifest = verify_payload(base_path, "manifest_payload_sha256", EXPECTED["base_payload"])
init_manifest = verify_payload(init_path, "manifest_payload_sha256", EXPECTED["initialisation_payload"])
m2_manifest = verify_payload(m2_path, "contract_payload_sha256", EXPECTED["m2_payload"])
m3a_manifest = verify_payload(m3a_path, "manifest_payload_sha256", EXPECTED["m3a_payload"])
m4b_manifest = verify_payload(m4b_path, "manifest_payload_sha256", EXPECTED["m4b_fold1_payload"])
m4b_sidecars = sorted(
    path
    for path in INPUT_ROOT.rglob("phase2B_M4B_FOLD_1_DEBERTA.zip.sha256*.txt")
    if path.is_file()
)
m4b_valid_sidecars = []
for path in m4b_sidecars:
    tokens = path.read_text(encoding="utf-8", errors="strict").strip().split()
    if tokens and tokens[0].lower() == EXPECTED["m4b_fold1_zip"]:
        m4b_valid_sidecars.append(path)
require(len(m4b_valid_sidecars) == 1, "expected exactly one matching Fold-1 ZIP checksum sidecar")

require(file_sha256(m2_path) == EXPECTED["m2_contract_file"], "M2 contract-manifest file differs")
require(init_manifest["upstream"]["t2_manifest_payload_sha256"] == EXPECTED["t2_payload"], "initialisation/T2 link differs")
require(init_manifest["upstream"]["base_checkpoint_manifest_payload_sha256"] == EXPECTED["base_payload"], "initialisation/base link differs")
require(m2_manifest["upstream"]["t2_manifest_payload_sha256"] == EXPECTED["t2_payload"], "M2/T2 link differs")
require(m2_manifest["upstream"]["base_checkpoint_manifest_payload_sha256"] == EXPECTED["base_payload"], "M2/base link differs")
require(m2_manifest["upstream"]["initialisation_manifest_payload_sha256"] == EXPECTED["initialisation_payload"], "M2/initialisation link differs")
require(m2_manifest["upstream"]["initial_parameter_sha256"] == EXPECTED["initial_parameter"], "M2 parameter link differs")
require(m2_manifest["upstream"]["eligible_corpus_token_id_sha256"] == EXPECTED["token_ids"], "M2 token link differs")
require(m2_manifest["upstream"]["m2a_gpu_environment_payload_sha256"] == EXPECTED["m2a_environment"], "M2 environment link differs")
require(m3a_manifest["upstream"]["t2_manifest_payload_sha256"] == EXPECTED["t2_payload"], "M3A/T2 link differs")
require(m3a_manifest["upstream"]["m2_contract_payload_sha256"] == EXPECTED["m2_payload"], "M3A/M2 link differs")
require(m4b_manifest["upstream"]["t2_manifest_payload_sha256"] == EXPECTED["t2_payload"], "Fold-1/T2 link differs")
require(m4b_manifest["upstream"]["m2_contract_payload_sha256"] == EXPECTED["m2_payload"], "Fold-1/M2 link differs")
require(m4b_manifest["upstream"]["m3a_manifest_payload_sha256"] == EXPECTED["m3a_payload"], "Fold-1/M3A link differs")

T2_ROOT = t2_path.parent
MODEL_ROOT = init_path.parent / "model"
PRIMARY_PATH = T2_ROOT / "primary_tripartite_corpus.jsonl"
LLAMA_PATH = T2_ROOT / "llama_unseen_generator_sensitivity.jsonl"
MODEL_FILE = MODEL_ROOT / "model.safetensors"
require(file_sha256(PRIMARY_PATH) == EXPECTED["t2_primary_file"], "primary corpus file differs")
require(file_sha256(LLAMA_PATH) == EXPECTED["t2_llama_file"], "Llama corpus file differs")
require(file_sha256(MODEL_FILE) == EXPECTED["initial_model_file"], "initialised model file differs")

print("T2 payload             :", EXPECTED["t2_payload"], "PASS")
print("Base payload           :", EXPECTED["base_payload"], "PASS")
print("Initialisation payload :", EXPECTED["initialisation_payload"], "PASS")
print("M2 contract payload    :", EXPECTED["m2_payload"], "PASS")
print("M3A baseline payload   :", EXPECTED["m3a_payload"], "PASS")
print("Fold-1 artifact payload:", EXPECTED["m4b_fold1_payload"], "PASS")
print("Fold-1 recorded ZIP    :", EXPECTED["m4b_fold1_zip"], "PASS")
print("Corpus/model files     : PASS")

section("FOLD-2 DATA ROUTING AND TOKENIZER VERIFICATION")
primary_file_order = read_jsonl(PRIMARY_PATH)
llama_file_order = read_jsonl(LLAMA_PATH)
require(len(primary_file_order) == 2396, "primary row count differs")
require(len(llama_file_order) == 790, "Llama row count differs")
require(len({row["row_id"] for row in primary_file_order + llama_file_order}) == 3186, "eligible row IDs are not unique")

primary_rows = sorted(primary_file_order, key=lambda row: row["row_id"])
llama_rows = sorted(llama_file_order, key=lambda row: row["row_id"])
train_rows = [row for row in primary_rows if int(row["official_fold"]) != TARGET_FOLD]
test_rows = [row for row in primary_rows if int(row["official_fold"]) == TARGET_FOLD]
llama_test_rows = [row for row in llama_rows if int(row["official_fold"]) == TARGET_FOLD]
require(len(train_rows) == EXPECTED_TRAIN_N, f"train count differs: {len(train_rows)}")
require(len(test_rows) == EXPECTED_TEST_N, f"test count differs: {len(test_rows)}")
require(len(llama_test_rows) == EXPECTED_LLAMA_N, f"Llama test count differs: {len(llama_test_rows)}")
require([row["row_id"] for row in train_rows] == sorted(row["row_id"] for row in train_rows), "train order differs")
require([row["row_id"] for row in test_rows] == sorted(row["row_id"] for row in test_rows), "test order differs")
require([row["row_id"] for row in llama_test_rows] == sorted(row["row_id"] for row in llama_test_rows), "Llama order differs")
train_hotels = {row["hotel"] for row in train_rows}
test_hotels = {row["hotel"] for row in test_rows}
require(not (train_hotels & test_hotels), "train/test hotel overlap detected")
require(test_hotels == {"affinia", "ambassador", "hardrock", "talbott"}, f"Fold-2 hotels differ: {sorted(test_hotels)}")
require(all(row["source_family"] != "op_spam_AI_Llama" for row in train_rows), "Llama row entered training")
require(Counter(row["label_id"] for row in train_rows) == Counter({0: 637, 1: 640, 2: 640}), "train class counts differ")
require(Counter(row["label_id"] for row in test_rows) == Counter({0: 159, 1: 160, 2: 160}), "test class counts differ")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_ROOT), local_files_only=True)
pretokenizer_hash = tokenizer_backend_identity(tokenizer)
require(pretokenizer_hash == EXPECTED["pretokenizer"], "pre-tokenizer fingerprint differs")
token_hash, token_lengths = token_id_fingerprint_and_lengths(
    tokenizer,
    primary_file_order + llama_file_order,
)
require(token_hash == EXPECTED["token_ids"], "eligible-corpus token-ID fingerprint differs")

print("Train/test/Llama       :", len(train_rows), "/", len(test_rows), "/", len(llama_test_rows), "PASS")
print("Held-out hotels        :", ", ".join(sorted(test_hotels)))
print("Hotel overlap          : 0 PASS")
print("Train class counts     :", dict(sorted(Counter(row["label_id"] for row in train_rows).items())), "PASS")
print("Pre-tokenizer SHA256   :", pretokenizer_hash, "PASS")
print("Token-ID SHA256        :", token_hash, "PASS")

section("INITIAL MODEL AND OPTIMIZER CONSTRUCTION")
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_ROOT), local_files_only=True)
model.to(device="cpu", dtype=torch.float32)
model.eval()
require(model.config.num_labels == 3, "model output dimension differs")
require(model.config.label2id == LABEL2ID, "model label2id mapping differs")
require({int(key): value for key, value in model.config.id2label.items()} == ID2LABEL, "model id2label mapping differs")
initial_parameter_hash = parameter_sha256(model)
require(initial_parameter_hash == EXPECTED["initial_parameter"], "initial parameter fingerprint differs")

train_encoded = fixed_tokenize(tokenizer, train_rows)
train_labels = torch.tensor([int(row["label_id"]) for row in train_rows], dtype=torch.long)

random.seed(FOLD_SEED)
np.random.seed(FOLD_SEED)
torch.manual_seed(FOLD_SEED)
torch.cuda.manual_seed_all(FOLD_SEED)
model.to(DEVICE, dtype=torch.float32)
model.train()
require(next(model.parameters()).device == DEVICE, "model is not on cuda:0")
require({str(parameter.dtype) for parameter in model.parameters() if parameter.is_floating_point()} == {"torch.float32"}, "non-FP32 parameter detected")

no_decay_rules = ("bias", "LayerNorm.weight", "layer_norm.weight")
decay_parameters = []
no_decay_parameters = []
for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue
    (no_decay_parameters if any(rule in name for rule in no_decay_rules) else decay_parameters).append(parameter)
trainable_ids = {id(parameter) for parameter in model.parameters() if parameter.requires_grad}
grouped_ids = {id(parameter) for parameter in decay_parameters + no_decay_parameters}
require(trainable_ids == grouped_ids, "optimizer parameter grouping is incomplete")
require(len(decay_parameters) + len(no_decay_parameters) == len(grouped_ids), "optimizer parameter duplicated")

optimizer = torch.optim.AdamW(
    [
        {"params": decay_parameters, "weight_decay": 0.01},
        {"params": no_decay_parameters, "weight_decay": 0.0},
    ],
    lr=2e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    foreach=False,
    fused=False,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_UPDATES,
    num_training_steps=TOTAL_UPDATES,
)
require([float(group["lr"]) for group in optimizer.param_groups] == [0.0, 0.0], "initial warm-up LR differs")

print("Initial parameter SHA256:", initial_parameter_hash, "PASS")
print("Parameters             :", sum(parameter.numel() for parameter in model.parameters()))
print("Epochs/max length      :", EPOCHS, "/", MAX_LENGTH)
print("Batch/accumulation     :", PHYSICAL_BATCH_SIZE, "/", ACCUMULATION_MICROBATCHES)
print("Updates/warmup         :", TOTAL_UPDATES, "/", WARMUP_UPDATES)
print("Optimizer              : AdamW, foreach=False, fused=False")
print("Test monitoring        : PROHIBITED and not implemented")

section("FROZEN FOLD-2 TRAINING")
training_start = time.perf_counter()
global_update = 0
training_examples = 0
training_microbatches = 0
epoch_logs = []
update_logs = []

for zero_epoch in range(EPOCHS):
    epoch_number = zero_epoch + 1
    model.train()
    order = np.random.default_rng(FOLD_SEED + zero_epoch).permutation(len(train_rows))
    ordered_rows = [train_rows[int(index)] for index in order]
    order_hash = row_order_sha256(ordered_rows)

    epoch_loss_sum = 0.0
    epoch_examples = 0
    epoch_microbatches = 0
    epoch_updates = 0
    epoch_start = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(DEVICE)

    for group_start in range(0, len(order), MAXIMUM_EFFECTIVE_BATCH_SIZE):
        group_indices = order[group_start:group_start + MAXIMUM_EFFECTIVE_BATCH_SIZE]
        group_examples = int(len(group_indices))
        require(1 <= group_examples <= MAXIMUM_EFFECTIVE_BATCH_SIZE, "invalid accumulation-group size")
        optimizer.zero_grad(set_to_none=True)
        group_loss_sum = 0.0
        group_microbatches = 0

        for micro_start in range(0, group_examples, PHYSICAL_BATCH_SIZE):
            micro_indices_np = group_indices[micro_start:micro_start + PHYSICAL_BATCH_SIZE]
            micro_indices = torch.as_tensor(micro_indices_np, dtype=torch.long)
            inputs = {
                key: tensor.index_select(0, micro_indices).to(DEVICE)
                for key, tensor in train_encoded.items()
            }
            labels = train_labels.index_select(0, micro_indices).to(DEVICE)
            logits = model(**inputs).logits
            require(tuple(logits.shape) == (len(micro_indices_np), 3), "training logit shape differs")
            require(bool(torch.isfinite(logits).all().item()), "non-finite training logits")
            loss = torch.nn.functional.cross_entropy(logits, labels, reduction="sum")
            require(bool(torch.isfinite(loss).item()), "non-finite training loss")
            loss.backward()
            group_loss_sum += float(loss.detach().cpu())
            group_microbatches += 1

        gradient_tensors = 0
        for parameter in model.parameters():
            if parameter.grad is not None:
                parameter.grad.div_(group_examples)
                gradient_tensors += 1
        require(gradient_tensors > 0, "no gradients were produced")

        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
            error_if_nonfinite=True,
            foreach=False,
        )
        require(bool(torch.isfinite(gradient_norm).item()), "non-finite gradient norm")
        lr_used = float(optimizer.param_groups[0]["lr"])
        require(all(float(group["lr"]) == lr_used for group in optimizer.param_groups), "optimizer-group LR mismatch")
        optimizer.step()
        scheduler.step()
        global_update += 1
        epoch_updates += 1
        lr_after = float(optimizer.param_groups[0]["lr"])

        group_rows = [train_rows[int(index)] for index in group_indices]
        group_hash = row_order_sha256(group_rows)
        update_logs.append({
            "fold": TARGET_FOLD,
            "seed": FOLD_SEED,
            "epoch": epoch_number,
            "epoch_update": epoch_updates,
            "global_update": global_update,
            "group_examples": group_examples,
            "microbatches": group_microbatches,
            "group_row_order_sha256": group_hash,
            "loss_sum": group_loss_sum,
            "mean_loss": group_loss_sum / group_examples,
            "pre_clip_gradient_norm": float(gradient_norm.detach().cpu()),
            "learning_rate_used": lr_used,
            "learning_rate_after_step": lr_after,
        })

        epoch_loss_sum += group_loss_sum
        epoch_examples += group_examples
        epoch_microbatches += group_microbatches
        training_examples += group_examples
        training_microbatches += group_microbatches

        if epoch_updates % 20 == 0 or epoch_updates == EXPECTED_UPDATES_PER_EPOCH:
            print(
                f"Epoch {epoch_number}/{EPOCHS} | update {epoch_updates:02d}/{EXPECTED_UPDATES_PER_EPOCH} "
                f"| global {global_update:03d}/{TOTAL_UPDATES} | mean loss {epoch_loss_sum / epoch_examples:.6f}",
                flush=True,
            )

    torch.cuda.synchronize(DEVICE)
    epoch_elapsed = time.perf_counter() - epoch_start
    require(epoch_examples == EXPECTED_TRAIN_N, f"epoch {epoch_number} example count differs")
    require(epoch_microbatches == EXPECTED_MICROBATCHES_PER_EPOCH, f"epoch {epoch_number} microbatch count differs")
    require(epoch_updates == EXPECTED_UPDATES_PER_EPOCH, f"epoch {epoch_number} update count differs")
    require(all_finite_parameters(model), f"non-finite parameter after epoch {epoch_number}")
    epoch_log = {
        "fold": TARGET_FOLD,
        "seed": FOLD_SEED,
        "epoch": epoch_number,
        "row_order_sha256": order_hash,
        "examples": epoch_examples,
        "microbatches": epoch_microbatches,
        "optimizer_updates": epoch_updates,
        "loss_sum": epoch_loss_sum,
        "mean_loss": epoch_loss_sum / epoch_examples,
        "elapsed_seconds": epoch_elapsed,
        "peak_allocated_bytes": int(torch.cuda.max_memory_allocated(DEVICE)),
        "peak_reserved_bytes": int(torch.cuda.max_memory_reserved(DEVICE)),
        "learning_rate_after_epoch": float(optimizer.param_groups[0]["lr"]),
    }
    epoch_logs.append(epoch_log)
    print(
        f"Epoch {epoch_number} complete | loss={epoch_log['mean_loss']:.6f} "
        f"| elapsed={epoch_elapsed:.1f}s | order={order_hash[:16]}...",
        flush=True,
    )

training_elapsed = time.perf_counter() - training_start
require(global_update == TOTAL_UPDATES, f"total optimizer updates differ: {global_update}")
require(training_examples == EXPECTED_TRAIN_N * EPOCHS, "total training-example count differs")
require(training_microbatches == EXPECTED_MICROBATCHES_PER_EPOCH * EPOCHS, "total microbatch count differs")
require(len(update_logs) == TOTAL_UPDATES, "optimizer log length differs")
require(scheduler.last_epoch == TOTAL_UPDATES, f"scheduler step count differs: {scheduler.last_epoch}")
require(abs(float(optimizer.param_groups[0]["lr"])) <= 1e-18, "final scheduled LR is not zero")

section("ONE-TIME FINAL-EPOCH HELD-OUT INFERENCE")
inference_start = time.perf_counter()
primary_logits, primary_scores = infer(model, tokenizer, test_rows, DEVICE)
llama_logits, llama_scores = infer(model, tokenizer, llama_test_rows, DEVICE)
torch.cuda.synchronize(DEVICE)
inference_elapsed = time.perf_counter() - inference_start

gpt3_by_seed = {
    row["seed_family_id"]: row
    for row in primary_rows
    if row["source_family"] == "op_spam_AI_GPT3"
}
require(len(gpt3_by_seed) == 800, "GPT-3 seed-family index differs")
require(all(row["seed_family_id"] in gpt3_by_seed for row in llama_test_rows), "Llama/GPT-3 seed match missing")
require(all(int(gpt3_by_seed[row["seed_family_id"]]["official_fold"]) == TARGET_FOLD for row in llama_test_rows), "matched GPT-3 sibling lies outside Fold 2")

primary_predictions = primary_prediction_frame(test_rows, primary_logits, primary_scores)
llama_predictions = llama_prediction_frame(llama_test_rows, llama_logits, llama_scores, gpt3_by_seed)
primary_metrics = classification_metrics(
    primary_predictions["y_true"],
    primary_predictions["y_pred"],
    primary_predictions[["score_0", "score_1", "score_2"]],
)
primary_confusion = confusion_matrix(
    primary_predictions["y_true"],
    primary_predictions["y_pred"],
    labels=LABEL_ORDER,
).astype(int).tolist()

llama_summary_rows = [llama_summary(llama_predictions, "all", "all")]
for dimension in ("polarity", "hotel"):
    for value in sorted(llama_predictions[dimension].unique(), key=str):
        subset = llama_predictions[llama_predictions[dimension] == value]
        llama_summary_rows.append(llama_summary(subset, dimension, value))
llama_summary_frame = pd.DataFrame(llama_summary_rows)

print("Primary predictions    :", len(primary_predictions), "PASS")
print("Llama predictions      :", len(llama_predictions), "PASS")
print("Primary macro-F1       :", f"{primary_metrics['macro_f1']:.6f}")
print("Primary balanced acc.  :", f"{primary_metrics['balanced_accuracy']:.6f}")
print("Primary accuracy       :", f"{primary_metrics['accuracy']:.6f}")
print("Llama AI detection     :", f"{llama_predictions['detected_as_ai'].mean():.6f}")
print("Inference elapsed sec. :", f"{inference_elapsed:.3f}")
print("Test use               : one final-epoch inference; no tuning or selection")

section("TRUNCATION AUDIT")
truncation_rows = []
for row in primary_rows:
    role = "primary_test" if int(row["official_fold"]) == TARGET_FOLD else "outer_train"
    truncation_rows.append({
        "row_id": row["row_id"],
        "dataset_role": role,
        "scientific_class": row["scientific_class"],
        "official_fold": int(row["official_fold"]),
        "hotel": row["hotel"],
        "polarity": row["polarity"],
        "pretruncation_tokens": int(token_lengths[row["row_id"]]),
        "truncated_at_256": int(token_lengths[row["row_id"]] > MAX_LENGTH),
    })
for row in llama_rows:
    role = "llama_sensitivity_test" if int(row["official_fold"]) == TARGET_FOLD else "llama_not_used_this_fold"
    truncation_rows.append({
        "row_id": row["row_id"],
        "dataset_role": role,
        "scientific_class": "ai_generated_llama",
        "official_fold": int(row["official_fold"]),
        "hotel": row["hotel"],
        "polarity": row["polarity"],
        "pretruncation_tokens": int(token_lengths[row["row_id"]]),
        "truncated_at_256": int(token_lengths[row["row_id"]] > MAX_LENGTH),
    })
truncation_frame = pd.DataFrame(truncation_rows).sort_values("row_id", kind="stable").reset_index(drop=True)
require(len(truncation_frame) == 3186, "truncation audit row count differs")
truncation_summary = (
    truncation_frame
    .groupby(["dataset_role", "scientific_class", "official_fold"], sort=True, dropna=False)
    .agg(
        n=("row_id", "size"),
        truncated_n=("truncated_at_256", "sum"),
        truncation_rate=("truncated_at_256", "mean"),
        minimum_tokens=("pretruncation_tokens", "min"),
        median_tokens=("pretruncation_tokens", "median"),
        maximum_tokens=("pretruncation_tokens", "max"),
    )
    .reset_index()
)
print("Eligible rows audited  :", len(truncation_frame), "PASS")
print("Rows exceeding 256     :", int(truncation_frame["truncated_at_256"].sum()))
print("Audit dimensions       : role × class × official fold")

section("FINAL CHECKPOINT SAVE, RELOAD AND ARTIFACT FREEZE")
final_parameter_hash = parameter_sha256(model)
require(final_parameter_hash != EXPECTED["initial_parameter"], "final parameters equal the initial state")
require(all_finite_parameters(model), "final model contains non-finite parameters")

device_total_bytes = int(torch.cuda.get_device_properties(0).total_memory)
peak_allocated_bytes = max(item["peak_allocated_bytes"] for item in epoch_logs)
peak_reserved_bytes = max(item["peak_reserved_bytes"] for item in epoch_logs)

temporary_parent = Path(tempfile.mkdtemp(prefix="phase2B_M4D_fold2_", dir=WORKING_ROOT))
temporary_root = temporary_parent / OUTPUT_ROOT.name
temporary_root.mkdir()
model_output = temporary_root / "model"

model.to(device="cpu", dtype=torch.float32)
model.eval()
model.save_pretrained(str(model_output), safe_serialization=True, max_shard_size="5GB")
tokenizer.save_pretrained(str(model_output))

write_csv(temporary_root / "fold_2_primary_predictions.csv", primary_predictions)
write_csv(temporary_root / "fold_2_llama_sensitivity_predictions.csv", llama_predictions)
write_csv(temporary_root / "fold_2_epoch_training_log.csv", pd.DataFrame(epoch_logs))
write_csv(temporary_root / "fold_2_optimizer_update_log.csv", pd.DataFrame(update_logs))
write_json(temporary_root / "fold_2_primary_metrics.json", primary_metrics)
write_json(temporary_root / "fold_2_confusion_matrix.json", {"label_order": LABEL_ORDER, "matrix": primary_confusion})
write_csv(temporary_root / "fold_2_llama_sensitivity_summary.csv", llama_summary_frame)
write_csv(temporary_root / "fold_2_truncation_audit.csv", truncation_frame)
write_csv(temporary_root / "fold_2_truncation_summary.csv", truncation_summary)

implementation = {
    "training_loop": "explicit deterministic row-index batching; zero worker processes",
    "loss": "torch.nn.functional.cross_entropy reduction=sum",
    "gradient_normalisation": "divide accumulated summed gradients by actual group examples",
    "optimizer": "torch.optim.AdamW foreach=False fused=False",
    "scheduler": "transformers.get_linear_schedule_with_warmup called after each optimizer step",
    "gradient_clipping": "clip_grad_norm_ max_norm=1.0 foreach=False",
    "zero_grad": "set_to_none=True at each accumulation-group boundary",
    "inference": "FP32 no_grad, batch=16, softmax model scores, argmax logits",
    "test_monitoring": False,
    "automatic_adaptation": False,
}
write_json(temporary_root / "implementation_record.json", implementation)

del scheduler
del optimizer
del train_encoded
del train_labels
del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize(DEVICE)

reloaded_model = AutoModelForSequenceClassification.from_pretrained(
    str(model_output),
    local_files_only=True,
)
reloaded_model.to(device="cpu", dtype=torch.float32)
reloaded_model.eval()
reloaded_parameter_hash = parameter_sha256(reloaded_model)
require(reloaded_parameter_hash == final_parameter_hash, "final model hash changed after save/reload")
reloaded_tokenizer = AutoTokenizer.from_pretrained(str(model_output), local_files_only=True)
require(tokenizer_backend_identity(reloaded_tokenizer) == EXPECTED["pretokenizer"], "saved tokenizer backend differs")
del reloaded_model
del reloaded_tokenizer
del tokenizer
gc.collect()

payload_files = sorted(path for path in temporary_root.rglob("*") if path.is_file())
payload_registry = {
    path.relative_to(temporary_root).as_posix(): {
        "bytes": path.stat().st_size,
        "sha256": file_sha256(path),
    }
    for path in payload_files
}

manifest_payload = {
    "phase": "Phase 2B",
    "step": "M4D",
    "status": "FROZEN_FOLD_2_DEBERTA_COMPLETE",
    "upstream": {
        "t2_manifest_payload_sha256": EXPECTED["t2_payload"],
        "base_checkpoint_manifest_payload_sha256": EXPECTED["base_payload"],
        "initialisation_manifest_payload_sha256": EXPECTED["initialisation_payload"],
        "initial_parameter_sha256": EXPECTED["initial_parameter"],
        "eligible_corpus_token_id_sha256": EXPECTED["token_ids"],
        "m2_contract_payload_sha256": EXPECTED["m2_payload"],
        "m3a_manifest_payload_sha256": EXPECTED["m3a_payload"],
        "m4b_fold_1_manifest_payload_sha256": EXPECTED["m4b_fold1_payload"],
        "m4b_fold_1_recorded_zip_sha256": EXPECTED["m4b_fold1_zip"],
        "m2a_gpu_environment_payload_sha256": EXPECTED["m2a_environment"],
    },
    "fold": {
        "test_fold": TARGET_FOLD,
        "seed": FOLD_SEED,
        "train_n": len(train_rows),
        "test_n": len(test_rows),
        "llama_test_n": len(llama_test_rows),
        "test_hotels": sorted(test_hotels),
        "train_test_hotel_overlap": 0,
    },
    "training": {
        "epochs": EPOCHS,
        "max_length": MAX_LENGTH,
        "physical_batch_size": PHYSICAL_BATCH_SIZE,
        "accumulation_microbatches": ACCUMULATION_MICROBATCHES,
        "total_examples_consumed": training_examples,
        "total_microbatches": training_microbatches,
        "total_optimizer_updates": global_update,
        "warmup_updates": WARMUP_UPDATES,
        "initial_parameter_sha256": initial_parameter_hash,
        "final_parameter_sha256": final_parameter_hash,
        "save_reload_parameter_identity": True,
        "training_elapsed_seconds": training_elapsed,
        "epoch_row_order_sha256": [item["row_order_sha256"] for item in epoch_logs],
        "no_tuning": True,
        "no_early_stopping": True,
        "final_epoch_only": True,
    },
    "inference": {
        "primary_rows": len(primary_predictions),
        "llama_rows": len(llama_predictions),
        "inference_elapsed_seconds": inference_elapsed,
        "primary_metrics": primary_metrics,
        "primary_confusion_matrix": primary_confusion,
        "llama_ai_detection_recall": float(llama_predictions["detected_as_ai"].mean()),
        "score_language": "uncalibrated model score; not probability or calibrated confidence",
        "one_time_final_epoch_inference": True,
        "test_used_for_tuning_or_selection": False,
    },
    "resources": {
        "device": "cuda:0",
        "gpu_name": torch.cuda.get_device_name(0),
        "visible_gpu_count": torch.cuda.device_count(),
        "multi_gpu_used": False,
        "peak_allocated_bytes": peak_allocated_bytes,
        "peak_reserved_bytes": peak_reserved_bytes,
        "device_total_bytes": device_total_bytes,
    },
    "artifact_files": payload_registry,
}
manifest_payload_hash = canonical_sha256(manifest_payload)
manifest = {**manifest_payload, "manifest_payload_sha256": manifest_payload_hash}
write_json(temporary_root / "fold_2_manifest.json", manifest)

checksum_targets = sorted(path for path in temporary_root.rglob("*") if path.is_file())
checksum_text = "".join(
    f"{file_sha256(path)}  {path.relative_to(temporary_root).as_posix()}\n"
    for path in checksum_targets
)
(temporary_root / "SHA256SUMS.txt").write_text(checksum_text, encoding="utf-8", newline="\n")

require(not OUTPUT_ROOT.exists(), "output root appeared during execution")
temporary_root.replace(OUTPUT_ROOT)
temporary_parent.rmdir()

deterministic_zip(OUTPUT_ROOT, OUTPUT_ZIP)
zip_sha256 = file_sha256(OUTPUT_ZIP)
OUTPUT_SIDECAR.write_text(
    f"{zip_sha256}  {OUTPUT_ZIP.name}\n",
    encoding="utf-8",
    newline="\n",
)

final_files = sorted(path for path in OUTPUT_ROOT.rglob("*") if path.is_file())
for path in final_files:
    print(
        f"{path.relative_to(OUTPUT_ROOT).as_posix():64s} "
        f"{path.stat().st_size:12d} bytes  {file_sha256(path)}"
    )

print("Fold manifest payload SHA256:", manifest_payload_hash)
print("Final parameter SHA256      :", final_parameter_hash)

print("\n" + "=" * WIDTH)
print("STEP M4D DECISION: FOLD 2 DEBERTA TRAINING AND HELD-OUT INFERENCE FROZEN")
print("=" * WIDTH)
print("Output directory :", OUTPUT_ROOT)
print("Artifact ZIP     :", OUTPUT_ZIP)
print("ZIP SHA256       :", zip_sha256)
print("Checksum sidecar :", OUTPUT_SIDECAR)
print("Primary test rows:", len(primary_predictions))
print("Llama test rows  :", len(llama_predictions))
print("Tuning/selection : NONE")
print("Download the ZIP and .sha256.txt, verify externally, and preserve the complete output.")
print("STOP — do not train Fold 3 until this artifact and output are externally verified and audited.")


PHASE 2B — STEP M4D: FOLD 2 FROZEN DEBERTA TRAINING AND ONE-TIME HELD-OUT INFERENCE
Python       : 3.12.13
PyTorch      : 2.10.0+cu128
Transformers : 5.0.0
Target fold  : 2
Policy       : final epoch only; no tuning, validation, early stopping or test monitoring

ENVIRONMENT AND GPU IDENTITY
--------------------------------------------------------------------------------------------------------------------------------
numpy             : 2.0.2        PASS
pandas            : 2.3.3        PASS
scikit-learn      : 1.6.1        PASS
transformers      : 5.0.0        PASS
tokenizers        : 0.22.2       PASS
accelerate        : 1.13.0       PASS
safetensors       : 0.7.0        PASS
sentencepiece     : 0.2.1        PASS
Selected device       : cuda:0
GPU 0                 : Tesla T4
Visible GPUs          : 2 (cuda:1 unused)
CUDA/cuDNN            : 12.8 / 91002
Deterministic kernels : True
Numeric policy        : FP32; TF32/autocast/GradScaler disabled

UPSTREAM ARTIFACT AND CONTRACT CONTIN

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Initial parameter SHA256: 7840923fb74e4a523e042f2c50410580cd7f2b1a34601c48fc328c9b5f42395b PASS
Parameters             : 184424451
Epochs/max length      : 4 / 256
Batch/accumulation     : 2 / 16
Updates/warmup         : 240 / 24
Optimizer              : AdamW, foreach=False, fused=False
Test monitoring        : PROHIBITED and not implemented

FROZEN FOLD-2 TRAINING
--------------------------------------------------------------------------------------------------------------------------------
Epoch 1/4 | update 20/60 | global 020/240 | mean loss 1.014257
Epoch 1/4 | update 40/60 | global 040/240 | mean loss 0.863063
Epoch 1/4 | update 60/60 | global 060/240 | mean loss 0.775422
Epoch 1 complete | loss=0.775422 | elapsed=288.6s | order=096cda0d94aaa033...
Epoch 2/4 | update 20/60 | global 080/240 | mean loss 0.534556
Epoch 2/4 | update 40/60 | global 100/240 | mean loss 0.475052
Epoch 2/4 | update 60/60 | global 120/240 | mean loss 0.446042
Epoch 2 complete | loss=0.446042 | elapsed=291

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

SHA256SUMS.txt                                                           1415 bytes  0814ed565be51e80bf3692e830a3c7673dc464347df301106af0f5c778796534
fold_2_confusion_matrix.json                                              191 bytes  2223cb490b91ec38ffd2125367f3edd16aab8ea08acb62ae5c7ea93a7f9a2247
fold_2_epoch_training_log.csv                                             901 bytes  ef01e57650c4f8a71e46a1ace1ed096e069f87d00f897ed5032efad97fe0c3a4
fold_2_llama_sensitivity_predictions.csv                                42325 bytes  8fc51ec3fda5b8373f01aed4264c80b366f91a9f33b98b47e42e54955941de05
fold_2_llama_sensitivity_summary.csv                                     1358 bytes  9c7d6a4c1d3b8b4fab51d442687aef8352d2377507f1e9d3080669a24a8db9e0
fold_2_manifest.json                                                     6177 bytes  9d9bc23e90fa24161b417ca7abac40a726f88c861b96d5ebec1a089743c17717
fold_2_optimizer_update_log.csv                                         45223 bytes  bd3c1b5fe90e23f

Current Fold-2 findings:

DeBERTa macro-F1: 0.862241
TF-IDF Fold-2 macro-F1: 0.870386
Llama AI-detection recall: 0.702532
Hotel overlap: zero
Final parameter SHA-256: a8bc5a8bb52011d0baa7c495756ce3ae8cd2f51c1a4821be69eb9c5181a274d8
Manifest payload: 3b5418f4ad29cc660f1ef275efc8708199299baf3526524078bdfcc063ae737d

At the displayed precision, TF-IDF is ahead by 0.008145 in Fold 2. This remains preliminary; no overall comparison is valid until all five folds are pooled.

Item	Value
ZIP SHA-256	a88923f6f6a7c7b8b6ab9c658e2a4fa3f3b194ac13847b0a9c05e4daec18039f
Manifest payload SHA-256	3b5418f4ad29cc660f1ef275efc8708199299baf3526524078bdfcc063ae737d
Final parameter SHA-256	a8bc5a8bb52011d0baa7c495756ce3ae8cd2f51c1a4821be69eb9c5181a274d8
Primary macro-F1	0.862241
Llama AI-detection recall	0.702532

Running score after two folds:

Fold	Hotels	DeBERTa macro-F1	TF-IDF macro-F1	Llama recall
1	hilton, james, monaco, sofitel	0.8249	0.8635	0.5125
2	affinia, ambassador, hardrock, talbott	0.8622	0.8704	0.7025

TF-IDF leads DeBERTa by a small margin in both folds so far. Training loss converged well — 1.014 → 0.216 across four epochs — so the model is learning. Three folds remain. No conclusion yet.

In [2]:
from pathlib import Path
from IPython.display import FileLink, display

files = [
    Path("/kaggle/working/phase2B_M4D_FOLD_2_DEBERTA.zip"),
    Path("/kaggle/working/phase2B_M4D_FOLD_2_DEBERTA.zip.sha256.txt"),
]

for path in files:
    if not path.is_file():
        raise RuntimeError(f"STOP — missing output: {path}")

    print(f"{path.name}: {path.stat().st_size:,} bytes")
    display(FileLink(str(path)))

print("Download both files before changing or stopping the runtime.")

phase2B_M4D_FOLD_2_DEBERTA.zip: 572,454,897 bytes


/kaggle/working/phase2B_M4D_FOLD_2_DEBERTA.zip

phase2B_M4D_FOLD_2_DEBERTA.zip.sha256.txt: 97 bytes


/kaggle/working/phase2B_M4D_FOLD_2_DEBERTA.zip.sha256.txt

Download both files before changing or stopping the runtime.


In [2]:
from pathlib import Path
from IPython.display import FileLink, display
import hashlib
import json

SOURCE = Path("/kaggle/working/phase2B_M4D_FOLD_2_DEBERTA.zip")
SIDECAR = Path(
    "/kaggle/working/phase2B_M4D_FOLD_2_DEBERTA.zip.sha256.txt"
)
PART_DIR = Path("/kaggle/working/phase2B_M4D_FOLD_2_DOWNLOAD_PARTS")
CHUNK_SIZE = 100 * 1024 * 1024  # 100 MiB


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


if not SOURCE.is_file():
    raise RuntimeError(f"STOP — source ZIP missing: {SOURCE}")

if not SIDECAR.is_file():
    raise RuntimeError(f"STOP — checksum sidecar missing: {SIDECAR}")

sidecar_tokens = SIDECAR.read_text(
    encoding="utf-8", errors="strict"
).strip().split()

if not sidecar_tokens:
    raise RuntimeError("STOP — checksum sidecar is empty.")

expected_zip_sha256 = sidecar_tokens[0].lower()
actual_zip_sha256 = sha256_file(SOURCE)

if actual_zip_sha256 != expected_zip_sha256:
    raise RuntimeError(
        "STOP — current ZIP does not match its checksum sidecar."
    )

if PART_DIR.exists() and any(PART_DIR.iterdir()):
    raise RuntimeError(
        f"STOP — parts directory already contains files: {PART_DIR}"
    )

PART_DIR.mkdir(parents=True, exist_ok=True)

part_records = []

with SOURCE.open("rb") as source_handle:
    part_number = 1

    while True:
        data = source_handle.read(CHUNK_SIZE)
        if not data:
            break

        part_path = PART_DIR / (
            f"phase2B_M4D_FOLD_2_DEBERTA.zip.part{part_number:03d}"
        )
        part_path.write_bytes(data)

        part_records.append({
            "part_number": part_number,
            "filename": part_path.name,
            "bytes": part_path.stat().st_size,
            "sha256": sha256_file(part_path),
        })
        part_number += 1

manifest = {
    "original_filename": SOURCE.name,
    "original_bytes": SOURCE.stat().st_size,
    "original_sha256": actual_zip_sha256,
    "chunk_size_bytes": CHUNK_SIZE,
    "number_of_parts": len(part_records),
    "parts": part_records,
}

manifest_path = PART_DIR / "fold_2_download_parts_manifest.json"
manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

sidecar_copy = PART_DIR / SIDECAR.name
sidecar_copy.write_bytes(SIDECAR.read_bytes())

print("=" * 100)
print("FOLD-2 DOWNLOAD-PART CREATION")
print("=" * 100)
print("Original ZIP SHA256:", actual_zip_sha256)
print("Original ZIP bytes :", SOURCE.stat().st_size)
print("Parts created      :", len(part_records))

for record in part_records:
    print(
        f"{record['filename']} | "
        f"{record['bytes']:,} bytes | "
        f"{record['sha256']}"
    )

print("\nDOWNLOAD EVERY FILE BELOW")
display(FileLink(str(manifest_path)))
display(FileLink(str(sidecar_copy)))

for record in part_records:
    display(FileLink(str(PART_DIR / record["filename"])))

print("\nDo not stop the session until every part is downloaded.")
print("No model, prediction or original artifact was changed.")

FOLD-2 DOWNLOAD-PART CREATION
Original ZIP SHA256: 37ababdc8808dac8112afce24d6cdff3e1bfc3a20004662783b8f176c06df7d0
Original ZIP bytes : 572454896
Parts created      : 6
phase2B_M4D_FOLD_2_DEBERTA.zip.part001 | 104,857,600 bytes | 1cf40dbd4cccdb7cb353c008d985907a85febaeb681b80f474825c996d082f2e
phase2B_M4D_FOLD_2_DEBERTA.zip.part002 | 104,857,600 bytes | f5e642f9ec506fdb5522b132cd47ecaad1ce05c14943b346c7489b14e7ee9a99
phase2B_M4D_FOLD_2_DEBERTA.zip.part003 | 104,857,600 bytes | c85f37908b5e1873d57fa051e329a98a606ffea70c848a1e8acd326a17eebe5c
phase2B_M4D_FOLD_2_DEBERTA.zip.part004 | 104,857,600 bytes | 558dfa57fcbbdeaf98aa59dda7fd21884779a4545cf8b0785cf7b44af2b3da97
phase2B_M4D_FOLD_2_DEBERTA.zip.part005 | 104,857,600 bytes | 030a360a23a07e02113101ff4a7b1dadbc5c777792c58c21970e181c7b28e298
phase2B_M4D_FOLD_2_DEBERTA.zip.part006 | 48,166,896 bytes | 7c8f74a2491789400de2e37cee664fe0ad8ed2a0f2a4435c9b0b82e0986d435e

DOWNLOAD EVERY FILE BELOW


/kaggle/working/phase2B_M4D_FOLD_2_DOWNLOAD_PARTS/fold_2_download_parts_manifest.json

/kaggle/working/phase2B_M4D_FOLD_2_DOWNLOAD_PARTS/phase2B_M4D_FOLD_2_DEBERTA.zip.sha256.txt

/kaggle/working/phase2B_M4D_FOLD_2_DOWNLOAD_PARTS/phase2B_M4D_FOLD_2_DEBERTA.zip.part001

/kaggle/working/phase2B_M4D_FOLD_2_DOWNLOAD_PARTS/phase2B_M4D_FOLD_2_DEBERTA.zip.part002

/kaggle/working/phase2B_M4D_FOLD_2_DOWNLOAD_PARTS/phase2B_M4D_FOLD_2_DEBERTA.zip.part003

/kaggle/working/phase2B_M4D_FOLD_2_DOWNLOAD_PARTS/phase2B_M4D_FOLD_2_DEBERTA.zip.part004

/kaggle/working/phase2B_M4D_FOLD_2_DOWNLOAD_PARTS/phase2B_M4D_FOLD_2_DEBERTA.zip.part005

/kaggle/working/phase2B_M4D_FOLD_2_DOWNLOAD_PARTS/phase2B_M4D_FOLD_2_DEBERTA.zip.part006


Do not stop the session until every part is downloaded.
No model, prediction or original artifact was changed.
